# Kiểm tra cấu hình Kaggle offline cho notebook 1 và 3

Mở **phiên Kaggle mới**, chọn đúng GPU dự định dùng và **Internet Off**.
Gắn các Dataset **models, packages và PDF** hiện tại rồi **Run All**; không cần chạy
setup notebook 1/3 trước. Nếu chưa gắn Input, vẫn lấy được báo cáo kernel.

**Mặc định kiểm tra môi trường đầy đủ cho OCR và restoration**, gồm inventory mọi
distribution, dependency, ABI, CUDA/native ops nhỏ, API và runtime đã tồn tại.
Đọc metadata wheel và dependency closure của các runtime plan với kernel thật.
Ngân sách tối đa 900 giây và timeout riêng; nếu hết thời gian, log ghi `partial`.
Không cài package, build, đọc toàn bộ weights hoặc inference model.
DocEnTR/DE-GAN có inventory size/source/license và Keras backend Torch/HDF5 API.
Thiếu package trong kernel có thể đã được cung cấp bởi overlay offline.

Giữ `RESTORATION_GPU_CHECK='off'` để chỉ gửi log môi trường. Các lựa chọn
`'build_and_probe'` và `'unihir_smoke'` vẫn có nếu bạn chủ động bật phép kiểm dài
trên Kaggle; local luôn bỏ qua. Log/JSON giữ riêng metadata và inference,
kể cả khi thiếu Input hoặc bị ngắt. Không OCR hoặc chạy cả corpus.

Metadata, file trong Input, import, tensor GPU và inference model là các mức kiểm
chứng khác nhau. Thiếu package trong kernel có thể được giải quyết bằng runtime
riêng; báo cáo này không tự kết luận cả pipeline đã chạy được.


In [ ]:
from pathlib import Path

ON_KAGGLE = Path('/kaggle/working').is_dir()
INPUT_ROOT = Path('/kaggle/input') if ON_KAGGLE else Path('kaggle_upload')
# Để trống để tự tìm qua manifest; nếu có nhiều Dataset cùng loại, điền đường dẫn.
PACKAGES_DIR = ''
MODELS_DIR = ''
DATA_DIR = ''
PREPARED_IMAGES = ''  # tùy chọn, thư mục có dataset.json; chỉ đọc descriptor

# Chỉ điền nếu runtime riêng ĐÃ tồn tại; GPU check tạo overlay riêng khi cần.
RUNTIME_PYTHONS = {
    # 'unihir_restoration': '/path/to/verified/environment/bin/python',
    # 'tadisr_x4': '/path/to/overlay/bin/python',
}
RUN_PROBES = True
CHECK_OPTIONS = {
    'verify_asset_hashes': False,  # mặc định chỉ stat kích thước; không đọc hàng chục GB weights
    'check_existing_runtimes': True,
    'quick_restoration_checks': False,  # True nếu chỉ cần nhóm probe restoration ngắn
    'max_runtime_checks': 8,
    'max_scan_dirs': 500,
    'max_asset_records': 12000,
    'probe_timeout_cap': 180,
    'time_budget_seconds': 900,
}
OUTPUT_DIR = Path('/kaggle/working') if ON_KAGGLE else Path('work/kaggle_environment_check')

# Kaggle-only; không chạy build/model trên local. Điền DATA_DIR để thử inference PDF.
RESTORATION_GPU_CHECK = 'off'  # off | build_and_probe | unihir_smoke; chủ động bật nếu cần
RESTORATION_CHECK_OPTIONS = {
    'books': [], 'pages': '1',  # [] chọn PDF đầu; giới hạn cứng 1 PDF / 1 trang / crop page
    'steps': 10, 'ref_count': 1,
    'gpu_ids': [0,1,2,3], 'placement':'sharded',
    'max_memory':'18GiB', 'activation_reserve_gib':4, 'entry_reserve_gib':2,
    'max_source_pixels': 16_000_000,
    'load_timeout_seconds': 900, 'inference_timeout_seconds': 900,
    'total_timeout_seconds': 5400,
    'runtime_dir': '/tmp/vtm_image_runtime',
}


In [ ]:
"""Read-only environment inventory; no installs, downloads, weights or OCR."""
from __future__ import annotations

from datetime import datetime, timezone
from email.parser import Parser
import importlib.metadata as metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import sysconfig
import hashlib
import time
import zipfile
from pathlib import PurePosixPath


PACKAGES = (
    "torch", "torchvision", "transformers", "tokenizers", "accelerate", "safetensors",
    "huggingface-hub", "sentencepiece", "protobuf", "numpy", "scipy", "pillow", "opencv-python", "opencv-python-headless",
    "paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex", "mmcv", "mmcv-full",
    "mmcv-lite", "mmengine", "mmdet", "mmrotate", "timm", "einops", "flash-attn",
    "triton", "pyyaml", "pypdfium2", "pandas", "tqdm", "pip", "setuptools", "wheel",
    "packaging", "ninja", "cmake", "uv", "diffusers", "peft", "omegaconf",
    "pytorch-lightning", "lightning-utilities", "torchmetrics", "zhconv", "fonttools",
    "decord", "bitsandbytes", "sentence-transformers", "opencv-contrib-python",
    "vit-pytorch", "keras", "h5py", "ml-dtypes", "optree", "namex",
)

PROBES = {
    "torch_cuda": ("torch", 90, """import torch, json
print('stage: torch imported', flush=True)
devices = []
for i in range(torch.cuda.device_count()):
    x = torch.ones((2,2), device=f'cuda:{i}')
    value = float((x @ x).sum().item())
    torch.cuda.synchronize(i)
    devices.append({'index':i,'name':torch.cuda.get_device_name(i),
        'VRAM_GiB':round(torch.cuda.get_device_properties(i).total_memory/1024**3,2),
        'free_VRAM_GiB':round(torch.cuda.mem_get_info(i)[0]/1024**3,2),
        'compute_capability':list(torch.cuda.get_device_capability(i)),'gpu_matmul_sum':value})
result = {'torch': torch.__version__, 'built_cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'devices': devices,
          'compiled_arches': getattr(torch.cuda, 'get_arch_list', lambda: [])() if torch.cuda.is_available() else [],
          'module_file': torch.__file__, 'native_file': torch._C.__file__,
          'cxx11_abi': getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', None)}
result['gpu_count'] = len(devices)
result['peer_access'] = [[torch.cuda.can_device_access_peer(i,j) if i!=j else True for j in range(len(devices))] for i in range(len(devices))]
result['memory_is_per_device'] = True
print(json.dumps(result))
assert result['cuda_available'], 'CUDA unavailable; this is not a verified GPU runtime'
"""),
    "torchvision_ops": ("torchvision", 90, """import torch, torchvision, json
from torchvision.ops import nms
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[0.,0.,2.,2.],[0.,0.,2.,2.]],device=device)
keep = nms(boxes, torch.tensor([.9,.8],device=device), .5)
assert keep.numel() == 1
print(json.dumps({'torchvision': torchvision.__version__, 'nms_device': device}))
"""),
    "torch_numpy_bridge": ("torch", 45, """import torch, numpy as np, json
x = torch.from_numpy(np.zeros((2,2),dtype=np.float32))
assert x.shape == (2,2) and x.numpy().shape == (2,2)
print(json.dumps({'torch':torch.__version__,'numpy':np.__version__,'numpy_bridge':'OK'}))
"""),
    "numpy_cv2": ("cv2", 45, """import numpy as np, cv2, json
from PIL import Image
x = np.zeros((4,4,3),dtype=np.uint8)
assert cv2.cvtColor(x,cv2.COLOR_RGB2GRAY).shape == (4,4)
assert Image.fromarray(x).size == (4,4)
print(json.dumps({'numpy': np.__version__, 'opencv': cv2.__version__, 'array_image_bridge': 'OK'}))
"""),
    "transformers_hunyuan": ("transformers", 180, """import json
print('stage: transformers import', flush=True)
import transformers
print('stage: AutoProcessor', flush=True)
from transformers import AutoProcessor
print('stage: Hunyuan class', flush=True)
from transformers import HunYuanVLForConditionalGeneration
print(json.dumps({'transformers': transformers.__version__, 'hunyuan_api': 'OK'}))
"""),
    "paddle_cuda": ("paddle", 90, """import paddle, json
result = {'paddle': paddle.__version__, 'compiled_with_cuda': paddle.is_compiled_with_cuda()}
result['devices'] = paddle.device.cuda.device_count() if result['compiled_with_cuda'] else 0
if result['devices']:
    paddle.set_device('gpu:0')
    x = paddle.ones((2,2))
    result['gpu_matmul_sum'] = float(paddle.matmul(x,x).sum().item())
print(json.dumps(result))
"""),
    "paddleocr_apis": ("paddleocr", 120, """import paddleocr, json
from paddleocr import PaddleOCR, PaddleOCRVL
print(json.dumps({'paddleocr': paddleocr.__version__, 'PaddleOCR': 'OK', 'PaddleOCRVL': 'OK'}))
"""),
    "mmcv_compiled_ops": ("mmcv", 120, """import torch, mmcv, mmcv.ops, mmdet, mmrotate, json
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[5.,5.,2.,2.,0.]],device=device)
scores = torch.tensor([.9],device=device)
_, keep = mmcv.ops.nms_rotated(boxes,scores,.5)
assert keep.numel() == 1
print(json.dumps({'mmcv': mmcv.__version__, 'mmdet': mmdet.__version__, 'mmrotate': mmrotate.__version__, 'rotated_nms_device': device}))
"""),
    "pdf_renderer": ("pypdfium2", 45, """import pypdfium2 as pdfium, json
doc = pdfium.PdfDocument.new()
page = doc.new_page(32,32)
bitmap = page.render(scale=1)
image = bitmap.to_pil()
assert image.size == (32,32)
image.close(); bitmap.close(); page.close(); doc.close()
print(json.dumps({'blank_page_32x32': 'OK'}))
"""),
    "torch_attention": ("torch", 90, """import torch, json
from torch.nn.functional import scaled_dot_product_attention
assert torch.cuda.is_available(), 'CUDA required for attention probe'
tested = []
for dtype in [torch.float16] + ([torch.bfloat16] if torch.cuda.is_bf16_supported() else []):
    q = torch.randn(1, 2, 16, 64, device='cuda', dtype=dtype)
    y = scaled_dot_product_attention(q, q, q)
    torch.cuda.synchronize()
    assert y.shape == q.shape and bool(torch.isfinite(y).all())
    tested.append(str(dtype))
print(json.dumps({'tiny_sdpa_cuda': 'OK', 'tested_dtypes': tested, 'model_inference': False}))
"""),
    "diffusers_cogview4": ("diffusers", 180, """import json, diffusers
print('stage: CogView4 classes', flush=True)
from diffusers import CogView4Transformer2DModel, CogView4Pipeline
print(json.dumps({'diffusers':diffusers.__version__, 'module_file':diffusers.__file__,
 'CogView4Transformer2DModel':'OK', 'CogView4Pipeline':'OK', 'weights_loaded':False}))
"""),
    "transformers_glm": ("transformers", 180, """import json, transformers
from transformers import GlmModel
print(json.dumps({'transformers':transformers.__version__, 'GlmModel':'OK', 'weights_loaded':False}))
"""),
    "peft_api": ("peft", 120, """import peft, json
from peft import PeftModel
print(json.dumps({'peft':peft.__version__, 'module_file':peft.__file__, 'PeftModel':'OK'}))
"""),
    "restoration_dependencies": ("accelerate", 90, """import importlib, json
names = ['accelerate','safetensors','sentencepiece','huggingface_hub','PIL','numpy']
result = {}
for name in names:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps(result))
"""),
    "difftsr_dependencies": ("omegaconf", 120, """import importlib, json
result = {}
for name in ['torch','torchvision','pytorch_lightning','omegaconf','zhconv','einops','torchmetrics']:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps({'third_party_imports':result,'difftsr_model_code_tested':False}))
"""),
    "unihir_dependencies": ("accelerate", 120, """import importlib, json
result = {}
for name in ['torch','transformers','accelerate','fontTools','safetensors','einops']:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps({'third_party_imports':result,'unihir_model_code_tested':False}))
"""),
    "flash_attention": ("flash_attn", 90, """import torch, flash_attn, json
from flash_attn import flash_attn_func
assert torch.cuda.is_available(), 'CUDA required for FlashAttention probe'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
q = torch.randn(1,16,2,64,device='cuda',dtype=dtype)
y = flash_attn_func(q,q,q)
torch.cuda.synchronize()
assert y.shape == q.shape and bool(torch.isfinite(y).all())
print(json.dumps({'flash_attn':flash_attn.__version__, 'dtype':str(dtype),
 'module_file':flash_attn.__file__, 'tiny_cuda_forward':'OK', 'model_inference':False}))
"""),
    "torch_build": ("torch", 90, """import torch, json, sysconfig, pathlib
from torch.utils.cpp_extension import CUDA_HOME
print(json.dumps({'CUDA_HOME':str(CUDA_HOME) if CUDA_HOME else None,
 'torch_build_config':torch.__config__.show(), 'Python_h_exists':pathlib.Path(sysconfig.get_path('include'),'Python.h').is_file(),
 'compiled_arches':torch.cuda.get_arch_list() if torch.cuda.is_available() else [], 'extension_compilation_performed':False}))
"""),
    "torch_restoration_ops": ("torch", 90, """import torch, json
from torch.nn import functional as F
assert torch.cuda.is_available(), 'CUDA required for restoration tensor ops'
x = torch.ones(1,3,16,16,device='cuda',dtype=torch.float32)
y = F.conv2d(x,torch.ones(12,3,3,3,device='cuda'),padding=1)
y = F.pixel_shuffle(y,2)
y = F.interpolate(y,scale_factor=2,mode='bicubic',align_corners=False)
torch.cuda.synchronize()
assert tuple(y.shape)==(1,3,64,64) and bool(torch.isfinite(y).all())
print(json.dumps({'conv2d_pixel_shuffle_bicubic_cuda':'OK','dtype':'float32','model_inference':False}))
"""),
    "vit_pytorch_api": ("vit_pytorch", 90, """import vit_pytorch, json, importlib.metadata as md
from vit_pytorch import ViT
print(json.dumps({'version':md.version('vit-pytorch'),'module_file':vit_pytorch.__file__,
 'ViT_api':'OK','model_constructed':False,'model_inference':False}))
"""),
    "keras_torch_hdf5": ("keras", 120, """import os, io, json
os.environ['KERAS_BACKEND']='torch'
import keras, torch, h5py, numpy as np
from keras.layers import Input, Conv2D, MaxPooling2D, Dropout, UpSampling2D, concatenate
assert keras.backend.backend()=='torch', 'Keras Torch backend unavailable'
stream=io.BytesIO()
with h5py.File(stream,'w') as handle: handle['tiny']=np.arange(4,dtype=np.float32)
stream.seek(0)
with h5py.File(stream,'r') as handle: assert np.array_equal(handle['tiny'][:],np.arange(4))
print(json.dumps({'keras':keras.__version__,'backend':keras.backend.backend(),
 'torch':torch.__version__,'h5py':h5py.__version__,'hdf5':h5py.version.hdf5_version,
 'keras_file':keras.__file__,'h5py_file':h5py.__file__,'HDF5_numpy_bridge':'OK',
 'generator_constructed':False,'weights_loaded':False,'model_inference':False}))
"""),
}


PROBE_GUARD = '''import sys
sys.dont_write_bytecode = True
def _vtm_diagnostic_audit(event, args):
    if event in ('socket.connect', 'socket.getaddrinfo'):
        raise RuntimeError('Network disabled in environment diagnostics')
sys.addaudithook(_vtm_diagnostic_audit)
'''

DIAGNOSTIC_DEFAULTS = {'max_scan_dirs': 500, 'max_asset_records': 12000,
    'verify_asset_hashes': False, 'check_existing_runtimes': True,
    'quick_restoration_checks': False,
    'max_runtime_checks': 4, 'probe_timeout_cap': 180, 'time_budget_seconds': 600}

RUNTIME_INVENTORY = '''import sys, sysconfig, platform, json, importlib.metadata as md, importlib.util
records, seen = [], set()
for dist in md.distributions():
    name=dist.metadata.get('Name',''); key=name.lower().replace('_','-').replace('.','-')
    records.append({'name':name,'version':dist.version,'active':key not in seen,
        'location':str(dist.locate_file('')),'requires_python':dist.metadata.get('Requires-Python'),
        'requires':dist.requires or []})
    seen.add(key)
modules={}
for name in ['torch','torchvision','transformers','numpy','cv2','paddle','paddleocr','mmcv',
             'diffusers','peft','accelerate','omegaconf','flash_attn','pypdfium2','fontTools',
             'vit_pytorch','keras','h5py']:
    try:
        spec=importlib.util.find_spec(name)
        modules[name]={'available':spec is not None,'origin':getattr(spec,'origin',None)}
    except Exception as exc: modules[name]={'available':False,'error':str(exc)}
try:
    from packaging.tags import sys_tags
    tags=[str(tag) for tag in sys_tags()]
except Exception: tags=[]
print(json.dumps({'python':sys.version,'executable':sys.executable,'prefix':sys.prefix,
    'ABI':sysconfig.get_config_var('SOABI'),'machine':platform.machine(),
    'include_dir':sysconfig.get_path('include'),'Python_h_exists':__import__('pathlib').Path(sysconfig.get_path('include'),'Python.h').is_file(),
    'all_distributions':records,'module_specs':modules,'wheel_tags':tags}))
'''


def run_command(command, timeout=30):
    env = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1", "USE_TF": "0", "USE_FLAX": "0",
           "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True", "DIFFUSERS_OFFLINE": "1",
           "HF_DATASETS_OFFLINE": "1", "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUTF8": "1",
           "PIP_DISABLE_PIP_VERSION_CHECK": "1", "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"}
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE'):
        env.pop(key, None)
    try:
        result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace",
                                timeout=timeout, env=env)
        return {"status": "ok" if result.returncode == 0 else "failed", "returncode": result.returncode,
                "stdout": result.stdout, "stderr": result.stderr}
    except subprocess.TimeoutExpired as exc:
        def decode(value):
            return value.decode("utf-8", errors="replace") if isinstance(value, bytes) else value or ""
        return {"status": "timeout", "timeout_seconds": timeout,
                "stdout": decode(exc.stdout), "stderr": decode(exc.stderr)}
    except OSError as exc:
        return {"status": "unavailable", "detail": str(exc)}


def read_text(path):
    try:
        return Path(path).read_text(encoding="utf-8").strip()
    except OSError:
        return None


def hardware():
    meminfo = {}
    for line in (read_text("/proc/meminfo") or "").splitlines():
        key, value = line.split(":", 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
    limit = read_text("/sys/fs/cgroup/memory.max") or read_text("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    current = read_text("/sys/fs/cgroup/memory.current") or read_text("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    limit = int(limit) if limit and limit.isdigit() else None
    current = int(current) if current and current.isdigit() else None
    totals = [value for value in (meminfo.get("MemTotal"), limit) if value is not None]
    total = min(totals) if totals else None
    available = meminfo.get("MemAvailable")
    if limit is not None and current is not None:
        available = min(available if available is not None else limit, max(0, limit-current))
    quota = read_text("/sys/fs/cgroup/cpu.max")
    try:
        affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
    except OSError:
        affinity = None
    cpu_model = next((line.split(":",1)[1].strip() for line in (read_text("/proc/cpuinfo") or "").splitlines()
                      if line.startswith("model name")), platform.processor())
    disks = {}
    for name in ("/", "/kaggle/input", "/kaggle/working", "/tmp"):
        if Path(name).is_dir():
            disk = shutil.disk_usage(name)
            disks[name] = {"total_GiB": round(disk.total/1024**3,2), "free_GiB": round(disk.free/1024**3,2)}
    return {"os": platform.platform(), "machine": platform.machine(), "libc": platform.libc_ver(),
            "python": sys.version, "python_executable": sys.executable, "python_ABI": sysconfig.get_config_var("SOABI"),
            "cpu_model": cpu_model, "cpu_logical": os.cpu_count(), "cpu_affinity": affinity, "cpu_quota_raw": quota,
            "RAM_effective_GiB": round(total/1024**3,2) if total is not None else None,
            "RAM_available_GiB": round(available/1024**3,2) if available is not None else None, "disks": disks}


def distributions():
    records = []
    seen = set()
    for dist in metadata.distributions():
        name = dist.metadata.get('Name', '')
        key = name.lower().replace('_', '-').replace('.', '-')
        records.append({"name": name, "version": dist.version, 'active': key not in seen,
                        'location': str(dist.locate_file('')),
                        "requires_python": dist.metadata.get("Requires-Python"), "requires": dist.requires or []})
        seen.add(key)
    return sorted(records, key=lambda item: item["name"].lower())


def offline_inputs(root=Path("/kaggle/input"), max_dirs=150, max_files=80):
    found, visited = [], 0
    if not root.is_dir():
        return {"exists": False, "files": [], "truncated": False}
    for parent, dirs, files in os.walk(root):
        visited += 1
        if len(Path(parent).relative_to(root).parts) >= 6:
            dirs[:] = []
        for name in sorted(files):
            if name.endswith((".whl", ".tar.gz", ".tar.xz", ".zip", ".safetensors", ".pth")) or name in {"config.json", "inference.yml", "bundle_manifest.json"}:
                path = Path(parent) / name
                try:
                    size = path.stat().st_size
                except OSError:
                    continue
                found.append({"path": str(path), "bytes": size})
                if len(found) >= max_files:
                    return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
        if visited >= max_dirs:
            return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
    return {"exists": True, "files": found, "truncated": False, "visited_dirs": visited}


def diagnostic_options(options=None):
    if set(options or {}) - DIAGNOSTIC_DEFAULTS.keys():
        raise ValueError('Unknown diagnostic options')
    result = {**DIAGNOSTIC_DEFAULTS, **(options or {})}
    for key, default in DIAGNOSTIC_DEFAULTS.items():
        if isinstance(default, bool):
            if type(result[key]) is not bool:
                raise ValueError(f'{key} must be bool')
        elif type(result[key]) is not int or result[key] < 1:
            raise ValueError(f'{key} must be a positive integer')
    return result


def diagnostic_json(path):
    path = Path(path)
    if path.stat().st_size > 16*1024*1024:
        raise ValueError('Diagnostic metadata exceeds 16 MiB; not read')
    return json.loads(path.read_text(encoding='utf8'))


def diagnostic_payload(output):
    for line in reversed(output.splitlines()):
        try:
            value = json.loads(line)
            if isinstance(value, dict):
                return value
        except ValueError:
            continue
    return None


def discover_asset_roots(input_root, configured, *, max_dirs=500):
    """Bounded discovery prunes weights/objects/source trees; explicit paths take priority."""
    markers = {'models': 'models_manifest.json', 'packages': 'bundle_manifest.json',
               'prepared_images': 'dataset.json'}
    candidates = {key: [] for key in markers}
    root, visited, truncated = Path(input_root), 0, False
    if root.is_dir():
        for parent, dirs, files in os.walk(root):
            visited += 1
            for kind, marker in markers.items():
                if marker in files:
                    candidates[kind].append(str(Path(parent).resolve()))
            dirs[:] = sorted(d for d in dirs if d not in {'sources', 'wheels', 'objects', 'snapshots', '__pycache__', '.cache'})
            if len(Path(parent).relative_to(root).parts) >= 6:
                dirs[:] = []
            if visited >= max_dirs:
                truncated = True
                break
    found = {}
    for kind, marker in markers.items():
        explicit = configured.get(kind)
        if explicit:
            path = Path(explicit).resolve()
            if path.is_file():
                path = path.parent.parent if path.parent.name == 'snapshots' else path.parent
            for child in (path/kind, path/'kaggle_upload'/kind):
                if not (path/marker).is_file() and (child/marker).is_file():
                    path = child
                    break
            found[kind] = {'path': str(path), 'selection': 'explicit', 'exists': path.is_dir()}
        elif len(candidates[kind]) == 1:
            found[kind] = {'path': candidates[kind][0], 'selection': 'discovered', 'exists': True}
        else:
            found[kind] = {'path': None, 'selection': 'ambiguous' if candidates[kind] else 'not_found',
                           'candidates': candidates[kind], 'exists': False}
    found['discovery'] = {'visited_dirs': visited, 'truncated': truncated}
    found['pdfs'] = {'path': str(configured['pdfs']) if configured.get('pdfs') else None,
                     'selection': 'explicit' if configured.get('pdfs') else 'not_configured'}
    return found


def diagnostic_relative(root, relative):
    value = str(relative)
    parts = PurePosixPath(value)
    if ':' in value or '\\' in value or parts.is_absolute() or '..' in parts.parts:
        raise ValueError('Invalid portable asset path')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError('Asset reference escapes its root')
    return path


def inspect_asset_records(root, records, *, limit=12000, verify_hashes=False):
    """Default checks file presence and size only, without opening model weights."""
    missing, mismatches, checked, bytes_declared, hashes_verified = [], [], 0, 0, 0
    for record in records[:limit]:
        relative = record.get('path', '')
        try:
            path = diagnostic_relative(root, relative)
            if not path.is_file():
                missing.append(relative)
                continue
            actual = path.stat().st_size
            if record.get('bytes') is not None and actual != record['bytes']:
                mismatches.append({'path': relative, 'kind': 'size', 'actual': actual, 'expected': record['bytes']})
            elif verify_hashes and record.get('sha256'):
                digest = hashlib.sha256()
                with path.open('rb') as stream:
                    for block in iter(lambda: stream.read(8*1024*1024), b''):
                        digest.update(block)
                if digest.hexdigest() != record['sha256']:
                    mismatches.append({'path': relative, 'kind': 'sha256'})
                else:
                    hashes_verified += 1
            checked += 1
            bytes_declared += record.get('bytes', actual)
        except (OSError, ValueError) as exc:
            mismatches.append({'path': relative, 'kind': 'unreadable_or_invalid', 'detail': str(exc)})
    return {'status': 'issues_found' if missing or mismatches else 'analysis_incomplete' if len(records) > limit
            else 'files_present_sizes_match', 'declared_files': len(records), 'checked_files': checked,
            'missing': missing, 'mismatches': mismatches, 'declared_bytes_checked': bytes_declared,
            'truncated': len(records) > limit, 'hashes_requested': verify_hashes,
            'hashes_verified_files': hashes_verified,
            'model_inference_verified': False}


def inspect_runtime_kernel_requirements(plan):
    """Compare the measured kernel to a plan without importing ML packages."""
    mismatches = []
    for name, expected in plan.get('kernel_reuse', {}).items():
        try:
            actual = metadata.version(name)
        except metadata.PackageNotFoundError:
            actual = None
        if actual != expected:
            mismatches.append({'package': name, 'expected': expected, 'actual': actual})
    abi_matches = (str(sys.version_info.major)+str(sys.version_info.minor) == str(plan.get('python'))
                   and sys.platform == 'linux' and platform.machine() == 'x86_64')
    return {'state': 'kernel_metadata_match' if not mismatches and abi_matches else 'kernel_metadata_differs',
            'mismatches': mismatches, 'python': platform.python_version(),
            'python_platform_match': abi_matches, 'expected_python': plan.get('python'),
            'platform': sysconfig.get_platform(), 'model_inference_verified': False,
            'note': 'Metadata only; missing overlay packages in the kernel are allowed. No installation or model load.'}


def diagnostic_wheel_metadata(path):
    with zipfile.ZipFile(path) as archive:
        names = [n for n in archive.namelist() if n.endswith('.dist-info/METADATA')]
        if len(names) != 1 or archive.getinfo(names[0]).file_size > 4*1024**2:
            raise ValueError('Invalid or oversized wheel METADATA')
        body = Parser().parsestr(archive.read(names[0]).decode('utf8'))
    return {'name': body['Name'], 'version': body['Version'],
            'requires_python': body.get('Requires-Python'), 'requires': body.get_all('Requires-Dist', [])}


def inspect_runtime_dependency_metadata(root, plan):
    """Resolve the planned overlay against this kernel without pip or imports."""
    try:
        from packaging.requirements import Requirement
        from packaging.utils import canonicalize_name
        from packaging.specifiers import SpecifierSet
        from packaging.markers import default_environment
        selected, problems, checked, extras = {}, [], [], []
        for record in plan.get('wheels', []):
            item = diagnostic_wheel_metadata(diagnostic_relative(root, record['path']))
            if (record.get('name') and canonicalize_name(record['name']) != canonicalize_name(item['name'])) or (
                    record.get('version') and record['version'] != item['version']):
                problems.append({'path':record['path'], 'state':'wheel_metadata_differs_from_plan'})
            selected[canonicalize_name(item['name'])] = dict(item, source='staged_overlay')
        pending = list(plan.get('requirements', []))
        roots_explicit = bool(pending)
        if not pending:
            pending = [name+'=='+version for name, version in plan.get('kernel_reuse', {}).items()]
            pending.extend(item['name']+'=='+item['version'] for item in selected.values())
        seen = set()
        marker_environment = {**default_environment(), 'extra': ''}
        while pending and len(seen) < 1500:
            raw = pending.pop()
            if raw in seen:
                continue
            seen.add(raw)
            requirement = Requirement(raw)
            if requirement.marker and not requirement.marker.evaluate(marker_environment):
                continue
            if requirement.extras:
                extras.append(raw)
            name = canonicalize_name(requirement.name)
            if name not in selected:
                try:
                    dist = metadata.distribution(name)
                    selected[name] = {'name':name, 'version':dist.version, 'source':'active_kernel',
                        'requires':dist.requires or [], 'requires_python':dist.metadata.get('Requires-Python')}
                except metadata.PackageNotFoundError:
                    problems.append({'requirement':raw, 'state':'missing_in_kernel_and_planned_wheels'})
                    continue
            item = selected[name]
            if not requirement.specifier.contains(item['version'], prereleases=True):
                problems.append({'requirement':raw, 'state':'version_conflict', 'actual':item['version'], 'source':item['source']})
            if item['requires_python'] and not SpecifierSet(item['requires_python']).contains(platform.python_version(), prereleases=True):
                problems.append({'package':name, 'state':'python_version_conflict', 'requires_python':item['requires_python']})
            checked.append({'requirement':raw, 'version':item['version'], 'source':item['source']})
            pending.extend(item['requires'])
        native = plan.get('native_sources', [])
        return {'state':'issues_found' if problems else 'partial_metadata' if pending or native or extras else 'metadata_closure_satisfied',
            'problems':problems, 'checked_requirements':checked, 'truncated':bool(pending),
            'optional_extras_not_evaluated':extras,
            'root_requirements_source':'explicit_plan' if roots_explicit else 'planned_kernel_reuse_and_wheels',
            'native_source_metadata_not_resolved':[r['path'] for r in native],
            'note':'Default dependency markers only; no optional extras, installation, native build or model execution verified.'}
    except (ImportError, OSError, ValueError, KeyError, zipfile.BadZipFile) as error:
        return {'state':'metadata_analysis_unavailable', 'detail':str(error)}


def inspect_offline_assets(roots, settings):
    reports = {}
    def inspect(name, root, manifest, required=()):
        result = {'manifest': str(manifest), 'required_missing': [], 'state': 'manifest_missing'}
        try:
            data = diagnostic_json(manifest)
            result.update(inspect_asset_records(root, data.get('files', []), limit=settings['max_asset_records'],
                                                verify_hashes=settings['verify_asset_hashes']))
            result['state'] = result['status']
            result['required_missing'] = [r for r in required if not diagnostic_relative(root, r).is_file()]
            indexed = {r.get('path') for r in data.get('files', [])}
            result['required_not_in_inventory'] = [r for r in required if r not in indexed]
            if result['required_missing'] or result['required_not_in_inventory']:
                result['state'] = 'missing_required_asset'
            if name == 'tadisr':
                shards = []
                for prefix, index in (('transformer', 'diffusion_pytorch_model.safetensors.index.json'),
                                      ('text_encoder', 'model.safetensors.index.json')):
                    index_path = Path(root)/'CogView4'/prefix/index
                    if index_path.is_file():
                        body = diagnostic_json(index_path)
                        for part in set(body['weight_map'].values()):
                            if (not diagnostic_relative(Path(root)/'CogView4'/prefix, part).is_file()
                                    or f'CogView4/{prefix}/{part}' not in indexed):
                                shards.append(prefix+'/'+part)
                result['missing_indexed_shards'] = shards
                if shards:
                    result['state'] = 'missing_required_asset'
                result['prompt_embeddings_present'] = (Path(root)/'CogView4/saved_prompt_tokens_nocfg.pt').is_file()
                result['prompt_note'] = 'Notebook 1 can derive the fixed prompt offline from verified base assets'
        except (OSError, ValueError, KeyError, TypeError) as exc:
            result['detail'] = str(exc)
            if Path(manifest).is_file():
                result['state'] = 'invalid_manifest'
        reports[name] = result
    models = roots.get('models', {}).get('path')
    if models:
        models = Path(models)
        inspect('ocr_models', models, models/'models_manifest.json', ['tonggu/layout/finetune_infer.pth'])
        requirements = {
            'docentr': ['model_8_2018_base.pt'],
            'degan': ['degan_weights.bundle'],
            'realesrgan': ['RealESRGAN_x4plus.pth'],
            'docres': ['docres.pkl'],
            'docdiff': ['init.pth','denoiser.pth'],
            'restormer': ['motion_deblurring.pth','single_image_defocus_deblurring.pth',
                          'real_denoising.pth','gaussian_color_denoising_blind.pth'],
            'unihir': ['model.safetensors', 'ae.safetensors', 'llm_config.json', 'vit_config.json',
                      'tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt'] +
                     ['font/'+n for n in ('KaiXinSongA.ttf','KaiXinSongB.ttf','TH-Tshyn-P0.ttf','TH-Tshyn-P1.ttf','TH-Tshyn-P16.ttf','TH-Tshyn-P2.ttf')],
            'tadisr': ['tadisr_cogview4_realce_17500.pkl'] + ['CogView4/'+n for n in (
                'model_index.json','vae/config.json','vae/diffusion_pytorch_model.safetensors',
                'transformer/config.json','transformer/diffusion_pytorch_model.safetensors.index.json',
                'text_encoder/config.json','text_encoder/model.safetensors.index.json',
                'tokenizer/tokenizer.json','tokenizer/tokenizer_config.json','tokenizer/special_tokens_map.json',
                'scheduler/scheduler_config.json')],
        }
        for name, required in requirements.items():
            root = models/'image_restoration'/name
            inspect(name, root, root/'assets.json', required)
        reports['difftsr']={'state':'excluded_by_configuration','required':False,
                            'assets_still_present':(models/'image_restoration/difftsr').exists()}
    packages = roots.get('packages', {}).get('path')
    if packages:
        packages = Path(packages)
        inspect('ocr_packages', packages, packages/'bundle_manifest.json',
                ['paddle_manifest.json','tonggu_manifest.json','system_assets_manifest.json'])
        restoration = packages/'image_restoration'
        try:
            plan = diagnostic_json(restoration/'manifests/tadisr_runtime.json')
            reports['tadisr_packages'] = inspect_asset_records(restoration, plan.get('wheels', []),
                limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
            reports['tadisr_kernel_compatibility'] = inspect_runtime_kernel_requirements(plan)
            reports['tadisr_dependency_metadata'] = inspect_runtime_dependency_metadata(restoration, plan)
        except (OSError, ValueError, TypeError) as exc:
            reports['tadisr_packages'] = {'status': 'manifest_unavailable', 'detail': str(exc)}
        for name in ('document', 'unihir', 'learned_documents'):
            try:
                plan=diagnostic_json(restoration/f'manifests/{name}_runtime.json')
                reports[name+'_runtime_manifest'] = plan
                reports[name+'_kernel_compatibility'] = inspect_runtime_kernel_requirements(plan)
                reports[name+'_dependency_metadata'] = inspect_runtime_dependency_metadata(restoration, plan)
                records=plan.get('wheels',[])+plan.get('native_sources',[])
                if plan.get('bootstrap'):records.append(plan['bootstrap'])
                reports[name+'_runtime_assets']=inspect_asset_records(restoration,records,
                    limit=settings['max_asset_records'],verify_hashes=settings['verify_asset_hashes'])
            except (OSError, ValueError) as exc:
                reports[name+'_runtime_manifest'] = {'state': 'manifest_unavailable', 'detail': str(exc)}
        try:
            catalog = diagnostic_json(restoration/'manifests/asset_catalog.json')
            for name, group in catalog['groups'].items():
                records = [dict(r, path=r['path'].removeprefix('packages/'))
                           for r in group['files'] if r['kind'] == 'source']
                if not records:
                    continue
                reports[name+'_sources'] = inspect_asset_records(packages, records,
                    limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
                reports[name+'_sources'].update(revision=group['revision'], license=group['license'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            reports['restoration_catalog_sources'] = {'state':'manifest_unavailable', 'detail':str(exc)}
        try:
            records = diagnostic_json(restoration/'manifests/embedded_sources.json')
            records = [dict(r, path=r['path'].removeprefix('packages/')) for r in records]
            reports['embedded_upstream_sources'] = inspect_asset_records(packages, records,
                limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            reports['embedded_upstream_sources'] = {'state':'manifest_unavailable', 'detail':str(exc)}
    prepared = roots.get('prepared_images', {}).get('path')
    if prepared:
        try:
            data = diagnostic_json(Path(prepared)/'dataset.json')
            reports['prepared_images'] = {'format': data.get('format'), 'version': data.get('version'),
                'snapshot_id': data.get('snapshot_id'), 'pages': len(data.get('pages', [])),
                'restoration': data.get('restoration'), 'state': 'descriptor_read_only', 'pixels_verified': False}
        except (OSError, ValueError, TypeError) as exc:
            reports['prepared_images'] = {'state': 'descriptor_unavailable', 'detail': str(exc)}
    return reports


def runtime_candidates(configured, *, include_existing=True):
    result = {str(name): {'python': str(path), 'source': 'explicit'} for name, path in (configured or {}).items() if path}
    if include_existing:
        for path in (Path('/tmp/vtm_kaggle_runtime/runtime_config.json'), Path('/kaggle/working/runtime_config.json')):
            try:
                data = diagnostic_json(path)
                for name in ('paddle', 'tonggu'):
                    if data.get(name+'_python') and name not in result:
                        result[name] = {'python': data[name+'_python'], 'source': str(path)}
            except (OSError, ValueError, TypeError):
                pass
        path = Path('/kaggle/working/.image_runtime/preflight.json')
        try:
            for name, item in diagnostic_json(path).items():
                if isinstance(item, dict) and item.get('python') and name not in result:
                    result[name] = {'python': item['python'], 'source': str(path)}
        except (OSError, ValueError, TypeError):
            pass
    return result


def runtime_probe_names(name):
    common = ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'pdf_renderer', 'torch_build']
    if name in {'paddle', 'ppocr'}:
        return ['numpy_cv2', 'pdf_renderer', 'paddle_cuda', 'paddleocr_apis']
    if name == 'tonggu':
        return common + ['torchvision_ops', 'mmcv_compiled_ops']
    if name == 'difftsr':
        return common + ['torchvision_ops', 'difftsr_dependencies']
    if name in {'unihir', 'unihir_restoration'}:
        return common + ['unihir_dependencies', 'flash_attention', 'torch_attention']
    if name in {'tadisr', 'tadisr_x4'}:
        return common + ['diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies', 'torch_restoration_ops']
    if name in {'realesrgan', 'realesrgan_x4'}:
        return common + ['torch_restoration_ops']
    if name.startswith(('docentr', 'degan', 'learned_documents')):
        return common + ['vit_pytorch_api', 'keras_torch_hdf5']
    return common


def inspect_staged_wheels(roots, runtime_inventories):
    packages = roots.get('packages', {}).get('path')
    if not packages:
        return {'state': 'packages_not_configured', 'wheels': []}
    try:
        from packaging.utils import parse_wheel_filename
        result = []
        candidates = sorted(Path(packages).rglob('*.whl'))
        for path in candidates[:2000]:
            try:
                _, _, _, tags = parse_wheel_filename(path.name)
                wheel_tags = {str(tag) for tag in tags}
                result.append({'file': str(path), 'wheel_tags': sorted(wheel_tags),
                    'metadata':diagnostic_wheel_metadata(path),
                    'compatible_with_observed_interpreters': {name: bool(wheel_tags & set(inv['wheel_tags']))
                        if inv.get('wheel_tags') else None for name, inv in runtime_inventories.items()}})
            except (OSError, ValueError, zipfile.BadZipFile) as exc:
                result.append({'file': str(path), 'state': 'invalid_wheel_metadata_or_filename', 'detail': str(exc)})
        return {'state': 'tag_and_metadata_comparison', 'wheels': result,
                'total_wheels':len(candidates), 'truncated':len(candidates)>2000,
                'note': 'A cp310 wheel failing kernel cp313 tags can still belong to a separate cp310 runtime; tags do not prove CUDA support'}
    except ImportError:
        return {'state': 'packaging_module_unavailable', 'wheels': []}


def readiness_summary(report):
    """Per-test observations only: never promote these diagnostics to model readiness."""
    groups = {'notebook_1_classical': ['numpy_cv2', 'pdf_renderer'],
        'notebook_1_realesrgan': ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'torch_restoration_ops'],
        'notebook_1_tadisr': ['torch_cuda', 'diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies'],
        'notebook_1_unihir': ['torch_cuda', 'unihir_dependencies', 'flash_attention'],
        'notebook_1_docentr': ['torch_cuda', 'torch_numpy_bridge', 'vit_pytorch_api'],
        'notebook_1_degan': ['torch_cuda', 'keras_torch_hdf5'],
        'notebook_3_hunyuan': ['torch_cuda', 'torch_numpy_bridge', 'transformers_hunyuan'],
        'notebook_3_paddle_in_kernel': ['paddle_cuda', 'paddleocr_apis'],
        'notebook_3_tonggu_in_kernel': ['torch_cuda', 'mmcv_compiled_ops']}
    result = {}
    for name, probes in groups.items():
        states = {p: report['probes'].get(p, {}).get('status', 'not_run') for p in probes}
        result[name] = {'kernel_probe_states': states, 'model_inference_verified': False,
            'note': 'Kernel imports are separate from staged assets and private runtimes. Passing tiny ops does not verify full model inference.'}
    return result


def collect_environment(*, run_probes=True, input_root=Path('/kaggle/input'), configured_paths=None,
                        runtime_pythons=None, options=None, checkpoint_path=None):
    settings = diagnostic_options(options)
    started = time.monotonic()
    inventory = distributions()
    versions = {}
    for item in inventory:
        name = item['name'].lower().replace('_', '-')
        if name not in versions or item.get('active') is True:
            versions[name] = item['version']
    roots = discover_asset_roots(input_root, configured_paths or {}, max_dirs=settings['max_scan_dirs'])
    report = {"schema": 1, "diagnostics_version": 2, "collected_at_UTC": datetime.now(timezone.utc).isoformat(),
              "hardware": hardware(), "packages": {name: versions.get(name) for name in PACKAGES},
              "all_distributions": inventory, "probes": {}, "tools": {}, "offline_inputs": offline_inputs(Path(input_root)),
              "prior_vtm_runtime_exists": Path("/tmp/vtm_kaggle_runtime/runtime_config.json").is_file(),
              "asset_roots": roots, "options": settings, "assets": inspect_offline_assets(roots, settings),
              "runtimes": {}, "complete": False, "status": 'collecting', "execution_limits": [],
              "model_inference_performed": False,
              "configuration": {'input_root': str(input_root), 'configured_paths': {k:str(v) for k,v in (configured_paths or {}).items()},
                                'runtime_pythons': {k:str(v) for k,v in (runtime_pythons or {}).items()}},
              "build_inventory": {'Python_include': sysconfig.get_path('include'),
                  'Python_h_exists': Path(sysconfig.get_path('include'), 'Python.h').is_file(),
                  'sysconfig': {k:sysconfig.get_config_var(k) for k in ('CC','CXX','LDSHARED','LIBDIR','EXT_SUFFIX','SOABI')},
                  'selected_environment': {k:os.environ.get(k) for k in ('CUDA_HOME','CUDA_PATH','CC','CXX','CUDACXX','MAX_JOBS','CUDA_VISIBLE_DEVICES')}},
              "note": "Inventory phase only: metadata/imports/tiny ops, no installation or model loading in this phase. Optional build/inference results appear separately in restoration_gpu_check. Assets use size checks unless hashes explicitly enabled."}
    def checkpoint():
        report['elapsed_seconds'] = round(time.monotonic()-started, 3)
        if checkpoint_path:
            path = Path(checkpoint_path)
            if path.resolve().is_relative_to(Path('/kaggle/input').resolve()):
                raise ValueError('Diagnostic report must not be written to Kaggle Input')
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_name(path.name+'.tmp')
            temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf8')
            temporary.replace(path)
    def bounded(command, timeout=30):
        remaining = settings['time_budget_seconds']-(time.monotonic()-started)
        if remaining < 1:
            report['complete'] = False
            if 'time_budget_seconds' not in report['execution_limits']:
                report['execution_limits'].append('time_budget_seconds')
            return {'status': 'not_run_budget', 'detail': 'Diagnostic time budget exhausted; partial report retained'}
        return run_command(command, min(timeout, settings['probe_timeout_cap'], max(1, int(remaining))))
    checkpoint()
    if not run_probes:
        report['readiness'] = readiness_summary(report)
        report.update(complete=True, status='inventory_only')
        checkpoint()
        return report
    commands = {"gpu": ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
                "nvcc": ["nvcc", "--version"], "g++": ["g++", "--version"],
                "cmake": ["cmake", "--version"], "ninja": ["ninja", "--version"],
                "gcc": ["gcc", "--version"], "ldd": ["ldd", "--version"]}
    for name, command in commands.items():
        report["tools"][name] = bounded(command)
        checkpoint()
    report["pip_check"] = bounded([sys.executable, "-m", "pip", "check"], timeout=60)
    report['kernel_runtime_inventory'] = bounded([sys.executable, '-c', PROBE_GUARD+RUNTIME_INVENTORY], timeout=60)
    kernel_inventory = diagnostic_payload(report['kernel_runtime_inventory'].get('stdout', '')) or {}
    observed = {'kernel': kernel_inventory}
    report['kernel_runtime_inventory']['data'] = kernel_inventory
    checkpoint()
    for name, (module, timeout, code) in PROBES.items():
        if name=='difftsr_dependencies':continue
        if settings['quick_restoration_checks'] and name not in {
            'torch_cuda','torch_numpy_bridge','numpy_cv2','pdf_renderer','torch_attention',
            'flash_attention','torch_restoration_ops'}:continue
        print(f"Checking {name} (timeout {timeout}s)...", flush=True)
        try:
            present = importlib.util.find_spec(module) is not None
        except (ImportError, ValueError):
            present = False
        result = bounded([sys.executable, "-c", PROBE_GUARD+code], timeout) if present else {"status": "not_installed"}
        result['data'] = diagnostic_payload(result.get('stdout', ''))
        report["probes"][name] = result
        print(f"  {result['status']}", flush=True)
        checkpoint()
    candidates = runtime_candidates(runtime_pythons, include_existing=settings['check_existing_runtimes'])
    seen_python = {str(Path(sys.executable).resolve()): 'kernel'}
    checked = 0
    for name, candidate in candidates.items():
        python = str(Path(candidate['python']).expanduser().resolve())
        state = {**candidate, 'python': python, 'model_inference_verified': False, 'probes': {}}
        report['runtimes'][name] = state
        if python in seen_python:
            state.update(status='shared_interpreter', shares_with=seen_python[python])
            checkpoint()
            continue
        if checked >= settings['max_runtime_checks']:
            state['status'] = 'not_run_runtime_limit'
            report['complete'] = False
            if 'max_runtime_checks' not in report['execution_limits']:
                report['execution_limits'].append('max_runtime_checks')
            continue
        if not Path(python).is_file():
            state['status'] = 'interpreter_missing'
            continue
        checked += 1
        seen_python[python] = name
        print(f'Checking separate runtime {name}: {python}', flush=True)
        state['inventory'] = bounded([python, '-c', PROBE_GUARD+RUNTIME_INVENTORY], timeout=60)
        detail = diagnostic_payload(state['inventory'].get('stdout', ''))
        if state['inventory']['status'] != 'ok' or detail is None:
            state['status'] = 'inventory_unavailable'
            checkpoint()
            continue
        state['inventory']['data'] = detail
        observed[name] = detail
        state['pip_check'] = bounded([python, '-m', 'pip', 'check'], timeout=60)
        for probe_name in runtime_probe_names(name):
            module, timeout, code = PROBES[probe_name]
            present = detail.get('module_specs', {}).get(module, {}).get('available', False)
            print(f'  {name}/{probe_name}', flush=True)
            probe = bounded([python, '-c', PROBE_GUARD+code], timeout) if present else {'status':'not_installed'}
            probe['data'] = diagnostic_payload(probe.get('stdout', ''))
            state['probes'][probe_name] = probe
            checkpoint()
        state['status'] = 'probed; model_inference_unverified'
        checkpoint()
    report['staged_wheel_tags'] = inspect_staged_wheels(roots, observed)
    report['readiness'] = readiness_summary(report)
    report['limitations'] = ['Missing kernel packages may be supplied by a separate staged runtime.',
        'Wheel ABI tags and tiny GPU ops do not verify full model execution or restoration fidelity.',
        'This inventory phase does not execute model weights.',
        'Optional restoration GPU checks are reported separately from inventory; no network downloads.']
    report['complete'] = not report['execution_limits']
    report['status'] = 'completed_with_probe_results' if report['complete'] else 'partial_limits'
    checkpoint()
    return report


def save_report(report, output_dir):
    output = Path(output_dir)
    if output.resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Output must be outside Kaggle Input')
    output.mkdir(parents=True, exist_ok=True)
    json_path = output / "kaggle_environment_report.json"
    text_path = output / "kaggle_environment_report.txt"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    lines = [json.dumps(report["hardware"], ensure_ascii=False, indent=2), "\nPackages:"]
    lines.extend(f"{name:24s} {version or 'not installed'}" for name, version in report["packages"].items())
    for group in ("tools", "probes"):
        for name, result in report[group].items():
            lines.append(f"\n[{name}]\n" + json.dumps(result, ensure_ascii=False, indent=2))
    lines.append("\n[pip_check]\n" + json.dumps(report.get("pip_check"), ensure_ascii=False, indent=2))
    for name in ('asset_roots', 'assets', 'runtimes', 'build_inventory', 'readiness', 'limitations','restoration_gpu_check'):
        if name in report:
            lines.append(f'\n[{name}]\n'+json.dumps(report[name], ensure_ascii=False, indent=2))
    text_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return json_path, text_path




In [ ]:
"""Kaggle-only restoration check using notebook 1's exact offline implementation.

Inventory remains read-only. The separately enabled GPU check may create a
private runtime, compile FlashAttention and run one UniHIR page. No downloads,
OCR, training, kernel package replacement, or corpus processing.
"""
from __future__ import annotations

import ast
import base64
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import time
import traceback
import uuid
import zipfile
import zlib


FLASH_GPU_PROBE = '''import json, torch, flash_attn
from flash_attn import flash_attn_func, flash_attn_varlen_func
assert torch.cuda.is_available(), 'CUDA unavailable'
torch.manual_seed(42)
devices=[]
for index in range(torch.cuda.device_count()):
    with torch.cuda.device(index):
        q,k,v=[torch.randn(1,16,2,64,device=f'cuda:{index}',dtype=torch.bfloat16) for _ in range(3)]
        y=flash_attn_func(q,k,v,dropout_p=0.,causal=False)
        reference=torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),dropout_p=0.).transpose(1,2)
        cu=torch.tensor([0,16],device=f'cuda:{index}',dtype=torch.int32)
        variable=flash_attn_varlen_func(q[0],k[0],v[0],cu,cu,16,16,dropout_p=0.,causal=False)
        torch.cuda.synchronize(index)
        assert y.shape==q.shape and variable.shape==q[0].shape
        assert bool(torch.isfinite(y).all()) and bool(torch.isfinite(variable).all())
        assert torch.allclose(y.float(),reference.float(),atol=.03,rtol=.03), 'FlashAttention differs from SDPA reference'
        assert torch.allclose(variable.float(),y[0].float(),atol=.03,rtol=.03), 'Variable-length kernel differs'
        devices.append(dict(index=index,name=torch.cuda.get_device_name(index),
            max_abs_difference=float((y.float()-reference.float()).abs().max().item())))
print(json.dumps(dict(status='passed',torch=torch.__version__,cuda=torch.version.cuda,
    devices=devices,capability=list(torch.cuda.get_device_capability(0)),
    flash_attention=flash_attn.__version__,module_file=flash_attn.__file__,
    fixed_and_variable_length_verified=True,finite=True,
    weights_loaded=False)))
'''


def check_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.tmp')
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf8')
    temporary.replace(path)


def check_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()


def check_offline_env():
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE'):env.pop(key,None)
    env.update(PYTHONNOUSERSITE='1',PYTHONUTF8='1',PIP_NO_INDEX='1',PIP_DISABLE_PIP_VERSION_CHECK='1',
        HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_DATASETS_OFFLINE='1',DIFFUSERS_OFFLINE='1',
        HF_HUB_DISABLE_TELEMETRY='1',MPLBACKEND='Agg',TOKENIZERS_PARALLELISM='false')
    return env


def check_tail(path, maximum=2000):
    try:
        with Path(path).open('rb') as stream:
            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))
            return stream.read().decode('utf8',errors='replace')
    except OSError:return ''


def check_stop(process):
    if process.poll() is not None:return
    if os.name=='nt' or not getattr(process,'_vtm_owns_group',False):process.kill()
    else:
        try:os.killpg(process.pid,signal.SIGKILL)
        except ProcessLookupError:pass
    process.wait()


def check_logged_process(command, log, *, timeout, env=None, report_path=None, new_session=True):
    """One process group; preserve bounded log tails and kill descendants on timeout."""
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic();last_notice=started;process=None
    with log.open('w',encoding='utf8') as output:
        try:
            process=subprocess.Popen(list(map(str,command)),stdout=output,stderr=subprocess.STDOUT,
                env=env or check_offline_env(),start_new_session=os.name!='nt' and new_session,
                creationflags=getattr(subprocess,'CREATE_NO_WINDOW',0))
            process._vtm_owns_group=os.name!='nt' and new_session
            while process.poll() is None:
                elapsed=time.monotonic()-started
                if elapsed>timeout:
                    check_stop(process)
                    return dict(status='timeout',seconds=round(elapsed,3),log=str(log),tail=check_tail(log))
                if time.monotonic()-last_notice>=15:
                    phase='running'
                    if report_path and Path(report_path).is_file():
                        try:phase=json.loads(Path(report_path).read_text(encoding='utf8')).get('phase',phase)
                        except (OSError,ValueError):pass
                    print(f'UniHIR check: {phase}, {elapsed:.0f}s; log: {log}',flush=True)
                    last_notice=time.monotonic()
                time.sleep(.2)
            return dict(status='ok' if process.returncode==0 else 'failed',returncode=process.returncode,
                        seconds=round(time.monotonic()-started,3),log=str(log),tail=check_tail(log))
        except BaseException:
            if process is not None:check_stop(process)
            raise


def check_payload(payload):
    raw=zlib.decompress(base64.b85decode(payload['data']))
    if hashlib.sha256(raw).hexdigest()!=payload['sha256']:raise ValueError('Notebook 1 definitions checksum mismatch')
    value=json.loads(raw)
    if value.get('format')!='vtm-notebook1-restoration-check-v1':raise ValueError('Unexpected definition bundle')
    return value


def check_native_receipt(python):
    folder=Path(python).parent.parent
    data=json.loads((folder/'install.json').read_text(encoding='utf8'))
    verified=[]
    for record in data.get('native',[]):
        wheel=folder/'native_build/wheels'/record['wheel']
        if not wheel.is_file() or check_hash(wheel)!=record['sha256']:
            raise ValueError('Compiled wheel missing or changed: '+str(wheel))
        verified.append(dict(record,path=str(wheel)))
    if not verified:raise ValueError('No compiled FlashAttention wheel receipt')
    return dict(status='built_or_reused_verified_wheel',native_wheels=verified,
                note='The separate CUDA probe executes the currently imported extension.')


def check_collect_logs(runtime, output):
    """Export logs/receipts only; never copy runtime binaries or model weights."""
    runtime,output=Path(runtime),Path(output)
    if not runtime.is_dir():return
    spec=output/'spec.json'
    cutoff=spec.stat().st_mtime if spec.is_file() else 0
    active_native=[]
    try:
        current=json.loads((output/'report.json').read_text(encoding='utf8'))
        active_native=[Path(r['path']).parent.parent for r in
            current.get('phases',{}).get('runtime_build',{}).get('receipt',{}).get('native_wheels',[])]
    except (OSError,ValueError):pass
    for pattern in ('build.log','install.json','preflight.json','worker.log','ready.json','response.json'):
        sources=sorted(runtime.rglob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)[:32]
        for source in sources:
            if source.is_symlink() or source.stat().st_size>10*1024**2:continue
            if source.stat().st_mtime<cutoff and not (pattern in ('build.log','install.json') and
                    any(source.is_relative_to(p.parent) for p in active_native)):continue
            target=output/'runtime_logs'/source.relative_to(runtime)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)


def check_inference_evidence(summary, validation):
    metrics=[m for m in summary.get('measurements',[]) if m.get('variant')=='unihir_restoration']
    if summary.get('failures') or not metrics or summary.get('variants_reused'):
        raise ValueError('Fresh, successful UniHIR inference was not established')
    for item in metrics:
        worker=item.get('worker',{})
        if worker.get('state')!='ok' or worker.get('peak_vram_bytes',0)<=0:
            raise ValueError('Missing successful GPU worker evidence')
    if not validation.get('restoration',{}).get('validated'):
        raise ValueError('Output manifest/pixels/geometry validation incomplete')
    return metrics


def restoration_check_child(spec_path):
    """Executed only by the Kaggle job supervisor, never by notebook import."""
    spec=json.loads(Path(spec_path).read_text(encoding='utf8'))
    output=Path(spec['output']);runtime=Path(spec['runtime']);report_path=output/'report.json'
    report=dict(format='vtm-kaggle-restoration-gpu-check',version=1,status='running',phase='hardware',
        mode=spec['mode'],native_build_verified=False,flash_attention_cuda_verified=False,
        model_weights_loaded=False,model_inference_attempted=False,model_inference_verified=False,
        offline_requested=True,parameters=spec['options'],phases={},pid=os.getpid())
    def phase(phase_name,**details):
        report['phase']=phase_name;report['phases'][phase_name]=dict(status='running',**details)
        check_json(report_path,report);print('STAGE:',phase_name,flush=True)
    def passed(phase_name,**details):
        report['phases'][phase_name]={**details,'status':'passed'};check_json(report_path,report)
    def guard(event,args):
        if event=='socket.connect' and isinstance(args[1],str):return
        if event in ('socket.connect','socket.getaddrinfo'):raise RuntimeError('Offline restoration check blocks network access')
    sys.addaudithook(guard)
    try:
        if sys.platform!='linux' or not Path('/kaggle/working').is_dir():
            raise RuntimeError('This GPU/build job is Kaggle-only')
        import torch
        if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable')
        devices=[dict(index=i,name=torch.cuda.get_device_name(i),vram_bytes=torch.cuda.get_device_properties(i).total_memory,
            capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]
        gpu=dict(name=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,
            capability=list(torch.cuda.get_device_capability(0)),torch=torch.__version__,cuda=torch.version.cuda,
            python=sys.version,devices=devices,gpu_count=len(devices))
        report['hardware']=gpu
        if any(d['capability'][0]<8 for d in devices):
            raise RuntimeError('FlashAttention 2 CUDA/BF16 check needs a supported Ampere-or-newer GPU')
        sharded=spec['options'].get('placement','single')=='sharded'
        if spec['mode']=='unihir_smoke':
            if sharded:
                if len(devices)<2 or sum(d['vram_bytes'] for d in devices)<48*1024**3:
                    raise RuntimeError('Sharded UniHIR needs multiple GPUs with sufficient per-device headroom; no build/model load started')
            elif gpu['vram_bytes']<75*1024**3:
                raise RuntimeError('Single-GPU UniHIR needs >=75 GiB; select placement=sharded for 4 L4')
        if spec['options'].get('mode') not in (None,spec['mode']):raise ValueError('Conflicting check mode')
        passed('hardware',**gpu)
        phase('load_shared_definitions')
        payload=check_payload(spec['payload']);scope={'__name__':'vtm_notebook1_check_definitions'}
        for i,source in enumerate(payload['sources']):exec(compile(source,f'notebook1-shared:{i}','exec'),scope)
        passed('load_shared_definitions',sha256=spec['payload']['sha256'],upstream=payload['upstream'])
        options=spec['options'];packages=Path(spec['packages']);models=Path(spec['models'])
        if not spec['packages'] or not packages.is_dir():raise FileNotFoundError('Configure a valid packages dataset root')
        if spec['mode']=='unihir_smoke' and (not spec['models'] or not models.is_dir()):
            raise FileNotFoundError('Configure a valid models dataset root')
        cfg=dict(packages_dir=str(packages),models_dir=str(models),data_dir=spec.get('data',''),
            gpu_ids=list(range(len(devices))),execution_mode='sequential',
            output_dir=str(output/'image_dataset'),books=options.get('books',[]),pages=options.get('pages','1'),
            limit=1,max_books=1,variants=['original','unihir_restoration'],crop_types=['page'],region_ids=[],
            resume=False,allow_partial=False,max_source_pixels=options.get('max_source_pixels',16_000_000),
            max_output_pixels=options.get('max_source_pixels',16_000_000),max_storage_bytes=1024**3,
            timeout_seconds=options.get('inference_timeout_seconds',900),
            load_timeout_seconds=options.get('load_timeout_seconds',900),batch_size=1,deskew=False,dewarp=False,
            layout={'dpi':200,'reading_direction':'vertical_rtl'},book_layouts={
                '1':{'dpi':225},'2':{'dpi':226},'3':{'dpi':226},'4':{'dpi':181},'5':{'dpi':181},
                '6':{'dpi':226},'bien':{'dpi':200,'reading_direction':'horizontal_ltr'},'kinh':{'dpi':45}},
            variant_options={},runtime_pythons={},reviewed_regions={},kernel_torch_source_policy='warn',
            model_options={'tadisr_x4':{'tile':768,'overlap':256},'difftsr':{'steps':20},
                'unihir_restoration':dict(device='cuda',seed=42,steps=options.get('steps',10),
                                        ref_count=options.get('ref_count',1),
                                        placement=options.get('placement','single'),gpu_ids=list(range(len(devices))),
                                        max_memory=options.get('max_memory','18GiB' if sharded else '80GiB'),
                                        activation_reserve_gib=options.get('activation_reserve_gib',4),
                                        entry_reserve_gib=options.get('entry_reserve_gib',2))})
        cfg=scope['prep_normalize'](cfg)
        original_assets=scope['prep_assets'];original_overlay=scope['prep_overlay']
        def observed_assets(*args,**kwargs):
            phase('model_asset_integrity');result=original_assets(*args,**kwargs)
            passed('model_asset_integrity',model='unihir_restoration',identity=result[1]);return result
        def observed_overlay(*args,**kwargs):
            phase('runtime_build',note='Offline compile can take a long time; build.log is preserved on failure')
            python=original_overlay(*args,**kwargs);native=check_native_receipt(python)
            report['native_build_verified']=True;passed('runtime_build',receipt=native)
            phase('flash_attention_cuda')
            result=check_logged_process([python,'-u','-c',FLASH_GPU_PROBE],output/'flash_attention_probe.log',
                                       timeout=120,env=scope['prep_worker_environment'](),new_session=False)
            if result['status']!='ok':raise RuntimeError('FlashAttention CUDA probe failed: '+result['tail'])
            data=json.loads(next(line for line in reversed(Path(result['log']).read_text(encoding='utf8').splitlines()) if line.startswith('{')))
            report['flash_attention_cuda_verified']=True;passed('flash_attention_cuda',result=data);return python
        scope['prep_assets']=observed_assets;scope['prep_overlay']=observed_overlay
        if spec['mode']=='build_and_probe':
            # Model preflight also enforces UniHIR's full-model VRAM requirement.
            # A standalone extension check needs neither model weights nor that worker.
            observed_overlay(packages,runtime,'unihir_restoration',cfg)
            report['status']='native_and_cuda_verified';return 0
        base_worker=scope['PrepWorker']
        class PreflightWorker(base_worker):
            def __init__(self,*args,**kwargs):
                phase('worker_preflight');super().__init__(*args,**kwargs)
                passed('worker_preflight',probe=self.ready.get('report'))
        scope['PrepWorker']=PreflightWorker
        preflight=scope['prep_preflight'](cfg,runtime)
        report['preflight']=preflight
        if preflight['unihir_restoration'].get('state')!='ready':raise RuntimeError('UniHIR worker preflight is not ready')
        passed('worker_preflight',state='ready')
        if not cfg['data_dir'] or not Path(cfg['data_dir']).is_dir():
            raise FileNotFoundError('Build/CUDA passed, but DATA_DIR must contain a PDF for actual inference')
        phase('pdf_bootstrap')
        scope['prep_bootstrap'](packages,runtime)
        passed('pdf_bootstrap')
        # Reuse this exact configuration's verified preflight in the same job.
        scope['prep_preflight']=lambda *args,**kwargs:preflight
        original_worker=base_worker
        class ObservedWorker(original_worker):
            def __init__(self,*args,**kwargs):
                phase('model_loading');super().__init__(*args,**kwargs)
                report['model_weights_loaded']=True;passed('model_loading',seconds=self.ready.get('load_seconds'))
            def infer(self,*args,**kwargs):
                report['model_inference_attempted']=True;phase('model_inference')
                result=super().infer(*args,**kwargs)
                passed('model_inference',note='Output pixels and manifest are validated next')
                return result
        scope['PrepWorker']=ObservedWorker
        phase('page_preparation')
        summary=scope['prep_run'](cfg,runtime)
        phase('output_validation');validation=scope['prep_validate'](cfg['output_dir'])
        metrics=check_inference_evidence(summary,validation)
        scope['prep_export_images'](cfg['output_dir'],output/'images')
        report.update(status='passed',model_inference_verified=True,inference_metrics=metrics,
            output_validation=validation,summary={k:v for k,v in summary.items() if k!='measurements'},
            fidelity='unverified; a successful run does not establish correct restored characters')
        passed('output_validation',status_detail='pixels,manifest,geometry validated')
        return 0
    except BaseException as exc:
        report.update(status='failed',failed_phase=report['phase'],error=str(exc),traceback=traceback.format_exc())
        print(report['traceback'],flush=True);return 1
    finally:
        report['finished_utc']=datetime.now(timezone.utc).isoformat();check_json(report_path,report)
        check_collect_logs(runtime,output)


def run_restoration_gpu_check(environment, *, mode, options, payload, output_dir):
    if mode not in ('off','build_and_probe','unihir_smoke'):raise ValueError('Unknown restoration check mode')
    if mode=='off':return dict(status='disabled',model_inference_verified=False)
    if sys.platform!='linux' or not Path('/kaggle/working').is_dir():
        return dict(status='skipped_not_kaggle',model_inference_verified=False,
                    detail='No model build, install or inference is permitted on local Windows')
    output=Path(output_dir).resolve()
    if output.is_relative_to(Path('/kaggle/input')):raise ValueError('Output must be outside Kaggle Input')
    run=output/('restoration_check_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
    run.mkdir(parents=True)
    roots=environment.get('asset_roots',{})
    spec=dict(mode=mode,options=options,payload=payload,output=str(run),
        runtime=options.get('runtime_dir','/tmp/vtm_image_runtime'),
        packages=roots.get('packages',{}).get('path') or '',models=roots.get('models',{}).get('path') or '',
        data=roots.get('pdfs',{}).get('path') or '')
    driver=run/'check_driver.py';driver.write_text(GPU_CHECK_DRIVER_SOURCE,encoding='utf8')
    check_json(run/'spec.json',spec)
    checkpoint=run/'report.json'
    try:
        env=check_offline_env()
        if options.get('gpu_ids') is not None:
            # IDs are logical under the platform mask, never physical host indices.
            ids=options['gpu_ids']; inherited=env.get('CUDA_VISIBLE_DEVICES')
            visible=[v.strip() for v in inherited.split(',') if v.strip() and v.strip()!='-1'] if inherited is not None else None
            if not ids or any(type(i) is not int or i<0 or (visible is not None and i>=len(visible)) for i in ids) or len(set(ids))!=len(ids):
                raise ValueError('Invalid restoration check GPU IDs under inherited CUDA_VISIBLE_DEVICES')
            env['CUDA_VISIBLE_DEVICES']=','.join(visible[i] if visible is not None else str(i) for i in ids)
        process=check_logged_process([sys.executable,'-u',str(driver),str(run/'spec.json')],run/'driver.log',
            timeout=options.get('total_timeout_seconds',5400),report_path=checkpoint,env=env)
    except BaseException as exc:
        process=dict(status='interrupted',error=str(exc),log=str(run/'driver.log'))
    result=json.loads(checkpoint.read_text(encoding='utf8')) if checkpoint.is_file() else dict(status='failed',error='Child report unavailable')
    if process['status'] in ('timeout','interrupted') or (process['status']!='ok' and result.get('status') in ('passed','native_and_cuda_verified')):
        result.update(status=process['status'],failed_phase=result.get('phase'),model_inference_verified=False)
    result.update(supervisor=process,output_dir=str(run))
    check_collect_logs(spec['runtime'],run);check_json(checkpoint,result)
    return result


def package_gpu_check_logs(report, output_dir):
    """Small review artifact. Private runtime, weights, archives and spec payload are excluded."""
    output=Path(output_dir).resolve();gpu=report.get('restoration_gpu_check',{})
    run=Path(gpu['output_dir']).resolve() if gpu.get('output_dir') else None
    archive=output/((run.name if run else 'kaggle_environment')+'_logs.zip')
    temporary=archive.with_name(archive.name+'.tmp')
    selected=[p for p in (output/'kaggle_environment_report.json',output/'kaggle_environment_report.txt') if p.is_file()]
    if run:
        if not run.is_relative_to(output):raise ValueError('Log directory outside diagnostic output')
        selected.extend(p for p in run.rglob('*') if p.is_file() and not p.is_symlink() and
            p.name not in ('spec.json','check_driver.py') and p.suffix in ('.json','.log','.txt','.png') and
            'image_dataset' not in p.relative_to(run).parts and p.stat().st_size<=10*1024**2)
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as zipped:
        for path in sorted(set(selected)):
            zipped.write(path,path.relative_to(output).as_posix())
    temporary.replace(archive)
    return archive


def make_restoration_check_payload(notebook_path):
    """Authoring only: select original notebook 1 definitions, without executing them."""
    notebook=json.loads(Path(notebook_path).read_text(encoding='utf8'))
    vendor=''.join(notebook['cells'][6]['source'])
    assignments={n.targets[0].id:n.value for n in ast.parse(vendor).body
        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}
    raw=zlib.decompress(base64.b85decode(ast.literal_eval(assignments['_VENDOR_SOURCES_B85'])))
    if hashlib.sha256(raw).hexdigest()!=ast.literal_eval(assignments['_VENDOR_SOURCES_SHA256']):
        raise ValueError('Original upstream payload checksum mismatch')
    vendors=json.loads(raw)
    revision=ast.literal_eval(assignments['UPSTREAM_REVISIONS'])['UniHIR']
    worker=ast.literal_eval(assignments['WORKER_SOURCE'])
    mini='\n'.join(('UPSTREAM_REVISIONS = '+repr({'UniHIR':revision}),
        'VENDOR_SOURCES = '+repr({'UniHIR':vendors['UniHIR']}),
        'WORKER_SOURCE = '+repr(worker),
        'RESTORATION_CODE_SIGNATURE = '+repr(ast.literal_eval(assignments['RESTORATION_CODE_SIGNATURE']))))
    sources=[''.join(notebook['cells'][4]['source']),mini,''.join(notebook['cells'][8]['source'])]
    for i,source in enumerate(sources):compile(source,f'notebook1-shared:{i}','exec')
    body=dict(format='vtm-notebook1-restoration-check-v1',sources=sources,
              upstream={'UniHIR':revision},worker_sha256=hashlib.sha256(worker.encode()).hexdigest())
    raw=json.dumps(body,ensure_ascii=False,separators=(',',':')).encode('utf8')
    return dict(sha256=hashlib.sha256(raw).hexdigest(),data=base64.b85encode(zlib.compress(raw,9)).decode('ascii'))



GPU_CHECK_DRIVER_SOURCE = '"""Kaggle-only restoration check using notebook 1\'s exact offline implementation.\n\nInventory remains read-only. The separately enabled GPU check may create a\nprivate runtime, compile FlashAttention and run one UniHIR page. No downloads,\nOCR, training, kernel package replacement, or corpus processing.\n"""\nfrom __future__ import annotations\n\nimport ast\nimport base64\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport uuid\nimport zipfile\nimport zlib\n\n\nFLASH_GPU_PROBE = \'\'\'import json, torch, flash_attn\nfrom flash_attn import flash_attn_func, flash_attn_varlen_func\nassert torch.cuda.is_available(), \'CUDA unavailable\'\ntorch.manual_seed(42)\ndevices=[]\nfor index in range(torch.cuda.device_count()):\n    with torch.cuda.device(index):\n        q,k,v=[torch.randn(1,16,2,64,device=f\'cuda:{index}\',dtype=torch.bfloat16) for _ in range(3)]\n        y=flash_attn_func(q,k,v,dropout_p=0.,causal=False)\n        reference=torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),dropout_p=0.).transpose(1,2)\n        cu=torch.tensor([0,16],device=f\'cuda:{index}\',dtype=torch.int32)\n        variable=flash_attn_varlen_func(q[0],k[0],v[0],cu,cu,16,16,dropout_p=0.,causal=False)\n        torch.cuda.synchronize(index)\n        assert y.shape==q.shape and variable.shape==q[0].shape\n        assert bool(torch.isfinite(y).all()) and bool(torch.isfinite(variable).all())\n        assert torch.allclose(y.float(),reference.float(),atol=.03,rtol=.03), \'FlashAttention differs from SDPA reference\'\n        assert torch.allclose(variable.float(),y[0].float(),atol=.03,rtol=.03), \'Variable-length kernel differs\'\n        devices.append(dict(index=index,name=torch.cuda.get_device_name(index),\n            max_abs_difference=float((y.float()-reference.float()).abs().max().item())))\nprint(json.dumps(dict(status=\'passed\',torch=torch.__version__,cuda=torch.version.cuda,\n    devices=devices,capability=list(torch.cuda.get_device_capability(0)),\n    flash_attention=flash_attn.__version__,module_file=flash_attn.__file__,\n    fixed_and_variable_length_verified=True,finite=True,\n    weights_loaded=False)))\n\'\'\'\n\n\ndef check_json(path, value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_name(path.name+\'.tmp\')\n    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf8\')\n    temporary.replace(path)\n\n\ndef check_hash(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024**2),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef check_offline_env():\n    env=dict(os.environ)\n    for key in (\'PYTHONPATH\',\'PYTHONHOME\',\'PYTHONUSERBASE\'):env.pop(key,None)\n    env.update(PYTHONNOUSERSITE=\'1\',PYTHONUTF8=\'1\',PIP_NO_INDEX=\'1\',PIP_DISABLE_PIP_VERSION_CHECK=\'1\',\n        HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\',HF_DATASETS_OFFLINE=\'1\',DIFFUSERS_OFFLINE=\'1\',\n        HF_HUB_DISABLE_TELEMETRY=\'1\',MPLBACKEND=\'Agg\',TOKENIZERS_PARALLELISM=\'false\')\n    return env\n\n\ndef check_tail(path, maximum=2000):\n    try:\n        with Path(path).open(\'rb\') as stream:\n            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))\n            return stream.read().decode(\'utf8\',errors=\'replace\')\n    except OSError:return \'\'\n\n\ndef check_stop(process):\n    if process.poll() is not None:return\n    if os.name==\'nt\' or not getattr(process,\'_vtm_owns_group\',False):process.kill()\n    else:\n        try:os.killpg(process.pid,signal.SIGKILL)\n        except ProcessLookupError:pass\n    process.wait()\n\n\ndef check_logged_process(command, log, *, timeout, env=None, report_path=None, new_session=True):\n    """One process group; preserve bounded log tails and kill descendants on timeout."""\n    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();last_notice=started;process=None\n    with log.open(\'w\',encoding=\'utf8\') as output:\n        try:\n            process=subprocess.Popen(list(map(str,command)),stdout=output,stderr=subprocess.STDOUT,\n                env=env or check_offline_env(),start_new_session=os.name!=\'nt\' and new_session,\n                creationflags=getattr(subprocess,\'CREATE_NO_WINDOW\',0))\n            process._vtm_owns_group=os.name!=\'nt\' and new_session\n            while process.poll() is None:\n                elapsed=time.monotonic()-started\n                if elapsed>timeout:\n                    check_stop(process)\n                    return dict(status=\'timeout\',seconds=round(elapsed,3),log=str(log),tail=check_tail(log))\n                if time.monotonic()-last_notice>=15:\n                    phase=\'running\'\n                    if report_path and Path(report_path).is_file():\n                        try:phase=json.loads(Path(report_path).read_text(encoding=\'utf8\')).get(\'phase\',phase)\n                        except (OSError,ValueError):pass\n                    print(f\'UniHIR check: {phase}, {elapsed:.0f}s; log: {log}\',flush=True)\n                    last_notice=time.monotonic()\n                time.sleep(.2)\n            return dict(status=\'ok\' if process.returncode==0 else \'failed\',returncode=process.returncode,\n                        seconds=round(time.monotonic()-started,3),log=str(log),tail=check_tail(log))\n        except BaseException:\n            if process is not None:check_stop(process)\n            raise\n\n\ndef check_payload(payload):\n    raw=zlib.decompress(base64.b85decode(payload[\'data\']))\n    if hashlib.sha256(raw).hexdigest()!=payload[\'sha256\']:raise ValueError(\'Notebook 1 definitions checksum mismatch\')\n    value=json.loads(raw)\n    if value.get(\'format\')!=\'vtm-notebook1-restoration-check-v1\':raise ValueError(\'Unexpected definition bundle\')\n    return value\n\n\ndef check_native_receipt(python):\n    folder=Path(python).parent.parent\n    data=json.loads((folder/\'install.json\').read_text(encoding=\'utf8\'))\n    verified=[]\n    for record in data.get(\'native\',[]):\n        wheel=folder/\'native_build/wheels\'/record[\'wheel\']\n        if not wheel.is_file() or check_hash(wheel)!=record[\'sha256\']:\n            raise ValueError(\'Compiled wheel missing or changed: \'+str(wheel))\n        verified.append(dict(record,path=str(wheel)))\n    if not verified:raise ValueError(\'No compiled FlashAttention wheel receipt\')\n    return dict(status=\'built_or_reused_verified_wheel\',native_wheels=verified,\n                note=\'The separate CUDA probe executes the currently imported extension.\')\n\n\ndef check_collect_logs(runtime, output):\n    """Export logs/receipts only; never copy runtime binaries or model weights."""\n    runtime,output=Path(runtime),Path(output)\n    if not runtime.is_dir():return\n    spec=output/\'spec.json\'\n    cutoff=spec.stat().st_mtime if spec.is_file() else 0\n    active_native=[]\n    try:\n        current=json.loads((output/\'report.json\').read_text(encoding=\'utf8\'))\n        active_native=[Path(r[\'path\']).parent.parent for r in\n            current.get(\'phases\',{}).get(\'runtime_build\',{}).get(\'receipt\',{}).get(\'native_wheels\',[])]\n    except (OSError,ValueError):pass\n    for pattern in (\'build.log\',\'install.json\',\'preflight.json\',\'worker.log\',\'ready.json\',\'response.json\'):\n        sources=sorted(runtime.rglob(pattern),key=lambda p:p.stat().st_mtime,reverse=True)[:32]\n        for source in sources:\n            if source.is_symlink() or source.stat().st_size>10*1024**2:continue\n            if source.stat().st_mtime<cutoff and not (pattern in (\'build.log\',\'install.json\') and\n                    any(source.is_relative_to(p.parent) for p in active_native)):continue\n            target=output/\'runtime_logs\'/source.relative_to(runtime)\n            target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,target)\n\n\ndef check_inference_evidence(summary, validation):\n    metrics=[m for m in summary.get(\'measurements\',[]) if m.get(\'variant\')==\'unihir_restoration\']\n    if summary.get(\'failures\') or not metrics or summary.get(\'variants_reused\'):\n        raise ValueError(\'Fresh, successful UniHIR inference was not established\')\n    for item in metrics:\n        worker=item.get(\'worker\',{})\n        if worker.get(\'state\')!=\'ok\' or worker.get(\'peak_vram_bytes\',0)<=0:\n            raise ValueError(\'Missing successful GPU worker evidence\')\n    if not validation.get(\'restoration\',{}).get(\'validated\'):\n        raise ValueError(\'Output manifest/pixels/geometry validation incomplete\')\n    return metrics\n\n\ndef restoration_check_child(spec_path):\n    """Executed only by the Kaggle job supervisor, never by notebook import."""\n    spec=json.loads(Path(spec_path).read_text(encoding=\'utf8\'))\n    output=Path(spec[\'output\']);runtime=Path(spec[\'runtime\']);report_path=output/\'report.json\'\n    report=dict(format=\'vtm-kaggle-restoration-gpu-check\',version=1,status=\'running\',phase=\'hardware\',\n        mode=spec[\'mode\'],native_build_verified=False,flash_attention_cuda_verified=False,\n        model_weights_loaded=False,model_inference_attempted=False,model_inference_verified=False,\n        offline_requested=True,parameters=spec[\'options\'],phases={},pid=os.getpid())\n    def phase(phase_name,**details):\n        report[\'phase\']=phase_name;report[\'phases\'][phase_name]=dict(status=\'running\',**details)\n        check_json(report_path,report);print(\'STAGE:\',phase_name,flush=True)\n    def passed(phase_name,**details):\n        report[\'phases\'][phase_name]={**details,\'status\':\'passed\'};check_json(report_path,report)\n    def guard(event,args):\n        if event==\'socket.connect\' and isinstance(args[1],str):return\n        if event in (\'socket.connect\',\'socket.getaddrinfo\'):raise RuntimeError(\'Offline restoration check blocks network access\')\n    sys.addaudithook(guard)\n    try:\n        if sys.platform!=\'linux\' or not Path(\'/kaggle/working\').is_dir():\n            raise RuntimeError(\'This GPU/build job is Kaggle-only\')\n        import torch\n        if not torch.cuda.is_available():raise RuntimeError(\'CUDA unavailable\')\n        devices=[dict(index=i,name=torch.cuda.get_device_name(i),vram_bytes=torch.cuda.get_device_properties(i).total_memory,\n            capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]\n        gpu=dict(name=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,\n            capability=list(torch.cuda.get_device_capability(0)),torch=torch.__version__,cuda=torch.version.cuda,\n            python=sys.version,devices=devices,gpu_count=len(devices))\n        report[\'hardware\']=gpu\n        if any(d[\'capability\'][0]<8 for d in devices):\n            raise RuntimeError(\'FlashAttention 2 CUDA/BF16 check needs a supported Ampere-or-newer GPU\')\n        sharded=spec[\'options\'].get(\'placement\',\'single\')==\'sharded\'\n        if spec[\'mode\']==\'unihir_smoke\':\n            if sharded:\n                if len(devices)<2 or sum(d[\'vram_bytes\'] for d in devices)<48*1024**3:\n                    raise RuntimeError(\'Sharded UniHIR needs multiple GPUs with sufficient per-device headroom; no build/model load started\')\n            elif gpu[\'vram_bytes\']<75*1024**3:\n                raise RuntimeError(\'Single-GPU UniHIR needs >=75 GiB; select placement=sharded for 4 L4\')\n        if spec[\'options\'].get(\'mode\') not in (None,spec[\'mode\']):raise ValueError(\'Conflicting check mode\')\n        passed(\'hardware\',**gpu)\n        phase(\'load_shared_definitions\')\n        payload=check_payload(spec[\'payload\']);scope={\'__name__\':\'vtm_notebook1_check_definitions\'}\n        for i,source in enumerate(payload[\'sources\']):exec(compile(source,f\'notebook1-shared:{i}\',\'exec\'),scope)\n        passed(\'load_shared_definitions\',sha256=spec[\'payload\'][\'sha256\'],upstream=payload[\'upstream\'])\n        options=spec[\'options\'];packages=Path(spec[\'packages\']);models=Path(spec[\'models\'])\n        if not spec[\'packages\'] or not packages.is_dir():raise FileNotFoundError(\'Configure a valid packages dataset root\')\n        if spec[\'mode\']==\'unihir_smoke\' and (not spec[\'models\'] or not models.is_dir()):\n            raise FileNotFoundError(\'Configure a valid models dataset root\')\n        cfg=dict(packages_dir=str(packages),models_dir=str(models),data_dir=spec.get(\'data\',\'\'),\n            gpu_ids=list(range(len(devices))),execution_mode=\'sequential\',\n            output_dir=str(output/\'image_dataset\'),books=options.get(\'books\',[]),pages=options.get(\'pages\',\'1\'),\n            limit=1,max_books=1,variants=[\'original\',\'unihir_restoration\'],crop_types=[\'page\'],region_ids=[],\n            resume=False,allow_partial=False,max_source_pixels=options.get(\'max_source_pixels\',16_000_000),\n            max_output_pixels=options.get(\'max_source_pixels\',16_000_000),max_storage_bytes=1024**3,\n            timeout_seconds=options.get(\'inference_timeout_seconds\',900),\n            load_timeout_seconds=options.get(\'load_timeout_seconds\',900),batch_size=1,deskew=False,dewarp=False,\n            layout={\'dpi\':200,\'reading_direction\':\'vertical_rtl\'},book_layouts={\n                \'1\':{\'dpi\':225},\'2\':{\'dpi\':226},\'3\':{\'dpi\':226},\'4\':{\'dpi\':181},\'5\':{\'dpi\':181},\n                \'6\':{\'dpi\':226},\'bien\':{\'dpi\':200,\'reading_direction\':\'horizontal_ltr\'},\'kinh\':{\'dpi\':45}},\n            variant_options={},runtime_pythons={},reviewed_regions={},kernel_torch_source_policy=\'warn\',\n            model_options={\'tadisr_x4\':{\'tile\':768,\'overlap\':256},\'difftsr\':{\'steps\':20},\n                \'unihir_restoration\':dict(device=\'cuda\',seed=42,steps=options.get(\'steps\',10),\n                                        ref_count=options.get(\'ref_count\',1),\n                                        placement=options.get(\'placement\',\'single\'),gpu_ids=list(range(len(devices))),\n                                        max_memory=options.get(\'max_memory\',\'18GiB\' if sharded else \'80GiB\'),\n                                        activation_reserve_gib=options.get(\'activation_reserve_gib\',4),\n                                        entry_reserve_gib=options.get(\'entry_reserve_gib\',2))})\n        cfg=scope[\'prep_normalize\'](cfg)\n        original_assets=scope[\'prep_assets\'];original_overlay=scope[\'prep_overlay\']\n        def observed_assets(*args,**kwargs):\n            phase(\'model_asset_integrity\');result=original_assets(*args,**kwargs)\n            passed(\'model_asset_integrity\',model=\'unihir_restoration\',identity=result[1]);return result\n        def observed_overlay(*args,**kwargs):\n            phase(\'runtime_build\',note=\'Offline compile can take a long time; build.log is preserved on failure\')\n            python=original_overlay(*args,**kwargs);native=check_native_receipt(python)\n            report[\'native_build_verified\']=True;passed(\'runtime_build\',receipt=native)\n            phase(\'flash_attention_cuda\')\n            result=check_logged_process([python,\'-u\',\'-c\',FLASH_GPU_PROBE],output/\'flash_attention_probe.log\',\n                                       timeout=120,env=scope[\'prep_worker_environment\'](),new_session=False)\n            if result[\'status\']!=\'ok\':raise RuntimeError(\'FlashAttention CUDA probe failed: \'+result[\'tail\'])\n            data=json.loads(next(line for line in reversed(Path(result[\'log\']).read_text(encoding=\'utf8\').splitlines()) if line.startswith(\'{\')))\n            report[\'flash_attention_cuda_verified\']=True;passed(\'flash_attention_cuda\',result=data);return python\n        scope[\'prep_assets\']=observed_assets;scope[\'prep_overlay\']=observed_overlay\n        if spec[\'mode\']==\'build_and_probe\':\n            # Model preflight also enforces UniHIR\'s full-model VRAM requirement.\n            # A standalone extension check needs neither model weights nor that worker.\n            observed_overlay(packages,runtime,\'unihir_restoration\',cfg)\n            report[\'status\']=\'native_and_cuda_verified\';return 0\n        base_worker=scope[\'PrepWorker\']\n        class PreflightWorker(base_worker):\n            def __init__(self,*args,**kwargs):\n                phase(\'worker_preflight\');super().__init__(*args,**kwargs)\n                passed(\'worker_preflight\',probe=self.ready.get(\'report\'))\n        scope[\'PrepWorker\']=PreflightWorker\n        preflight=scope[\'prep_preflight\'](cfg,runtime)\n        report[\'preflight\']=preflight\n        if preflight[\'unihir_restoration\'].get(\'state\')!=\'ready\':raise RuntimeError(\'UniHIR worker preflight is not ready\')\n        passed(\'worker_preflight\',state=\'ready\')\n        if not cfg[\'data_dir\'] or not Path(cfg[\'data_dir\']).is_dir():\n            raise FileNotFoundError(\'Build/CUDA passed, but DATA_DIR must contain a PDF for actual inference\')\n        phase(\'pdf_bootstrap\')\n        scope[\'prep_bootstrap\'](packages,runtime)\n        passed(\'pdf_bootstrap\')\n        # Reuse this exact configuration\'s verified preflight in the same job.\n        scope[\'prep_preflight\']=lambda *args,**kwargs:preflight\n        original_worker=base_worker\n        class ObservedWorker(original_worker):\n            def __init__(self,*args,**kwargs):\n                phase(\'model_loading\');super().__init__(*args,**kwargs)\n                report[\'model_weights_loaded\']=True;passed(\'model_loading\',seconds=self.ready.get(\'load_seconds\'))\n            def infer(self,*args,**kwargs):\n                report[\'model_inference_attempted\']=True;phase(\'model_inference\')\n                result=super().infer(*args,**kwargs)\n                passed(\'model_inference\',note=\'Output pixels and manifest are validated next\')\n                return result\n        scope[\'PrepWorker\']=ObservedWorker\n        phase(\'page_preparation\')\n        summary=scope[\'prep_run\'](cfg,runtime)\n        phase(\'output_validation\');validation=scope[\'prep_validate\'](cfg[\'output_dir\'])\n        metrics=check_inference_evidence(summary,validation)\n        scope[\'prep_export_images\'](cfg[\'output_dir\'],output/\'images\')\n        report.update(status=\'passed\',model_inference_verified=True,inference_metrics=metrics,\n            output_validation=validation,summary={k:v for k,v in summary.items() if k!=\'measurements\'},\n            fidelity=\'unverified; a successful run does not establish correct restored characters\')\n        passed(\'output_validation\',status_detail=\'pixels,manifest,geometry validated\')\n        return 0\n    except BaseException as exc:\n        report.update(status=\'failed\',failed_phase=report[\'phase\'],error=str(exc),traceback=traceback.format_exc())\n        print(report[\'traceback\'],flush=True);return 1\n    finally:\n        report[\'finished_utc\']=datetime.now(timezone.utc).isoformat();check_json(report_path,report)\n        check_collect_logs(runtime,output)\n\n\ndef run_restoration_gpu_check(environment, *, mode, options, payload, output_dir):\n    if mode not in (\'off\',\'build_and_probe\',\'unihir_smoke\'):raise ValueError(\'Unknown restoration check mode\')\n    if mode==\'off\':return dict(status=\'disabled\',model_inference_verified=False)\n    if sys.platform!=\'linux\' or not Path(\'/kaggle/working\').is_dir():\n        return dict(status=\'skipped_not_kaggle\',model_inference_verified=False,\n                    detail=\'No model build, install or inference is permitted on local Windows\')\n    output=Path(output_dir).resolve()\n    if output.is_relative_to(Path(\'/kaggle/input\')):raise ValueError(\'Output must be outside Kaggle Input\')\n    run=output/(\'restoration_check_\'+datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S\')+\'_\'+uuid.uuid4().hex[:8])\n    run.mkdir(parents=True)\n    roots=environment.get(\'asset_roots\',{})\n    spec=dict(mode=mode,options=options,payload=payload,output=str(run),\n        runtime=options.get(\'runtime_dir\',\'/tmp/vtm_image_runtime\'),\n        packages=roots.get(\'packages\',{}).get(\'path\') or \'\',models=roots.get(\'models\',{}).get(\'path\') or \'\',\n        data=roots.get(\'pdfs\',{}).get(\'path\') or \'\')\n    driver=run/\'check_driver.py\';driver.write_text(GPU_CHECK_DRIVER_SOURCE,encoding=\'utf8\')\n    check_json(run/\'spec.json\',spec)\n    checkpoint=run/\'report.json\'\n    try:\n        env=check_offline_env()\n        if options.get(\'gpu_ids\') is not None:\n            # IDs are logical under the platform mask, never physical host indices.\n            ids=options[\'gpu_ids\']; inherited=env.get(\'CUDA_VISIBLE_DEVICES\')\n            visible=[v.strip() for v in inherited.split(\',\') if v.strip() and v.strip()!=\'-1\'] if inherited is not None else None\n            if not ids or any(type(i) is not int or i<0 or (visible is not None and i>=len(visible)) for i in ids) or len(set(ids))!=len(ids):\n                raise ValueError(\'Invalid restoration check GPU IDs under inherited CUDA_VISIBLE_DEVICES\')\n            env[\'CUDA_VISIBLE_DEVICES\']=\',\'.join(visible[i] if visible is not None else str(i) for i in ids)\n        process=check_logged_process([sys.executable,\'-u\',str(driver),str(run/\'spec.json\')],run/\'driver.log\',\n            timeout=options.get(\'total_timeout_seconds\',5400),report_path=checkpoint,env=env)\n    except BaseException as exc:\n        process=dict(status=\'interrupted\',error=str(exc),log=str(run/\'driver.log\'))\n    result=json.loads(checkpoint.read_text(encoding=\'utf8\')) if checkpoint.is_file() else dict(status=\'failed\',error=\'Child report unavailable\')\n    if process[\'status\'] in (\'timeout\',\'interrupted\') or (process[\'status\']!=\'ok\' and result.get(\'status\') in (\'passed\',\'native_and_cuda_verified\')):\n        result.update(status=process[\'status\'],failed_phase=result.get(\'phase\'),model_inference_verified=False)\n    result.update(supervisor=process,output_dir=str(run))\n    check_collect_logs(spec[\'runtime\'],run);check_json(checkpoint,result)\n    return result\n\n\ndef package_gpu_check_logs(report, output_dir):\n    """Small review artifact. Private runtime, weights, archives and spec payload are excluded."""\n    output=Path(output_dir).resolve();gpu=report.get(\'restoration_gpu_check\',{})\n    run=Path(gpu[\'output_dir\']).resolve() if gpu.get(\'output_dir\') else None\n    archive=output/((run.name if run else \'kaggle_environment\')+\'_logs.zip\')\n    temporary=archive.with_name(archive.name+\'.tmp\')\n    selected=[p for p in (output/\'kaggle_environment_report.json\',output/\'kaggle_environment_report.txt\') if p.is_file()]\n    if run:\n        if not run.is_relative_to(output):raise ValueError(\'Log directory outside diagnostic output\')\n        selected.extend(p for p in run.rglob(\'*\') if p.is_file() and not p.is_symlink() and\n            p.name not in (\'spec.json\',\'check_driver.py\') and p.suffix in (\'.json\',\'.log\',\'.txt\',\'.png\') and\n            \'image_dataset\' not in p.relative_to(run).parts and p.stat().st_size<=10*1024**2)\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as zipped:\n        for path in sorted(set(selected)):\n            zipped.write(path,path.relative_to(output).as_posix())\n    temporary.replace(archive)\n    return archive\n\n\ndef make_restoration_check_payload(notebook_path):\n    """Authoring only: select original notebook 1 definitions, without executing them."""\n    notebook=json.loads(Path(notebook_path).read_text(encoding=\'utf8\'))\n    vendor=\'\'.join(notebook[\'cells\'][6][\'source\'])\n    assignments={n.targets[0].id:n.value for n in ast.parse(vendor).body\n        if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name)}\n    raw=zlib.decompress(base64.b85decode(ast.literal_eval(assignments[\'_VENDOR_SOURCES_B85\'])))\n    if hashlib.sha256(raw).hexdigest()!=ast.literal_eval(assignments[\'_VENDOR_SOURCES_SHA256\']):\n        raise ValueError(\'Original upstream payload checksum mismatch\')\n    vendors=json.loads(raw)\n    revision=ast.literal_eval(assignments[\'UPSTREAM_REVISIONS\'])[\'UniHIR\']\n    worker=ast.literal_eval(assignments[\'WORKER_SOURCE\'])\n    mini=\'\\n\'.join((\'UPSTREAM_REVISIONS = \'+repr({\'UniHIR\':revision}),\n        \'VENDOR_SOURCES = \'+repr({\'UniHIR\':vendors[\'UniHIR\']}),\n        \'WORKER_SOURCE = \'+repr(worker),\n        \'RESTORATION_CODE_SIGNATURE = \'+repr(ast.literal_eval(assignments[\'RESTORATION_CODE_SIGNATURE\']))))\n    sources=[\'\'.join(notebook[\'cells\'][4][\'source\']),mini,\'\'.join(notebook[\'cells\'][8][\'source\'])]\n    for i,source in enumerate(sources):compile(source,f\'notebook1-shared:{i}\',\'exec\')\n    body=dict(format=\'vtm-notebook1-restoration-check-v1\',sources=sources,\n              upstream={\'UniHIR\':revision},worker_sha256=hashlib.sha256(worker.encode()).hexdigest())\n    raw=json.dumps(body,ensure_ascii=False,separators=(\',\',\':\')).encode(\'utf8\')\n    return dict(sha256=hashlib.sha256(raw).hexdigest(),data=base64.b85encode(zlib.compress(raw,9)).decode(\'ascii\'))\n\n\n# CLI child\nif __name__==\'__main__\':\n    raise SystemExit(restoration_check_child(sys.argv[1]))\n'
GPU_CHECK_PAYLOAD = {'sha256': '5783754fd6d5a1315fb619c4869aa2fce987e4d17e4e83fdfac502d6df76e4e3', 'data': 'c-qXp+m0hyk|6dg-ZjvW9!WAt#+_NLoyf?j?3j+cC^EBVx|v*GC+Q4&2kFjscgCe+0+_db(u1Id^q>dLVzDnPNH3BgVFrQ#wSJ@*3+<~uL3TM;zmSZq?8VkhXE@!@#f}|2cFgRU*?IhK8Yc@s`>yxhZMJAeaTbi?_@>iLf;5X0KMUiiIi3aMo91ojyT*5Eyh_GF`dx4E-86|8j^|BR*(wP<&j}aHILRD8f<~xb`Xv(Ypo7_$=)>zDy^FV}Z_d4+&aXaRynpBPo!W9T4OffiZJ36mc>tZ%zC>T5NicQbi?<B#gSnT*-qODg>S_3M&~Qeh_`cyJq7%3GC34`uFW=$+eGKm%e>x4Lz!}E@JdDF2%F>>5U+eoj_b<jf4=<pJhgyFGzb5c&+QR;@g^Osp${gx3b!Pr;;P_4!FPrmVnmKV2UWd_3CthV(DomYKL?8qc2gc=${pi-mmtmH+xIK6cBXcHkkUD@8C%9k6Y2fUBPJ`q#l<UDK`Y`uL!Q9Da0eoG~9@22^&z;vFE}UC`z6xY>JV1WFYdAAN9^!*v%mFQclZKPQYeW#g4yOuFPQUM*v<0&8za)Tpi=08b;nXvy>5N=w7aOi8@MDfYW(@~EV$;LY<~qk^?KO1+chrE@!Hl4+Xs~-!0btXDu$c!D$1Pfvl}dnUT2RcnjwUrhfnE~8tE7$u*Kksp3iz||@0k=+Np4~WuvF_h$UK~iy36128NLmW>RaO=oY(K19d62PIQs2Oz18Sggb|Us>wNEYfa<8RL3@Zr@u@R(elWl>(dCCgKR+SKoF_?~)Tgx%0G-3*aUQYy6A<Az2qq~t!E0Hx)PK~iKD~rmw_$MS_~SpU053E_X^>@Mbe%eNu84~u3T^|0aSp3&{s6Tmt8p-KYXTFaFkARbi6?<s8ggG?c9Qsa9<hwH|0!7oqGFaj6e)pQYaurP?76LR96|3{y>|8HmFy9gzCWLvl|`-b92OtW5YIRb4A>UVe7Hd7!V|s&AT7gwx7}`(;Hc9U{{dxWz#N0?hkngpWpT}BLstxNd=~+G0O%xF@uP_Yh@Auxpbr=?EP6m9_FPLXTu*{AJjRh?PX7X+`WJ|CU=w0WVJ@6040hy?Z=5iaeJo)@j=Dvr<opQz`C^s$X%-~SID$EwuIBS5f`GIbIV&I{DDFhCDuD$8r~uMK13DPbR}<{>ZWct)!W{w`PXq&+`XTfU3@e<_a&9?q0i4P1Z7_yygoc<>+dlBS0J;|g`2#RG<e{+L#Pe0gL(a21v5PDpc(dp_FerW$o#RgQ;o?#jyub-Po5jmitt&ZQ#Bc#wk34M+r^d*vJ>g;lBs<^toi;ufW}m*STX40Dh!8-yH)k_m(sn3m^e&uaz=38#cs<Lin25d0L8<yYOtZQwi{I-m@Due$mW8+6L_GREU_XVrc2)6Sl{0p1goY5?f(dT@2m?VL;=Kv7Q!I2AuOe>9l{14A+mK{}U=D`BZCh=*9s!fkAEI1zYhV!O1zxtYhh^aWs?f9d0YruOul|~42BRAO2Wuag(JIhy#U%BqNgU?#-a;t;2fQ}{2|<_P7iJM`*a!I`5v9~`UF2>C{LUyC^8WrVj)EMNsy}{cEmrd^T+Scrs<H)zTO6YVMOP>R$iEL#wc*SjmT^`W9r_K~^+;|#*7vdcK9QRc_nbfnvQ7LPN8U0HVe)|}A6%MNe8PFH*KvP^=Ib5(w}XG%_}6TA1oSiEW?`SJk8xwerpDNg=oD#MeC<?x_3zi!<*%i64N#En(Zt*%nAjn%s6iK~m!X757vu#jKwsMWOMB?Lup#H`?&%VHcHNHt(!shv04Hf3Lld%76im1iSH0a+Z=sF`Trg$^orxY%u}f4yP>Pow^%Ck!*#dt<|G?s)NZ_<}yLt&@!;0^7cSirb<Xir735>J;SmOe;)M`D!X;@SO=E{S{T^7Fa4|vNoCEgB-q^h0_@>EZOdYZymYjco5=FkkTReFYZIJcxRc(|p(M|=VA87>cYNbjiY@$Tr+?g(XI&)gBCeR8ER>8H#T2SX{^EtYdcJ_tN@!O&cig=udWZ(tUfsr5H7HN~fy*@(KdUvWn?Nyu2#Muwb*KUxA30V*e@28BGR1d7wrBc!m5^U_(uJ{u%3UE>(GO5UF2wqyw7Ynmps%5p|ka})VhZlf$~!YJ>>3CzMx1mcp>U}2DpwS2om12VQ5LIm#Y{wS?4fFZ&%=mgF7p?){E-oXo>Hhow8n&@AC^iZ#3B_9^)Ev=|lyWJt41C)G=Z0F~2S(gAa?blsaUtRZEjSLn<^{pxXgX^R<0n#6cA)}&$!v<ynyv5CJK?UT`{c*4WaormyKpk=>aSoDTItLOo=9Tt>dAKYW3*L`@kCMR|6!-u;3y#&buufHRUF0gu8Xn0L_>|VMEMZ^+{6xqemWCCc&1$9~0+<Q1Sq!{F3!*5J-~wupsX=IWAot-rw<sRMie<@=;I89p6A+k)jX?d2X6t1QT&t+}LAf7_!eoAw1zGYPvw0d<wbo`%6lqi}Ys^|vaG&{85c2ED^#PDwDRu%pyF&JKalf`8Mq<>3fuzjRf<A#jdGD)gb;At$QJBUV5MMa<#O84w#4vNogE<^zeYuvDI;mC_y{Pk~_J8!DN=}Bd6-h<YFu6z{h^}q4N+mQDv`YhbGe7mSEU9yYKyP<`0_!UBNcF6_CAAFw)G7>M(O`--^4hHVO;|8^v2L_ySag(Y3Dhzlr6Yx>7TYxXvJd`^OGN8htw}hY1_|o6bycw;Yq&UJthAuOtoJkpy}4u`YrK+SUlzML83G9|5KGV^XpavyIr<7U=&#c3EXl+KALSFm2YjTzPc}d>rPNZ{Y@6H$IgBx0DG-VA1d}`wl@NC69?s1OnT2udp{A=wPId#$!<bbF+**1nD*GNO`#|n-Jg>71=h6fMLD%as%PXiDuN7DtmGaDw@2}oIvASm{doK4;w^Yo&#dRdxlJX~tAqK=)U49~Ih@?}{(|18)Q;KI^KK%u2AU8b{JWZYhK{Y}o1S0T>Vu5R*$p)yYVCY7fSJJcGqG~2x63~h&S#*UTQbe{a218i_SCbU2$yxxNOhVt&uWh}bzDlz}jkf=)qWDpjRMK8g35f8PXV^o!LGgJYNGMMM2(2;uP}2B?i*_*zCX-;o5-NU|Hieq94tX%&@B#icR<*fu3#*Qy)jv=v5T-IuEHoSqKjB<3AxuC7O?{L*ai)j`>yNlY5UPp?6MFgz#}fpxv~7{HfjrIDM6f|q8+oaY0X%tF03ZBYKb-qO&NZ>OpfMIPUz^8wawpY&y@W}CRkV!fEZ~g^?@(YI1EqQsAlq<W8!FDh)=AQy@t<&DX?8=_`9qvp!984*hK~+}COH^BSr+)Z=8-;GUCBs?vWc|PP3Lh8@_LxUt{NieO#hH%q8y5L^Kc}KexUOsVXIH#-?NkoT0nXu`w=!a+YWmQ+ch9EWlLum-8fj~r7&BgIZ~Xuu<eKp27VMri2B|4XIBocgS2JZB1>(J4PzS=&p(E9z`pqLURolr6Car{(~FWcE+rSo&#i2>T8tu?oO-=;h^A)z*A*?bt^y)ZnETe{>ASPvz5ghcReut$QlY+g*(^cr@o~bgb1b<o;sDASJ9oMofdD+txWbe`uN%W?z42FR8v4<z`6{V%PskduxNA+j9BvDRXf#zJFRx>dZi@{_bAK_K_|9$5=^h*)_1%J8DP|9O4~Y2DjhD@Ul4tRJ0>r)#Di>QT+!7+V_Lr$70{=ctD@ftT5U8701ki*7E2fF$aJ|al4J^2HwM5-Gg)h;yldR^j9)Q$+K|l@?kVOXqi5sZVIxNBhEq!JOoXMAOvv~B71*w$i@W>H(h_DaSdhDPHKBJQ)5W@GQmFx~=bnDZ}kt^+pp#7uSDa+zTu9aD?m6>j3mTv`Sh)zI%=nH`G*dkuyZ*IHf$W9){#dl;pNmq+{{g$jwxWq`W#nBHQptC#Y1$EGqKd2Y9l3`i0FeZKkOx75VG~^M3_hdH@T%+NPQLU89bFp%R03(wam;4?^e1vv49$=H;F%IQW3)Edzk9Qo;^?0oT-VfF2X+*j;G0uBEPGj=zamwcJrTMjFoMz<)0aP+hG+r4HlnoSXRu2?4MObR2E#KFSa(rU_6*$<<1C__hooT!U<u-_5oKWXF!=)v5U?C^nm5Xv)0(Y;0BmCLGF-mi!Iok#}mfnCBxN1@bIIt9n5#kHB4IeSX<2STs=!pS?TB19}B~d3fStu|?583DxxjG6%bSBe2lsf?IxQo?7Zx#ufr|JUY9lI?kj7+xJmgEq@<XYOJMZ<*#p~9gojtm>mPyHm5=S7AwMoAfrS!`ROvdLY&a5|<kJ(~Lwvf>cv+b)O#4dZJN4pCZB__B;1+yEA4po%IKDFOoE&^GbFiYcjIaUO)S768Y^V4?-O$}AKs?2EBj=khO{5J?fu)J$SbmoQ$pSU+9*kv%6+ZLe%z_72>V`lA3QLg+n+KWvESf`MsL4{ncs4U6^3_N1$aJ=|uj*@5i_hD5ED)W`ieCVXShrGRUS%Ar_oH+p&O0NT{n!(-!P7}*ca-iCTKCaYS7Ne3xh>hUm<ZaE=o6NuohSWOBzpjRXrndebQtCLOvO{G%CnS=?89daey!~XpM|GS6P{Lm#uUzorHapVS)<PJA+--S82@8sAxvz!O~_Rvx8H0f%Nzz*tA{smg4sS|u2VMnesR*&R96wnz@KxbXOp*t+?1)IaBxJiPUn%vx+g)s(i(Caljg?`9zAhAR*QP?J>D#_*o4sHOEF2wnnpQ7TSe*Yd6MD-)q9&JgW9@81<TXX$$H3Fe<46>wt^`M`#a1jV42gKv?Dgk{qYq1kus`MRh;6ohG&+h}6v^WuUaMK#k{WR5u)vJax4FlwDRB{@DvNhJ)7XQ@E1PF=I2K{ajd_NOqRx4ukCHuo<K~lc_hA$NPratI&Xc3S6MbJb0y6ZH5z`woR45IQ;uMPhB7x?cE|7i|ipoZ(!@bGaFxGkszga9w-(4>~EBIP6K!4D5+xqcTXH}IEwa{i9sY&bU{^V1t{Nj#3H*8~fe5x`7BP-v0N;ah9$k7t2*6Fk)2LGRcmI6CJLeIYYOVA2{qOlyzW+mrWb58Z_~-1?JtZ1j_qt#@fXT?S(bbcj+z)Wg37Aev+b@MCBxGPO>#({6MQYFGqBht>&bi2FmQ{{Q^ffBMJ&3+Cld|MWk?2KmydQHjN${^`F8oeXzVOMk(R(S<Ne2nN?aV1zdU?QT&DBvTP@&yn&qS!F#9uyJBKCGVev#8hq;lvj`{8kRvER=kIczE5Rk*Db{a>$Rq7Mupy-7kwcjpw{BMtij+egTF^~yap1m0#Lx70;L9%dJ2L~eaXO)u?l@Uryu}=kgx<fA|l~NvSPIN&}`vq=n)an1VK-|?L3qw@mQaseV9yN$Kw?q!qpM~*s8nE@eQR<Xra=@4Zh|5+dODIdGKh>1K{+l2XsEp&|?Ud-bk-Xv<chGQg92dhnAp4P#db6v<`Dc(kxh16`@rl%F&_S&~0Ipy0ip44e$h#fDZ-O@K;b4La%#}Z($Hay<Dh_wO-C<umWJ>|M;(g8D_H}cCtVIyJ+`%9*<f;r7$hopZ@XRX3qFe|M+iOIJelib2Izn{~BGxpFjQ6f4?Lg{qbMN&iqgR^xuWnG7<|$!Bm(#F$AF?&R~)vb2z4NL4cotGHFP-Hi1Z$z@DaIbZrc|u*7JEmB29z$pvh{dIaeZ0wx4LPbY+4Qy>!q#8&doO$?J1bg-V`UL*S|sfY(Fk`San9u2-k3?_uhE`l;aH^A=`_^4>haHeIfna(o{Qn_%0Y5hJ%H={t3cL6`9cv<3Ii+=K{O(;F*QS|xLa?bwvzc}fi{_%f<NqwY#pK3NQ;=xJM$C*V|lL7-<t}-vTo2?`9%kWSC^e;2#=8yl5To1DZZCFMt*gNhIfBZMj{Ez>zK&W6bxG2j93!3p6`w`lv&Loa;Po&kWTC+C_EYb=<>OyzIf^X!H;XHU3XFsCoF0m%?mCaz$J(xf~Vp~tzk9W?xvB9JT@1{NzT)*|&P76))F4;J0p7-|j?ES~Tttp~H=9)8aP{UsVDIzxd<3Gd#yEe2Xe9Bm#t2CHYEo?QoJX|zqeM691ws5xGyc;5w3nT8tULN!bwn>nj^am_Q>xHBf!)n1pF#Wm~c)r4Xn%R5LVIb%L3YJ<=VED-x4Dl1DDJ{ZN(+wV@NhWf8;qA4CxXJCd_KH!=)Dp?71l5M~0fg$m0kJH;OIdO?q<3C*S?t}Vb8;oAldf7=b6R9&8w|@#^sLtc{cMqDm)?cYR&BRdA~{hpeoF}I=ON6u%l;=VJObyfv^F?x{?2dyJnYpN$*oC({&~3Fij8R23RBM?(K%31vMMbm_BS6d$qPG7=m5x(^&D!Jp@gO{!)SHyu=w*xFy@(AZFdo6yERbaG*svZH;?avr0&A|8hwR-nekFF6!2KFlM@00HL4)%X2cNOtER*h*9%k3>>^G{20>@FT9mQA2&fil{^ttMgD>L0-C?c49g5=!Yr-~aj8%H!237FWED8LHsR=efxFr9*c>wHCYGYIk-Sb7g^IK?4BK3N?@=$9D64>u=L*QT}d;;Bt<ABs_^|DX4I6J6{gfeFyUsGh#B)H{T0Ob<ZYv|PH^N=u_1xbkh`~oPTpt+x6h?%vsK!14R4}7u?(;#{$xj+LV&2Wrs<AYos47~Ji7eDQ#Bg;r%J3OJThpx2Hf&A`}x$%2p`sL$I(GsmG6upsM>zBV`=TG@!Z9u@A51*Zj*Qv8ufp$59MFo(C5#Tb4qbRs$7cYrd(W3I`P`K>l=-C!~_WAXx_tVA4i&vND-s|(9F3!$B$_m>3M^!bHEwVkTCJhs1iI1slsoapM`x)FyO-&ogR9$FBeI-Mz`h_d5*{Rt=Ce2TXwl%V~3#XxyS=r<{H&s}F6m?|OlQI4>JoX>e_g^K()_yxENBfZENGV1Nh|$nojzPvVGODm_Sh;=fRxVxXM&!lg#KU!r4sStp9Y(kW;l~ozpF|28KLb2&<+q4w{4I)aq*aigJ}|BNAjDn+9QXbh<GK)h#A+5hcyj*W;Mq90s>pSsmZIps#kSwNYdQ&Uy$C1y1voJm-(<sJKKKdf$ibWdRr?zbqjU)}#Zc&!Jt{FD#Hl1!jWg&`KvNP%$ZX@}p>8wI4_JaM#~|y6^9e|>f;}V8{1hjkk+NF`@XH|hIbe?I<33PJiAec)*fQL3@LeDJFm{{<%nda61wkYB+z06s_6GYjh03(XtBK#@Vjdj~)^Sw{GjgBH0IJe@x51_Q(;C+4J%+F>Uh*E;pUz`HtJm*$?%ftz&fONQoCVIdFnKT>QlKOh0<{PC{jh->Z{Gk0U5*4m_z9?^@kLK>7H9t4qXY7aN-z~mH0}b-q1)o}i(mmWVzz@gE;a+@JZvfiWI=6=pcwm0e-xslVV?-Pyi;8Ys@VVs5$Gx&V5Khv4*0lC&@hE_)tamp%d{>KL?4yPCGG86T4;epR3Ct-0D2Y*0O$b(0(f(UI&5<c|1w1k8$e4-ipAz2Ws0W+x(M#&V-SsFlo9&1RW?1Yv9T8l_fOhT8H2pd7)(L?1ToC@B&{=az{e+0iT9Q{GM2hKXm%_Sj+R9Rr0d=Jw@?5dQ2|&*V|K@)Wi!e7Xk4u#TBBH+79*pG9<iO54h2v(i4|dsjtFO<bpwkARyi^}6*b9)fYJEy`VT7#yNst!os}9<dE!Z6S|Du&fJNXO2?lSb&T=)H!{%(nC1QyHmpd?`EOo>r;eime6~eT`GrC6rL}9|q9+oo<%aO75?1hr7mRU)tBlHad^F%0#5-{OL0w`#Z0@o0PlF9>A(kL6wqtwKJtBeEm;3YAT&4Rorrx<bz8<yKE*m$u0g6}rcfT44OZc`$XqHl>1Fw;IVXlx37KLJ77wx~m|Si<c)kN6c202UynXyV}M;W4<ve1V*d-+Rc<u!E;<T)nX#cSwxbkeurRjw;7&f#QKECjJg_+vmp?m&m(eqp-2eoL7DtoYSKnP-FO@!|+;$Eub1~0YU|!0vc|xtTEajjY$ODwZ`+nPt3w(7_i=Il;h?OV9;{1v|3(3f?74DTH329^L<|tx7gO@r>bqeLcAkm+7lj=aI`IQ18$1F#~-1Yr<x534$k>Y4GhNf+);`L9pxc_$wMHPOXa`{D~72Cfn-u)s;*l^y(yTLrbz6Zbu|ZWe!5mm6b&^Z#UeK$woXdPa`MyZ)y3(%Pu|B5=Vu?ub9n+hEXopZ6h?j${w&>R(Oaw*1*-&)Om#YtTIJi<@%TLYgaQ|acmtY8@bB)i^Xum8i&tmwyY0@g^HccAK_AozB;l3xC#fOXMZk$+lf;daoVvqX*jj7oKsy0BE&Aoqoj2c{zSF>cZoUgLr_=84zjS^RB*3d!B10!(!23(X4WM;$uq~96U^E9!9(LJp3!}pgmx8)2Y<EB`$-+<22YM0TQn17~0`Z{JbyV|?X#OSUJc0c)#MN?*Ym?JTY=)Q9t!My+3wAtG%e<!E;*;{)HL@*maKDI1p;pIpe-_9lCjmkmMyL;rJv<;yITem*-2Og}=K!X3XvObmm#4ozcP{;rb4d^YMKf%-B#(}`%TYL9!T-GbF815qqF)gU1>eZE-Lt&7vA~BA*{CC0cA(`gy5Gm{_x~K-@2}~85=1fZ@5%N!$MUdtymaO#!2NI;!TWcYZ#AZ`=MT$Sfnv}VnIWF97XG)IWTYl&4@eP3NA2vp4IIgH#=RMgt6cYdREqdU%|$k5gK!NRPxLK;c=HuN0NV<_B@C~&hGFiHy!kg<?-%Dpf?H7m_UwP-p3#Vq#0%e<g#L9DW8uxiAAMsWM$ZKz!O))Q=35c~H0exYxhT;<vK|E7C9lbU(2MZHJpB3FEln92;pkofH@B3%XA{p#{nagS)@?zt1R(hRk*hl*ZX2*Nk8ulJ#Aqf@tNV?@(Hx-t+J4`0b!QRyQ4{K|(&kMw%|K$3@tLS{YkX7#G79G?Gx+mgx<ROhHlchKQN>@{pAroL>7vO^J%b<~M!~(eNRw|xdk96)<Os!ieC=Fu7i3_du*jEaIzibUPl|>8aw`bO%RdK<Z$Dn0eRv7#&SC+J1DEo(kGeb>6;dDc6jtCy>xt$ro=?A)bpHnGHvc-FV>jqQ;Ln?yC5Yu?h8|TepkJ_hF3x^KOXtPS4o1Ay&j%O(ygPhx3#E{uxz*h=c9-LCD&C-7avUxJ6O<DqLerMs;$9l!4;~4u=d|3z^<wd@0K{D;j>NKl^Y*QC{`RFaqVN`U*wheH8nq?Ifmg$aXor;?PP1Ri0N%sn*-z-co`r<EIi?JQ<{fIj;Y6{sd`SF72wLz1+TDE6zpJO#Op1W9-mBEma=#YH4?zORYrX~YzCl$C4Gj}ix^Ia$>^l*Lq>N$=fQ#bhbiTUZ#KnxgbP*>@SSEaW`K=~giH#0AG~zKyx=N=0ICx3B>LhTaYWNIDuUAm<o9*~Hz|d!L+T<=A4xRiZu*O-s`dZdPsl8@F8m7HG`p|Dei6!v-9=lny6Tl3l3EB*_=J*^V!4)*nJpXuwmV&FR*RRk-aKFEtuhPxBf96lZH2KPoKb^h~KVCU!@%2w(aJTOuAZO<ep3!D7Q`aaN`7*^~ED}d-u${ni%y`oliDaH-E`i#g0HTZ6Z+Abz|K7%L$>{hAXA@5+@qEUio;QQCio#i#SX~HGxelTrks;uD>)P6npQG^C7gx@!(>Ldr&f81SwAVIJz)m~{a6GhC`U!3mUuihJ3R0LXe7gt|=k=$LA3)7mW{%PD=9ywxexV7j8WyAB?O_3yJc;lA9Kh(7w}s{#7<j9|$@8L(@Zhrg<x_>r3$`_ksKDfAc*;!9J|N72EbAxDPP>i9?{I<{ZS>B%5yC`{5%xI>!}?{QYLvZw{pob~&FdfEIovQGyi#oZbD#o=A|9{OKLc2}ZOnrv=TQ;ed?k>$k9bNt`Bngaym<ZkD*?I|+WNN#YBa|L1Iik(HF!^7qK}`iemeiH_woJbtFv?O&DHzQANX8m?_|;$9qt_k!Q|w`KMXo2$DQ$B_b@m<Iy~4r8jq(Zha=xVJ~;|Hhttkv?C(zx{b1bg1bf}%b{jK|yy+kSd%=m{9R<gJr@ik_+WY(CN&8@b5(G!%wtsTG-|hPT!O^5UKJoX*fgixTgVFKi#6LLn+mrTGw>0vPJ6(9+ne6Yuqu=#M2Zx89gS~^3!(bHbA9u$`U1+QAx5p>by$-;AGC4XKpBzjk!O`eIw{&<sI`qc?$hf_CbbPQM?0160gJ80MFbz7vXguj29Uq^-fBw;dAM8!0-O+S>aB|c=I_R{If+M4)!|7!2q;uTu?jN*|eSd#4o%#pkU_3tXN5Rp4r+u<NJ~-SvI0R@qFu364=x};`vUl9+jP{NLYKhF~`=jn5BC0bP?E%Qs(PSUSd3-dT9CgS0ouh;90StO?3Sf4}hyLNw@lnw3`t9S+;a;cP@u{V~8cgcvcc(v{UR<7{$Jo{R$4~FCPCs3|f9JhD|MctkuX%pwcjOuwpltD2OEH@~Hh^hbP=&uvD18*mV<~a$+Iz}r53DQ+D1T#|JYdYC<0s=;n1R}}N`hUX8-V~iTA`zpa~mY#G^8}k;?xFsWroLu@Mx3jY6&}k;G+gtn`PND?d|SfgMz*qwLsh2y?FOGm%rV;{F_e#@F$WX{mCi{vh7FlE_Pt%gOr`*VmT&I*Z8!?q1=yJR-+t(We}^8Re^qYch`!1J|740F`jG;qU~<F-`+pjZ??M!&35PH@Su6jL-DgX^-$Wmet6b!mSGezenWK7aio`lBZ0~d>7|pC`K6c2(#H;TX=qf7^#mjHXU>=8L0`aEDDS%1)VGIOQN;xJO}fgU1n)pTKu$2IFrDF%E_T?=W&rv1jIS<W-+)<wy2fqrfD;6&A&B}<PuhzSjT@PV4#OOIpgk5j(xI;ee-wx*nM$qbK+leNltIgzf^t27=$G?oe|`<hS2kPprAi>A1Axv_1BCe)-N#nRNTf=_Xxe^Fo46?z8J0@o+klKA72R@f`+a#40OzLCwJ>8&yjmLIHAI#s7@y32XPwWpA3=#TqJ@Ket9_&Wl9Ab6TEjA7H4^n3PLE8X*cgcB=o}?{;Mhw69h^9ps3FdLF$+mF4X<riuXA}Y%o%nvGEIb>AP$i&b2PDQC9I4IQt?EpZW*POx1*I-<de*uvA47ZV2#|P=mUeQoTV_NMZ^#w_->1iuY+_2eCfVk;}9dABNzJN<|zQI=2}j^%Ae#>HOFCAPc6cF%4W4XwlWYc4Be7Y5l-SON5DzH$a^eaWhvSg)X}=gLTC@R3|`9k#`)TcK=Wkf3PL#%O2zOREey8<8EZI{817kci|Z%)7QHcu9)GJBDQ!>-Cm4b%=&b|;!V_3DQDet+;y-ej^g&S%E{2IGm=~HZ2E#3<O8Fe;Js`MM^o=kndKIaS%=X(snElCvPFP!9hMUmhg_Ham<g#gQx$s-sPOd3epRq=i8xd8c^3&QQcJ|bJq}wB2I30LYT|U(scIhwQJ+5D2RSkXlj^J<$9`QE+u}7$V*N87r68QCu0lk;gf?K3P8@At+h_^x=0(M8td5Rr1h&z+xI(o$6D~wEk6f0e(UF0RC$wU(5)5f&GpE*yqSUkJdKt$S}S}qLtCg3#vr7*a~*j!Pb@TS;VZkz)s1S<`_qje_hT)Eo@2binS5LL05rMi5ur`G6%Ss!-@8Bc@#r|8j(p(U6N(RS05Q&3=kw2*mXD2XZ=o?#1RSC;u|5p6LQ^0p6q4ouuyz~txJ!ZF1}Fptd@8!A>kDtgiufwF5WL}A^)C(_7W%60y?pRP{5v&++uA1}^MFVUem?@LqgYANwCDSCV;{|n^~p<EISw}8sydE<E8II*_QjE&>m-1%zZFjF2I%CkG`FwHu+vQLRBPiEE~g_IsXlvw!{M<c0>GSeVaU!c`L6+%ibm!UXEBH!sszXlQz)9`xX_xIW@zN?7xCiUB`;|Ay5?C*73ZQDPC<;oCul1cv<swp3fey7zvXgE5*bH96F`!7)4JBnkae4+ilPQ$r@{xG*#ztcVD&a~q}b%nfN0F?w$oY2w-Flc0aF4OPw_~=c#^Z~Lx?n$5X8`$VzVSj-kFVNE-w585^Wj*_pUf1<f!0)`@Ine!KI^ETxzi)sr_mk@YZ;8TJoTHx4-??jJ9uCV6-2cqoiG%Y>)E^i}+ZBbH?N%p$63tLk)?9f{)^X>gTZAZoLR?)@9_yz4Jy4LcfF+$2@X>JOIl0Gle8TNoC%fD*AI*B4trX{%@*(0q2mGcZRzTI!D@QEp&Y|xc;PpOL<k%Sy?3aMJ&cTMGQEKWYNTOh_vAJ$lmK|O`tC5ByM+smD%QT$F5tM0n%8wh@w@uV!0Qe+B*nwFQE85wSjLFuo8qUs+;9yTygra0?m`4tNme<}8x-R1Y#ML2^Rf6WVbq7ziTx_B(zTmFumPmT=sZPn7C<cz>6iND7aP&PVG_K#ehVaA4(lL=+4o+_fYA@&3ooKZf!E0j#G8II<V&}AWK(qfl&btKg4Qj@kuNBk47)ZElaH1-Ko;?ikxWj)A+H`T@o%3HfU5xDLDD*TtlwV2obW@|8jGc`kAGKG;*^&{>Ihi;Hw9a0ug<YdKMfc$inw>^xn3KYku}Pvxi?|kW2?N(XIM@uH^21Y8)M>RaxpB;9u@&Iy*z}{J)a4a$(Ijc0+Fyq;^m76%NO(fJMzb0jL=n1E;PMBBex3H_ZppVvcdO!SX*OC1jkcR7SM0r9j?v)Mt(l>%j}dXuZc`}>Ns?;`TyZm23&hFv{H|Y$O3XL>J0+A%ka~%h4Nl5e#&9ffg}|l2p>h_uN(|#N``nV%h?dNrJ*<S{;%~GhR5}VRkQw7Jn1Kd_QGsR{;tpiFFMK2}1zS7pO-Bvq#8pGGXPJdOIoFk?NbnQW-QG0rSISe=V67rQfEmRAy5#~d1}i~^lr8NkEhEsqB6+g-$)DS;ql1#!)DDQ;$i^{!79OZyGa@T#jTA_QWvpuTv2;pI!G6y&YRLzZRM#w8Q&beiN@m+$+1paM^ok|6o9<#)=m)#2nk(BuE40I$d5F|@1)eF%8hX1(aDb18`xY+k3i#>p;9&1?Bbz-VZ^LPrIaU(}yU&8_Ac5{$Eo!lf@9R-k>^LR!;%;pi`EYjXNNlf%08wjDoP}1c3<#VeKCx`q0@NsFkdM`pmgRLzZpr>&ma|xoRm#>wB>QQ*?)C^mhgO`$(P{#6>8@;%MeQ=qV`w>*3)E737_8&srebPBn$4_TPjAt?^&@spx<&X*u34^TRn2<YTop%=pQ*ie$wZOTCic#<O++ekk|%dJ?wzkC_*xhxpLN_y6+|*2IGzTpLjc(q_5>Z3vYuaVS1N|~+sHK(y&QJ;EX0|`7`+{9L$?c{xWAESZy}Cty?iX}nDQBMdd1&&t>v^v_!JneB1ySes>#t@PMrfui$Sw(Nz+0bm*jTv9FD;tHjS}zpS!YA`?!X?ESeedmLLkK#oKu>f3(nv?K|#8{#0{$`$^(IG%?cy30>$J5H9H2Gg5y8KV6|fxeWXp=jtP!lH7r)Ky+`N)-s<EH@z9An|*n|x@9RkR&T3}HM)dt!RvI(K9IG|_*7<Yk)NN6hwmaQPHntIA8O8FDGXTrr{O3Rc@jk;^NiT4ckospJnA~cM~v^@*>Sq1_0XMTmG<ozTbhW}<D>F<I7<BF!NdFtV?P750q(t3B<G_hud4GfPlte%XTUIsbPssIZi!u*B4a!eSVp%5HB8;xwbLMvZ@bp-Xth{AU^0+sX<dUvw+;Czt+Hs@B5TUwzPcqHo5%AAqoW31kKC0PuhA6;&{=B~-}6O90yNP4MSV2lB$FWd1by_e=&Scvug|Z%tMeZ(&(A(d4*>Ldo|HBQ{e4}VYB%r0uv>t5yo3Soix!<Vc>p`~CwJ(JuVg`O@hm$pP~lya7mTe-4H@;-aCXo}<!2tJA?mH4m3iE7W+B6?8&-`DRIBJy+mxLc`F3Y`xy3vt_6J<=xa6oDry1{aoQF5MWdlS$K&{goT38}=(LBBr8K>s4i(p>Q#P3<?=CY?@Cn$H;MF#{F5e}%=yG~tziMO9NXv-;fDz1Lp*lT`1fL+z+{-_~1?<S0^JL{niW|Nuuuo95a_T0*K!6H9e?D(2*C@=>-9C-#(7v@nBY+R>AH(oZX)DOf0_K4m=GG{K%ueb-EOJjC-VX`0e0Dr?4D0X;5hwBc9GM7Hv1elI+vZ;Hngc;eFtuUI(MNT<v@w@2CHDm`<rZ=;lTOEkoP3rP(!>NnDVECd@%1%WLmQ$7CH4*)enN5WflTqfxOtqwkW=Rh6^MLB0s+*^Ex<4g>Ezr0`nE{b0T@*601y{4Pl*i<!#NHK7TH|>6AQQUqRZg5Rhw9|`jc9|GuEBV?%NCTXw+$l5YccStPCYang`hjSxF|IyRF;6%M2}W}sb!3}cZMLg+?#?8$Z2?WFWDA%uqC=~<sQk1!)kg%!4p%SD`;Gg=>^t+ChF>y%U7ZFyRQByAa!%SHv9A!aUtfu;oJnlauP05GvgU8$s``}NI__?V>P$H#i-0K{W0jNyiGg+@N@>gQvIe^=w9PctQ+GRbl^4io}%Eui6vq*iwm-%LZ6(!lKf+r8j{!_nzv+GbYyjc_2uf?5iPm-rsOP&tu0+cY{n6UNn^548Y*ADrPxP)e6r$hUp{g+LU}~Bq-%(9T@x^Qu9Zu>hts}TW)C*UWWHD<JATcWS8?5woLigwHrSYB`-{<aHB_x9FLk0)m?^4`+tZFQ^@72d6?btQG`OML;;WI=j$y+o_e7DqPi``};n35d*B#0YP%&JOaOFXvK0URvm=YnnqEk<m?9*>d?X5&$Yr62PDpa=eelDbO3S*jqpev#*$)!6b$1!>Ht8o-uwk<TAU^*4SzD7~OpTs4p{hEpx)`dJwk{x+KVX!H)N4CN|eq-P18&9YOS+YV)q1y7J;LbC|FdBJ1xrUvewtsoMMTrowQ8-4Q8x{$0k_Elm!oXfvruw2=TvIhvN(lJ)ME*@BriHaWGM3puW)Q$j$gW2v{kM=X>VgAX{MIuoGh^;7>*}S&PAPGP-pqdvk~n4Au&li0&1PDn$t&J4K&#(ucP`#Yuls!g2(y81YM@cLx2`@_rC^(c<!<bx>=3tJF`%mi_l!KD2wm2Yvxxdfm1J*#+^De5nsT&|QWdy7Tdpa}aOX%b9TesZ_8JtJHBw%bS#)+pBjuKroW5;O0VRI01UMGX4epP0r*>>~3qs!o0ku`om=xR{3fID!+@17-a*9Sx51<l@j<zzTA*|=II%^*<LUo5bXS(pN-n{a#X>Tbq^`#zd{;9`T(RqJZ`7qT#i^vdUJ;V@0k%?;>P6E?Ay5bgjHBLs-SHf&K|HUan$_JR@EY93{^EGi9$v}BeWE;V2r#-r`c>3T`GBDt}z}7)L&H``dLmw*B_WUbOU`h@WO0iPD$nxsiTJ$Pq4asM{vbNXF-}(2SP-(Mu4h(Ey(VtjAJD;pHybdC%hx_+Z>YHQMg6kFvR0Va)wZuU6mc&51Vb~A=@=K8~=%bEsoi#hdg8MOZE-Ls}qzsWNG-$WFCntFG=)v(3{^@k~(S`SWr(NPoEV^mp-`pf}W)sR;S{LXw#_(5E)m77rQ?13M?FlQ#y;ZMU@or!^t#%a;lXBcPYg<4j(}d#CJQz_Q9}JPspn<;eY?LUM?$+^wRU|ValMqKs*+?q0@E^Pxa5azs0dNnGqL|IR6h3;u7qTK_$en=|L~CKeTRtRsSVd=5Y{~=JSURgZCTmojjA4rX`^(@qnD@;o$zfJxdhzC1Fzv&N7HJe8`ZX9O5Q5FT3+P(J6HKiT+(?$W2!Mrq7{NYz@L<c0Z!pP1BygFQqo%?*ZkQqXjuDhDLCCbs3SXhCvVt|4&|*THp^w*{QV$JcVlb$F4exp_66T!lKDR|aQ8!nXA~{5!8rm7S%R#T%g<kOH&xU|v(Cy{l;4j$Do8A1YZf{r?gMbk^A!A7O*_3E27UyxiQ09VWT{W<qKYtVkD7Gh4nH;+obMvBKb~3rceHntbp&l5bjl%!df{V3s#<v+c7x1o#Z_eIdzQ6J=PhWLGim6@!bpGihy+^T!iG*qt_7;u9dA~M|SBa-`4Xs@WcY%*O7b8d!KCTER=m5ndPh7GlzUPygl;8f%hm~TSARc|B@+)Kw=N$7&;{MzBS08@uonKyFeE4|YaLUh2`53|&hs?6Wt*e8{hx7ZoIZ=#K?&9q7J$AtDZqgjq3K;8;GxLHszNHHW7~)+dlblGXPG0b?>B8aW3s~FY7$(1}r;dLMyaITKUmD>f?!B`PS^Vc8t(dkglqN)<G_+ii%2ODt5-v)rTzV>_qzbyvTC}(8C{v`N9I3P2TvfE`QNvAWg&WTah>C#J;e|$}1P^aZ7B5GxIM7fly>#mz)Ucv(9t84`hl&%=bQ;j{ww}ybC6}IYbKE#i5-B>7jbca0YKU0SyFt1XqaZ!kv_13dM*hg(fZ52xXP}^3GbI2jlu`xgGq_;I8e;eO+&u%*J+EQ&PW8U72O}Zih5zZ-SLYvp?Y+8qcY5_(fZllb{?o_L>!ml&eH}1No(^2$IZ(gxm`ddfFl}N3nYhfeGjqzUU>@Zsqr70lO**2P?8i_OS#%;P1sFEwDGxki=#rS$D_Hq0O<}P@6n@@@RU&gqE=?;y9k{0kf?hggK?1=J?^T6SuIBI^rB_sReRRyl6MClaa+7PLXrxnC#NZy3Wy6jrB8}^m+d*0pb0J{cni1&+whtGVvcLtlY1;%vU~4V}aD8Bc3_w*t%iIp)mQxi{p`t^!h;CC>c=AfID^>Zb5RO`R;Ut@(rHX#d=mbDpKC8<(Kj`iTwSsqq9o|k@cuV&Q2{e59jt47z;8N5QJkg3!lpfc&|MU{?e~s}<umxW@BciUAv)KIM%Xbnp*CYF6kcR>^h<Do8F^^|Ws$;Ma@@s=MAH6o9bQji#x>ZJ5j$!}02c<2f>=^Tx4q7PiI%Xe@S81j;hnEgsM&^$}*!0H_WGYv#CPT|GjSfCfMPI;tvi42VXvqNHT%G<luLN1X%({=7Th;-zDY(sJjZ?L^DTtLbrejJdk+{73G~mYhE<I>&&>0SgYdFhz=0_wf%7|r8>AS=trK+noTQm@2Iu6WkQ(HSK44^F^kLI3_wYeCLFGr$>foI`sfxpC9UHJb5fTdL5p3Z~9$090{rKTnouk_<+U%uw9U`6Du!OF*4ropOcRO4#+$|DHtO!NljRbTHPKa_-6xXxczl`(@Xc1+nt7DuI0K6labWH;5^qK?&{5wTIGj9Gp%ME<snCyo+v;6p*o^juKT282`Ck0+coVG2u@@K2Z1knBKL%!vbnHH3lf&_>BATf+3EL!o749R~De8BHFDCLnmogg-1>Vh{3A%|1{oBZ7*!W<&Q?p;TKQlaUCneI}W^SO+_*$-=>ojOW;1HOUFe0@R|ktRcz^%=*hZnYqFQuwH5w(0oJD`CG|jg4;+w$VNirpa89e)=5Ie!ND_wAB8XCR{Kq*Q<l_5i%xr%EK}x2SsHk0nievl5>c{n!xwN}RAZIOBGu3xJoS5F1^B@$>}uBvfowXKo0PpyRW$#G&YHP?pqr{DIYd3D<@2IY><X&4@_t!c^-Bwb$B6YsyV9iu*GFB{<BrHZ6;EOc#~50>rX2Dl*g?a@Q$@3DHJt*e`Kgf8Eqp~^I3@FL?iF`4*i6`86*d{&e3J%uBy`aBnpIp>k6`r`RXALPY?~G@$_(~v?StYZyVlUB%Y4;Pf50qfSZX~tay58a7<e+KoyJ$bjp!s2l{}M7QL>ycNZAIvvdG5LAjL*`RN{XsM#&kKO4V%}Eh;mHft2U$qtbRiN5gnw8?D?v^DW>Js6WD6P~nH^^=U*u;taJn?ss!z^{D7dVK4vBbQ_zZd$ETnBWZ=Tx?nqCQ-PtZJ;wv71YBqj!L^$e1^ND1=!0wqEwm)C+R0G4ImJ`ycU-SU(fFVdu#$?u)<#pEDNBN#;oezAsE(pEGJkMTr@97_O0tH$v4#6?nAs8g1`f#TAY91M?f{baElF{n2O}Z-mxtLZiis*4+ECC}PYu-FWDZH-Qqd&9*@riDK9R(k%m^t_+m{d6i<nh*CnG}*nv-BI=HW;gc=*i%r0we<5yjB{L6aV&vYdK_$Fu=jCH)kUpz=J;{B#EGs^4L(9`SD9Ts|o3W(ki5C?DWw!}$RE{f9UW?>|sM@hyEw)ia(~eTgpKp1wKvUY~wC{doS#`|<tN+tW{Id%n#UoK(7rAg4iA%a{G>{OaSy`*%>dlmB$~{`I-{@#4+9(@&qT&at^s;2*XRJI5XW<fJ<}KA9X&;P>9NJv#1m4v+l<zkM=2JU$5y4^G-AC*#rd@aV|zc8-tzZZJNW)DZW7rLgxUqCZUXuNlBIN*O0pFD=6!g?;s%Zd-6AfBc74m{Qux9;Qn{xcpTX3#&Pa1#G<H^(h<Zw8ej-8KF?I906ONDyMqFEP!acorDRS6jXn>5ZO?c&1=#{98;L3=w3kT)>5{&r{NN1<d?&Jv#K{<W$+1){|B@K44uE@RFQr9BQ1Q_foD}fx<fe)=}_ie0cMA}Iv@I!8Bg#E&=?h;mmutmp<eq-QYodT+GBAH_kvcteJF+iiY69ZTSz8u6e9tI4J)CW`-{=U?>Pf~_vFy>tOs3|t(DsiQ~Sev9y*qK=n3n2b-XPk=fi}rT0mYD56Gzl;YERXFw#J!QrV;EXDV2r9*Vy?Y_~Zt4V5WS1U6=THz<O>Y#oqjB%%bgEKr^U6zy^!b}+CxT4UizEA%6;9CAyI@M4b58TiarWb8XOg5wg6l=NL#N*V%dE#ZYKKnp*kp&wMYEO|ywz(OCj;swGOCQATWs{vk8>X^h)!T3}tzbY(hP)+5R1^P#|4xTfe(nqHxAxL>1AnI4HG$SRXUPltYR!MmYG-hYAKDg>SX#2IyshN9=IUH3__JynV{io)2r0OnM*o<?hADib`jX(r{>sx*(;(N8ky%6K#m0W{zL|JZ1b10sXvFcGlXM+5ENaa~gu8VwvQ0puvKT)z$Nq*FvdwKOSk}`~KdlIz%a_qd(L*Fv{jM}**VUO&PM;wc;tqUo&WjHA;`VHq|;#{bh6sTnNOVo0DQnUXKELu+XBNzd(d#k*Lu;Cn2;l1uwSylPsV%?o$ouIkx4>7TwXxJHa8+%<VX_S!MJe7kN_kDAVp%*#&;$+15K@nG8bXY@OFnG)xj5XX=2%_zEnAM3JU{DXeMqaqp(DU8mE$XS$X^!|#1k65s6Fd~AtbEjBq<oSqi=af!k=}JH=2ogH_`Nk*EtcXtD0UGZoOyma4nrAtngR#$5E2pnDy{{HL4~Q;8c3f#yx0Wez3;rpkGLKN!qvWffxV^$>P<XBEjODsk81@eQ6(saPv{Eh^V&T)tXGc8ZOwvvn2G@XL-Vaqk$|WR)FV{tQYsB%Q?3sc?w!p@(cnrgRMG0SWQ3`RQitzQ=-G5Nn#W_x^cJ$&h6Ok-O1pfW1M4Zi9?&(a6(T_xC}pI=2Hjk|KNVOr+A+CqhSz+{MBTX9V4Vh242@n_T7VSCh(AJ-$5{4USW4PAQ37pFGclxI$>dOTyTnFawGXffo$r<qRa#55zKM5+yeHl-CsV$QA{8%;8xgdDqU0N%cf({o;3xN~fY%_p4U;$`$Gn{H!-tDY>-X{jfCyKME|LmA=HmJuYfHMeyw=0Yw$fiy)oVRwHT0|X;kO@N|9J8Ft#|S6$M-Ie6oZTGLb)Oj*^Kg0fLh!VeTai&chDX_<&vo*bop^3`P1B>;bBhM9`qvR-{P_Y^;uCIto|=wIJ2}Yijr8*dbQraL#VzwvkhXHlGvyw5kk;{9_0(u<%2>vElWWOm{hKX0eGVnm@%w8mKa;Dnn;xipSdw9;TrEYWX_bLtiUHhTL!GTn(&#N{^R*=pkqgYBlW3V3-1r4iP<p$(T)u<_oJJ@(aA2oEao1z!#wpHq+@>x*vERM(8~lbXo|53=q|zJqc#OK=&J-kjBFjn>4Jj6v=%p@%+&dZ+?Rs;5Zyv<%z$Uu$bn@1<bjfkLVe`y9`Y9A>@DIluu-+L#j=*K(<0!-Rzo1?3jqI_`(tj)#s%hb&t_tIs)1lAAZ;}8p=5WnHnOKYDrpmum=WjoYCrSI3|@*vQFOo%VTdj?Bt~VV%h&)8>On?ZVi?1<RP^OLNk()@Qo33=i!fdI*_anW4zefhVx+f@JQk2;i~EMe2<cffWt5wwxwp(TtnA5Q+<JvbxOlJ7z&2tZ){L~^vH`eX`|$2fjdMZ6s!BPwqJQXG*cTRO5Qs@Gk*xx%ih@P~zXhf`t;Hi$f79JZsDmLtPxSQET1MBkXRaNSEzmkinafp3NjTZiRj(*(uUXIK^sftP(uW5Miv1ZqnkcMCTjy!xLKBl$wNkij77Geqzi8$$(7H)2g{8B`HIc+XK6W4p(4d&Li_#@A5xWj1rgcrwCV!|px5(oozwkJ}8O+@?(2{Gz!tb_ASX{xV&1%9BD9%E*TF@cm_Dl3v`2z)e&c-*^9>ux3-hXsR#unGcrH((%=u#&5H(4!ZxGE+>00H5{#p_)xj+71hW#C7vr4t1KV&2UCC1yPqp?Ss*yzL%CR8rR!L5!`jWE!*(f|}GH{1;UUL^ze2LKe*DFkk(CA(T)Q-~~9UwexzR_&%MlU=>>~8}uIIxvYRoJR(b5+J7R;TP);BOnN&`a^XSzjz)*UY_e-WbrfTWp7?(G8}Hq#%d_8LwDejR1ku%W9A)!dz!kp2gI^#C?xrE$UA{m2jd%XH&fl58-<hYOAkOFdc|aq%p|_!-Z2nM==mNsFLfZ4N^Z8u?I~5q(0vDg(71Wjz8-dmc*#=T}U6^4|#X|Y3b-sCCded@z%ZC9fRXqVsMF?Kpn8>0@wp#wUK0l^uVUfnARTTbVh3X|9h11RC(nd{Ip?bT~af^lul(Wgc@BLw17Y@>R`^C?6(~_{Q=WmvSX5}$)ICfMMED<#v<FFR54tjrfwM%z!%PSirkTxuSb=Pok%g0{DS-m)x#?vNF2_;VwkV12Z2J8@fr?bXT)+c#67mhuMRtt=<4CDw+?A~`8&IA1Kz6*c56adV3or$xdQG8#Lrytf>o@gl9t(xPy-LC7p){(<~2WaBH%Q^j#R<I}(Lt;P&j~P>tghd{Akb@*_m3a7%qul=BVeCh7M8<NuF^=4Y(p_%P;;b;6Xfvbu?R$dX-Es=*IMn-2&J0g!bpT#Dr<;>6D@ZkE(G9G4aM^HT5HRNclI@cimKb6fWF}99oWhU{VuE2=vZIU$fVNaAoPfJ6N8Db&Ov^?RQ(8{4gp#}vWhIMp?nmC-e*ocIMyl}TYLssf>-O9Bn*5-TmC)3RS94>=P>K+qvs5FJAiqS?h2{WF6kEe4V<L5tJHc9-rIKVr9O5VboqfJ;C@DJdS$=F~v(;i0`5~+vaaUa7WK|Bd_{9~Swypw-j)$pkU7o%>``!DGQVA00D}CqjCV1!phsyj6G*yW@c<I_1Zme@bx?wAK21u7x<xc!+@^%*C6R%J7=uXwTggk%bBAA?(f(3Tqy^kLhn)El2B=J&pb%|iXsE_9q(sqwwqpreV@vs0WTC`{Ek5=e9nU)6Z$93!MIClm+<Elf`8J8}X#C!qtNuF-k43{_oOYuCopLeS$!eQ))bps?2M12!31N$&Zse%0(Mo7ykupCS^=y!OfGiQU<4b@8h#aL)iRqKNx`aA>SKvkR;%^ys?N1VT+<K+&wa5vQFuQWR_Z{77=gb>Sk0VD(OO9NxkvKwSstkEo3C7cF~dC5z?X~S%BoU05PpgV!(OS#{P5lApQ&JS`*O$Qf?-LECFinQlkg<-Vfy(CGk^9Ib~7}j0%;9ajMg7Bpy9Roc*J7x=#&WF}cK<+hbx`|>fD!p`!dCx}`=~I21hzBBfi$I|5lF)RH-SSVIsiu3dUs3Y~)Ev4hWgZ7#3^uHaM|Eiw9y?e}u;*k?yu32LbXtyPs2q#P1v|abYPlGAyhOn{%VP8-M=`nHZngJd(fTvDAye1z>ysmXEde*YA}|vw=B<Q|2ACVzzk@N}4}tXB1lEI2pJDXKnZN3y`z8Xzqqx__fp3UG1h?q*OrZmk45loMOX=K{4{^LO2LK3X4uXN;?O()`unNJr3*6GR(zJ<GzjcsA?@a>cgsd?Qh_79jV;{g<=z`Lg%ga>ru3EW(-WXGFcJi%5$KN{)Z@vro^x|1vU<Ip-3`jo_fftC2uEYg&xWmq2t}RfCi#stT0!%EaTj-cP2KesjTwOOw(?FAuio3nWB%n-H5DzjW{$=t^%$!gK&)$JGF~Jn=S`?xRvtyt}Ozb^?#SOZ>`(eXG-aX>3$HagIU&x+u>$M!w!}P{vGV)l)Xo@Jh)y{hqP)+UXyaJ}2#ORHxg2Fhmk>jM}M1>7Rd`&%HyPTK6#SDB;IbZkIcDai9N(IlSjy;P#lzA#4LZxcMXoVk$Bg7|u6pNp}x$YXu!+Z;CVI$vr+Np#SkOvG7wipr+Es<X=a9Q(FDs%V1;KpbJuPJomt9z9FX#emimRTViEwv)&_6rL1w4}qHJ)`K=qCzMX*6F-NJfp}Ub`~@v<KX1|&lrs8u<i%{kHVsB`I^EvLs<u<w2n0zLM|b;UL|h1GABH`4EAja(ePL_N~Jl@Se}Pfv;gx<l}ue|JX_I5A&TTutWxoz86k#!%E`hjQtS26pgm+K1Y!r+p34cvk(l6fx9sywC!D<&%78hJn-C+%0C}wT)a&wg;sGL)ax<%YiWw++C2lNk$l+Ml8N=cL4cq1h>ZzC<lR(q78RE7(tvw=iPu?Rn)@$$44d2F9@jU^4&KS^>y_*ILaRdR6c=jMvW_U|(EKeo=MwT6DkkF_SKGbGJ;x+Ha{qz`#)hB?44MV5dNQ6su@x0P^OkHsTDy0mIK=rbP|I|e#BQdbH?Q9LJoJI3&04Hpd!G~=ixHNfNfzL>qYS)+B4ZiOh?@c7*ND|5}dOZ*7WL6P`ELv}p?gn}wKY$>NDty1jcS{gY>UZ)jA}7SgP=mlEi4AcuTW&I9^>7-_=iID;P4o>T!WC@O7U}WpK=Z9(-opn$L1^oBE@^O9&hJ_FS`C!}iixqPp>_>&5X0P)b<okMcJ=Af`Mycv@!KnzB69xn)0NZwp;m$mp@Q(8DX8edis1*ny2J_J06Fx~+(+aGR_LuySu7mQsUoScgc!8*;jbD08X9X``!)9}M@Yqq;^|CM*ltm1z!!2d;SK0i^VNjp16iEZ*A;wDP<=5!z{&|R(DIq93U|<gc8S>$@M_zdtzP!9Yj_T)A^Qp&o1%{5(VHn?kdjrScB0<V=#edm`&+P86oOLkHcVHP*Mx*2+a;*ZbKGDR#y+lWIGWMq&m1<VjoBg{%=%O?4B)de))W(eC{OK@t3gB)263CwpEbjyw&!4qYjau#!>3}t>&I$maMP)mf=@B3CwJqes>%y<zXD?P`GK`nuiMFHsMUs@b4I)^vlYI&IQtC$bM5g9Ad17U@g1@~ZZMAwD~$_fwoaDuTxA!ge0e}vu;K*Q;CQrJ0g1GN$Q8Y}?u;*=k|_7SkeZ0i-s512WQaw`)Pg18ZK*-#Y87E{nLJ-YkhF)IA36RU*$;Gb3*#Vfwf7*soc*->ZngM;Iys(DOAM@1!)HI9PH}grxx7LE0OK+Sb-}giosCT6z;Y4qImgADt}Al!Y)ax&ov4RJ(QqQJGtfyQB^YaRLl6iQgG%1tbsWwmjz**pv$E3iFj3sZw%<V0AMr}?I(E_^5L_3<!})_1D_y)Nqn?i+-6pg)w*+XeJ<ctYUj{Oqv&Qk9xEJLkI@hcVnW|dnIkJMMOBX{~s6iR^ucLf<7dg=t`s;MX9t5la4~{tz1iU}OZ9VXl@l3z#cAm=UQ(wSdPKfKTT`xOJmJ~6=y>pvWWurpweA)QIH;AfZ?eacvhWcu8^-A9+PMQ**kmPYzK$vTQV%=?+uQ{~t(6rj%i&^*FEv{uXZpwRi`BrnN_V=^z`+Oy-zO_8X8_N&(YVMzrUzD(cDUkfTIp#-BUlTQ>!-SplfbZk6Im3(?LM8edRml5xT)~X&u0`{OVhSYz-YV*PP^(Ce@DimIOjl%yW1g5YWTsM8FN^ijQC{q$Rgwd*!YyQJ`o)bw`=LWqa!rt#<H*rTP@6bK>96yy!&anvNY_(Nt@c=a(P6u0)MTGRz+2d8TJB)SjT?0|W|Tg%T^n3nS4V+82fOmM!xJbfKCl!Ua>6rU*;sz<$<;?H*{~3TOH_=D(ilkR*_z>Znj4?&*f<kTZxrFLsUXC8RaBaQn70Yjk23g3RmM*Krp!o&ok6xQwWi@3>&OOEi&EQ4vD+}Z%Ik&3{WE1c8_&CfkY=!0t}v4I6MDh^lEQnET^Y1viEN}`r_uRlu+N|If{;9XjUp(yAa2^vgmL4Bh9eurpT?eLm~kxoS;2C)&AqT!Y-m#Ob6UBSWm!FRvGi=~crid%+GaSMr(jB{>|P1ef(_#Nsz}jo_W4X#CLNjz@*2-g9>9vQLBpkHP$%=%eTULJZBX&h;8TCV@(){<FJ50ku$p{H0fSyV{E<VcwAqEJUqia9Dh=90X^PQ+8J{u3Aft7-RRH8qVuB>W?5dP21x{y-$=pp!x{*%Sm0J`IyCk%?)G|OUIOkc-K8T2`7A}~28R59PcnOT<=y)#)mz>UoR{)K($)*R&N)wB|bFnwPsPY><`5iMGjA77r#7)9#EJqFKLNZrGI!?kRyb+Nz=+!J^@NkQ*aqJWc-N<PHFGPkM1Z2cC>6T>h4*wyyvBEc<JTCH==`7AtLEeR;R_KMf>Fa_N=t3q@3PGjY*F+Gi-Vm?_4r1n5j~Y#IY2FwSFu|pTURPFJq>fp;Jru<@xm7S2yQ+O|WfV^y$X%=F+%gAmuvZU3Bv=oQ=Z5m1O7wXcG;WPm#bRQM1U$YSST)w~Y%(H1)^Ma{tgviV@@ZOmB1$ChRI65|p1Fo2w|0yaSsL*@xOziM$~;(dJU8J$)f|whBE*cP77;|O1Co}0#RvXeg}>pG?d&2tESd<l49god!L(lZ3~<NBvZR$F^Z}oNQ5Uu&(Id(qbeZ=}@K6+2(kGg_&f1AOU5vu(RlG_?W|TmRlPP<x%mTcQDG(UhHNAhh7upu7JK+5lg0jZ#uPoe>M8gFIet*UI-HNuzOK%fhar=^@#27s*UbxOU1P0X*@mc)5BTwJ@tY41e_$C$4L_z&pr-_DIvOo2^bOPmZsSY_hP{<UtB`d8)^Dv!R`jYfFCAb`)9f@HQR4Qh4z*+`=rl$fgX+x?6K93Q(cu~E`-gBya!Y6XVcrtelLC0_>pFcngs}${zE$4j%KRI6JzdRGl+-l6;V<EZ=N5P$AvUfS2VA68VG3W)|g%wYRmKS(uo@m}2tvrZF+^Pd~B!*%xD}9MDo!Qu_iOhf~%nOkZRN%KV?~Iz9LOJ~<mrAHed(`Bvt{|fY0IX1kFJrqZOP=CIJfWU=QOEOSZ#kK!n)+NpV%Uax9cC%u29&xF@igeEDnof+{9I(o<ypa*5>A&iB16ScvTKp6-2nxWkP5V#vSlD8U)i)FCB?`Syww)nb+6T)jH-2h81z)7VgA5hHr8anPp)A1$nWxDI|2>LHG-gSh{CQN2q_D%rH6MhvjSDRELvMLl|sxmp0%-ImGQ=D&;^UcwW$)l^Fp@Y$;>{++b7j-Li258B|vsYSLxJIrfV%?7$xdk7;jXX#Gp}`Gr~0zstGzl?iYnqyt6im?^4M-SK=lvNpASR$0su^t}>*LUAB`Ox)V?&f3W4toU1GrRiUfB@{}kf?ouzd;$~5TNc&P2U$H5JgYwv!UMt57i=pl8aJoa`NADK37h~fnQ4GXaTcG98LZc$NZ%Wk;1V3_Es4>=Dw_56L!X($+64l3Y^q(xlTIu0>l_rBSCs!6QR+QsKD&h>L9Auz&Qo6F%Eugu4Q!w%=8B{osYzp>KPS+|YV|tyBmDFNNza$1)G$i8XbF>zZkx7b#8<Xklc)X$&wXRa}lu|U+qJg2rpVliTh9#k>oayBTg!9nPWn9ULX+2A{X3?xCvLacQ)gIyBc!LSRfHwF&JYl>?J}sd5T6j+u!%0968WVD}(7Xu!FZ8b`ckKv)7y}UWDL!7LKiz%Qqkqctj?J^6h{+$%Xp1V5;f=t08xA{}f?M4>8ZM{<yu9tf5|%Llr@G=gYcV%TtFOvn@+{Mcgiu-UtJ`{x-Cj0ZYPT<)Yuq>oxt(&D^R2dRY;=BKfB%v2(=c$=aDEf;MLF&oEMwlFrNB-+9vEui^9f0zx{<>Y{Q?C=B`K=QZX1};wXfDyVfpkc7FC*|>*V!AKetrZ;I=&Gxe6aGyrwItubV4|2(tiZ25nWP>DY>*V9KXPTV~FQidAn^W)9FOOM+%%$lZbyJknb284Fd(Z~85byudqiqj{?Ct!$JwlPMhw16ef;AyArOetEl*u{ql2Bb9(uNCM`-N_oB^tQyZM!Q<1@HZTqf`S9ohgGVc5k9U`?Li?t6T;F;a*6f?OW#!nS&VEDQ9+~h4w8vxbcu(u+$x8boOf<WX8n|Dut%*je69nZzC`%7i-LB31jlR}zh4TDLZH0QM&)y0z8IWx^!DpYDKC_YhIqI7I&DQ$<toxhk$=NU*%hyFwqlixN)64aYo2$vRg)k3&J(~^3LTbNyd%(}yt8`64>38Zrm*x%T$pzf!G3DGsX*iq7T~(P1GXSMTDl}A?6wK@)m*%K+aTEeM$`lIoe7SItyiP#IV<{E1jwfI0P}x5Fs%}DME7mvhRY58JW|5Jr)amPagWm0oN}idgmQQdWpWO8g$*P80f^>x2rSWP*BbM*)*Q8J`mn+MiTelp=KsqN%|0}~iaeafoV29)wn{ryw;sO-m*)ew1(yg?hJeG`e(*&DZq$IB0$4wjU_Ts;R{*4Odb2UkWzC;(sKDv1W-FzJXJf$+vF$+qi0CI~a#I~3>e6HIVzu5+<EpU8LM%S{4s|5Om*aqZ&<?#jU{wuhgl&^{BkQVyduG=YTEEojDiVSNoYGky;#Is7OYX#t#ZXmKbjw&eg1W_e8#6Wq2JZ(=tc7DNX^+!_GcAPw=+eU}#%Jgh~Ez8)gV-NArg^X2(nNb_`L1o5-zPPY$Q(((Xy-MHpsSsWPg%-~6zU~@vQ=3e6BEXguA&O`$HQ{;6aozG#6mN%xIIR*cE<EdGL86e7k|nWS1)*BCRT}D5C~i0;>Hgs*quF-5v_1}1sFz345PFe?JZLjYRDU%55%b}pm0MRnxKtO+1r$~Vt4+%z)$&`Sx(N!t{bB4GQ3Mtqbl4%#heW9La&V`!2FL<J)yDUCBygE^hU#$2lv}4oQ*oJzPwd;8oW=INOyY4s4tY}gfN!N9H0$>9d}#2_jzqs?WvLbp(yEq}pTnc!ZPBX~bSz6y9x?n@nL>J&>}m-%CGzRhbJo9UhALZ&Tg^@RycVJ)E136YU-GpWXEt80oo!Q{<<XqFWHw!rT(_dBb;d?IxZQo!ojytPqSaS<`{oJie~!Bd%{I9s^$c889>B14Cf%!wGFi1{84tP&hGezL3N?LHt5Ih2df7z}uo1Jv6g-F&TY86f%UUhXx@`4XK_R;7HFO~h0d1H)GId}Wh~1MJZ)NyZK1!FvUa7xf!>MiHk77PkF`@dnz9yY=J^6hw2nI}hLvlpRId9QB_R|__&or~qA=!R;R2b`Rpotj0c@(PxvE_?(Cw^7igt}a@=hQOToH0L&ZFznP^E;o?vA;*@BT&PR$BDNtg!-v$GIH*Zc~w^z%%y`UXJDnpmA51)onO9Fu2j2^l~6D>Ntde1vjS1Y%EB$Z4cLjR!d=2_#rD0h+M<Oo&3gub{Yt=<#NMYQ3+&w?OStk{>a@?vAo30?bT6{-aM^Qnt|xu;2`XLv!Y7cRl`es5bFzH`3m(yV|5IaHb1u{ZC3-H!=)P)7r8AN!fu7XpgkYy;Cjb=k1|8D#bq5<_P!=n`FJqGT*%cuXoPdbceC^;=V&680q}YJ8_4(tBlP`W1mnrey)y}4(X=}`bJbN3O7q~6_Wj$Lh=XCyxm0g`hP`ud61y%OZ6bVR~n-q*9W*Wg^iK|`7=qzVqA%AZ0hnDU>1$;^&;X25LT#MtGXUS>#3wbgB`=jWBXZ2}!`Dpcts_0YCdAxi2@|{*9irR<!4Z7LAe;f~Vda9w-@AgWK>HM#Q1<0x2xDv?ncT(6h?+lXcBtKiln-F*lHVCj`d+7I^HN3PU;|GPtVe$&fz+GQA5;A%7Ygr3pSl1;X*<u&XvDmFJV>w2jVI+?eK${N^Q9?>Vk9?jUMwo{MITSK^%W0FQ_5InEoyT7$vL*vL`mJkl=v&n^nLbwEX077ulBLS83oF%5(NqZb%kABi%vJh(M`E}b=gg?_l=fK~f)(F6A{q8z3m_Xh#El41v^^}1or1Q}8xbghEzvd9a>_M{@@o)nn>NJ&W&R{g1s&3vY;V0TS`0@<px9^y8dRk(OGUnY^AJT?l!zylOu3AnmNA^lUPH+4Tg2JtKk~-B>mZ^8XcO~JiTspPrW=~@_>LDcy-CH=ru$~9W6jD!QLuV0O2>jy1?nvQ@eSD-%S9*{Mlzlm_k!`F{N2R|@8hTUSLd&t5AWWjFU5KeoERe!vRRla&M{F!NiEg4luR(?RFIb4)$wz<#BX0Gx@{B*g+#-Hl3w0&jCmJt6j1n6nQt=77C!>46he{8ko{B9xJ(qoRbev|Z5bD932e}bX}JidFI#!O>>*ok`ICvXH^>O35?5Bm05^m=*$ugFX_rEwkm1cYs>5P@a)}p`3?g>RWS*}}0Z;*-5Gl&xvt35WwkgtzZfNz^YOOpDU1#o2j3{Hd*1tlE$?uV0Bh^%l$D}l~`yu|s?leI))UH(|VmH4rvbl!<ui|pN`S8e@*O&3Z9I=}_Lxt0JDXmWJc?15qktw@2OL^`Ub`(b;SYGRQ;S$-2<VPx*DPKzl2=A+siUEs_2C5$JbHf<dz;P?RVuU*-ip-k>o~hn8oE?$QTO-lUOC(`5je!V_p<Tcm;iExhiI#zWy>oI55^0D2pU)iIg}@4^NXkS4>(Rpq188_}uXB9teE+@EIV{>gHX3g1OiNbavp`gftQ0BEPhE#A@76>hbjeXa-EO+ZzfN2rM|K&soZzs^FIN-uFYXtVyAzNDZJI*YFmTUlFKg5*9vcZ#M0u_5w$OM<2}!qAN=W(_yD`mpL^&QX2b0{S!<57=kbp!^rY{kQ>>isje>%OoIDPl&BkpQ^BWxX$UQvPpdgu&YBjp0VLb4RqypJEw&pt}0Re<H{94Y<u)5ZID-jAmjm!GfBf%Z>7oquE_s}D$-@ck5g>_j>SSqo(lJYJ8ofT;>cbJ}hI^^7}D*T#alWAN3nEID%ueyI>qDnds%Qhbh70=T-iFCcG{A@)<plaVcW5p{!%S9TIyx}ch0(?zTJV0>LIg9Ii}HB<W_1AV0&z0y#C1_yLtT>8w-mEHR6D)Da%J<1aNeiD0%q0y-@UP`8bZIDj)Qrw~9U~sHdES<Yq5TS;Ev9I<`ZFntGfBjx(M{eI|C5as|StAB0>atL-82?|p%@$4CADU(leEUsSk^!Kzb+o_Ho`IzDsDHIpZ;-g|GriARDUI+|adcE(yclw%wl4ULVO<$R5ujGaZ0iItrCAL0w3a)g+x84Qzd=QU1J$Fr|4<K6eesm);ch)pJrL8FYR9LGAjuUS{KAT$`wII^ef&Qu6yVkjdb=+3SYK)r>at8lS4M=j2@ni@0CP&pR1pPXI4_F1d_^?Y$!RlWJuuz9!^qTE`AC)}s{+vU*6>%6ue7Izd?TG7@=?Z8#^NjmG^6=8>E0E(3JkwU9Bv7aD`_n^B^&b&y<P#n#=!D+LveptVl9hnlLyucqmqDf*VH8ql)MR05Z@y6k^;f*I=-{>JYYxMM-B`Tze$!hWYU~JN?`m(B0HrZP5I8$(#Y&1-7YATlv-fwmE}TYyV$HxJ!N^U*;zN)ndCNzFk|CZIjS&F7f?m%4D4x2DGes2_*U6=M8Z2UEiZ)y(xKb}_|5^iWT6kQe>AZwrxIpRWVSSCRO=u-`L4ChinZ|k+qx$t3+U`gtO=fg!c`VfNb%2@)Kk2^2$QA4=!Bq?-h|6#fC8y!9og2NoWFCb1*@&^(le1PWs(B8%qa)-o?MgZw}|V3cPR4X;x62HNZ9-eJMN#|i2G-<;0hl5^~T#W>!&~`(Ij5{R%(~Uo@Hh>pwO+eTA4ei++>^-??S~4C7c#d8TJK`y9|1n+c9rKiW!xH%4iT=RWWs1j#SBE!-7~k23sPrLF_HTxXhisvT8{!xfA6@4y(MKLT;clj}3KV+8iTRY5l%U9>49TPGp?n)MQm6UKv$+nnB|&a!~;MW$sxr^(L!7>(fQ;k&vP+?RCz>L$W;{lUJQm2r0tYHq69$Cb-5;m*(p*Ez5AS9U6^h)*&y45j!27b<)!3lNM{&R)(!2)s#|No|c%_^Y^gJ8{5B@f>d5cT*4KpAw@u`R$$l=d6&BR<b}LLXt`csB;Am@tdtvp*aie0Cq;smsWi>~i1s47=m*hNy+h(9@bW@sQ}@s(uxlaxk<1R(_Bln(9GE63^T8ZH%7jn`xO5h*G*sOeVv4?f1t?dg3ahB>@Qo5{Qoo@plsi-UrKf5;@?eeEyf|0mANf-?wrA*TujRrL8$*$;tiy$!ZW-zt$(F^_(@UUk|2m5CHc1L=WD5uOhZV#MbPE7+%rz{GEG40L6@{}fu~_AkCaK&o*U#V^S*$7U(IwJ>ep3>CMPhJRUmzHU8HThp!>Am@HECJ-*r|lq6c3J{ouyNflSt+BC>UVR5rVD3&wC|vsz88^!APag#;77g|7vFf<Aj}-SiK@wUL%(quS!}oOgVTGQUFv=b-~~`coYJta^zO=4?4O)M-?b+N(-qh^fa_USa-y~29}$iV+b+EUZd52126>-OD~kx$b`e(8YjWEA9;6BI>8)KoLpV*l(bMdD9IO@7_Rvbp3#+92{sxi#9m`-N<-mWQ-A_hEu_oICJy`hy!qzzoue9PiUy=k!R_!}v4#^)1xf@9u{0hL?_DtfX-dUX#??pKZwMW<40Jv99u>ayF9TIicm`y}*=}=v7=eTOjkq7I>F@<&;=a=R!Jd5A85S>BtTX&PmerE(P&BR@_K>dUfhR6lAxX-p!2C1~@d_M7F3X?LN}Pz&^$y;>VAdJWe7xQ;NIVh{Y9ty!neM)*fme6(pmv>oy2J}Vw_ELlZHziFkDoqXIZ|Kednp$I$j87ncgg+V;d?{ISs7Vu;<1)@jLb5-ewO6*MC&-Lktvjm7gIN;BFg~XgXeO>>B5|=Yl{j~FF$QB>rwID3EsAnQjB<sq*u0vjQr1(fgdltKj?HDPN#dkVM&YG6Twa>vii}jud_-*GxyMn1YiNv%$XYpW#n(%=cFPp3uxK3Zg^^l>J9N-Op8CAqB*$l6f}iG0BFNXRY~rPRGDV%u<GIzzvPM*E+Q^x@%1$*;*=2Uj6if7Vcg%m-wB%S6OkbozsOAZ+<Qaf%6*31I*wtt1@$8f6sGeeu=Rm{ujr6(S=6UF5%_Uu$Ha9m0$25A8aW@IbkHu<r3ml5oWnd=zo8I~cmg_`GB+FT=f1cUcOkLj39NYXDm=61x?*nbo>o~$y(1!ek|s=bsEA%vQp%{Z!Abd_Qjt{4{MM~<ctfT<gPsJz5`X7X8q3#qpc!3u(H8+yl&@`i-QpFY>?y<>A_~J2R*8~AI%*=wJB4rO=Sc^5%z@2YGJeIkWb_I-5OdCy#q<P1X!dmc^K&&XJ<PB2Y^3yL_B&<gbjxmhHI#=!*F%vE_n~lqTT%8bzs8N{MtYyacX7gQ*d6HiHo~n{votR?x>9qj9=RQ;709}beR}>IAWb^231rS?GKuVpG|I~IX5@Kx;wfcs7Rrhr0bk{yE9>-}4>(HCUI`BS*p+nA4Td1WY-%N$18hcB<VyJ?oplVxINdcW@fyM*Sz<1}7#PKdr2{T0S8?jhvGU&3Uxaf4+n-;9M48PNG{zu-`5v)uLB1cwk@MY%hRX<*(RW{GVc-SJG@PS)|CA#m1XGgH2<F^!)KkeQkUMV9pq~;8<yB}nh1m8HrkK|;-#r2HWemia6b$re`pRPI#>-k;DiVjtN&Yt8`O-y2QNB1tSAt0z1R^i>*B%F{6N9z+>^HJq)r<&Op)|u1wvDxQQ96dHIA+>50wF*5-z-4L$lX?Q%MxO&G^fF4T}Q-Rp}37kGSPygT1bV-*tC`xH##9#@Vy1k7cr{0wA?23jb#OOW=w8rC^MMW8)dlRY!pc$4TVfUk(A|~kSRNowOi_w>4wZWeD0N9h9=JA*y;`-<*!C3ON2LE96o3h&{Y1KEb|PNGmeH)b#L@RiQRzK%AMFQnRhOg3{8db8!`0SfTuGu<S&xLZG#;Sdji;ej6HL?f!QGKT$Yvd=@-g7>zX$<gqoEELJLPv_H6@Lf7z3w6pH*y%PcDmiPYVb;i^PT?#c4hD{M;oI4dn!9QE0Y%J{E6t+$fcL^bJ`rdK7~#>=yT;>w~`W_qGEfufqXomxJBgOzbX+gnSdXwEu<n3iPrl#6E;&=x#yMRX1(ZY+@sQo-v+KKkIY_{ai)X-%@!C)>`{&;7>fGc2jr$DD*oiSkmA1r~mv{3uO~A;63><?$e>t>4badczAW9zhv#%>uKqk`E$JGJF_4NF4N0P*9$_`W8-fGDdhXw*d&*3*7VoPhr}xeR%iASuN44M}A?gGV$-6t2eKp*Hwy|>@MIQuN-wSI&uc2@zNOuK$v(>)kHXBv1}_HjAeU$SZj;GM~7ReboSx1;C$hNf{5iw(R4MRQ{eIL8z>?xD@G{b2V}R4;ttOF+W5uw&ZNvH%A$9k7piD*iZXRJj09O4uW_$YhI?@>j4(|L(=uSdQBa5*8jY5Wa>Z~C7W(sK$GJ@K66+jeP|?Z_XXb1;K^K{_io29bV<`Y!>)k_s?pjC3pA-&J<{<#;3QIXOC{6UB${5aLBA5afq+Bq@5>ji%b_>HdAR`HE(6Z!<ZV@2shFj8k$f-rY6n&brm(K{T&lOZcPS?zrnM;f*q$i_6Rwt{Pdv4cdN#rDAErwK?im>`tnf#m$Y8ta^GaFV1$qJ7+J{imN>AG~Ex!<o@AuJVMHH-z*W2)SZOk%a%S%rNKaf7CS#@bp0<a`jn5((lUf}%x)l3n8R6T{+-arM^k^m)QeKDzphf{>w2`z2SiaI69dk=Rl^D_@d;HfxzxQw$eoe-|pm&()Mzx)WBmUr}qD4kgw6%Je*{x_3nEL(p7(5AkwG`PT_U@K2}J?%<1dcfVMlE~lbwy^4#d&03yviB*aypKWFlWjTdWE$6BGK`~)WVV_3`Wr~1apcn%tI1L+su!hQZ&v%hyh)}KC7fRs2AbtRU7J3A6Wn6Rp;_~ZTrUc6N$m_6Q>L<__aZ==p(=UlmQp{f798nlofBmRk$#D8L<t9R(7+Wt<S@Ty}3{pxsb|&#%#A{M#=V5&KlL^J};(2UA2BYY-p(A!<{+z3Ci&t2-CdBq8G>t)!AaI}*C#t_Ryt#SG=@w=OM01d|;?eJc6DxGE13X|6?=D93>o}{GTI9ZKeE0do$4^)1r*FNh^Pes-7Z=BxeU8FkUtFP@Fg-k)`g_y8!}j6cXnJ%qIypEB4m(HFlg`2D;CO%HAMEd)9PFW9_tW{i*YB^qkMBQUot@_zdaOY=uiu`x78sIO`_+KIfRqOd8#MMgh@^N82$@R~f0{Y-7)BfZEYiBFLuw%ZMCF*|P%O5C2>*8}q7Un^NOsx#sL8!FZ@aB!1k4}QjxW*olkgTDYOkaIm+!`a-5~k$-4FcjFB}F>{Q8T(a9$%!4d*93oqlLId?*+6oR7hL+Pn&;eB0ZvLwrrsbYXyH9^p|StdH$epxE2X%eUOiU}xt%y2eCTJ3GU=!j|j&odci#<Ny5k|Kb1qZ~nvo@jv~)|Lwoj-@UVUSVT1NtHJ5nrPFP951qGu2pE`U*)r|z?uLu&RyqrV`6O+H@$Sf<fJ6>uJuKVAvd!bu?yK{ztLyl{&mthFd3^oQij(VIEZc&&&3PPMw>sT}{au%P{osSDv7t9A)5JbcWvk6sd!1LES6|oXTa=wP^hsr!^zX-UybPL$-Tl)ae>6J0ySr<V6qe%m);QYbU%Nm5d_Vp9;bE7_K#ubPVB3UNF4Wv>b=u9nqoc#qSN{x#6Ze<&{x9+Q3<j9=?`E)j=3wDMG<4DDWeB911{*pRrJ84F-mBj>oA1uxf3Lm$R=atf1VLo<G6w#K#yq^C6AxFgt2x(BcSjG+Xxxk@yHHaC@FvWDy&6$2v$PV5Yar^?XqO7@{_b~w`(vjgQ1{Q$>;Yu)G#-ck+<=2?v;YjdyrAWso!_n=Lg#lgKf3-tT`hn3%fHy~zWja{AD#dEe|P?R6@j>O8LrAceh4rr-}Rc>u-nYxf7jRozWCiNh-NF_`7OSK-<R<5E{LFDMD_pu-~R*G-q~UJJ^~k>IcGDFDsU-Nupyckrp~8eJd22ev7y&^5_TEzu1W0br;z+k&*BAeaeR5{j|4-?Lwf!P%o#?$G2v)M+~Y5!QM!D||2j5k&?GR%bQwovbB1kf7AK<RUEp<swf*4i><Hc)rgTU*4U-f-Z<rx1U~33r70?y;2?#~7B(GPLG7Oi7$jmWmW^oh#I5W-hDj0EVXUDn7n5n1E#3z3qWbSMbCl)@RgFJv1GqDRcWF&1WJe<&DB6U#Ef;|~cI4E{-N=7a~>{yIP8XZul0MxU)dqnY0a+Fil^X}s0rX^Xt^Z)%1|LWg5?}EFO-!uVF#2(v6&GrF|;otxLzj8j!@M>^0tN`xR-Lj6KL1Byc?(DRjL7p*g#omTk@AZ$*s?}OfrtYu`=!^fKy*KTW<2VjQ|4Mmq{8&A#?pZ)k(p)x<v5~|R073vn*`ArxQ`22DU7Fs|)jbO-`cSqO%AzD%q83t{Wm^&@-%HBAyfQ_R;yLf9_+bF>lip8w5s^#fR!jHvfS@#x2y|6tMn*<PMn*(NMv6n=>~c-IEgCd|A)pDkb6(h^U>oMS8G>aOCU|aH(B{8?^oI8g3<(B}ah4i5*G|uS0z`wC44d#FU4TLV8blbfhU@Rz98?{_ciyxZlVVo#1Q-B8K|}y#X>su{S=&!63^<Q-z{3da2FyA$0!y))Ft?u(jV>T2Vqs?Sb)4BpZ5Ub@L%Q0~!|hID(H`>%fw-ehF=}(E2DB$Fq59M~#`}@4C<+W83d{_P2Zp$A6N&lIl9!(F(Lwf3)OjJ)p9BYG8QXd~j)s&00ST6>ArU4c<;}5cnWGl8$hD?KWK2XwxHayz$bAd%XfWx*0Nf|-hD_P);2u35V~3%^O?qYGA=7X6fn?pVIqb7bievoMiFRQHMKn(%n&;9vsHi~_!CngwP~fy>le{HF0BDBI9kD?hVe(zNpwfe76X=17LXaIXv;a@HQ=_=?(%&A2gEl*)PfJfN)!{xJ9ca>2&h&GV2T>iPL+s-YKdqpPyAleXkE>XX;AXM|p_ey+>;&hCHuiTTZnTIc1hXkrQ#H88C;n%l;U_~%|1^&-a_E0FsP4t+I`2kZcwLRQw!r3zC<Y$X?J?D_j>ZI9T*F6gw?Q0KX%metJj}t*0v~a0O`_XlmQ{wg4vn9O7wvhvlN`l2YVxVHPgVK0+V7$4GLz(r$X3@7YUJjKmUVUdhRx;}CI%bt-8jJo6IV@KjB!Ha=5PWG8e_kqZn`!|u5KQvYruDYfrH6H9N6nG$QPiAUx|A=@Xt+lTI37(xmvBt|EM@gebm}<MUdL=gv$Ub57Vs!(DFIC1I012#64h_>cx}m!;UtLRR$W^pzI@{({8N$wFU9usw;R0KK*|`{im-1<pu)~Jx>Ebh8v)^w7gV%4F3PA+Vax!%BL0=K||F*jR!2mT$t4aTMX;#FM6LD#~@H{Vz!%9B!JYu&_Tt3Ng%o-9VQAZu(7eR876J`1=_0Dq;f_2YYjd?g=Q<YAOTF;Ac*b@sDiT=mHg3WlbBmL^qQ<}wdb*g)MFMx9nA}`dLYZew2ybOBr68*VPT_54no|k4#n{pY$>UY_D3dOAgeEeYK7Y=gqflt5DT!9S`nc#(ALt?*ojs>T&oE%TYF_LJ6vftYP{@5!1nO^7+6ILp-?4pdG3$5+ulaC$~^0svuHz+|BuzY8)Fn@oKsDbU-<0Ba}9KZoWF9Zaq-f#moHqpSzqKN_XGobYxD<HooW?y9csmk_Cq|xmMlDZv$3??SUU5m#&YfRgKf1ih&mQw?>(-*PQu-Y0jVvB^ZUx@v@4#VfjV}mk)ka}>jFT6wREq~k`AKIyCGq2tccZ)u{tTtEpf%`n6fD27kR*uMdl+Ubj$q@sv3Ied8d~mz(LfDQ3>pVdVpL=+CVhiHkGYZeW(d*Wc4quJ#*{QI{p4!MB8!QZbTdB-nC~hU%Y(II|n*ml_e#@ZA4Pk=I1}T=sk7bd-le;tDsEKaw*2O=3S-bPGfq|Qx4G4$kqjZtUy`d5M}H~Z`|Dk@!j82mL{<hr+6bfU0`IdQeF};l}JC4i^e(r0%X)hwv9!<Qk_9vJXX@xV<Y9N4XCk^#aa0AG7XVXAv5%6)>8>wav?!~9RI#1yPUB!EB#LaNxmgODW73C<<w<+ttziE9fH_M)g~oaW$H4xeBR^Y+o#jDI2E{7XxlMNcBrENm%ozh%Veet$k9!#rP(ZUsY06q_ZI<Bfi)tl@Z_PwkVwXx6!Ig{t6qHS3Q>$~<?1%m8+GMLmVNr>4X&suLq?jRsZz-*=U9zw6-)R!c^KQdCZ72q7&pkVkdQ6%*(6G3j}#Y_jt=H74>cg;xXD9ExRs94>U5q7+U-(o6ir1<>Tt;pb`BZasNJEd#Ug1aCOAraB=f<8HJ(P@MEgMI1k~mZ<G&uLxFl4D=!3#OM@TT{d4TnUdB0l7Q)A)ENj(p|FfOI>XspTrP)Ns{UC<2BXhh>l(>}y%guMg2{`<w(63iNlvI17R5!x6v?;NW@;)x%qAq^E{ExZWkeg7Z=Okie)DQ*WWbihCiq~5t6lte|BNdSQ_@dZ0|C#5Uv?pD~u?N*j?2W@+<g9BgVxoBh|r84N+(SY*s*Ctcbh*noXI?-91(I^D8QJ2D`KCoDkLq&C`AXmgaHo-&6v$5|TKyMRfk0<T^o_7F@g#hT^Q51HG!06Ed&=`CfsPa%+k0R5l;x*zEOBP}BGQHi26pqlNi)(O<K_g@4$hj@tjvxfofk)u{z+?bA67;4c*_xTMoM~G^nF6NR;UJ;tI6-O<J8bnaq)e+cPPp;HHlSddoFhk{Ff6IHO&ZVs4$9mute&V<Sf;wTPyMNh&5#Hd^G<S>`5|Fp8y=i}A|Zq8tdIxHVjK_I`kv6b#52A<vB1lsFOt<xWrTB<hIj@PvocdA^bWZJfY5(?`MSvK2z|TI?)L%rD5m&r))Y0qgML+1ZJDMsMmIv5A!<Vih=mObDX)2#w<NjAismh#;uCQvD8ku{klxsU9GndiTSVD8G3IE)6RxC=Fde^uq5!)H>%Kg>AhR#SLQOm+TFtN<QTl%&#T{3J+H>)MmO-|3DzBK`bn}K*1D?$q@TxjOKQ5^K^ob@o5Dq65PI8hiYH4QaW8vE3;?l-{zxbUE`0==C^PJw~IoTreE?d|vu&j^@?(~LOqvz+Zf}%+N2Ta%|i6lJj(&Rv-CFBTYh@h`@64BsGyP@WN4wsitRR1;}1UOG=%M{aTQ{fH5hv~9N6gni=9P#?(`2w&CD-jnVm|lc(KzIs~V-W}i$q{6(j|+C9^^oSURErGVnYzV{6P{Q+RqAExGsa-pzTizcd#}X3@jl&w5;mW?>G|iuo}tN<v@0u?VHt86XASrBK#*W@aT)$Q4gWm`|DA#V9)H0*>#m_g<D;xXQEIZUs_F8%xYxp+YM(38-gO|<V5DH&?6%0l1*2<W)63-1N3*^xXMQ=I`Q@?9FK05pJf2M+9rtu1JZipKw%(l1Cfy~@tK5{sEgRhL0#4kMEN@H;QzoJSCe4dPYbtWZY?S#l(pU!!C)j~$igvnSORe>qn9>|Sm?kODvlAC$T0>XR22j-F1^E5num0%2Uwsk&?^#k;`>M)Fyw@)51j#|~g#{V7a!QrM|M$&jt}y}c1_GnD;0S)12*$Y^DIS4hel_437vMN~fpzdzUXzXO!WxUrNFf?MdY-XUz-9Ogb_K+~xRi=t3YBfjIjX&yR&#b{;dw_@$)4wjwCw`(o27FU>!sudyh_l!gfjfIx1OwiGNmr+2&ft|<)Y?ofTTpNml>ZM>8Xk?C}(6%62otxjNS13jrJHM$%f=k#mh}Heq(zu+E7I50QV3Q=A}UUCStCr4GP5vc;t;@Es@DusfjghgU<gaLR$ozH1X!=#qEZ|K9&6<?NrW%`FTE5aLFeV4Hj}tT9BR~6fxAG@!&=gsq2fdc!Ghs$lNR85LG9b#T!q20qQm=kM|eO?k@!gOTqqfaIhT6T_aWk^!_iL-KeGez_6B+`Ha&SI6cB$OH|n8OLFB>h_x`@xlkedwbRE{>pX4(F=r9Ikt!9tAB|Ip>+?hqjf9NIX?PFuRfXdLCU}dWB2Xr;*gFO^OcZApH@k=*S#`Ftt+h`r;@&OO$6MmKJa=8=>RXe0=BY|7cL)9vEn#$Xq}K9h#<*EFV&k4BB$~|E5CO-z_C{~3z_HMnx>?|H;AnM@iWSU_r3|bj?4g#PK(<qaAQ1EW@BZ<fZ+SO>ySm{(nAEHL>p)`Ke@-2G#D~`B4CjwkPWS#$)8Bs0JJ;Ol_4hhaYdcc>c2M*Dxu-5&aR#bjWM2#YCe;Kqzy&JEjN4T(!un5Nyy2{N^TKCuRWDw7_QI8LbAcD+)iAwAR<(m@+}m!pquyS)kk)uQPA>MFOXrJQ5+!-XtI>A&blCgUCq7+VH7#3Fng2%~k24|xbU}D?R7W(AFAx?V$}l<An31-6;evPmZ!kpZ+C}fN+Tv89NqK&qn)f7S){`H`xB)s~yg=&*C{CbiZq?K<AKGMyqR!su<KAB-&GD!fwZ^q(Pl!v|Tkrq*6$|rNEf?D(JtMdo((3Fst6=}udJ*0rStg1|4Z`>~PM8ZZQ%Tr+FiFyA9_Z=T7=1s?D<LS7_CDtOPBK=7d@qWL5O2aV2fzOc1Uh;^A~cti=byt!sw&9CPL;w-NJs}HGC(@PvW<W0-t)&9CHVqtkHt0d_9((D@F-Rc2jEIP4tQ)BKJw2Z9xN>(<G=9L-%ueD0)}sZ9|FM~yjcuuW?hA5ucxHJ!9t2Y{um$(`XX*XUyULfeg5fOLeXzBoPe=M$aQM4-L}UVrrQKd=-GZ}47&Q#igyJOrk&)Z!cl-p0d26Je}3FyFVQdN3G(CE2r{uSV)p(f*u59yXuBVH7eI0Y61_MKkh-X%-(&|PSv}!}ql<k@W6mp&haFEMGs$*0uixs&Y5k?e+R`T<KXZCvcgayjpi4@L=>8{`mmaTH;h)n>FTm#)q)z$>zZ!NyTrp7_(1pD6J^s~zIonD2ms6>H!<ll%J1>O4cV*P7A)72vh(<!Gh&nMHAf?dZo7XS?byY-UUZ%KhVB+D5cW!{G1**%nMK<Bq<XK3(awL5k0`Pu!jbBG{{qhx&35MKAT1A*Jw6HNB13uJxy;?ZJ2pV+iZ*`*mhI|xo`a?QA!`$-hr^E=4W3pZ623UH-B%<%WaE~@R5eD6nuZI5y@Vy2uy$K$;+J%mj2`?@OX@=t@>@X;sfd|s)b|Y@>2j0#u{wvEW0qRu1J6q>P7-9k*I>;M{cds7{>`KD*jk-yMRaU7U9R_!QX8_kL-v3nq0We;y=-5&Uyl5I+{|-}9*;uef5ET~&e^K|Gs8<ZMn0+-7LXrbCim7jTV`)nUu}0ki%Ou3US})hqL!fcs`M_P<m?w<1x(ZA20&Sro>0=%(HxE`c+{uu`qZsH2XM2NqG1S`HBZwRP6L|0!VFLfIvjaHQB?W@r2a$1Sm4L5%^B%*rVz$ew=&b|Elm;E`Wi-QJ>5aLm%tjtsW#Zr#&<9h#THUDBMtz^pS8YVx0J7={rb_uh)&$$%Uu8eBdEf5@p!CEQF~S{vOidAk_H2ErvhHq>bEOnuz7T4@B5(qiw4g&k#;gp~8Bx)M;Q+I%fjZ;!Pzidbo2DeEAqS_g`O79vD@vrwGVirvHRs<je*(ziw)dA<7bY~R9eeU|P0xME0AQ{i0O2EGZ(+e31o;0w@v~x7VuX)kyo1OGL$5v9yInoo-W%L*Vp>ZoR>^Isx&&b7=a+%$niZYwak>(5GJ)0Iz{BLgvzTbS&n`K{Gt3S0tPg0v$;a4+HnzUq*i&hVhvxSizzYp}7bOi)ec($grLy`FK&X$5$^e;&p#Y_J^p|3I(A5HfB&Bdmd^_?Nb;xp7^YVKI<@ZE+pa6Cr4{eI3R##YJcbvspz4acIEmdlqp1whR4!SJ)DKh6_BmZU}gOAY>XcBbETgXHvVD~!b6%n-BLVprFqL(w&OPJ0ysy@zOItShcv^NnT$q}35=1w%Sv0?9lG`<7pK%o@KE!0Q(wTdpFd!>Nx6#-Rv*Fd@?3maHX#K|l_N0e%<B$F9*lF@#?@`+BZ!FX*>GA_^H#km-aOqd`k2vEzJR}ieVYuu^Z$Ml&a;i8TyT$;JEnZuDS!7MC;?OO7X1!Wx%`HB+YU2wyQ?O5_X^VExtdnfpYpm1k4OA$rU3eY*y8_2Ry&`*24g5YTj2g6qq7C*u&;^In;BJ7%r7<|ox?%)}8SSC`kh=I~aL9*9#(XgNRehMH^fWZRW!aNvt&WinhM-Gm)jyIuQFJ1B|Esip*B57)dvZv~Ld{Nb^qNd060io+T^}aPNH&_Y*wY523e@<=7Ce12#WQBV3DE2^mT$hCF<9%d!{X$YPX;K>v<BnhPPI-6U`TCuAzIOkY-=3&N`iU*6>Lj}qe71-qiicl*<NgOfx%<tZBZ#A;$vy)U9tUICL&^reXhc^SF}xMeCWmVRR4T=UP=Mt?SBafZrM&ss=)^mbub0rLCd+MH!w(=;wqnqgeWks>GwZUP0c&&BxVHromKSws<yt*WytDB_on#v%dbB@iw4!Es;NJw5Vw+2N_NoqRy&OkOnxKEik%SWJ7+%9%GUPm|;Xio(9#q(y0#9=;V1^-F<8bXZfK#q^e2<=)+MvJZ8wC)oIy7X&62SSKI>Pp$#6zSy+^np-JJ4+RK`7w$gvrR~nh`Wgi76_i`R{;nglWq8bQyQqgn?JcO#I?v28=J9DHwoojWW68g=DKY*Vl~XSE~%%YW(ODrsft;>n8e`O`CnJ5V-X<N2Y;**VmW%81InUULJ05GRsOGYG#L|^YzA4=g+OpVMq)6LQw&j<6&(MOp9JZ_fIzu(5|6eU}~~<O%iB!4%=HsAPl}bo^(1JN|MX1){FLh#HLO~u^Cil>uE}LJoAKyy8Y!dX97?BRk0w}FrYCKrX)@d0R*RoUs*l9UUA?{bu+j)r%$X)MV4FGYEhD=7&OFDe+ML~Pyax-hH_9s5Dj`E)s>^BrImGSdU?!=EArDC(pgEnzKe$cuwqGOt9g-sWVfQNb`~_8KPAASSlvyD1p{t$Gi6Rt7@qh-+!Y({=*CI!+%&+d4W}>#?lz=~vnATNpFukUj-!=b+Pa!mpDbwEEzYEG1}vpDG*Gm`OxY9pp7VQJLfTdRguM|CEm4b4Fu~NLJ{m9?BS|bd9U!$UJY5!)?Qb<w_f82w&T(M5)LYX6a}imPk6VqD6G&ZX6&lplhd{NMR7%yOtWr85>iVM&Pw8V_eUPPW=_m;i=$#4-M$!Z*EP~7IFr2>+0;iuO4Ln>;R(BmJmE0b~`h+A?mbiU#^f~<O`)L^+6!#qy0`RY+>QX^frUqox1&ybuY1xsC`u@H*KkqHp&PX$#vX@@#k1k`w=-iE3EM=|^zoPEZ(TYwd?jPV@+~iwt&E4hn0k~sufbmGZLEh#Zn?eF++EGWLTKZL#-x~M0CkI=4vRtLh()$PBud+BA_J4Mw?rOM?d(D?x+tJmyce4$3@b`@f)f&v5h5!0;YY`xfJCWgAVTF0b9R`9ixsvqRfem&_6VydI%h94BKj;A6nSwtCzzcM;Xw)aC5-C(tP$lCbi<1@!2M`1ck88!)ZLGO@K&}{AHYi>IR0t?kOQ-j_;kLGph-T=Hm&%HFi^9f8tCQh@0%Z&l5_|bf$x;EuJq-}hFz$jLu0cwWs)e8!(gF~rm&Sg9?$>0;h}eZVY^wl+t29XR0Ffk0&FC*rASVzF_X$z{R-j7mBMY@yWg^eFcnaWM1RkJ6ep*r7-N6k<iqhIVFx0RWV!@RSHwV|OW;SZ_EG4ke#)eg(cK-5}%hxWQyRorRGupbmC3rLO*p^nng6&;k5X7aMw>CC{yw8nGe|_!(7BZU3M~t1Za+kUAS@>^bgP2=h&}jCAMOvx3ey+!KPKX=Tl-gi~hspvks%_V>4<N~E%#=wHp;p2U%u;r;OK%C|4%zMCK6Bu)+@oO!#`<%q%Q2)QV6re`<G0c=BCoRLdsH@}ML>qCF5oFfZ6z18b;Y9_nH<y4{=#OHqlTOb>{}L>*E7dQ11Eb<4s@YcsG^3VjQvVY_OxPjwI+p8UG~%}BE(ExcG)b##Ydg@+|`K8{+T|Q#q{Z0*J)xFPI;uB62chYZnzUQq)hVJ*wU7Ez%kXR#dzcrN`1DVupRt3(S<!pRW9UJkG7(aKp%ZpB%n-;L)`N48Mp<}H0Drczl;D0%I$j4p!AO%X?|^)xR#9NUNwyqCYbSBGHi`Ki9D@FQ7+C!S^k(&SimK^n?#ib0+kgq+slL@yXFSM39Gc0Rx)CA`oq0&*y4nXyW4Cl5Eg^_l4H1)7v@6EJRBfYP#3$*{6zdX9Y`NK=!lcS5|X?*vD&3a+*#~79sHx9`X<H*Ml>V1VW~j6K|Qt*lDMEpiY+E97`})DCt}n2R*vA=^wPdsUiWqP1rbh-!mu0}V#EY{u)KWXAX=?1udEmB1858_Dl^HVlsx5q_>1oye)*-lFMsXcmw)-;cR#rQhu_tOoVxqtpWl1$S9f3igHi$YrUq#`BxroC8BO@1S|m%xz3h@2I``lF*}XTvfB(&2F<kEd;SYx|{YD^k@AWS+gzmoe>YYD)U2RR|TGmAFLW*uydx!77dhgXY7}Uex{r1iWKfeFg7v;#h=4s%XZ>m6Xiju7M@BZtHcYpfxdq4eO(Am2`c<b<$U)=ru&+h;Be~X3S{x80G_dot#XCkDg6P=3pgubGfOreuWEISK15Dft3so*SGjv9;(_ka84`yYHYUCcgw?UloqeydX=6`u1dK4+~}aY4w8EQ<J<HPG~Rw^ONbi}(Q(=|?XfzVgk(uYBX)&)x)nWfSGjJKsnNhNJq&_g*^u;XmJb=UvInhu?bZ{y)5b_||*(zxwvQH@<QBhc}_d;aA`N<9lB-_zUKB27jH^XP8M%m`}ff3H05=SN`eFJ6}Ki*7xpy_idQJfbrdL{`~$2{{#~_Md!u$*6gc!B%#kL?V8P$S~oRBGnc&-b0Fp2m%n)U=KCN1`wu|AtYs&3Pn%v~Vm1T4{59FB806yV67%KSI$m1vwii69ElfOEE{)2<)7Dm=tT7FNEvA~9BbyX(G-WqXbbDRf6$B~gLz%Jfgi<dEe*$|mj70p2>I0z-x(Dn=5l)2@Kgx+8<;0J2;zv30qn!9rPW&h*ev}hGN~#W{A#M5~Y&Qu-x>sD)aXPXapj;-p4y7_v?}TPzDxxJ#naKrXs~@rWD|T~_TTIT>HlM~cKQ@!xMJG)OcG1UJijHJsXSuqvAwCFN9Irv9?rD=uxg`|fm1)!@cEKsLg$+F^;}d_)S4x+WRQKqX^1!J@)t4ThhyQ<SetGe+PX-p;ZWQ+FtBbWWfhYc2cQ)TLOF4BZP0>*?w^-24U|63GV?!Dm0I6Isfy4S24b^jKZ|bV`oWanw*Sa<~8O`%yNPn6ID+SBIR7CkxAO>}O)NhPvZPog&SfRK2{3NMWRc1NLs(MWn@M#*PGXhoLmh4aq6;w(iE>AOQQy(w1fT>@oOrf4scMqqpRpb9w+;trk!Ri^M9Yioo%Lv|WE&Qw%QS33#0BQtNZx6$RMn;J6+9Ow#kU$QdS+|tD+fT!=18+5jgonRQSByCAxaT26j5_A0YX~?y;hfghF&7u?iGXN&Xl&6X7U{u5MQu+F5Z1J27>J4$rDCd-88vk?!BthXiL2(4x;VMOD~0qWRSmlqD__PUOxHY9h+bXz7|2kr7|9v>13@pka+sl>9(iFYu<+<0|AnUl)sphiFC7A92RG#@Wle9z?6DjgWpc!l=-?7?*}FY09FSjF4(1_?^jn*|fLBA)3dr;wku@#V<+VWdeft#0qBu%gm2C_99%Pnor={B&aqu-FCPr-@1kmoI5<0v{pW1lfVs#Zu!J508U2c6{8>w!ej=*eo#={1t9c}K4aodT8y{N+k_tF_uY_{>jVmXATynF9{aQDSu-u>ry_#0LOz3BsS_DBJuL6pbupeP;KIlWxbpsXr|@o?^P;H|7P&~)%f1MC7NvAp~Juit<Ff2LTnY+|@qrZwR$!%&by;(RTzyNZ0eyNm}rZt%Q|S3_8)_LndE3^Jz_7)Nq?v6SEqNgo6NHuK=z7CK8g9)Wi?@UjV}_37~aZ{K<6#lx4smv+l&WT9@nso~j45{S<Ce}M=5Ne(V4Q;S4D&@~)_aP;-#_*-N{0ylRt@^)D}v;^EVr0dABr(1Cc$O81CbqNpb>lolk-(f2n-oin|3g@4>aq-fP#*Ir)Ub%GPR@TWVpu$_fyYuch58waQ{jdG!;Sc|C_z&Ou<9jb3zVf9HUwr+;FMj9V*Z=3efB*X7H~#D1FW++FCb*OxK{M7d^pQ8yGeoJay}WK2<{ac|tyTkZx9;e+_%vnrU8^64)v{sX&DX*Nr-2XM8iR!SWJMRrNqfAt)rr_iXfekzV4NN64ZQ7ez$W{$vHNrQo3AkH9=`OI!>|44op;_n{PFj3U-rviy#MAqhrfCE@U3_5yz`B_-+lYT*Z%E~?|lsh;ltOzb@xZ_-u>El?)>gQ4qy4^-LJoQ_nSY*!TIs`Km7jNcRu*>z3;vTKR$f@HO(v1mdNb!UpPz>$MS0e9&%$n<wf|p|F#Z^m*98A@WXMU9%>N2s6YqDT(!gz4<XEPHfaUaiK~DY=?EzEz@7^&T6;~b_6V!JOsrTX6{$T9DfV5(iJy#DPf9hBpL(OlfpN4|Ia4R-qRj<SZ(pgW5KJZ@;gng8*NqAH`LuyOR|85vB{0^b)z7cH%BA`wt)UXZAnjwIp#7+TseG3&+t$%pPC7wtdNj)@z$L~Gk{zU<+scIJCnMR^^?7^B;Eh0%bW(q@In02&&MUw?Vw!6ZQs;@qz6hh84+GUhp}wbQ5mm+9R}c-fHY+)&5?$=bR=@8Ye(f9gfA;dd*S~cC|NPhCFTd{njd%FtSML7vSGCE23A(f@hyV7ad#`*2YJnWO_x8UWzW3c}xm$tx()Uv|{#r*PpZT?}AV#l$>BE2j!Qs2Vm||7{q;Gxi@T;lFWu45J6P%G&C7IUCPBN|E4Cli)-`8>G%He<h;Qrfh^OfSiesSlWf4u+8Z!_w^3gS8EVnvVP(RhoWN+o=PjOrB&i@QI3`|kICBgX}Bp}Ykz-g)QO40|?kci#WTz4zaQ?()9<`1`C^!2E2PvJrDlY$$F_4V|FKkdNv~1tf}pW1^_?w}BsgVw{qJpG&Ifxak^&B(kP4#JxcXibUMocBu=Sov^plpj4Rk#oB4{!abRtX--t70yviG4EKKZ?!DLE|L`B)fcJzsXbtyXeFOBo6rDT2|NY@V{wxh}Az<J7`@7%$%^%-;-E=pucyd47yZ?vp9De!dLREa}##86-zVWle_r9f5Pt1PWGxl(95{d-=rNf{6{XHD8-yOd4PaO3hf1f_7Aoes6I8_u7cYpKt;qU%;5eS98D=eph$7n-T0nT8-qX2L5@0MIuX(U#>(HL{Lu&ReIzkB%8pQ4=kmmh)7%t(N0*)RSczQP+CH~yb88Tmi=KX~=<hd<G2oD!D_SE=oxWFSJjhW<p+J5lsb6g^XqOGij@<Gjf-I7w)Jb9bR$b6yo26-O0x0WJnCP+GhW^yX#n87RssMM$~wbm%pMv@MnK*fmv{^37&JVmD0c$%au&wkW`wnHXRdg>o8twn>g31#6^AsmzT*pR}|Xiqy2m-|T_l1f`jxotI1)vUcj-dFh9Tzkm1M@4s{RCqI&lm9VqFdHBtr!b?;TzWR15qFELbhZ-(>?5pBb4zf!GXxj6`CKiOIE0-HsXb5bUu4~eL3avLwxqaz3Fc?fXGm5CgF^%S0AWd>r$^x5G&3YrXK0MeiE|10I>>>$sq8vMqlvCqzS9#)B-xw##&C}`<%<XZMj1<7(aGR1iXPs2fI_Ay}mfvo5<ym$3E#Bj8C!Kgxe7eq(SF<Vy0}Nf{UoOVYQ9w8UiSLwRTYP4T>*dAK!+@?Dp)_3`+{&Xw6k=xxZyOz|cW>Yw48Ts0;VFC}KH-&*f8mh#1~u1f76#M}F;ja(`+}PtEN|_&71rXUAup>-6V@mNTp+TGG{R2DzyU9~2O*VU*<P>Kitqy6xJMb)p<_G_(pi@7CB++HL`;Fc?*O{I4<nz5tF&lWmr+$oaIqUkg4Xy`ZaW%}jcrVBwi}Ps5@yDabl|ArT6&ACn)yYJG}JH$U)>0&+I5bbmQ|T!Tvu>czRS+lm2MjKWhd%Jr`2+5XX|>`v%n#l>NeqAxoePSb?wQ{M<Qv}XdeuSU}jPkinMFTutqX4y*l0vw5Yb&>ht_#bt~|sxwYNE8;w{P52ji(zv5b!{SkL#cEb(^n=DeS+PYc<Wf;h1Fp1mo)~J432Y`x<D3R39Siqs*NIsYg^~GA&SK5|Zo7?X6H!(S5V3SbXDQhg;xD&^iRNS*Hw_Px<J>4u<<d3hzs+cYC@E*OCVNMO`b~}#}soa0-zYgE}9jO}s>)t=VefJxDrMv&Z578!m^L2Pc+wph5MYH;=@3J!NJ=FStU;KLjB9Iu55N%`+!dgx!miR{Llrtzq{euiqXLtq$MaP&=2gCjyK~1hdFc&5&Ka93I5oJ5!cc#}p8Z&u)brArX8l3OF`FUeLDQu#yEWSd_fHf$uxeoN<=ATU-N^f?+h!eayz0u&=a~cg4%#8-mpCc7QSe}Qhd-S^|ZVtarU;I{jXE-Jk;<vp>!Rg(kuSy^XgGlME#I5AMc2Mw<^onte)llaxPO4-y)Px8-g@A2m-9>IdX-Ay_7QfDK>W;InDcA4ZO<8)9o=Appb;UTCAiDCn3)^e?7r)En7%Sf6mFZBGtTY+b<<;sLWKM5e2U=qr%Eeh}%_;Cr7x76EW=vYZ7I_!A%d7Hi%yHV089^OjSCrs88CRC7Kd5;VJkUwat6v^OQYlh-YeL=K1Re){<@R3GTgC*E+a0`@HXM)oFjRo$P(xRF@4(n^RuJpQyz?E<4DrS*3@*MBZgLc@lC$sMDJpvlrZc;_u^r-tw-F_*X2%CN52(?LMUCQ(-H2j!@l17ju_j?ZBQAv}_<JGVUrrZUiMzy7e1h~Uc{+K=?+x?#9iykzi>F7P>uxusV|9z@h&Oa43tOG>KD_R4E+9iKT)BMV(zTnHYI@IQU&ZB>28QzbRgO?iVGK-mM)As#w20)Z>`Q*4xYr-Z3d0EF6ncuy)~8G1*;jEsuP56~!mWq}B_=hkC@JvlF64wgBwHi7nEjQC7>lvSi|j5j{)B}p>`FHgP_W;ji|5HhFLkR1o)GDY^J4Paki8yLQPsE2w)zBXazlRuR(F>qF(&t1h4<@<I}X}!z_2xUw1%E@Hm}D1@tpwi1r=B8ixTp&7bk6he}&Vy;#JRz6D+zjVBeePCLzrDZa;4M`)0WMxzVV19x?XqL&93nI*(j6NIBstIi))3i}J}ht|Ed`ardBCdurGp4=@gwe4E?&l+;fzGq0q8P*<P@9xC_jN`#PNvCN=c*~)F!#D#uuce&+jFti6PYR>A{&P<@}&IAhR(4TJr-%blm8GM;an~#3Gk;8Lr|3?ie_S+2=yW8U1ZR5j^_^@Mq*cBgk`3H3~EKqK_xvkxBBN~$e%D)Zf$K0mZ^x9AyFyHi+JRg2nyyn_m1*4Kj@%FesPJ9!u6voa}Fm^Q<yHmj+s=`>NIT&l~m|CONA2ndSF_UF_X9e~a6C%5hz@*5u8w?wX0>g-qL&tKvj?}H=UP>KEy;<L^<ed=QFW?O+lOEOilpRP*vEQ$lGU`T@^rF!bq*TD=`0)rXQBZ9|FNC`r+E-YtF+Dg7%v8Ua?HVwRqAroSG<%t;wSf`*OWCqT#ZVV6r^Q*}I$rIBH@o+?9pVf7uslg0^cKr^Vp;2i4&^AlfU&k%<}d2@a5|6h^zcXxp3-UN3Gi(6mR#s9=FeMgD&W@2zVV#$malDx(KpgX?AMZ2{I(&7G>FTSLtM5XPD}B9iHSmWsUkV#R@)Fv7o`Sy+}hJAuh$3d_@LVbn>B8Vp=CMmQI$r@1T23@s}DvF@CX>Eo!uyL@%|LLLCW~cr8!Hk2rJsIA#R<}y;Xs?sD#stA<<VlDDKlXa*N%@h*vD}+!(k>Q7(Y`q6tIxlntjN&9yp?pB+4yQJf9^O~ErE{cRGfm$dEc2cha@-*US5Ky?5URllraa%ywSt%%s>DOnA$X6EVnKa8bc_29Q1uS{#s+$=CmI`#8&?xrT?2b6BwHaC+=tZm*;+ULUiP8hzGjM5xa9(0wyFpH;vNj()0(`Y@XQac(r(#7o1^K5?SB$$xb#dbJ$F{=mE3{i{j*$FV>nS$mriKx{`AIxIdU)CYUzPQJ%4J*-{^_q9jh=+(9ooJU`2g_7(*N8igqX5|4hWVvTy~08YhD|)en09uR<F@Xs!OTjkQcxbd*>HD|^JK-9D~>*i2pU!ISD2<Qu-6r#x<s*3$%4$fEDIXN%z?GYh6QS380cSD0X_owkDWnK4871#%xPZrmRxipz^TIuz$?0k=$fnB$}EvT@z|Y9`DEBL+T(94pF}rqwK|TxMy*V$i2{sj4Ky648dbe>5-MG-hCov{R&IIrartOiXe0qzxh$B}0aMP1Nt-Qiz*tV66bN3M``mFT5^mEza}8kOuj)zw8p8Taq#>)xCga3nMj8}|w-jNFJyto?b-$ggR+rXYvnr;GS9}Ku<Vehsvbd75N1V|#)>B$`HJXM6Tc4*FEGbY2QmMcZv?vw{*gsa-Wxy$-Aj8B<r*%#NK9BVh=48|iP4YBNY-`%JxK3+Xg3(cRT8DC~Y(C6PXU)@(bJXe+RaX?;VuleETE_Ise%55OW=gZ)9gIhjqH6`IQiACu<@y}rme?L{stz^t-PGEI3YLY!dXctFnm}PMwC+FGAWu6}skJpmw2=O?Nbnrxd*i(*V=Acd)75QaQ{hB&acC~%L6*@r9^@Jj=9&#Zh6%gxGi&iaep_aimI>W+`p{V?`lwiB7!PtS6Zl-#E^(?ZP@gjl&VA`Ho7D7}cUv9+J?`NR+UOP>4=Tu?co#d+8AS$OkA+S@7=8?lqO=Mh4@I1@z>yX3=L!|@EYYf~GoXJzS5AdHHf^m~2F*ggl(h~rr@HLGc#v%$6qyEU-SUQkSu3Z(5);=6R!n8bkbHzXbR2G@Kl-U~oFsABE9Z?D5j>dSnx9+%4WvEp?KE0(mmSot8-rY|sY*#FuB|i`Sq`7p9RB+#6PP8G>-4vG!=b!Logn+q1(+lW;Meu0U3-q47^a9FwIs}Ff8a0H&VV3hja94zPAu^<;Da0WP*a2u;enPO%c+B3i_XO<fXtAiG<vJt5D!?4mmT&s;h^%6Mxcdg-ag&MMM|BusaOwY%(uGa^TMiQ+dTKx7KLCnu)dMLk``dbftfHtlUt&&{?@GXoQ2|ZP7EzdwIV?&Vc0$tlwzx?_|DhXho73m2-9s@;>{I~rtZD5Kx1r2eQVWFf(4Zt%3ED7u}M(UvG-h>i*nmn=Maz5vC3)qgI)O?9$q)mbgTu>=}0pED=9m%@#iodyoJD{cH4H-D#L5q(bOSM3PhD?T4WOf9}Hn}YEZx{G}6O_9Njp<`xm%x2h_xM(1&{<w4>%IYALS?1B@Y3u60Jeg99K<gS-m(GKFAiAcTXzwKn%uf8;%{LU_UBzd(6Cz}N{wd~I#CHV1^oln?2P+UNUmTC-xo7o}OvD*QX|ed1FJn7K+WPD%$}Q0P$_Go^l-cFsYu2Spj#_(UT!V=5?@h$9Pzkq=^zo%(ODpou~}rYyPwyAV&3pqN4Tg}El@tQ5;<9?x4FP6v+NG=7Hgla6I#Y59{G3&iQHMPYS`l&Qzy|LX-y!g7xIU$Xk4#Y`;LPCvf1WED}Q5ENNjI`b*V!NF7B6&MV5TT-MR(2%s(iSEn6sA#EE)1ky*L(KQC3Jb!gCC<(!TO(!YD_Y8N;#*Y)p$8$Vv3i*9kj|_5NsAe^&5=0v2AW>H-Ls75Y?-GvVkEE)5uWHLKh-V|tI*LF<AlxH+=Dn`BTv~73gR@wd3?Cg?`_4<C3=wM==e>e7w*O*QTj9ZN|dIDtL$ho{eH4Pya4J#*tv350%3=DbSO9J-LrK2k^rW@BbXBDMuU=3<JKIf^CJq<^4t`AnLaA08ZQa_Bl3vnBPQ-?aT`(hHx3a#QCizMd%)>$J*z<zMm*!>igw;4J(7}@Wx$b@fIoJ_(V){Gb>dBE8-LLy@)RVZwp-I8tgy_J?lcAH(hR&?7?%=w&+zM+0z8n`Pq7rIx}zaK-3|a6S~`R4L%eeU&$5Z3wre(t-eY#A(5lXl!m&Zo$cBB9;n}sVPPCuKnhOBUVT7`Pd3w5GvZEd}s^MRHgJPB7^%8cJ4l7D7!mP$y9cl@aeDsr;@QzzaBiaXSS-@JE;!1p5pr%5Jw+^U!C2xXW3*+7B0#1Bc$~^>EuUrQT_vq*#j3_^2eyO_|wOW`9;5tW%y&-{Ty=u_uNyA=N=B95B)WqA2yK#bPAuwS8xk~I$<6g`9)ai5^jD703j08r$f~-#<vok*7Sim~KvY4>!wC2?mya;odE}Kna30$!m1hD)Yc&A|f6Z(;TF&+(Ab@9M{)EPGh2gKElQP^vzXZQ8H!y;rLJ8^-LTENInFh9Suhetn4?Tw1U_8}lSKp>yfhm2w{SZT4b?E+5+4i|g*U_%?F#nK0>m>f_nCLgRu&B4XC&3CT5&6FYM7n~_?TsXVAfcL@u(c!>Pc?8{g>T~wZYQ!Ph-D>p4x|u@b81%wvFI+R-H)OfwBS!27D_U2@f|{L<ADqT><ohh)!zR-~9?7`0kD0ceu(v&?t1Wm|79=41$=Y2}8_efzR-t0t49r<WZHD+lePI@F#~?f9t$7Tt!qe(^YZH@bfz{ULuJ*wuihB+Ko$vtoqDwAFSD-;V3Mt2oT@$~`WssHA9kRS!I)$AFK3sD};+8#SvKfq5w)iZoWi6;s;4G)M8gn>55n}dy=C!k@wTT+Hx14xA=L*nB&p;AtqKD+B&!G1FJO#@+n$4niBjQUs?lu^!yl--C#RUbru?(EVwhXVt7*_8)(GI*)@Br5GE*XXfJf&jbCQK!dC;JKOD*(1M72h?;`S@CM0@f?%oRG3d2y#O-Xa!v}Ctbm6`AnL-hA6-aml6%4Q)($Vxuur9BH{f_ic;yj8n8(&1h&4iRWj*UJj&S=C_CWBD(M{cYy}=LE2rI;xg=}aW)z#lp+Yc~NUB5XI!lY<9!rf#^fdEcDll?Z3z*snsU{9Tv4vaA{n{SC#%{4PS-s6TOn5Da+dIdQNHE_Y<}~U=y=`<DlF7PCr)|~iE;T>f08`1eSn@9*>IyH3SCsL55m|+Mt!O_gsKC}KL<kCAaRo<mkm|tZgY2PrZz&5sny3Q0)=e0G)5xJrmnJ%nauh&;LQa%J2fw4-DGF;~EW&bp;E1;L=m{K=ev+gV6HWz=?)iSdlZ`SbP$%33C6cIPR8Ps2Q=PKb(M7CQ{ql7)dr%q$eV8YLl41dT-f?y0a&MGfT^lKw%si2kehRWhghs}ih)U$Rh_Q>H4T;4UV=`47iH*ps#AYON-g@qUizR<riV3=77Q&+spG?s=VfAh@HT%<Uhf4m5?>M$ZBKsH9dWlapE7_|d5SOYKDP(2D)90`PluL-EnDn4nE)>!65|CI?w2ZRw(?sey+J40DTtmpPZloQI!e*P7MtB^fEU;N(Rc&Dz@Mrn)MGw5ab~KE<#o8i;l_Y6?658o>1V2^d-WD$#Z}<AcEc)ww&nblj6dw}2{fl26hsmZWogij#U^yUVIS*#yFlxCXJP*7OlS9!&^U`ao-3X}?l%tc*e2xeE&Lz!=B`>RGgVXzHat9?u6P*y_$=IZ91IDz^;6H>^n*AJN*oC;kh7*%VS?)5hrV`l`&{}j63`L{~IE;1iu5x}1po#Y~CvFzdmIn4H7>7L2&z`&FiTrG??9rL)Lpr~F%<WQP2VxVRHl28jx5H}aB+aCrW9^ubzZi@x=Caomdzgqw%E*^tFMF|BvV;yi?Mbg)Qu7u)N==X3g?RtG*X%?>GMM1)#i*DqppN8=%*98Q_mQyGQXhxvQDd7GH7dm=if=4B6)pFo*@&PW#TGf5#bopib}J;g>V0DsbeM>h>{eDCOWAr>(^eJ`_p&0g?U>88xzwe5P;_UhxiRG{Yhj@1_JN|m2N^VKMZ0m+;k-ea#&73Rpu0~v=!m87T(>xD`)TJf&iv`R?$Zps&or)Fx>jd61m35g)gE0(kf{&#ybeNJSDv}>>Bh~=e|w3-GrR`!b=+aW^EuZzb2CU8HhB_)?<VFM_Et?dBe?mz&=}`#Dl3z}Sd(OIt(~|;<YVvR2wRI8i|Sph1ay{a(Q#RT(nR`D8}&J0kbvp^s@e_Sg0n=%`F*?fN-Z8mUB6PapbO`a?wWIPolvIm((VQ&W=ZOnKI7yqUdruysb?rLf%D9X=w3bjnRU@P`&>_d1_d0NPn*<H=Pp;tRBpHrP9?H*%lR3-FO6Hqhez%9-4IRDY^i$dnT@z-*(vS5F1$~QuKaN4bSm_V4KjWEdD@D4RgHRPgnbQ!dIY+|bMe5}G_oL9$8sJfk!`Xct1E%Gf+y7m@b4b}yN==0Sl@SSL{OGPD7puKSMs3q2>-Z;(q?V0*&giOuAXi04Q@9vf`*l=$foeh+5xui-v6j%&r`0g@q=(2Zmx33v?O`3AsqSIcOeN)GqI2NdDra&2oB(t0^Lr65vfqk0wn;rcHHLef(sze@p?^HaemWg$;Rc-0>G+O3Hb;Wn{@@EM(;<nt3VB+ZhoiPG}gK<x4#Mr$Wd{MICG21&$cCsNTY74>0l~00Yrx$bz)ai0Gj5AA^;gwU6j$zL4y*}BPE}I!6D}OZuN>A1HAA+Ythh%`CI#NSeD4niU}=hx1P>lm!=+MHn^TK!&TbU#)B57Ae#iAtPepp5WEe-3B(##mpb5H(mG+1?#f|QtlJp#^BJseJO|m%xVmbL<Z8op{V^zZfYlxE%xd%V>d2X=Uv!5o_p?s5xHC-9kRA*bLBw~je2ea@rp;}oZyAL{JWgEai%ZohI8S-kp1E~t#oKL+>aG38XrVQ#?>CmJ`;A55@jkn*)RO^@Xu!LqX**0Bi~!aeaaOxYEv1xib-Bt>UggkG+OuQSX3qrhMzlYGwNlmo(gi?3kt=~YBU|yvUbh`Yl8_KXyXkpr@a$N9c{_|8SmLTCe@oS3vZoT?fma|1T%eK}dE?v+X}Iw6BqZ-x(_~8B@zP<Iwu48T1uRM!K8aUeLLunKus=BPjYb2@cNuu_e%xetwaoRfgLCdwrWCOcWUCYDK<=RBt_??Bds3jN2gx#rN}Sr%hrjG-snn)eSBi@jT%u|12-mpJ(6s1{PkU{;E5Un$PTz2ZES(+Lfqz<!*Inx)jN_1!1UkOCo1h(ve4q?p<I+DlEh*5`)t(EzXd5q8;T|}shqbE=>v9n2{Cs!2L05RD#*lG`=Oi|epetT|ikA}UIIiT}s6_V#@rh<K)v92D70C;a8lcy6KpXreESl#F?sr;SdxR4Q?^BY5Dz3$|)I%Lc(-Kvbi6tAOu1h<vO{QUWsgeWig}jd`>C9otskAwaLNm#T%QxrbYFj25BcX?*U@DMsh(nIog9!mcNkw*cnJ0~980imCy0B8(eS_MKcSbs5%^8kHx|wB0En)-=S!JmZYl%DVa>1qgd{(iY=z#gsTxE>}Plm9APBY?Q?~yjLYG5~qGoD|+rcjpuR(dUcB7(~9VW?y*npSsk$;UazQgmQ_1_kDE3fXy&Dl%1C#GwzRLn=8z9K~hWSF&rYY8<nQVbowFiWl&>zK;21MTJKl6(X~OtF@F>q&o#x4Y@T_$&VKh#B*)s<Z-S|boH}i&>!`5v?|3B2ss0r4~w&{phr5yR!zGi@t#&QuNN6ir4`coDvjk~rVrZl=C;zDF|ccMBF2pE`uWCpFpcR_GwIW}mW8!BW85u}#&;cLr{CKyfT=P^E^y5#oU|3;r^v-#xDv|4wq#0H16ILOt;kx5dpI=#m^YL!IJI$;r2<=Sr}h4d?=<xr+rzMBM{G8lcv7vwJSjxbyckZrz&a&LvZ5SzD-pFRBjAD(Yh`|zr7<IB92c53`uVktVbAP!rZzE~o%z`~63wb>bJ#kv^4c688=EtDli7b6hznkmS3PF<wQi52;eqAd(rIH(H&%yST&9@JYsBThvgK7C1j8Aq$s((sHf`$0b5K(Asb@+h1AxJab>lg(QvK27H?cI&bQ0GjATaQZw)1PglP`lB?3N;^P%}O_g<obFRqaP_GBXPbG=EkoNRK@yB{YBAIqp!#9eekVo$ZXRT1)K?3XFo55lCVpmR;Y(PG7G1%Pergk&TcBw?bxNjPPrt%h)1f+#gy~#8lh4f@+xz<e_Qww;JM7;`K6$Hit&U&nRycb(IwTFuhL=^@Axt$oI$eAT+b&yz{lY%GFt9OpJmvIj+qhwNbj~++V5$j<?H|!0jBfPL4UtX(ygQ)XFpF1!tlPVn|YXLGCP;hQL>`4_qh~g?q@MP2W@;32xAPBwJT1lP76)alKslu-aDMR;?v9CF5(TR7%Da>+`e0CMai5qj!~pd5}HR3*d;!pcl)UGFGW6<wArv-hXVQf@AFw1Yb2pI0hmZ>;QwXoHc&Ad_)x|a3++hUF1<%n#MzL+~)kO4Q*>8TaaDBjb*(WsT@kdDj`CuhQ$LW^B}oh83sJVjZq9$6GT~bwF*u$aAjOD(|TXBM$Ob3OAlEAAx@k)vvcAAQyrfVXSN;|_0*DjlCIn_&yo$4_bNF~4;6c^%7fP@PZ`*M`rfMP50Cg5w5pS|^1#lH9CI-L<bnK02;!eIfPcE+{h0*rmkQc14A?JEgGmDQj~bdkd04(3i|PvGq|!B7s4(@vdES;jtQ_r#0J5hOZizWK*&%^HIUjdc1gGNNHdL=ko!is;wf~G<+E)6$hsviNMl<qh2QvB+R-cuUsUiaPY;CTux<rbH!XTGFMSr#&K<>43QsKnu{iopcK8nZN<Pw35`qEVi2Rt4<AUAoIm)uGbIEAA;Kiyv?`^3M~;cQUGtp0T1eGbE9wN8?2duhe=ZDU;Q(|7}-yQ@a&Jt+OT(xvl6ujmVl0Ik^b#`4qS5y5uJ)Cea&@)IBVL+>MJ^Ap7Zg<7Hs%cdz?1K|-DIZJk~8wPR;2BDyRj8!Wd5L#t%xtv!Tt!Q^p^V7ZZhygRV)sz+(5NKfo!gOB8qGGM)y1dmR=xu;2?nE#Qu~h(u=Z0OfZQcETnKFK<AEw%>P>?`lN_*jQ=VpsORy{{6&Gf(<>w(*^Zxkx@>&sJ1mN$4+d|I#M;#6)|z&ckr@pjI~V2kKv7vp3XBX1WYjB>9f2u0@cX|^^Lft0uLke3;Y?%{CTx)bjRDEXv{c0ugSR;Bl4j%yR=Bi{fiyOmL9v*cv+qj>WJnUFb>F>%uV$d{9`MmH`+qiC4;l?rqIniVPWQ(;Xim7{ZM&**sO*{0GagglejR<g3G^1CVcnDO&ID*U6>R4bRsJ!9spaQb~_pLgc?a?U)Ud^06;&6wmZpJ67Icl!2FR&Ek=EvRHm{;Bh(%qY2u)wNUJ;kSNp_oY_=3`@Rr@AWU;`~BO8@BYHM&%}v7)%KFaU59m)&7=pHaK+j~a-5b{ww3T=p1HtTD9=hLn{XsoYcEa!^MZM|_u-G;xc|XxAAa@a!{7YJ{SSV4_l=(&{^aX-zxl?!SH5!R-LIT%jm=_fY{H$bBk!dBX>N_NX<cS(jCph>IN?3*R+Q~t$hPk#N5!KCC5D+~bIRF>xe<1JdtPjAWN&<-y5ggwy)wZ*l^CTz@6D=|w6k|ME5qjYIL~HR2?ze;y<cQ*0|m6Vld}^POtA68+~wqO6}vZ-@g!@wrF;+=JLsikKlDxbf%C$w`Mle%D&<L<v$K@4TcEL?MS-L4GUY(hcaYM8xiiC2_`z>DeWZe5%B?4bsVCihGOGR=9GI{txO2-w`Cfbq;aK)S9?UMr%suzVvdu6v@A}7elcC(MK#|toXl`wn*W6vIdR~-QP^`~7bm0^I_W0M8k5-d<$hC=&$AUiTvJ5l5ahwtv;m=qqD=2GTpjpUg1@=QDqcgW@c@-Xnl7GbP0c<-es!*wI(}}k!r`u~i&XZ#$@hG(kKFdwhjH@s~Xec||G5PqHAJft}X}RuK_)SV6vucZKF`OpDsMjBM8{N2<7F*0LM%xWuG8)o#-t5jb-SLiv=_Xj{k#gxDy8@9K2v|E56ScY+cuVX0jg1Haxz+$mjGQ#XPL$gpmTbr&S0=o`(1dk_8yOW)1`A0rDKF&$u$XI-yD-fjIoNKZZn2SyyW8gIcehK76Qxg`%`hpTjyOsT()5Fb^^ZJ8N_UPrL^R~ul*Mp>PA5_(t)>(2H<S%S)oXsrJN(Um-2K5@ciwsN&O5(8eC^kV|N0w;z)8b4J8>@oI1wGf40NFth&!yHImeY;8Zh&c5Bqix_r)I0UELOKCVI*(9MvT)ZMqSjehLhvL4P>%2CdEifq&JKXw<-T+mT;{F$1nE2|N~cgeJ@AcB|{BQK%`tm@jt}cIuS<QogMe+?pmz$^noOl&EPUp+#z$0_yY4*^zfvljTs6QfH66C+hl#C1(<CU1T@By74BXiWDBAhI~;n#j)Kk2%j>I*~NAna_6-n_FC~4LYHqCXo>W6dF17(F!3Yr<}FJ(s!y^<ouQW1vz$`dAcT%tHywIKRmx}YJHo9Ow(eH2cWf(?1)JMb>>y9EM?I5`XIqo2>yHH$$jC9CT~(Z#nS0vRjaeAv)o}4Z-p4Y)^tgiDuesn%7RdTgfRoM?Ss%)D%x06q64_&^>uyfTSl8ubkW)Tu95cR^0A-J2*2g?xt}%4H&&S6eLC5<NU2>0KUV%)m5YyH*Umg3aOtr(mtGKdZ9GId2fQ|sl(Q3R7tg#z)`ptNBP;kMH0)N~yW6nt7z^hq(V{ZXkU#>vo%p+W}JtzjO5)PEi#o6?q08-g54Y4{OqYq=cAogSS_I$(v>67_OCP}e0p<CrRoFTJxX3WL`F+0CVk)8V>{2>pUCu1h=hne|Ajy6KG@P&L-Zj6&7CHzmAB++mr*S6ZY7XwVcBAo$v1Ju$(5Q{GgRV-(e7I_b^CW^P$1I|g`nJAQADoryFn@7ke(efpt<xfK3Kkgy$&LH=h#K0>aI&l-J41#xzyImFj&Yq>#Md-Tb=AA`tz?_P*%+dIEYTliQfAe{GQYXx8d@o1wmK?=hGg}ACBvYZpbjY6)uIOQjwPTJLJX$<i901kF38#=$tFvbi7WvbdKofYsX5#faf~)EXj<T70w;qbQgGgsgos>Dt&zuyT@-*Y6hZj@46R6$uYw-Ay1W>^o<N;h5bYo&;xTD_iyyx$d3sEiRy3>?!U_BOoyI~jWvPft=gS-<(1N@Cg<nlmB)O0}Xb$M{C19i`rs4b+wc^i*<OHuWcl}fGI3A+Q|v6q6UX6iuvN)8#BV!=Vqds+LP<;0<`u&BKBs8OAb4a1<SmUQ8owSTJVMy}nStcu6LL$jb+qdd_XN2Xds9eLXY&wNdIqdoC9UTxm&cR`MTo!#G!hF;j|AUlJQ#cPHGZ`2+q6pjkEcjv(C_u#K&Jiw5I#M?}d6M+ry?Zu<E*9&*!Q8w8u(IA|bS$)c!#g&2zj_K;AXBK9;@@hr`rU2pcfb!U47{*EDUF(l7V{kRz=^V8#4Tt@qzqK~^ywUd;#C7*H%7Y=4WtYcKCd`g5$2cQ9q?fC&z|?*3!jn(g=BoSnF@Sa-USOv>xf?G3=tAK+V-?l}hZSr<wq{rU>Sv7jDRX$?BRQ|&AXMfdJM-4%N?BJ(Ryx8-4^K>TJWcIU=2<6=J<1>Yv3r=+f?e_>JpJ@Bii4l+ZMnyEe@DjrDYY+ZGe&+jI*J|jwnuFW_f|i(`VDq!XT{s>_d75tEE{MPbq7e;@i2QI3y;u>*)2lK)m<1(5b-d>MSF7UJ0q*>hHO;t+q36sT6V7;!{SEGJfU{p;BC86P1fYzX*ccGP!8nOX&_YzpdheOyc3ncNVDwI{7LMV6Vn9er%6G)vtl6={EoGJF`#SI4FK*N8F^f&QMM-Xez|IpWUwk2`N=XOQb5I_u8Y+)b8%yVBxWljhC~p<q$>s6;`+*HgH#YL+R?NsJ{Sc93sGEh?bUK!2^dP6tAf3n7i=p&Z>z_--BWPE#?iM?3cYFhmq{Bwwe;s?{<Q3MwX8R7S})oZ-m{~4&1T>&bLAb<on=`iCi9g|kcOy`nbgXT&nAc=28s2z<57~q)}3yn9fd7F(@SShy>vZwjn}ZhiEZguQYHPZ5$3#B@mWw;U13MMOf7v`<QGhhJEOSQ@5W)rrzY9)35HivU#ip+=t4vn36zbBcQg|Yw+*yaL_bEdT9v;UM);~yJ}?eQD4$mGILM=TB2BD?%103@xz0P`XsR-!B_l_!Kvnb2qt;2~Ji|5nn+=Clu#6k0w=|_u_$^Rrq;nz)$MG-|UB^77d9#=^tG?u*>E<vBcU&O>Y06Vu5rih~T&LVLVU!2ev$^B1#u(V4!K<XqXv1B%*Y#6WKEeSZd&g<!NVftSI?Z;NYu^x&;^!j$)TegUkd?z<-hX0!(h7q(=?+E@&bH9zuAZWn1a?MtT7B;$PU0R67pf)3g-fltIkLn94#!U5?cxY(<>`t(XlTXlAmv7M(hxW4Rx(|Wnsa0aCD3k~E4QLK%N&5SJy#ZRB9__(trabV5#;z*-09?I2uuxSUtGYq6>sC_R)hb<g@%Wvb;1J}7soA1bZ&Ru;7xw168oSb7PeZP6J2*K;VAKp;coLTFA2I>_hs&rcqA_K%31`O<D@P?Wfeo{b<vE|^kMDPSLxq%XOZ+$qI%9Obq;(zbH2OQ8I|QtvdxU^O~AQ~Pi8#sNN`=FJE(=@QNIZ+i5ZH=tq?O4N4;>f6SbHss=1jqNti4>?!?AsY~q5<TYF9I9aLPiGZh!t8oOUpFeH<=7B`mQ#1dS{f!9~rk98A-Tu18ZisI!-n{1{t(_~M|Y^51(A|^W)2AhPr$Wu8DceNtSX`b)h))-0K7_r#S+zt8j87=$d5znQW;Y&V(fuCwh_4KE!pwye{@|W{maH>1iRcq1Z@KIFfx}E>f*qU)0^_KL?SyQC5ND7V`<5=II_v%WpxbCdU4S8k`0#;9aA?`MyA73~N4;9qU^-sj${G)kno-6JQB)oql@E#!xKkjWWY{J6YSrCtn@hI*jwZXw7E8ZigybJxo!7$!#k37Fw@s<~tmqCYZMxf5ZoVeU;*2q>ti#^-|?J0&%B{gWlQS*#<{s2q?R71ThqZZX%pbM_-$~UiH{Ojr!;Qw9{RWE}wHNx8hhAZBM3*Py^sb0I_Jw~U^Pf_VxZ7`m;(B<6@6K^w$dLDF#?icVj52)FTMa_aWMMzc`&s3KeYZCS|(1*Aeb{PCUfIo~nQJ4VeG1#I*4Woe1(RSo*jXNE%^r;789fU=3H;g+tM?FyV+oRDSSy@=vhC1U-&?UPITb#DTD2YOtw+k2JTML_={^mkAjC%`LE?>BG?dGN0Xn$1GdoTM-XvP+nh!V`CNT5_>48pTBbi+|wdjVSP;J|~<_6F%ok|gj*hZ*TXWOwNG<ZGdzMR-GGa}@8=j7dbva~EzcKY5L{b;_$w`G+bmU-ZDPiK@*${^JQt+*40-obql&+#^fs@hJ}ox-tdjX?ozaI4a(3qi)n#2AeqU0q)9?5VlbQ_Hp2C591bvrUYKI!=)3C&g0%|wv;J1<x&;A3$iy5+ZhTJwiv)CS_slR+NsFiG__crZj`j~iPhTUn71pqNVkFV-k!JJk8pYplaW=MIT7$V@WTB#(HsN382%%#-dJ4@yd{l1@~0G|>e-tNQa6xU8-%w(bO8jzY6Rg)#;H{I&U#wNl-eTn3QWN!N_R|4vI@`a9Mh@_A@F3o>y;n{u&&ZU5#8#69QOYlM#|}(z?z?HlApc5^p?@bw5a-J#tBN)&X(8TQlhHx2s67<hL1Hb53q>+5UHhdNMR1eWri6nXv8!A)AOGTyo(Ex)e!97^kA19U{lDB@uF@7tXVM{OO+IaF)-|cdwocj$_@#9mLsw@NB@(ku`(+{A&~TE-ZD_4rNm3we#HyPyNm-SzS4MYMtsOxD;@>B`Bwwc5ollqpVO{ssN``9RqsaIp(a1fanOax<BwIenhSLWHk7X!aF=TEc4-m*%+IrTI2JIHfijfPS@g{I6XlK`Htt$K8JgmrpS4H2L92SUi#i{TMZg*r6^UH}tOpAyn24=$bEI((jxv{HaSxrK6{Vt5tV!4rqa}`Fh3uGMxETkeu%v*J!;wc^VsG*DftUCxLw!YkI2P1q2RPKi)bI6bS7F3Dk$Zw&Vh0{EHZL41>8Iiz%$D7#3F-~dbAZVAMofjNx!_Rx1eWNl!Q<_eL#ctcHDr$OgLH3~zSNh`=(bog2K&XY)FiNq;oz=yI|H_Lc9TGkaCggeM)Yb|Ko<&!epb5%dYf^`b~>%NCu}Fa@>$>Yfw!SSDgoB&Q5KTW&8$yK7*vAe((Fc~c3+Kx($BPFMW02qQp6_S(Gd4rbrLAr0BFOCe=ZCon1QHcj+1`e0uw-{2$P^CQ{bN!a>jcjl)_8Ziyn(ZLRowb_^JhpH%b^(RY9K6evKEgfYw@u$B_NG)dnsE<PT7cZw*7SZ)!J93kItC>1qM3;|p$0+Er4Ttau=JM#Fw5p&DJ_!|u47j)VY}(Rn6)pzl4;zlKst>AzTufQIhc7V$~#sC!}&QT;K1PrCmX?GMaJP(^E|*6DBiY2PTJof_yv)^dFjT_&wMHm18vgt`k$4}MgcSh>?r&HkpI86R#Ff_PIMZs)Bta6TeI*ss!Cb(celqe}NR`O$t{G!SG(?7QM(Loe1{8{;wQYPd>Uhn(H5q(LX@h0!ePN`9kT_du6&HkmFw<`H`37F8}C_3frn0|}y@Xt$?suBb424)%XI8V!9RFezIfEbn^}dLqy_ViE=T097_|ZKP7M#wt03vCzRssHex_RX{FRpX@5H2yCQQ(Xt^nm@XiY@mUoI$*H7V?b%Xz^~!aa$UR!a9Lu0%#fp1O)x_;floZMn7waH%D0jq!OIGRtXl$XT#H|0-6zICOBs7=UBFEPlhML66UP7A7T6tbhtB}cyV$B+%2S`tBRPm~3wHIo~wR+OO-CXju50#=C8g(YD3;oWn!I?w*ZZOGpE|D+EnO2HZv>T7)eMj2rKdDyZgb3Yqe=w^n4P##Y^vNu!ndrURb+K>mTYElI1o|1c;oU=em#tkPlvXh)j}*JW{5E-t5Tg#0H;HfXto{+pYa~A9U54+V43b%go(hl^5g?^_B{mOGjX_UYDx|ag7g^3JliWex_*F8=U&AbDu%-<T8a`iZ(K057qc2NU#FCjif3>_REA|A?TV7R*3&E2;Ay_Qu$+UQ=riF5-@nmWo*VJ(A4&C01ddm$u9GSf}^cY$y-e;gFsQzGDAi?p?)8p-J0EPQTY|Cf_UypN>=-dGJn5)YV%N9|J&yhEX+KiUyTv<@}S{$21Gua>#`#Q*)Bq$@fl_Ot?G8EF=NBdDE{RSHgW}n;}IvAkw!9QM!6VNY38neKo3l6kRs8ah)EJ(0dqiY{&|A&{kdGY#r+-dE#U^=!2VPhv6;(6H2iZHLD{e}fce`{4w<CsMSI`7$PwECk4cN#jXtJ`F><0@)j#fkTf*r-)TU<*jyMo0=~;!R^W9CkqXATml7F^Ow5eQR=ymmUQaA?b16E}s*KO3_WP!KBb%TpvcahQMr5i}2My-ne@68ir`;FZ*Ea99%LFID~^i=b$m9k3yp2w@P+5SfJgQZ^dhUp_VStcig;eD9@6ZY~&iPgI?I}ivT_rl!CGw?A~sSj^Z;*%UPwvW^>#fcPOycQS-6IPd!c(&tNENb@#gin1sIm8J)slTw_CVlSgo1&+F)?4)?1V?lBto@PbAp%tW9K<zFF-uhK{^6hqA)L-HeLULlS69s@V(gK5|wMvemgEid#|Hy9J)Fm9~ZyvrkmHb&V4Rat222t}+HsvN_{-2-Cxhvc<@!X%{zQIm->FrUZhKBM~(wm=uaLg>0;6@eh}2+ZK|FoH&9GgvCr@#M`)*$mLaaR;RTUJOF|AdWiF80ZPz7`?@M7YVBnw_H@@tDiVu-Hb-id3;>Z-mUvWuq)CWhHk+wTB)Ur`~{t|o1cfSp~Jcz4T0@wrmSwj#2m4S$%=2VO9z@^51ogC=G@Vz5R3T-VU4>Hg)Mw9G>d+dNX#0|ZUi@_y2t~dHN)Je;tKBPhTB?r>TbUoZZh@M--vsojli4l^QB-uNa+u_E}dEW)M-0z=~;vTb8vXTHng~NQsJDMcM+K)VvGw~)MhkfJDFRwfid#<W-bjMr;zmdGrb@sg!r?;PW7Oov_a?dq2OK{)CrLKD5MN-ci8b|I*lU4>heX?smmZ{?NgAS3h$WcMlGCO<yu}|UR)~CG^_}D?OIXLh3%eRwi>-Ahm3cWgTlK%0EJuW8p<~}hb9VA0@5+2BO;MXTLw(-!=@hy!3pq7&$UdApDLVF9+!YQuEgR=yFUg&rpSR<WYqAU8uq~^ZF!%;Q!SoU13+$WnG?W)c-=wljX)7EBu6ov;3^)?@}5Q^BOCGjt52T;V=*q()Y3*aTmyRm{F!qVuMg#idzeJzsn47fOHa-FTstz)<bYh<jfQ-Mpi^X3Y}A7#zPUrm+>vS_!Aai*j19Jtgh+XkxGD7&x&!=VGz5)MORP$VbPl|36!r$7K2z%*CY<}iv|z)C3lwn|j3zZ#^PUufw*|`u>XC#O?scz1S^Lx=n4#(V6At&|-CBRRy)bBPEi9j2JX2dbz5Ljj8a!5T?7;+*301_h8!)PKzC?m-oL=6j*^(}7f_Aw99o~R`cpE}f&f)KkwYenjjMwHi9DEM^-S1U9<mdHL->E0ydcOPx%|Ww3$cmVZEc#NLfV4X#?A~-+s>CGx#N(fI;s+3E%S;}}WFf9>yhhv#1Fr3o#T#M5A~6|pWOczU92dg|?N|X?WcTMFwpd$ScH)JaPb(a@=m3#a%#bX=j1+6oYG7WQu+vc5Wk*cJv*1eRY{7`ZA#P}@I1jrCfob4IdNI07QFZZgC%#ZW?f{ZR8MX=|aF5{_KNqILRzm<!e3L#7t>b~KJWfLe+vYNm2`DU(fsp4@LBr>;*mi;Up!$w*wlDB7cPai7%p)|C$Rp8MumjAfXL_9j%O}=}c0n`6J*5q9V8ls-Gb|Yb+T<F=QDYA@Vx47+x=X&c?EcD*f^~rlEJhrm6yw21*q*X;t4K7!rls*fe-JeQ8fd;HH~=@{Yl}`E*bEceD3ALs!K^p>*Dq!4A5*hvcq5zo8yDlI-mXKTVOBdV7pM$}sqrA7=vqF`1s4vKjBEwR?w~(r)+`<+AcQ=}#vu|FOGf)3PE`g2asZWg(8q)sVqi=*G6WQKY!4Mxu=QZ+X5gSUJ5j;J46&ryvaq8%2WGgzB|-++6-j~ilxzQTgAFZoX@g7k%!5uObH<z1-EbHqXT$gr-4M$JY_SVW*Tzr?_=FMooGU003;tq4UOaJ|SUett!=5Y%q(Z##IRhCi)d{=dX^QM;HlEfZpd!XzF1LiP)Q+SzW6=aGH?Vv+oV=b%p3e$_PqIOBN&-yLj0PheyBeb=XG+!{DIvsa+$6TR@^UYu?QTCANodq+0Q!vV2MotPTi)4G5}iTYz`dAPhA(dk7UxUczBixIqNJ%<9l>a*a`Tbbc9K@qYh+CW$3k6#<TgJkJVet|Wf(?%wF{0obMDJ%jfX7u!G*wpo6P9Oa`iS<o{z{<=>(CTN&A<@OfVUdpU|h$XRDj)v2-=a(k_O=M)1ZxMLz_kQ5Ew{6_x@kUDNdm762LfM9gSZvgsCiWXO56C4AT8?QIMm;hMLT$mmf4`X-|Zsf(+2s-`I<u6c69jJ#kGljxiMG3r|f%tXYf??J!A))5~M?Mtmx$hnNXhuwixZVQQgdO5qxNvFT}AiGRnhvV<^lC#I4n+!+FEW&*Tc5m|fv|XQU1|d}t2ez8-1FrrPSqxn$<UnV<ItG!A@MUN>v-6YGU&%IZm?RDp#W$4EE_s4l@oub4#$KkQLn*lIqBxT0BPKXj_%ovS9>s%Aw^TT&ovwB_bvkPl-cv$BQT&7Y2CAZ}oV$>1gSq(NvBg#=K^HB@q!Uxz$6nlPL1&L{Q<1joaj+f-n{GF%aR6r1Q+S7OX0+q^H$Qi-B3>gWl3*=z>0lchrn`gzDeKZ*c6&)u#^-RzJ%XeHL;+jgkIylLr8BR>B*A8XH0pOnRZ)v2JS2}se?V_(d)-OwRMl#Mj6qLXDc6d#F9HHtiPA_+X^+{b!p*n?a)$XzIE;;rjlsdFE!C^DXU}@p#;~@l$@f{U9ZgYNiFIM_OSTh|j1D@HbF&L8={9a%+Ze1Az-LwgLt1RSR6lXPphZ@zuJp;*idwCnYl5MX=sSC5yTi7Dnu1!K@N#dHzoJq+akjTMM;pp=?{g<Qz(atIc)QmhMvZWkruf5TEtnlv&(b!xZba{i5U-qkMd@x>o$Yo7?rozwPRbx}d#6s@H$mP`O#Sq7kiSjhC}BF`^%AMSw=NvGL9wH?PL9+d&*@o5mu8URVyrJkRjsSi{byz7bNI~ct1eR4Rh>Jgf(#E+-MnAVUOsH_#&>flE%91k)@Wl%t-66=c3wgjBBC2xFi<jlgu`9dNA30wQQCq=*kOLaaVG*OG?6bdXXO?Aq(fl=cMUKOY>Ld!@9g0@26v8JGepw7TRxpL=}*%Kq|O~RKF^Hix2BQx#ySOMO)Kk-tBN_(9PeD!T-$BNyNs^moJ@b3g|cR5#(SG?=6to@m{qh1tUg#*p~=ZCe_82y9JAW-&O#xb=jl&pAu-+62N#~k)V5!lm5d7cg>A`Ty)moU!oqr!eVv^AWfpPD5!XxmE;@apJA-|>yn$7f|1t}xf(Nr`DvyqXg<R^eqlEmy2!l$Kx#fg*<5c5`+*ZtHa0;^hrn>k1&I-%jNcn=&Oq@#n7tH!>`c4Qz9Df`jR}M!93hHS|I~JPnTw!8R9rsE;%mggAj<`%hgDCPWOGuceQpsYuVr_J#k;iHsd6jCx9T~j-q8!D)QX^2=ke&~d=o&^Tvk1!1fz5awQzR)-m9h`cq_*M|(35Bo@B@r2le?@{{zhLdF~A^9vtarxDOA8-s;2{<4!kRYcM${ns0bsDY`ewNNu}SFdvc#n;o<p9<5;+Qu25;6{pQE|mdr@O>qH}IgZX-pXx@<srgaB=n#Qetg;KD(3KHe7`4*6hga7!RDVG{1*D%>4M2i}q%ER_y@nk2)a7E&<)P<?y0=P3HfbP3sw*37H-f)}Sm0qvLV%|pwem;(7$;?9o^CYCDxOUEVv5S#6yG5!uD(jn2%T03)P?}cV0nF|WZ#ee`4DAX-{p_EnwNv&nev<`xHj#PVig`Coc6cG}e(=C~J+!G@t*{7qt!m&o0l6;g;bKfXBzPMG&_8cpyIiU5#?hXS*D_HlQd~)e^fU&pxUT}yo7gGcgsHC<HpU9;&F#*xxw15*>#Q6lO43KIo_=>D(CnudEwS7p+5D1^e&9sO6>Xn?;d2(SAOqz9cWhX4^#-jdhRT<jm{^}i0ZpU4+nZ#ctao%&ot?agAM3SYG>Bl;TcX5K4*sA(I_ShwWt!HMgfkP&r))NNyq$H2id&qlQ&|}$jp^igTO)TTFk<8{BECGOp(1n@y~No@!QdPjehFGl-3A9n29!+usNMM^@{j71QJsY<CV{aQu_K3shqxupbrC?%IpZA^4%}ohn>Ev7y0&&`Ge<d<<!`d1T1kC1J-1c6^yHJ5FI>KK?N;O5ty}QV<!7#~XYLf~WXwB+<>zEu(Ydql8`)E9Qo8wz?kj-QTL)$UM(4BwKsj(SIb&L&73ZB27DtN<rPNm`@C*T^52bq1gZ14pw_Hl4u#1p0#$9!CR=lcHGC7RWY)wJ4?Of1+-reyiLYWOh0<(%q%dga;V}n=tYhvJ*P_w~nd^d3e`T**O(TZ<IVvQC~5t|TYt7x4oti&SW8r!$S%o~3;JN@Pk&8)hi#}wq9$GpUJrAfNBXfuVFaCx10v}GRP&m_40a&?hZh!xo$DiTf=v<Q;5+*zuEngO5)K2_*zO1)B`;>==fmvb9|&bzoW3_-UO$TkPli?_1NxbKHi4ZTt|jms8YrOSJwaxZ9<^)zGCQ8W8O$kHH6Ubf}l%$m;bHEyjsKI^x*p<}RtXVK`#$r8?-Rv&wNcmk`*QrQXyz+o7b08z&Fae>ioeDgX8^PVf?yRtQ)?E1v5{oK6;2}HrRBv=<DtLHi_W#7gD&GN<G=6C*}=9w-ac=xE|W>Ql_=tjeBdpNb;s(S#uwer(HqrEH&9AAry1IM)vuDjT{EF^2iTDSXDOot-8)UxHFW#g6uugslxDXSG2M{X7?Me1S^QHZ#Prgq<PgR@rSwzLBEiz7iN_O)Anm3vY7HFJSSR*S}T3gn%#kTdQNR>2qPy6$u>=5lkp?pkn6D%PzrGmGrj9YbW_k(AQ7u0EuKJ?>#1)SQYtYQ>$RiWs`U3PX!{d!`}~={6ldugVTpF(qb3y}WRS$`pprgEMqs)pYC01y?QQtetgxmKVUvS>#e^#!N3`tOcHZv2D(l^VR;XlU?T-?>ZY8;w{;Y?Wi|%OlJq!wQS95JG_`GC1K~(u!5aP66h6p2HT8eg3?a)<~@UcX|{DMKxwLG%&a66G`Rbq8|)V}m_s%jA!>%gR8PVrjyFtV<{=RZpmBQFo9;rp((i3&I^BSK%GN63hB64J+DqmqvC&X8&nqR3{3+Wkn1hwl{()m`UrX(J@0G9I|K-0Pe*5Knum1GzORpTh^p(T+Ud-lQ9t>){UCrW|^&Ch{&}2Ze_{h4>mtlJUgRkCs=NpHwynOHVFWuhR&F83WH1k7!utKFAXd7=Gv=FhrgBot@$2P;W(chT_eH(B#@;g%^@3Pdh(chf}eHZBL4(v|Zff8=S**L;&-{S23-Y(bUQCcyS_%uR%rFg?Mmm4N3U&EAL&9+}WfQ^h1Ohs(w7IyCNHJsyTK~=s<m<0%zkyu&V3E6dzGKS^Ro2-I5ie;M26SS%o;ZG#2i$rjvj0|>f&>YWjltpcl;yFX%sGqlJxCWxkvUA*FosGafiKpDia)O(|Ru(8ZZ~o-N*S>t`oqxRhgSQTU^v3-UUc2+oukXD7jeGCEdHBWG@4Wk!JMaG6op--I<8aW1n_-SEAk^L7Nl54sLP8tI9TVC(2?{+43Oxx5JqZdu)}YWviKx)VQNltSg>j*cKh40<hA~f0;gToTB04=O7NRmSqm7fG(T5^vG~4Z&+giijoMpjU^eF&aY%OK(<(4y-a@x$cOZrY<kfZHF!z6Qews8_m`{xl$D>5I|*XA%Y!d&2T!bLh4)iXMQS@)b}%&jqO5bus(TYH4kF+8$T>UhPm1oeJIfxQnrxVJQU%S8819pRhmVLBUZW6s0^HW1nL>+bgsWfhcyyFdN;{kQ(*{s%w2`{SQ`cfb4gop*i>{Cw~AFX6RKz0uttymjZD7vbgYk_UWT4cj|hMuDmJ_6i*BGr(bzy}m*x!2`m_N|JE@gV#R%>dS{;{_WwLKY`J_|BElOjWti*S@S;g>1W-YA(>T<!B!n&Zas@c(S$%PA-nGGY}%*HF1b9#8qRJ}>YzlWd;>QA`TK@8g&d=s>2Y;-XJ^Xp*x<7}vvcImxb}pc-PxV8JFW(@JG6V`4js!5<Ad2{d~iFB%7wM*Z2pY*Y47m;Z^Hs~`11F>XFZtOAO7e?kYH02S}uXK0-j#399cpXZO&=cW+X4lZhzS@%E*xXb1?aYD=mrgx6sNEZ8E7;ZlkJ@U?-=pPEK8&oVuF!)D@)+U7hqG)cR46Ud^`IY0h84T6WvR9p3l9bNIDyfNuK7_g?mPcG0-{-PhThlOtLuN3<TsBU-1t!7%QP{H?XQ&#ZXQXYTgA5O|-4@74gwug|7mH6hKuugz6X&V`+v3!BjYIqBK4sl02BeFLXbTe#CvMyqvH%T{Yk?{gIDwSU$p>8#Z3K1k53lrZ_&&(K*1b=2Fh>%0y1rQ?tk{zd(T3T7f&=NUZ{K({lHS4~{G&g)p1(QTJ3Z?fxnosDX#6Kez-j2k7yWM*xgoJ{%{olL5D)w9}{6?1wgacdlQ>?0-F-4%k;U|lOOs;Tnagi7g+I>qqi49qakMftk%Y~{O82cfRUz-6YbFA)`yJvjC3PX%ke(tkNDk2;1$BkGgmPUpb8(C_sk<6`fl3~iR^aE);RC=i2_)hgR}ePy!CoJ0`(NoV6i!IqJ5zfi{MLtMlH1@bO;$D&y3>D4W*s;n^7!PMQrO{-3Fa9uHs^Wf*myUYT)07cw}>t&Fy+18EcAm5WvH!bvR2e`)2^{fxpT`X>PcE6tR0+dP#9<}AB#pYA59MeshqE#e&V=))3vZ725VKOS|YfQmZbs0cc`?n^sfy~uO-Mte=hjklPrjewTCDV;vSfiBIlbfvDwhnZl+znaN*j(Ay=2!>03M@3GceAw4pp=<L<fF0W4A5D*O&&sXu~Zu~>d%R3dXhB5oiyVZ%-}|D)(L0QiE?Qqy!*fS;=NbjkRewxSR)6Q;*jwIYki`?ajV(vsPPQJ<(E!iF|`|^aa2D-<744S$eTv>tPdFgEb7;@K4dAULYNkwz%IwR<qPMWzAoikXdJI&WlCW&{d7kq*_AA;tY@d3Ja{OF{z&pl(fB5!-*kfQsNTTFiA(InCH7zs)f|i8P2D(V$d^5oF_~x1y;VJjJJsb{Y{+;JlH83>yb~2Smklm=@4HmDrU;2O8_(VgFVNn~9xA9D<%CV+#G`uRQJuL*RSHY|E66@i<5Y*O4?)f`AEU1N?zmmNIYDNcs>uVzX73(8qFkTefjk>0@&wDoLw%yFG`^*%?m85dA6#D4qlq+Y$0?!0nDkk!=cT(i$r7)y;lwB5$}i)pQImVtYF3N%5d_!_w_Yxtgr9wEHaJd#)lPN<KJGgLCxLM%fpP9JHr;l;Xm7R-VYX|Y6>vA*<_=pEvJ{BP9GngUOyvD#5dbBiIX%0Mp5=~DzRwj}J8`?7xLyAQ+^#3#jwjB!lW@n6KHTw$Zn9bXvL0F&$phT^aA-UaY`5cKh+*Zf6UN1jR@}t1pwEjP_BrcxA25@B<S)?68y*>1#dvSkx@FcHgu_uBc6`_OK#c{h+tI9aBKHNi!>AQ;nAeBVEwrnn7QHO?7?rux&-QtQv}B+MCv^*9)>O_<E$f1_ely%O+(_v=v-CKLkcB-ujC-{Ye__sTUX6__qmsjWMC%#DBVqTNT@ovtZ#5ivF0YK)#flZU%vuNVHnXid&|G)mN#8k!%)5t<3sY%WRlzviXOz)gJ|dNdVQ*WMWT~TA*I0X%_3V@`ED^@+yko){#YfY558-iz1oJF8zzh;9y)k*Ao-(+$=3WN&hk1(px);UU?NLJMUnm8Sw-tkG?hS_hW|SlV%jprs3FMzR@;*98Ufe1+4IR!p!}Lo1Z#fp6!+Td6Op!Zp(hFjWJtvb~=azBT(kZVwj6hrV1hu`?jDXH~n~|5a!(jw7rU{xCylY3mG9)XRxJtD1A((6@mOh(w@|rH2z%|NRR`H4smsXoZ$&#-!8*4G{xj$z+{h*dFa5e}73f(YlkO7Lto`0ddJcQLe8jdnKQ%xvk+9X?<jEJ>dDX8+8H{}lt33BuSIFUqY(NY;z0@RpEm6M=;`ncaC$NoANKwL{(>Vl5zsj^rX%s8N%Ry(KPW&PX}__`<dch4+_8~@DDs{@};=7M?Ph(FFU;tDo^CkQ&8K9G1K8U296$fpY=K6+4awrjaDfjUyKJMr-Prpss^Cr;^)iBtMPt2(xuBZI|`SMAC45+9H9<5C-BkfTG~7AEVmH7bj1ax^Ntw#rHc6-%l>GYQu`LfGbPm7wC?k+QncG#ezPH%X3VqvS}NB~xseOtEP)lZ}%jZ=RHlZnhQ5;7A%}@g_y#Mn(D9-7>os$8bnv$}Nf`p3XSRKE*7LWgKa<!hOfx)XZRSU}Pv7Gu$AaK!8+S8wgMKhZkrnUb&jTjnEjyfDCtuVhbqQSJ&n`?23XK`?5B-uIuTu+j+p5)sX?8wC|Boa<VN4-T~&1zXBZ(hfW8yV+&z34wL$mA#l+Haw8LkVCE1tH7erb-3z<4{tm;gT)Occ*z));8o?s&R~#~OZH~{zwK=SabzFk0Ka95#^9B|`dci+<k!~kM^J}glfdl|i&*Pi2D<D#X6(a<bd2qnkqDDzPX%EG(tZJ>SQ(1!w+tG-v4$8`GQ$#f=iFdROA{}Cq1(9D3t23+{A`pbi+PLzI;>se+8Pdkk4JYu>HJ84$aaA~6D`~gDH8l>GKV@`2RqX5@t8nrM<z$=WWSit@+azpO$S5r0ss|Hf>>y~}@opBe$qzxA3_cM!xlg=tC*HUR>W!OVnm^z`OJ&;IUrs_R|3uF{=Sjv#0`C!GP*6-3HpA_xvyjBwop=B`E%145aPY{A_sA*l0;s@4G!8t!S@D(^mml+PwIlE8@iu6{+fRa84R%Dc=8YmyngOn(#u@MY!6<5lXrx{lwQ6W(EPyQ#qrtJY6?bB~TZ1a!yngYot5@PCjDJ+U+`?67D~^UM-njwFuj+Db5zA58TkSXjJEaqO?J)5+qo{{NfaP1><^eT&v8Yk6-y;;Oi)X6Ki!}-R8Lr_o_<I0<7<D3$0xb_jO*B;S3A{ZJ2jp4b8h1KghhgvG$Plb>H;g-Yb2u!kUVAheBr6LG+fZk`2^L0oq0MNy1+-kCJGEh{>Vh4#u-WNvE_6XJTex!h!li3BFV*zE%ia=_v4ta|l%O5NQK~VXh`WRSaKzRTc-$KHyX?W^UwYgIXI~VIYGHE}?}jKi5>fcvg<H!{UK6DSOVt=N@lD6^km@wpOHuPCJv|#I@aY1T^MK9``*jv@@$7^vgA)V(!tfdhpy^F5qQ3j?7U`Lg&Iub)-+{a|35IBJ!vEM44PeYRWHcgq4EU(;B`_$Fqd<Sl3%%71wJGY+#(K@WJVKyj<N<&X_rPogBH#{D$Qx$Wyj`m2!8${GL1DAmAEL#D)rfBfQIjzj7!BhtkP<K7iMGg+p%To(3W0CpQPc$S5<&B_K`bSj)X|r-1)zoF4$Q~B7^clZ9Ce^EFNwP_TZTp@!V0Dj^c|}j*z>onYrN6b?S6l|6LG0l-Gq5j9fYH1d+G7&^73QrzL<&1^aFNX$gft>JDdvH3;GakejZA|0QcI_Fb0-FPFvjo7924aWW_hu$>Y)td$?04Mh$vC8umL1k-6Bec{k8ThWA3R?Kf#?SfkmEpy#EF%mm6W%w#IA;C>FwLe-{J&sF;yV7+aC`Pk<w-F%P^93cCN$3JPgYc9rsR6?cW(SaBpF<3md6g!Tu2mO9WysfD!UvCtF&}kv_mP6(A;wP3%;<7fUf?u2S{J0j?0&gpdT2*c-R6M!EgjH^(a`69W@7<r|NY2B+f29b-#a3f$YNmVe9I=<g;<1Y$77GF(SGvKtx~98krZC-Ijjrx_5QIZ<vLs3;pL}8K;R{{J*2%KMVau}4=aWfMitzm_#FF|+|H6Hb%&dG>b@$9*cPV0n#B^2V<IBu1zx?uh%Ax4sI>xLE!EL+1KAMkq8J)Q)IakYrP~Frx>+-pPP>luL7@|=I$Da0K)hX&9q~mcmTnyM|16-;R(JzU`fO-QLQyM6QZ<fOy+}d`B$mek7X;v1)IaX|8|1<~Gcx+rGfJ!XaTl@Nool)DdyG7wVhR({_v=_kI^kgQrhOT?4^%>5S$tfNA36Sf9=a+h3c*ezeZAjAQpgQ;i*+~$pU(h$e07SGUmqeW_n3j@uI2;apqEsZIfR1fe7rsoz7yMoFZ)!(^j*g=63Lan1^ZsG7FUtUdE9VnD!?Qu`n8izTY?v_eqG?*6Y;D88L{L8;!ib4h1Cxnk<voDSqcA<K_Jx6r)y^DDiwRKlCE(bIH@r~D7U;C0ZHaJHyK*uucGF$pac4R22+_v96Ot5Z*VbOI2bgf6Tnaz<;Df+n`|26CC~&xts%>mb(=rf8;A8rTA!t5!0dY=;79XVbV)6=IA7|ED0WKOx5E5K44Fi}Cp&)6A$${BE7r;_F?erQNH5{OC_k}t}V-A*`6jlfHB}Xk*UooZP6_rY6rxS8;xg_p{%x@(u4!=~RK5LMtX&gltIEJX{obTc}9yJ}U-L~`HI`=#!<e|~sjJ!ug7gjCulJjo+dgLX&T`w(-YAH__=*sn|i6(Sm$Lj^)ZRUu)q$Gd21OgWM&{sm&9opg!XH}dxb`^J-oHxEI_R4wQ`Kqbq)E}HtRI6M4bcHKwp{t*ff7)=-tDkluYqibCdJ(%^d*_W^#itK)eT!)4;veg}YWWy-?uIG@>*4@TgM1WGa?S((a~uUo^1AH}_;2~JO?%U<N-CSOH`s<~y#s%d?sQd3(d$=RoWeb91?yPIAR9-CXssP@z~>O*fhnRsEa}(&G#?}fo{nimEni9gfu6Urr<kB?D7?KDk$igx70wwz-4=7@gXXvfn&XzQB>z|fnyT<Nrz+V}(1_J*ThR225?{liLPH@WN~E#`nt<&v$U+`rq^8|6=|p_M7LFIRaBQ~VAaAyxmq2Ec-TB@g+TmRy=ulfvKU#|`&^WLH{b()M!?En)c*!1)We>-eI@KLF?;jk!Huf!RPn2TWw&OS!NEBRA#l=#}1=Y2|<U%MiCc_$s)L6uAwm4hzS{ru}5f?<Vwnt$G%>ZLihEM~4!ob4H5(kHcmcT&v57OaYc%q;`X+!5H4D(iS!r^qb*V;S5`g+ijaGrFkQg)eYGy)QBC~GG`5l-6GC=7_mN+l5%$`e>Rv{HT<1&5I`V^(lp4wN-=Fqmgj#6<=N11TR+vMOnxTv;%yDXXlLJs)4P*#ep%zl6nuCNM93s88L?7JLw<$)o|PsvOo-Ikc*Xc@mvPvWW)K<h{xkV})3KTS0>;bGh(*+>o+w^7}%={4bTpNW06c;?+CX1zg?5H4kw4&O>?lqp~|;ODh=lQD+c3tWm{$Xr9b<JP4fQRe5vZu0m`IrI;15hT*9H;UYa)SW38w1D|>XHyuku>UFS&flzZ<nZWWA1yg}*suh5(YxUgFkTOj4w0-4D$6HfqCw+BI0~@q-moC@9@@&=Q7O;$@ZP$CZ<H*8Mi_AlpN7)n>mM|ol6$#5ptZ<j3Y?cb+JIgQ#zu3TX6NwiUh0&kKT+M@mNHMHk0DMu$9+NN*r0WHY>L~AL-RIBWL(t%l$Edl<2I)%#z5&wDR1g^xE@28910Ig%Gh)7=hfO>kRqA&iDrwy>4|<KUihsr=WN1?ez<;*50?vah?j5TtptXg917G3AbkPrm-$t$}@1C?m%1?mgNs|>m{_i8tFGE_oRRl9rV<%QwX)S4-rH!PlnZnQmY|AA?x|^ks8X0~{Mz`<RQL3<U%!@W<ZHHbK&@j2QSc&_n6ICkJPH5>^oA`+)6G?}OSjt#&8A%gzPz}aX{W+T1=oVWK#xu7V-kG@E7K%h7``%=B;IVI_TgbvvboE%Ot;MAr>Cgi7p8>NuOkSeNO%$MuhcSaOQZYNgf-qB}0&>T^OeWLfMQBuRSypCdjl%^s*p|nvb5v8O9yt9mKFYJ>&={~f`B~5;;!4ABdFXrLuO80Rq%}bO@hr{Ohvo3mqL*-)e|c<85K1uce@5GUnTvb{-Evzyfz8xHM~BD8q&0Tq6!UfB8O$L{FZ{Pc`WhLM4%_hm<*|U$-vWJ&3`s}JYn`W7v%1J`^HlHX_gLEu&R2~mZpXU*8hK~gE&!_eO;*AN5Ir&FqFTU~C~RQw;^GzkO<s1ZhlXN@4aF)l4XsrV4aE){iU~VE;@cZ8Oy&v@mf~HH2$c3s@sq}GhV}HwVYF)AQy62&uW*OI*!y~=0#5c&48n$A+ve3B3hg)<i2LnjJ(nCML(6Tf<7>>-PIzPrjRCJ37zghW>SZ`2BEXRA4>oTNV|G`p3v*ImL!g%N-HTlJ#%IgwZ~>B{)kHyh4hKJXTH|zADP$x<;tLArX5a3#P;RCgr{zxwQSHx#eRs~)_d9NQAvfH__<DlVdUi(RZqu-6#eG^Eec348HJ<8E#HG%|rehA5tjkQ1E$hH_>e9cyTE|^xggmv5yR07Zi#+UE(IT!ktU|+DExAyg6&DI)<1Sre*+HyEhQC^}`X%FmdJYmn4#`L<IeQ4uCFvapWDr7*U$oo*qMN>rxW;5ze%iu*Z>b<FcDqJhb00<8)JH)*ZEF5cr(6cDGN_YIk6LHkjJ%^XyVkGFwp_?Ed3h`;3#`!(y1Yxi*Q8#d1*}oY;Z5DAdDH6y_|om$9fQ_@$0o=pofoz*uAz(S2c-v*0=xR<(}hVB)~-dE>!MC$Km<L%mciq_9rmt4hF<GUx~%m4;^^h-*C#D}K#f1MHsY@5T@?81`3K)^{8}#oSSF-clR99zP+Uaks*r3G^iqL0+l@f)c!PE7Ha^r6y*WcmL}4A6j?W0wu>;ehmYO3j4bhcTM=XU`VhcMOeeEFZ)*e`0|JkLp*nwlQR}*=(wG6l>NeP{Do@>5Srr}s)gJZ9AoI0i$eG{(iGpv?o!%$a*ClQfXo?r6p%Uavv^PQXCIO(h~&<vL?`1Mo$3k|eh9~%Y<rD8DZVv^`X(kNwoCXMVrj>BY$nL6u1a&hO#+qucNY+r7QZz0^bh2KU|R$jl;tor?;EHr=M{b8}IoPJ*pM$+p8e=BF%0+y;MhGjgPJ<NAyk!R2D!v|UD@q>r*!vkaoBmQ%DnjVXvqO_t)_vf_J57O!F@feNYDjvZ@C46^<0;WEnop#%uoBjeAxFqQp!(!SEJJ+s7!3O-V72z(&Pk;X>fAq88`m5l{M<3Q!CMiA|$doVVDNf^itu3q$Kl=H!kNG~{3qJqB?+1Dj-BMe3|Mj_DYU9{qpnmmtKmWu3?adGWEC@H&80a^D@jGvR{m=QsKm5(_z)ODflfVAw{IcaBkR11xHc+SvLw)n>zxLA~{WJLe=Fk3@K%14|Cx89JpZ@Sqpq#HUng?#|&5!@&r@#H*pn)^b0jo`T&_JKXFGt3z27!Jo>cLx#z>~%3!K8$fbXsP~G}}v3#x*qe5||~TSE1*=ao|)X(e7mi=P}BCU}#@|4*z|Bw>ZJWb~>Y{4?_Gic)t}j{fJvt-3+rZVoh_hM#_F}n#FM14AS8+bU`t}_PYjXL&cDIJJxNVdcs@2=+ef@lXeuG!vE1C_q0Qz1GFcHvKVAP(jMF9W5;~lbYdfFfW>gE!c2sr*&A->_i(e-3{9%es)l5qBh6fof+Pddj9~Ti66rCLA-U=@np-~FkMxFGcD4(wgjlLopcrw+#a)(S2T@@u2n+un?6})F5UuQSKA0Uu!2vNBs*ZJ2)+5Y7Tfh)PNur~e@jz=iLZ3CQL702AY!m-8CU5}6Shj+MImUlN(R|HJBc|lgq`f|Upxfm5nZZ3ho8mRbo<w8G`?ERB9T|@!YQT>xJQ2Zqfc_*ce}z+7;S?)~DQm28B2g87EZ2psaN6;8Q9F>eu3_b80$u+Rz<R&FrutY*(G&DgOw!@x0B!%mD;BMc4C>l)?UK=yPmxuo|J6TDb!##V==8U1oQu9tjfjGAHUxgW%);#&0ic@WsG8AeLD^1Kw&N+f6JflRVb?p4k(`&VF+UMwPq*U@sOulRI{#`1Kh65Hbi6<O`5L~dnVAn~iW*T9L}iif&BC0yzh<f;=r=cIkxM}>6`y!pk(DPIM!3l?oJ~XqSGa?2L>JCSQm`hyviye|`26>O^UZ(ygEO(B(JGv%3D0gZ90+5V3W(4D@ISx#7k~EVxBtyg|LPBXHL()#JkxBv4|vh0nq2zL-~5m`*7n+FtF}+5SsZ2wW7US<@9+NZ&;Itmzxj(FLmP`~i-}an+m_u7&twz(`Q``z=bL~3zrOjM-~art|LXHU{+Dn5^`9@xpI-!Tu8?gj0GNOB<A3?)PyhX!zx>^|$-!3`xMiT5-rPkq{RXCOEk-Z{x3&zlf?ffy{^4(a(Ej|7e)VTR{t-NH!mq#o&42z6fAd?uE_FoPT8v{3dTp(UP|MwC)WW0bp8S2rOHZ|;Xg;VOmcJXht>FU5#_+dY$XFn1?u2|@^KrNtnakH|d3(^8MfRk>YpP_7Rex8<wZi-k^KwS9z%4%JQoN4e*T0qKznqVs7UTWf@octdI!e`){BS|ZXCJRUD-TY`YmYZX$zXBO_6kL>FJHFfDZ10(MXsbY2{z>18L12leHseT%;V)C>0_)sTSKDTpifV?9NIYLrGr~8T}{<G`q(L8!U~dr9>m!Rlqo}1x3yKnFL^pOBz^gi#wNKN9Be$B&NKA8u9}am9o=eZ(l@p8)+@Vfsy~}ah<NU`K`!c5Uo6?BXs)U5WQw;Tz?{#2@Gsu{_8$W&{@IUz%xBeB@JdcOdfubdUxVkhKejrnz53Z?7Tlvh7@8z4gKTRVAoT9v{ON!B*?;|a!7J6}YXB(obnD(#l&g-l6ka}^qw%<yFf7m@oxg^@p*AhH-fC@K-;$xZhc{+DP0{m9Kl@LA_U3>8Z8)_4=s$e^TYvlJ2Y>mKAN}YjKmI+QGC0Y8@*_OBfAZtM`{}>>^S}SmAO7qw{**tt)8=z0GBM&;fAr?p{|hh4Pk#6t!K-054t;}ey#{XNKj<L%al=roE`cHyS};OLR*o^F1{=<{o@ZUlrPfi;olr8p%*Rw*Tx7Hcgs{bOCe;e4l$yv3>nr@z|Kp#1{%gPf<~RNyf9!b6kJ3XM`=fNCC=d0ujSL{Xg+DtU>RrxvXB71mvtvWCn4Zcy@;x1dN{}S0##pb8H25LczR;cK%vkmL|Mjb%|M9<r$@}Sl{4?ZiUm<j_iPims|MuoLfB5(R>R;F3nAN?mh0DhSnmP;<K}#(apPp64cgph0I!l`~02oEL@#jvnM9)NR1LG{g-uLo}caujhuD5(?JlK=mLi1(>Z{jI<aQa53Ry?)E3^kSm_;M89)KaowEoe6q#YYA4DHPFjnyDbX2xZ!z!IQuuF9F<3hr`{pe<+EQ4}|TFomHpCXtsW4>Wb;;vNC{Y0;~ogZ6BqB7xQwKuwZF2!4SZ^#YraBICauzSYKRP{d4g3uDhhQDg|pkNVSj-`|}}c7xH2r0u*h$0iC#EHBZ_J$;}gzoYQak7p4L0Y;RrNam#n8e22=DtUc*)`HrW&T|F6Qdoyu4<G!&)8}Ck&Suqi%P9=m+sgq<X6cl*JW9Shkx3%43HY-M=;HeyDXE9SIyZE|<97vOD?&}QaRz3D^P+ah)F3>@!%=#i2UMTxp6a8VBl~8rK#vLvIJDp}}n;OHFT%#rCcK8`5wB@PO5p_CTrz7ihxQ;Vh^DzU>{k<>RSTn}d45^qUV-nUyyD<6u<AY#>*uG$+=4MENH4{^p$S?IhMJ}A(VLI9!q`}D+jC8xb1NeffLVaYsi7EOGUh6`)`6fM7WlBz4TD0Y!Iav5bRXB&~1DtHG!$PQ&B*!@yrMB~(no=+HQZIIzIa`H}#T<=ZZQ_yS5x3b|vY9<<30AI-O9K4LX;bI4Wpmn(IUln$K<c%=@(Zf`f-1kD$}dFa7ozeDS4Cy2D<>8y%83-cXf3E@ksOA9VJ(B4@yHj}v56K>Qf;*btx2JlIgT}BG8QzU^uZQ{Yx*!(gAQ421Hi9ay*1syp)xHTY#v+*jK<f22Jqbre)odjZ7ne4ALSJ5%0P?v&53G}jrM|ev`tB*RKxKR##z%*ytQmwFQ_f9UxV>YZPi_v-|F~Z$nmRfD#yPR%$~WF<M)z_T9M9zu_OMRGX9+E_;bqmb9zSn!M*;zwMWj2KRjrfg)`yLNwo@Z9e>V0eiwfj=xY3{;?GI73!CxBW5Od^^4V*_`^iF^tw;;_Y8>ibGn=H|%;ufFmuum3E~~GId)i)fYx()L4O(hYZ`V#}$$3TIXd|j#HBG5m_O#tJnaDhejiz2V%_WxSY19l>7EPtcT+nMH<F$qn!lhR2M&U>db;1R`bGG_Bqe-Gh-B?J((h)Mlp?FJ`P%tuSSHx;CRJhp2i5`(H9J_+o>%!Fz_PxMbd~BBO|G~fc{P%u<BIAGi8*l#dhj0G#UvCKupjY(gwS=epivQL~w4@|Q!o28;PTm$$F^;&W;gm3GguS;B!}_*2YON{6KGmCBEPtqK*UgUJe!CMix41xah?5bs`moX*A{Ym02~>Vsl_K+3EQz*Id_C8v)3IL|P0%WJfr03ij-1ac9W)ZPO#e(%QPr;BW>j{$V2XEP50TPdG(;+Wd>Mwqf{M`{2iNuKQb^q|?^ak5jv2TR(Jx0RN`=*NPh~b$>GsY-Y|uJGsR-pN(weNfZ<lbN&&ZpNt0Jo2x(0_t)m&{Y9F=I@ti~_v>Iph3dX8jcONtL9$Cc=~jr;B<pWoH%E8KFc<rhMD8mikywf){)4(8=|mfOpjw%B5^nn$pHX0_AL{_@v8|GodJn}cck%qUn!bz4JqwNnQ-U?Pwb&YWy-53`lYQ2*qoBR!X*fH`Yc&Y<O}GA`Eab<`7mGkEV4U?DO3q-f87V&;2BbG{9ReCyz(lH{#<`{(NK2kn-hghLyr=W9cN3!r>WoT|{`*BX#rnfu~IhT#dcJfKN|zo#vIe!JBqca8b<NB662>TDy$y}dGXjdB_+w=IjVu3yciwTt#%8?BvMYiOsz&QcAs!$RBBu>W%GYhcZ2iT|!~lU|gH1WJguYOS?bTE1$dSc_LR<=bKPP5iugv_J!LJK@iZ^K9`p{MGSraco#m6ti-|dLB;+H9M!Q`dPSHt7aujuQjuftq+2q{lEYB&;IT=nZ5tn-~RWX{@(8ZhYv@~N&9v5>J<Mu>EN$U4MUHqDf97s!Lj&{#Y3f^p}s6RVYwCwx@*O2qAVgX3L#LaSzC6R4)S>^SU0t#k+zn#RhM#_QE;z`uA&M&^=eVw_uabk)z6Dljk%3?qF_;!p^F=Wz1e{cGpr^&mOw;gY<Y*y!m214UYP-3pS;)<RsAo3v&JS?T;3Mm+YE4h^kbz<-0BxOX0?4IXfXDzV2Z5n)OfWCUT&QAfHe15(RYqiq)~SdLGWFAah-@(RfGkmtLo89S?Dv8u2WT0^tz@4*jg3{N$@)x8@2u?-lL)V9BD1J+~g;UH?D6+FV)nV&t|jcf>c9ufMejf7u}qyU11<fREOU#vGIFKM0F!?$bfMkZ@tyL^o%GJFORcvXI*;aiEtE`gFC=lvOc`C(YZ>wOur2!gJ)SfA}I^L`Ir-O-%tBl@L=4JVUp5OOlj|*KEC_QYY#b_{n~>81_ay7v*}iFdxEBAYaKEJQc)~UQF^O_RQ$^0d&!-T@7?*<gGV3LCAgo@@}bIiD%bl{k)8E!h7#v&+&_I(%x)j0`4Ej%<+3IZ)9;;f=oa}ZFOwOp*BHphv`kXF27SiFk;a!Q&sD;3AP&qa-#EHZ$h*KLBfn=t6aM_tR<Qm25>9DHgy*@WqAoqZBw(qj@JvD~>bzzkG|DM#=W9k48rS7%nwQyvnWIaF5RWvMlr2xv61flkI_ON&wr`-xURuszVB`d3jrTr(Y8<YyN?)Oo$_*!%bI0;>5@WqzOz%)s^M{|99!ugoXYTcVu>wO?Wp)&rq6NCXw<Nr@yJ8QT%KfF|sb0lhKBGp3sER~c$NMUXy3!JA^1S7iQCmu%9HVZ$gs%QU6Sgih@CKCq3XIuAT$|#`|7_%qeBTK%y<aTk^#t>`wwM5;Jk_H2!UIAJWWC%@D7sma;5O6feJI~Y6h$c=mSGDvXOh5$0^TA)b$4$RaH~l<$@<-d{v=7wN$S}}U;LrU-mWyW-rUn#Drxs*HAi5av#?AvUM58h-1-V(Fi4<TpWcW_b(H8%rovuuz7GqU6NL%*OaVW|hZrNZ@k_PBEi0g&M#DmT9BR)^4L{*$2YHEpI^-E(B<bcRX9e(i1)TP7fnqO6gY900_Pw1rctBo1l#w2)=Hn9XzcWmP5J<*QU$V5{FQ$~sOQgX>yu$?_1nG2tKFY|OW01iO)yy-V0zm;jk#H9}Ro!s@bYB*7N}!GnsGRo4aKR2v^K1y61!X=0!ZEdmNKmCS7^)8Z+Kms_cC#4_9v|0@cRS(1Y&I#k*4Gah$@c)s0pHf$qOG{v*LTdB-F<;9K1|<1HqB>DrM7$alJ<6JN1=bmZl>lO#bVQ9s7)4|=#+8r1jEk4dnvmu`ZUGdX?-VTrGq$-j}djLc&t#*7oFg!=%>4^OACAXc-D)8D}|ufSE6bO0NnPq_N~q9&WZ3{Y+(*7avD%GfNV2CO9b-7!Cl0QjB$57AML_+fntLkbK}CW0y@e{UO_QQQ?%SFQHC@Kc2BjrkpJ~W{LOq&_UPb%g2x9y!20PBcf5u0{<QK72Yq$pmOJFTGW)9>EVo#}EX+Nc^3^ZwxzpKbFVXcU4<9%6>x4>Q*zcz6gz`u&nHZ>ODjyRFT1ZS8#oGqK3d4LpAXH-kH%5`i;mpxKr~u?k!iPzVI2X~G;nV4Jh2yGW0V|<+c~H!OmT9B_j+SADJ3RPkTFfW#^4pkoDNxx-fOPKJ8^8m2*E88LjEce#DTQ8xPhco{@G;UE)kzS3^6_mn!dt4VcCWr&3wmhy+qYXm0p+L1$UuJd?c4J7iG$Bz*E;zkNL0Xw#0Z;Qqe$GqSo9AkMNTvWK4Cl6yNJ<XAEyv!o``NW``Wxi`^-^g16$^L8jP}ZJShqyB0x)Mlv33tmJ!b=!Y)`%>MjoMOWHet69d@=f(s~munlEPCnD%1n^s3qIz7pc;$phLJ{jz-cQ!Y!#qG_`)oU?4ws7r(oQ`X9LYF6R#Ir8&5+|g$+3CeDrAzz3BllpyJ=Fd5Fv%CLN&|lPFk$&{{`^wUgXe(X#dwV~dInVnqCmDAgz6VM^v<R*kuA9y2br@0F+{1G!*ZETc;os_FMQBqKAMk$GJ83P9iO4+&OUJ!uwJ+t0&7Ra|7L({YBS=>nuJ>h^PMK7{s*uqJy`=_8}UZR3l}m!gLFEeyt#@u!)XCz9QRX{`C-szRH;_|vVu`QORVJ3!e}8o1t#1e`qNRV!ox2CZaZ7sxbB4)>gPiMl4gcaVQf5L7{%RFoCZ1oP+mmu2z$G*lbSFMJ+WOZ1Db%u0ucytz9KYY56i^w@E(I#<8EK)*t^v!0At6x5_p_6YNLc5_33zc>KMQbv!fK9;rD``OiWNF0%0j<u(7#jd6palQ*5H_qT!N%cl~(PKv8SOA;yV{iHvdBpPrw!qG$lkz`y~#-9T%={4Rk38}MJe;l%+|0tG;<@?s#7^-1ygo(6@}EgF3xnufi*dEe~U!_W|=K=u*&;k3Fx$W}~7ANPU}hTzes&+cvE+MO125o>##9TPw((#;&3-~pA)PJo>12nN#uvwRpJVi#k)+4X7&6b$Sb8P$F<f}@+8gW4REpUDiNq+`u2u2RF(*!pyjfB|&HsmN-Ai}ZMDVMske)oU$|h~~_9*N)O@j+hPWNAreImT(=r!@PDeCbk_TGYG68;|chOiM)IooWgB3@5{&2bUIcAVX6==e6Dpddv7>QN9iU;eu(&*XY>f5$hB9;ErF}JBXyy1<^(QpaQW?dalLkWz9R`f(FTPnr4(El(8WBP0x7vaVnPVj#I4xD$_I5$j-vvomTZlBjews~<ACKj;p06UQ#cF*K@R|X;hj9*N-QppxRc;YotvPXQ+YbVrD09nAVqEO)G8~Vx^3V&sF#>>7oRj7qV1_849~tk1Z5kj(?g>5gZb19tZ$mN@U0mIHNMeS%=0->s@foGFxB|#kO?*;;uDHh3O?NwJav{A#Rv|DRNfJS`B>u*foasqJhQ@zU{z<jxxpfsMzIlVI#p=8M?x8rAMJ_LtDNueXXK$x-FYV&g(@VhUU7Eu)UMbxZHnXIaXL*$83xNhL0HBVeYcn+zjex2M9ir%DTaKGgax#vxKc^y{2^n~(L^z~3nzKAQ$OT#SR8+mLnf)i+aGe<J7VBXrZa69VeYqW8u`vyhrZel!dXEW_%uBMSi`T^A@o8b2R!2^b0FE+K1Citn#=e6vJzRAn8id;;<VIcLpA-9BxsNy<=SQ(*9tncf<7y+9!}4fJmFa37ihkZlf7y7@@z+iXSS=;?x{J?#ta_^*eRa;r$qjNeZydi8guRg_5iMnUp%zf>nh;l&9NNjhy%%SJ|4he&wr;PZZ{Ck2BKND*gElYx;}$+_-Rc)LHP8u+b#JTF_8%CB9RXE(X+)R+{m&b-Hu;MQX#%*hr}Z|RRAbJE5`8|1FVF3Ev5t_xP+)GYaJitf>}j3b9}ouhXHvfyJEElKx0r+wv=eaI~Lgj-4fL$v6}nIwAf8|^C8d~mX(M$dcEG{batTFs}DZ-AaL}5hDx1v!j=_+R$I&e#NtcVgp{+>Vdm9$VI}j9+corqtp)A#t$=}+ikGz;A1>%o=C-QF6dFaHUT^nB#wzV9vg~kJ4P<1ago)QRzkDTvQgpGUL}0=;MSMKWPSHFe$@j;_G)q!UhDF+=e#X{06U<?zWYncH&5^zjD$aXDGOJ1()@5y1<X0PYaZ^T3>YXmBZ=xnmOn0*rHLFQHC2S$QS)!8ncBKF}S}bVok{~r|5_)z~q#4zS7`yFk%^0c@|B=uvYL=OaAo5<(RaXe6q8bTP*G_lq9v^P=)^~d;9rD`k7<1!Lt*!+i#wjGj6KC|4#XB)CTd+`jn1^0l`;NJ4SFRi$r_+69dv7MSql{&h61S`2up7+VFK6%8A%@+6t~l)mX5v_*psVfdx?9Aq<f<yW?xyiovXzV7pgHp^RrD|#T@U)$)toBU3k-|qwoKlkT9SEVNzAjn>ms0?Ul30>3~`95`G}(eM53eZ)UxUFC^luFSL+D%6|I{_-xB!iT5oI63AIyQjdZ!lWNTo$E`hQu87h#jL9Tq5AVZC--%fj#IK8V8C~q-&N9r21$}Z}1KV6r$ICCLs3(kT3Z5Q&98MHj}ytYy6yKDy~jfke`7BhxpADP4{B}n#)-6m2JQ5vm%c!vb?$b@Hzc`05SSQh^ti;mEm?I=Te0`PY5R*AOpTH$eCHRf07LlK`@#CC~lUQ*$2_0n5)sh?#5U?HmB$s*QQM0~Z+Bgc%4Gjm$I#ZEwKdpURxN6)*+$rfv#57JK2V-IMWPkvdZfR9g~p_!>+BBb+zg4-UEZ@teZ9|aqcGAoUOXLG>mDEQ7;VYEy_iTzlQY|Ix|Rs*UJ0!kDekLxTFD}}eZ7u>${taJa7-+mD%z_dHOgRi3CE(^c!2x{Y2-Ot9E;Gjj#kJOt_@{@erZ15;-$!r_h57ROucK=iQ{ux~69|Pwskw==$x?fwX+E+gL^zpsq!J`MylKbCzbm!TFPai#PZo;<iG%U!McDqOi0}iv542nJ*AE>S2`-YbazIljjx0;`Y;>>_4^EzT^1@_6edYTl|BzuXGgE~oD@rA-PR6b6#$q@EY!dh$#P&LuZ8^J4!`dXMPOYVjRuo+Qfex0vky3S|^%Xa^dKYj2B@Z-}b$=y%y(5=s~z9fo}-1+p;{Rba8rIfFVYLh4#3Z81lSjHr?ss_xD5Aok8pFDj8Wm&pKZ`MAbUQC-^Q7lyXmRk+E#54{$3gkt}VBeigKzb*5zyT@&wHdOJWbo|TL*xkYV`DlY`=D{s=^RVfK^TWpBZN#J;W#Zr4Rkjd537>_f2Dm$H(<WXlRDa*3CfOcM%Sj4)&p_tH_F8WOK)#{bxO|)F#wVwZ5qzC5@M&)q&N;ctr$(dC<Me|mQ4ovh^}s}WxEcph@)j%zMRfN)!F<0>1x#^o<5d=e4KG<lVoKD=S1;n0LmPPC;j){^GmO0ctVC+FEP1Cw_Y<|zVgEP`l7ZLF^=~n1G8|8nZ*}vwFYqxE~G6XsFQ(oHm(}id^CGu@q^J~b_8sW)2QGNg=D?k_DC#n?clC2?Foz<zD!^*Iht*aydfj1ch{WKzGd*=&py5TX%N0B(xSBm*g&i{GWNZEuNUlQ!{Rs!3MJRaYvqS`yf+$VFzr-Km~Db!th-e*@sFYMm^c!^EuS;&^GiM7!uQA&d<@s;U6vxBR`(1bOQe=TyKkRJ5?)M;`93R&4BDu5%0+w1fguiV4~J}|S~)eaK%NZJasCE)xj4ESU%l?M+N!-_RMY1LM7wruRmc)~%QA*xK@;cvrC|eLQTXHkPH~=~0L8QNmZ*~qUD~SKNp?H0Yo|rfeC^KX78L306F33+Vm??(T&3z8dPdREQZ4vm2l0*8Sg_zYEtOdcV5qzyx{*YLMPc0ywl;T(K2@GJkjn}UkfMfST%xsX+9n48jesq^S^}E$FWSO}Y}zGgfvo8zZA3WtuzAmYW!szpD;QKhJ|cx`WD9gE0g%&Ar~wv<<0OJByiF_|o@<)z!>t$Y7rS%xnt`>H-Q$W^pumApV!`o@CErd%RnR6;M}n!^Gu0--NGnlF+=LAr1!t#LA|I4;K+P<!Xpn8?gw#`6^|Cn=9#I0Cp0rolD~51j;j}`BbH0G<{b1^h{VA8&%}$H)fQ49$hPEYmm||;^i0YGsF_oy|d`iAacFkP*y(numj;R=?M@2q>Lyu0R35+EtM;&5!$`P6vDucaNb2d<k@yu0XXhnURTNq`|7LIY159;<IW%k19ae_&n35)A!g^kz(h_3kS!}O(2(X%4I;)0L~!XxW6*oAxgAukFdTRu!Z)^+vNOuX>K*5eiUdIJeVQCeNgAJo^QwL=H5+3IuS-}235V$lTDfLTq+HyLloMY2Cl2Z||muZ1Tj{jk$6!D%BbL@k*v5@WAGx1L)0=v4P476i+dgWo|%9wJb?!=iuaA^J&nGD*pX(juSR#H|Ng5gqG0d>aKGoy!|=@Fi?}V+TVGV%S056<#Z+_octz7M(zEsvlz*_hB7?a`JsRfJx5+>2b6V8(_W*8+N4ozpm>yyY&e$-RqrOotxKM&K*c{1ab;vi-R*u=}#o4Vt-?}4`^Q~P)&<=iC+F|$aah-z;m)5?fIqtWNr{lHU}e|OXstqpO!Pd-LBCG+f$3&<!<61SV82}LuZ}k1P9;ruZ{fPifIm^OoqWF-7saNT}9dZP$8r)Bks8s(WQ0G37Ot65Gtq(D^|LhCiAiQ3co@Jn{N+}aIOc!_@7MN!z|v91GFV=2^01d$2D7<Obcn~VH*2ImM}_w^;sx0xcKjtnqx&j?+AMQx1q2r*MQbhHbPPn+{q*ziLf2_^?BtOsQa2_$w4|q*GLM%UcCuV6dLha{-j{B5PsDCgJEfv-5ZH?V&m+mIQu8taU8>jhK<G4VXeJG0kovfiK6C<Qg#`j14_co{!ta0tzkU7leU3j=b973nuN82$dDs1E+)tKDTGt!5bq`>t&vIjS)zhQ#z3{R?_~+X*tVwiHklQc0m})3$5IhlNtT2c#!K=(MLL2`pnywdG1ZN}Y$y1aLD=J&dPZe?tu0b{aXVaJ*G3-Jl&^Y=t-atc-$zz4L#f70qJ>bhs%uSb?Gt}PD>b9fyWL*1s}{PMlanIN?R-q`ySnv_zI==033v_po<D!jXombJgfK?gNwTHyK>5O_WOjUfkoOPB+EX@wGrX;jm*OzE#M3HM7SdA`7F5I|m>}aZ9U}i&qys!+*ud2Qgm(8!#ys{2&Q8)1DJS8cZwoj$+<^~54@?KC5Z-ZI%k5pvj3QJEin~TL3!JnaGSM8PCpUv*^?caa92sl)S|vwv<ilp<nQiS>;I0p|(ml798lIX}c6KT%*vvV7%F#cDr`6cRXq?GiJXNK4Y}UAxJI50usRrTbv0Po>LaiQME2cS}pkh0E36eNM29>u>J!OjxLMR^8AsZ`*Lj!inCa42;QNfoiPeK0YB^<`kG~P8s{j7V5xip|DS>>>*a#&O4(5fQNCec|Wn^@sI8o||NnhvWINxt}m|6))KVhY8xSI+>2mQ&;P&9FSnY1m9#IU%}Q(PU0KAuqC=+s6+;pkA;-?;&~igf<vaO-E6%o2O-sv}itgOAQZ-<8e*Xo`&2m8=3}gl_h#?m$7Y5K78F-R#oX!r<u@)T92W*7@^TpOER}&My;>sW>B+9md=vHBgvl1XQQ8TE6`~boJy!V+(O&TR<QPg@o~#Xa`Yf4+HC;I94kP%TUx<k22?pbVSW>CLj0}lQlG+!KrqHajk;g(s6(RIQjy@XBqTa+oJ}n}GekX~-J++4Z+uz<ZkI-NN0Us4jTQqEN9p855IAJi6rUS4%y`qUF8g%f7xIKAhzSN8aEl6?gP3MCjD}VLjcc2ABAh9o5f7m?=KT{D;su#Y<+VZB032WyNajIFO1fVu=E0*3&y4ptlu8^x^Bj+CfxFLE@DP|q-c!;nh>QoNbrsaX62VjDT}?;HQ%WhQ2NNovUvkdnMW@J?Zn+b&_!~H&6HWOGn@1{rS3GkT=sQ$rXrm*4TZzt9k@Kl^_nu!eL%p$uL6Q)1BuOj{f?m_)qV=m(<@OW|J*gklJ(h8;NF`gN)rHm1rc-3E*bQ9h4yJ4@oeqb@IHELDDBw5-(PN-iWkxJ4O!SmkP$1Lj|A%h*8ca3)a_BL%0pm`gWg2Vj)f1nWTOhl5VQDQ&FJbep(JrUg^>wS|<;=Xb&KFpFn?)#Sl1iRe4MzuKL4)<~3`=VqUDIw|b?UY7)-ABBATfSJY@dm@sN=KrHn(DEF4+}prdlW$wO7#}t@XP&hxc$()NEB94@Sc27JXmK94}+Lr$Mzo;Xdrgo>rl^CHA)V?zMN1U@8&Zy*<^Ms;l0-ZpDpu6;%4=(ky8O_Z4#o2>#Kw@}e?_D>d&Pk4txS^hI@4)O6G^J&PD_5BuC;h}Fb{dR?n9n}&8I>J@B*C|jiERKvr(p{n6Fs8uGzY+ntLN4E+ywH2U)GL&>&&6GPyeGxVgmjhT8c$d0qEs$CKw#o1tGpU+HC4sPWf|C}H!SNu4qNkO$ldwL+TY|4$|58Ow!0hbo==5s0{9Y6Y;Y{Lk*O*JgIyp(IQZdXLZ7?rA`hLKbHftO%sKI;AtaDUTrye-{F+R$(<4_;3xf>&MGd3xdjNw!d4Lx{nQcLz_O@#h<q5YZ1pznk~uesqf!!=wFW~w3srpN*!Xw*3=!SF7^<|&o9&WQHJE{@U;%+H+)v&5cGv3fz+xhtfCRcH&iwqz%n(id@vT+AF-2!UaCqIwfnGlN=FM}OER+LP1)OC%#b1RF{hg{&4MPzpL=LoTf&YdV%jlXUKTfshR-k`pg9tduIG$})xYOsHehk<zj)SYX?yHQLNG>N`0yH=tkbRa3rFg?W(PIFy>q1o`;T3`{RUgJd3xDmF~_LAILI6+9V-4Sg`)yvBpzDJxm}h-rl3EO)Y12--j_%3wkt3j>V03Xyd=jf^olx@?vr1&8ZLB1j9#S`Lo{BlGpZz55tzR0+m(fNF0j4Fs@i$upd~woGnkoUk>1so8qXM)P4U(rDze69t>CR<KTfaxpm`n55N2DU-J{PztAYTwpsh5yOj3-6*^vW+FmaN*}#69puAX4>#DgdJqyT{GSfL&{ubM>?B;)Y#5L4>pj^BF`uB?NTcU~wtJJ!PW?;+7c4_trNg)^_GV~drx%hsa`;PT1g2jJnim-7xw0`>szY(%vpSq~{a3~)km2gMfP8Yg6A?l!L^3<k5!IzSS~G&*!aF7(>o$j?AM*<1&UpRoX&v{pkfv}j(UtVo3Lw0Oi^?2BCD&d-U%A74y+}r5PO#p9#lFOp>T-Wx%uOSm`UaH5)n=Y_53D9=v>RTieQ}o;_q%&{Y2Jn+W`Qfg)_RHp3zH4<?-L8(M^8c})9r+&tk^FhW@lmYLDcp-=R%1z<6tglmxx(&!4EQ9L&-c{&+NV@g~F`mkaHo}Fwp{Rq0=&$k7vak%_2o~N_3?`yTg2nAzTkAR>85t=CL7))Kc9Xvw!?77-!j_<jfG{d5@f(3_w1a?*b7HZa;p2pxE=kHgoBa*~dvL&0&ydlsXB_j8JBk@sVWqkj*<4;~T}nZK%tX3YF~W6jUckr}Ker$1wC04PR*PIXINkQ!8<t8SBCf9Bs?c&*O4tVVIJc*Nf+`kPE$Xrf-(K+lB2j2~Cc-f~Sg-Nj4a-Bmu#u#m2`eep1{achBAVXu>LtPHaR~yc|@1y<dlKf$$DTQ<!41Hq7?m2ps177|T`2C-Jb1L_n@%G&R99A219`$bbyIOl$1>&(+b+CSubRbZr@>1C@w{Ag#dopo=i_#)9W*zKe$YT22`erIv10GS^2x!{jF7=a$bJ^=?|M-G1;e-Cd`y))cy}V}NeJ;BtMI?ynzV=Nsk}eV857y^Vm!jbP51jBENbVO=1eZin)BC}RG~&|$P!R7uq1wNN22n4>}{4UVU%A|lQwlnc;`ok$#-Gd9c$`kGG?EGl^>k#=j58+rIn<o1gza(gEvdnY7&CnS3(Bzq?$dnY9O#SoHJS+Yq4k|3W8ZEeIG@*Zo5Lf9?HFFvtn2Da$qd-tCqD>x^ra7?V;Ok|S5tYH`@jwqe<n<AD__JtP86l)KLmX^iXjCe+&&h}&$M!sfQHf|YQCd(Yn=IDAycH~uOaquy6#&oqBXGV6a%L>pDh5u8v+Sgf1$HBPCdIeJ&ozBR_1(TVSc$>{ca*?<;`id*mGIyL&S_+y3;9IW8JAViERyt)R!uKllMwteJfDwZvFkBnosOtNZIe&H>pe_d=%gcsf-T6{o#f1R!0PHm2#`m0*UvPWegUd>Zml)GO`}rl{VX{P2<S<MF;X}W~5VW@9x`vi>#P>DKT%3&OB3nDNz1FiDge|jdR7%{c#oI{m1$2|78}Be>4~beq97#hk_#@`cA%6rG<z`xjBh3F)?MjUcQQPISj88|}8?-H|orgreP70{c=<9_>T8o%fj6I|^*s47TOb&EGwI*iLQ`MjqPk=g+`-70k$lWiTB9Nw8f`}#g^40n5vvJFdc;3+G|0_0IV;om$@d<P1+woStOKf%9()kNdW0ryC2j=GYg>+Sz<~TAtA!fCfYDeXx98OEFIM<XV*Tr}X8@57J&+{~(bL7gMH93U+mG4V-3Ps(2&~Wdz3d%UmIy(_HPF2@@v<p`eo=uZLyBwmSA@S)hAr<nZ8^0he!g9)qpBH_?w?Di0sFU1&_UzsxjMb9dd3gKj(|b>~C`->TWhoqf$ibT;{zw(@gx=or>L{?vYo9LTE~f{ixO}U)6V)4nqcg^Z`x^PecG<zKDMhv5R(v6Z$r0Z-V`t_$>QYQl<U{5aNu5VxQVEO*Rn~je0PE}CFV)?)=VNsg!mc4_R0<1{QEuQlN+*y<UTB4i|4`SMM8S`X+YJ!zgnfJIeTLzq;Y1~Ac=%Y<X}Qj0xC^Nv;*+{&uu2yrIP9M^cGiMpJ56Iu0ByN1s*ivC!fc<x<8q>H)dUK#cNPe_7Z?8d5_fwu9@ti=_mTT|Oiyb|IIk?zQzrtG9U#E<&ys%gTBGQU&-V_&a`VLntr&>q?n$pI<%3LfN)p~)BRk~n&2-h*HQrc5*iQHoL1|!+e^bKNKCPa(GfVAp#?-1LF)tA4oL&ESPlF9R-<yNPzNAgw8?&9~x6m{DYVAp%Y0T24=M$1T<lt$RX~`SQUN9dIGGBT-ZXd6vqRBload+%t;%ajld&l%(kF)W~<i=6Zvy#*F$Pb+5mI9%0k`-ul5_8-EsZ1>=8M=y3IDxi5za<6uar$b-PVR9-a2ocWX<qCainVMU6EUDBBH^-@ULCo`j8lFaPK|yVr0}_-Jw1J&IGDM3$`L(b76;h`^9c*pgX&L@xl#pmV{L(q<|bdqb=M3d&GQPhW-;`vVzAO5i{KzayHE`#9qQT?o4V9Rgn&8pbWD1bo-EVT67W_Mm@3K!^p(LnT50m^`4VFGus2wk`<Y4YZVp#{I0!^VmNapir$JPOroI34?qg1cO$1{EbfXxEoMrhKv9An}75DT39Ce{*)^-WMaUm)5!JPcNY&YG8c+fsJtWouO87S6p&ntZ~s_{K8v93fI4Q{M&;;zM%hE1ne*X*J3?CCb&%$L4|n?$E;K1Peqldkh7sw2}~^($&5LS6GQTD<MM3!>dN#S2$h^?nmpT@LP3oMX5Zj_HcB99}jeWgP=wugv#00~e9yYb-ZfnW3T^m@P-HtYM*?LALdNmmMd~4dXUganKD7Ior%`NboA6jwPCS`t0_TXPB|z>9Z#f9({z<;>J=C$_c|Dal2<Fzv%76!5z$3A`&c5vMEa-21S2f-Or`z5}AhOeT_s)qBzWViPWmAD04r^QDN^4OjyqQluZ?bT);Ud<?v;Cgefd&ala?e2|iM_{Q$ISPG^{|V9aLySerV1<iPn1%OXevnH5GHs!Eiy6G|ho$*D+CFx_9rlo{(BEodEz9?V9=%T~n}%+#*%!RCIyQ<9lwiUwTMY=1sXr>a}7LkG+>+vBXEI6(Q`oobvlVKZN8JA_@{5D+X?S*ftm-RuD7k0XPL*DCRXfz85+KiuwV0;1lIb9jE@W?*EG^D3h~@j$8*&4e`@#%2~tt|I&1LzBTlk&(pP5$>b|!!Q<lEc!VOUzuf$I7cR1gdSK)C?zhVKeU@-e7pBgE$okRQzSBPdp79p2+ZbexB?tTCjnHfat*0H`AuK$mrw};DWfU!;A}t4lgs;>*vV;nEJ)d!pmH*90UFPi(i#Afo-BQfNnFMF_tFEI>P3?!S_2QQXN3YH+Z5OATDf3k@&b|62Ig%qKcUdw%>79617rmAx|9^7HAga6r9|Sezj%2Og_CiSl#^jTOV|Lq<eZz11hh(xm6V71B%!RGBK#($yrD2|2#k}iQ`M7j>4|JCXjB<K!{tPHizuzHO7i9{hO3(Q?J~L)m!z)(!GtJxWD0v}T4w`GWz&?a&WwdBt=W~EwzKJlHM`XLrn5B0vpgLpA*L{)RFnup7yjRJHS(f5x^gdyJ+BqzegOQ!ls%K)+z-AAT=temAY>FWQppkn(K_6A8`;r@r=Q^u9zFifv!@M*xGgE-D}WwZvF46z-PX+et}|=$ep~D{Ut%fwC!TAkF&r;4-3l9zi7OQfaHK)E@U=k&ya;w;&-g5M72y&XA)zRCXGEif#oc!A_Iy@6L$DC#k#|AajLy`kk1>ke2U*V-meF4!NzfxANuUaz!7Q>VJwR@Mh8IH?UEzd1xLbZ>Cye9u{b5m-FhQ5;{hDR<RwH;@bpac%y`NyqaCULLcwpwxt-*+-i${pz#KH2ntc<xb(@dl?tjK`X?`^}rY47y5+^JX#+RPg=s?6lXT)Be#2FIBVu3U+nS{pUBWR1pIH|3G`+^er^__g(OienG<Jd!=(00G?3M7q&bjL;B8XT1TRn2915JrR$SMIVH(D9f|D^Go(zm~cBEryQhNcp*t-l3Zv+g;drBU|P0&Rp!9SjLemQb<7e8tGO|HJ02T|5k}x57a(YS%HB-8FoQe=E@K3E2$;IfY$@P<#oDLSll+Jn`*gQlZ{KQf#O=+i*V`>6oprJn;HF9CW}zsolBms4!M}eR`*(trWV3r0F>J78i}ylZJ$*MchS)vJLC{D?v_nC5SSlf1<1yjM;ff1YQ4@+GOqhdo#jyhBm`_R7>l%`tMtY#SbP2KiWYb~`-C-ihL8OP5Q5ku)MsmGi=3v*aDRXd>tq%>swk91O?p3ZxoE3mgzeR!m6h^b=i-k@MIhAjTkiRvOeO@f&6_t=*xL0BN)4nibA7kDI1(!n;Y!_UqQrJD!u1UBiC`1HOFfUPJzfTtKvRCvPgEOaR=5gnF*DS2GPoAZJc3cF!M_IW+30Jbgmb0}|yz^;!Qe$4_oj@{{xGw;{xvkCFK0IQXUa&t0)}^8*T0#S0!dX^eBsR##>xV}Sf9sbVfRx35uMWLv1B-kxJ+&QJ6$O|__h8zE)BqB9B)%O@i)EW1d5D4G>SE9ZsX4@Q3D8k023%vT;&YNr<#1dakEI{$FxyZ2r#^|OSS8;QR`+V;A%%$$ZHIJ}8MTERz6|zIeN~F^_IMhN3(p}*6ag-6cuu-90EX3Uze>Z#2)*oaj6Ka|Klp1(`%gQZnNnIuw01OyaL+G=SK$D6_ItKWL<ZB&r<HEZQ10#LN7<MqN>cLGsyhSPtsY@MNfKDwsX=2)E0;V+9GmZwW!Qd2_L9vS2~HPK+)pQ!Pq8*S74%V?Z{$pHiTVcx&3ggtgYbsN;y`r8u|*a<LPjejONjrDu^z7a-h7DHN5vYE!WT7e(*h9t2(aRyIA+)DF+7<VAee8~BWd!kqYU$65N|y8ps~<4y;u1>X==4$Q5*uz2h^VCC$Z1AV<~;}7o%%lo*O-%MZjCO22g{xW5qAF1Q6q~jX=&hB-7GfjK&y)(V;yI9dHNtyu*+Qr}O^oe9+UO&BwKtO<Ww>l5_==VTybT9)fYLMbgDu16-sf0P5jTH=T1}1JkQ7auN>Y5;3JnD&5;-j+!!+av=ZU!A$p&XTPs1Q=MWbIRZolIBruY5BDC&s)fs=N@teY>FRz?CE+ap_6f!P(LuDR%z?PEN>*8<VYs42sui1xH`*1o;qD>F#ztGnLIpwtqszB@JM~#)Ja)b3iVodu1YBbYstZPyDKNZD2*u&G6ND5chL-7Y_0D6F%B-i@F<nF08#?u)z-Jd~L)!92sur!YV(uZVq;1o%?HXXaHnl|5wXoJ>zSo9z=+N4!(de`3zRxbY+hjHEgh9Sw``pT}SG8*8&*oOT6~Z{_BkB}DWJ+qePfA@zrT+uM#6e*8C>dszKg?@NjOc5Jvp_E<X16MNL~S~A2MNu#%DT3ML#?^1JP56M8Bi`mAGL(iLcJ!9m;LI3)@o+Xish6h&ee(NG0pw*t)2DSxIwo)hPIwYj0=bZfL_~ExykhTEVk7(bu5eXh1E31YKhWkHpwPS=<kf7Fk3rE!G~#?X&l&Ra27v?MKbK0U1q%NdFL~;(UohOo3$WJPVq$ybJy|O^A@WhvEDBMOr331%X`J@YQJ@>pysYrF`qRMGuK_?Dc>_67d5UGcKseJ6{868(F*>Fc&l#pS`SM^<-3IGG$`;pvu~%Ewu~;^2%*$DQkTfw>ODQb)bgaltv^(hucUOS_xO$1+GefbTqHtz*b+=i-i*95oXr^9o+Nb=)W850FJR2^YOT<T*o))X>s-|Gywh<_o~Oo<dMKk#eRwAb$G}cVDYr+pL^I!VQH}*>PCvdmr4`_?Q%Z4fbrb~zfr~ia&NOR^+Ta|u(P(^P{%&xWllT%8;j+2vo10-HA{}&l)UER(+h=s)<_jMQ3sKl41)YRPu5c2s88a1N27i>55O-kaTlAuwNIs&3qZ~;__|)7`c}=8LixhLba)5D|CMO*u_ckpiQ<P~b<m0<oLzl}7+oh!aGSB`RJ*=@<w^h53Jdr1K>gGcxvryON?kdvh_F%~*YPz5mEHgM@6u)RKDn+ai=J#;_4qG+5!8`yqn*AkYG>nw*E_K+i0T#KSzHd&D2*umpQQ5G~&-}8%3sR3emDjI|QKoB+OiM_(;SESz6SJ5$1jU0H+kEf}F^U&OJ2I-euFE!WV;+n~;1&w%=Tp1Nz#0cBn&zaWMHJd>mSQ;Z0{kg&>=;j$4}l%z$$+l5E3|@e7`Rp~dq$XS#Yey<ls&dnUt=_VmB-o1Oc@}^^Tu5VHqziyLVPpU<p>@Y!^`D+wv{^r?=P}ezX&FX&K{CkR5tP~fAF`MV0h`elgXGGW!w_glx}nl@$8JcB2B&^HAi+`v4y@L4zE=YcazIU!w!p<;P&bm6MG?z^$5?i43?+PimTFHeTlkfU9s-!YuH_JHM?hC-0lhs-slwEHP?E3W5-Jx8t*q=nrAlKX`rBOf0qmWXFRkTDV^Cd@CvOZ+~Ah*eM<)eOZmk?o~8cXIFh`sK#ipP3}l<;UCy^>5sW*O@35-w`1&owhRp4T9^zSHUQYbsR&*(o_j1N}i13Kc#3Qn{vp9jO&&Zvo$ijgg?D)u#Cb$CG8VQt5*>GKr-kQA-ElShX;BItnbjMWZ-pFOE!FmGkSYhjn=^Y}m4?pp$rlLTm5G}@^<R^I*$v&Coa0w9-b8?Dc|A`@6_z2^4cAhw29_kvk5L@wdb>&)3$%XnLQ&GbMFjC9;v>6`7kT;*)S4+#lo>CX7Yj9h1H4YM1LFkqRRxE`Jp}!rZ1jpJ-7glX;jgoE;f>Mfh#Z%4TD+b6NvEEMeqFk_k=9|-uypxpRwbF}=*`qWta`&5rE;9VFy2+hSAKicOQF8y&C)m+D70`vNfQIQV9Kb7r{fn(Pl6lmJ#eP1c?Qxii#P1kbH!_0npj~$2iNIs%o&0;jP&-YRimSt``Sz^j@xO=M2Qo!waB?=@2N=ftb2NgMJ`e%UL9C34grh{3lQ9(eXt2Xg9LK(!f^*#Mkn!|ddt*ZcY0<0;6|q74T|N^?Eg`KArF|dPW_woQ$>oax!vkuuwJQz+nK!a+Rrc=EO?;;HB^E&+mTwRqqgxlNU?~Bu7B9-3bl9Jx-*mwB6<}<E>o(jIAd^{fZ=;7nusuxxSLX|Rq+9fonk%KCu^G(zr#L^<D?3|>#&D`d!W3+GUom%)Fq-7f1ul(vnt2fE%ooI$8b@2p5)Fn&*{GPFQihiPfryJ$qE-HEb{dqh#%M={wyI>og6T#|m64uoTBiHD!ym<avKHtGQJ2#43A=ZaqKS(2w%6-TPG<*(#mWaCeBfr^!RP_SbQTycSdqgbo2_F@1{124v>m*|$G4w-oILv!Ptxz+d-Ckw-Q=@-4?g<%+0&NQ;!JiAts3Ag(d4sFzjg1?ga7&7la>o9aF?SA`ze2%!4e@RkXbUE1%33H_y%C_m|o4&AvyDnvTqoq4x8aqj->Yju8>=fB;6MVoZaV_CNoEsicMUWcZ$sVAoKqSu{txB@}<xQ#|r1807K9+z3gso)Xrnfbe0*hQ11iCE;@9Jog<i6(yGEX9A@b_nNJeY4#&4HC%{oj7N2DymHKFvX@&pB51v1NZ$!6HRhToqJ)jLGi$hgYH49kn3-a^MBEpnkegUQnm7YP0LX|HQobw@kO!l%ADx(;J<xT`LT`0sL_&46xTi!S_)deB+=>|#aS?!50A?{JRNP=oG#->K@3=xmcAcyrKP1pA_@>>Vs!gsabXAZ)&K+<T)SSuX+RAY7E@J1N{o{fB>^SM1J2!%!7P&&CWWhmY`5Zf`Q9UlBdYII%^xhee?!c0QfxGbA8M$Cb_=3%>|U~9)39<gd&k4#Z=X7xc1L{$a@S$COhG?b&}HX>WV6PdjJ$|6S<GFo=j3f@=i(%#zfODIF^QY%V>6pX4#TKKy+;*$UN5qf5(OWnB<iUHoP0DHlX6(fol>|EcDDdS~Dk+QshW8%`h7X|Oh>&f~{WTDoL<Cb2<lhgOMg7+wO;wh)~>uhwck|!UsUBR;~9nmhpHy^X<#(g-a;i}V*gBg597V>iN^zq$aUV8}lk8zo;Js2QNdwDk93T{uxe4`U@2oWiruFo_`*mfjiW}JAvSU-XV_xPeLw5Ou1e$U4_o)Z|U16~sQNf-@xV#Rw7r*mb2#|-M}1ox@<Ebqk;`*Io`PqY2(1TAl-;=P+*(wJ9ra#1`ldY#c=qrek7z}uS909q8W(31bh7DlMz;SXd!!Y{)l8K<KxNyNtd?x%Ndf0*2V@bKQ#<k9U<tWf8?tub;FhA<b;G0pO&D&9E&w4aq)F@6!xPG(%JigEtQ?O#ql`Sk9+hb%Qqo&vATU0y;|BNSXyj3c(T*|~B3rfA^tlY5fEPM+NR=-w|^kyFBN<Xh+!_kHxF#DAt4{(D6Kj_{x15dR&tzQ6rXkolVu2Z;YnUOjxxe;>Wx`G$P+{`cj(?+4%DDy38U{q3r$ohwwMyrRG3R?7FoZ+s1ZJZ*_*+j~Z#9?6#%Cn?>Z%sM7U(VN}V8Abk21iC|8C?d39hkkz-aj^@Az&|1TG@tFQ-K5GaDF=3cGU4bUaz6z^qDd)Jq?-;oz)~ENQm+D6QUC+__jTbTDI-rEq|>xd(Zndt>m;L?o9HmKhjNBk5AufgXh%?#EG=cxk`fZ3XOIIIhlPWyL`hcwyn4e(_?~;2Mk#7&P~>B_X(M!(nQqMx7#wEhJCDAFLr1@M_()iXZt!J+vBXH?lOMvHpuO>-xTqrmLfH$pm5s?C5<wyZj%hK6H`l(UdENnMN;ahEOq%XznlH9-na9|D=o`Ww_Z%pL;vd2BoWh-Gv-2&QE8f4sGYLP+6euec4U4@;Mb8ukuMBpx*>MJNSGd4x6MLS0__S-^L=dq;wl*tJVtwgra6OFz{{2tkH=N(?7XIs${2%{TEBZhFV{_TR(8~UQf2Ybyv7<yg1aBAkDaliyNu0Eq!((Q*a_s3`xpLExHvu~Avc>Bd+OV0JT{v?m6IkDH*GZwLK4F^x*w`*HM!j-d2XecAFb&7-bu^}&%cX=_urA0+h2P63;Vx#kD_fPS@yxUY&x}c$lkJw&rO<N-du`A}K}fiWV%F2qZZQm~64}(@F4cYPKZ8d^X610rZvM)4l^ZC7Q2yXh!bj;eN9<e^!4d=xw!BEly(1?(Q&ztiq$0un4(}19ir*(;E`9)T=?ie2TZWzjT8q=cAPiL_5!GlJK}NI=@Q=Aza#kH(usZTX4;$d0@jJlO&8G7{#@=Ax%1Z1)N_6s<#C;31b(9T*5Ao+?fMD(3ScC(q7UU^LU?AD;M~|O%G&vmmp(?@HL5gdE(FYhvib8<qhylxInC?x|>CDQa8BkPCoLP#!APEJh$v~N?OzDh%0>{EnI421~%0VJ_GvH{(!vB&TYXNW2H#L+lqXa`sxEE^zNn04%D2Ox{q-w;L5Sry25g~^&T>rMk*R*`Cg($%q4hx*0;Q$J4Kfj~|3sn%r?OQiCZbU)*=8aoBHE=;o!BS0i+P6AdTT-~CMO4kr9Dm}|uT+7a=-LYxD+6Ps$99Aw=oCMT)l%zAGHIQ`pJD{a*ox;m(;BRcW~)W)5@K(agML&nC&OA9*wZ3<ZH!S{r)3qZiet0NqYr!50^%wUemy=EaQx|%W2sRj57~0PD{sY22VoC3ts9W<af+c%3oGazMNZW{-8FuDnU=V?(@KVJI;Chp*Q-P_=oQ%!_b`<#>9L<*^7VtJqx&rEC@>fgNBL1P9++c*uitrgf9)p4AFv&6sYuLnyeixdDzIYSt`x#+b6vz4UN2{qoUY{fyn5}Hcd3jx7~?~hCoePZuYVto>0)n|oy@-ffp7H4jY5kQ5r<lhgWH17laXr#i(suNFm3o@SNTvd2~9DO6~}*48oFK1b#Hs;i}-!C$@Bs47G>=Kz3?0K)Qnva2ahn`z+^H6E)&^pRzvCA(m<$&uFp2N$RN8=8OQP9g1h9_RDJ;APae@j#wsqxaMEeyrv7YCBX7)`ltzS#291U<aE3Jkb`x6|O+$rO62y&qg=+?#mE0F0=P{;bNh#nB4TvJ~&dd<pI9I`7J{50$NGQp-u$^*nM#d+aRm?=0tHqSHWn5^o;bbWFnd2zIrkdzNXH4pDC6~fehdvb8vo-}agMOxk3Y6gx1JADIJj2wc*`;w&W#X|H^bC|GM?(tFrgmD7RPy7XS8Yi7D|8UP6?ARV3pZA?i=8<56zAqRFSCe1@J)SJL<L%*?<(CmhOG*E)HVFymR`au6w^fb16z>4_h#ztiTbHZegOa9>4`ant33r(S4IjV9J5UER>CROtMIWh5f~L%Ev|{LQ61kUiO}#;;YiO!SvC`6hD(+;IWSBLYWWS-8frFJc2z({eg+|OPW-NRphv!=vFof}<U3%xwkeRp%H{?X&2BEXSp6F<AP{aTCk5D+q2?i3HqB^7j|3~JqIz-Dza$8!AQvQ55<XW9g7-vr)v}f3H&n%<V2Bsmi2y&C8c(8qp(AG@55^&{j9x7xea$0R>jK*Jd74zf)~Ix|<pz_z{lR352a_!rOvMZ~4n`GwdxO!ss2hwbQZtx}6>1!eD)#mUqjga?7*(WRHA_;6Vgev*G>*jta1;aWAc-WNYH2CR-~nZ8-}0q!X7K{v*HL*9V}gfJujE*)OS6J*@akwQIHWV;Fk<fqy~Q}6Wur1|y><e?@b#)I?^!@!L@+?^QNGW|zT|7v1YT{D3UXJ9@2#*9a=lp}_`8!V5qCy7wRXH~qMtBziz3SSR|TQU0S7S&832}z;m9T<Bcqnr#E<H9=}$~Z8~VKLDX2xz?FJikp%SH0B88&>4&^dw5sL8^fGdSXRBWrxGYk_QkLSKzVAYgK;V!$(aIbhag{_vmTu^J8jt|LyPYG~UQ<0E85Ak$RZ(C+VmxD*2KD)Q2#d<@_rTr;h#DzF)m(&V43B-c!XQ-|u9Sv(ilqKMF&KE(!)n=x<ahR2pn0X~QuIaX;EL=qxQzZ+cV_NABP}~Pt67OS@7MKwrAio5S^bf|YD>AyFOMozAgiBO<NCsIrtLYT;!eI1}*r;I&o(@E4S_&o$BY``}C_=I0Fq=`>OulrZjgZFdsWp*aRIgrpVp>g7He`y?h67MPbQF@VxdU??C`|{PL$?CXox+?9@_|fI&z|o175nhAqd=0xtl)iJu`q)D4SII(Jv83nD_>5s2Bh{hf#V{`D9oTgWpR_QkWKEH=j1;&vtRL&6>DPGS4FbAA{9efojkTxlj3cwlh)SDH1V5FADyJ{K33HC(ynN8r`BwL6XR)Q)7hzZMX1bf*u4b}>htl;<K==xL_fE*hR>`aGBzIRi*$vl)EUvcTJIa8dpclfKSa;OVT-9s-Wt?EST%UzN%$L4JzL`^rEVIjsPUMr3T#s);}k{hM>mjRz_A}&!vvo#b(`0WioVzRYzp>D@Av>E#<<BHePvFW6uQAEM-~xUjWED(cbJZL2PyTlWt~|>(eOjWR&F6)Z};#l^>BxxeIX1r8}c2uyKq1q+8hHwHA9L8)EnoI+jKl(j}tM!+qS7#uBms;DanJc(*`jVn}*<KdRoU7ybwCXI3EwPlTbjJ`&g3fq>o{lsvuc)ZOT!^aL^(%L7K>%&HH@iebdzxJfNYE7qy+>^u2IhHN*;Y^&MomXbM=0t<Njv$sjI-ilR!Mit~al1XJ);-1z&VRz0lmcZ1GZXGC-*hSb<cyB_~|f6EtkE25D6>~5KA-e8W}%WjhTI97F3#RBO0CHP<bqR7Wi35TW$%y)^pczdywTAsZ+y;koe)dpm4xk=b%BWN{4wgR!^M26%@`OX@oL!~PsYmhA7gN+VcYbZHFPp_4Rrb`g;e0r9}xsPq{?o_5HtnT11VRg5%88V#VPtr+AalX;?N9o)wBb|t?evscY#aU8C`r+ey=*E<CSo(m&db!vAYOW;lP;VG`^`sGdangy~5TRMi<M&^+eenn+bZsJPF<hio;9|lh&*xfZfbr)J*PwW7tc5~>2bsFS*kxWRrTcTS&WJuMZsd5bO7V&Po?a*xw$*xjbCe_NWyolZInq9hIilmQagN4Z2BS*OGPvCEL>V);(u4^?2Tct3e&N$$pajWYcq?gBQddMrow6Bslzy|=lN&M=&YL_%I4i@fmUfYrPNy`YQ@03GhIBejPs43TV6@$5($go>1Lb#GVu~x;6UqCWCgSpJ;u+~^bi|Pj)oU_jC~@QZ<M9sL3Q+4O<54cFG4QxSdvpj`G7(aXrMq>M3U~>Fr80F3y~fsihTf8{;kXoHX^4ytNq8uAR%YBAPPS256i`}RKz`%~jt{bF#$Ii?%FQTMmCzZDr?RToEuQR6C$9SLTGp;#%^G$+@805lykX+IjbB&1p<fWuK|kK4G8WOpVwhh9iJ`Zg&8C2R3gTWw);?tD&jKF}|KYQA<R?@mX9jw^<=!w0^iY`{#CZuFgl125GAkkrAzTH&TN%7yBSHAeY~XcEQ?#dndhhZ$jDdbzs;&ak$7Pv4MnPyz3|u9iUs?lp1O{w>B2@)aRkR;=RK44#dqiD?f0EjM>5v*O8au0MU7lY$Cfe_p^Y1yb{rGsA&$5tIEra=JQd$xoJgvvGZU^PEQ*wWjPQS3}fN~{i+0S#KYoc+pvxXBEn)nIxR&VJjmKp(p>)giqtr&}@F9<f`js}md)u=ewND@T?6NxxkA%vD7MW|p5iOV6XZjQFwJ6>@F#jeFRzt+{esJqyjvgEqpfI^)7l+(&-#%gd#ysGB%^%h+d=!J?VQTaBG;bK0Y<wsdu8|IgJAQeQI(zcTVK{Kz(RJt3?Su>x&w0*6%!P*<TKo4z-JSDAVR@dKZ9UT<(cO=9C?!l>VrD`Hg21#Z{1q?C{HSJ;>sq@D`3c=cG4sNm1*TQ%n-M@pFPj^%ow2&LFd^MMH2u$|Ua`toJ6}YSu-`Sk?maIsVC2M=EmhZ#oRb2e2VC*hRli~VT&?k_KsqL<ntnA6EMlZ^Pu*Tm4*C4D}o5g+?&oAK;1L?3FFoJM5?H_WcE)D-l3wfpD<*x+7vbp8;*bp7Q^V>L!--g&+OZjeK*<Td@jn$niaCXk>V4>Wf*v;Y#bFgSm*L+rIi=|twal9~T-YSi5S45+7l-P&eGv7s%1U+lQdR4R)a6cS)2kkx%C`AO3*MKCr$k(=f5vJ5aqtERr`?QMaRR9l~QJGxZKYm|}h4$_5yfMD|-WXqwZ8zQWhsAMHQ2?fY-CA$M)$PC?vuurkIp0iD<b3{$nS?Je-~KF9hS({tII`W8+gUpoaKGym9XCDQLUA38*3jk~zoeGvmHLnwHgXkHc<@U>K}-*4JvI|{R1(?Q!L*p~9}q<@O*1xjpF<orc)0r%43<)_Gxi6Xv9M;*jU&?yp5IxZzq3GpXMuhm3-m>%jT+7MyW!Z>x5h2ULx|1CGgvczVV0F|NhQJ(CWjDyHFP6HyjAj?iu>Ylnb>7$3-htNuGOzkaY{Br)V?E(EA#zfK3SL8*d%_t2w|H?cJ}O79^Xsue0=ZDw;nwDs7`WbNUh|SkrRA3L(3}Q&rX4ty?vDCL*!&7?j#S>@11gHX89^F6DcPEH0ozbYFEk;cc=XW@0X(-Z#L$qDl|&MP$n$Mf%DmrbIDU%=oV@AQJl1GsPdoEdy$yi;8x_fXTZrceg7`OXqVWjWMoxV0s{(B{V{(nrodxrk}}O=QK<6jr!q7Rb19Q&^NNwjg)nDqF1e+`un>Qx+ow6xzpZ3{=A7>^hb^YF62k1LmGq9*Ui`1nF!NEmk7U4ESYxOUu#~>O4TH7zz*lIjl7LxbbT<NJ5b^I3D<?|*Mr-)Y`>dhB{bEWpeh2RY6j_T>tyf!8p&LxQK?~*ZSv2n>95z8Dp{+(bBIkWYHr#JI$%w3|Rt$rKS`q-8XsDCgGI3s!Xsm=~nl1Eln5JQy&&buI5%CS$bG@z|bTjSv3XHaff|w(=2*~1LH&T(Wz^E-+;@ef#F7fZ@$Xk|EEqOoL`Ix+g6z%bU)@YAEA(TE`F{tuXGl7K%96W|d+Lqcnq@x<kq_lWZus<wz;g%Gl)*zuPM3NwT;L_(P;8w}-shiLr)&u%@PakD_+k7aC>UbJOU=srjPZU^+`|9CeDb+|F-)fG{;9?&ub&%lF-ey2nrJ=NsfzSp+ZB8>Kiq!g$N2oq1k$!nrz%2-sEpCBgFGz#!o`upF6yyO3*hvrz_4%SbqukD^R6+6zku`$!Sx)=?La1<xD$L3r-Unrnj*<k!l&@3^U6TSC$}#ITpeQCxk=;?4Oc@!`0G*VdvnmQp$<D5)c{Uu71?>pP&D5$yP^C(NWxy`p@vx-2ZNDh?hnd(8YbZHfo1o;meSNLdxw=!AzPh<u%N%aC>F=0}v-<){lu}oxsp<@@=<P^*P!v$g6;t;9YKmoS=-^4lN*sBYF8Z`E+-ZF$b&<9jb#s&f&duavDhD<@8yg$W$@5(^CyBW}ymWm34(=j8WxN5#^U*FZW~B=d8f{v6F->{^%H302Hgh0T>y(`71&=8iH3=P<5i8}ckvql{WPX{k-?)C$8Ts9O#2UJEn9ok-z$Kn?Hey<Hd_5_Op?n(~rgF0k2-N`h?2^l{aig;&EU5jcnty&N2=h3LqhK$~25TZeSu0Rkl(EXQigwDO#8z4<F>hjki@<HWz&=uoFS3G)rj`ex(&Ato{&NDM8Vk5FQewejr+rvuiZJ(zFKzcC7_VRTs@kBrd^8_b>OB!$RaoX>Y$~D~BRb+y9jxdJ(%v|D24^jYekN@yD<<_&$QW$E>iEUHoGDF9h!Fy!;99$bCLnEOBH(Xl<Lb?pXFH|+nNTxgsh&(v(5!Ua&xZ44a>{HjoGAx&ipOy=UK>*8NjSu2`H`T8qLUz0zo2i;RYTvHxFdNtQfetK6-g+dW1Cf^NQ?Xhf0z85+L54-vtxV(k1ywW|1jB?Wq=N=C|9wO)ecjhp!7V+CWss4tJ{RZ+1kc+zxsDl=EFj0V|PzQQZ2++DV$F$95!@Y)h-%LiwRo#EQVtv-ta;pTcATxMeeG05Jr4RcXOalxv5}N5^SL1i}$g&?|dK$o=`+reu1XVf!73JnSQU{x>_^)Ro}`)V6E6&GnMhpkYuYua$^A`aGP5e2GNe2By1(yE=MwA-@Z(9DlO+LF^*1s<{F#1GuXJNLyJ+?`Y;*FuFJKzmWGK&z}9dW`DFl;AQIKKszlc30#@cqv-%8;@(#ea`$EQ`agWPR2eV{-d2Ommp2a2=7}S;A*m!UW@^qYVRWG}mMFes>U1BIBKS$ODjw@>7le)+UMNO<px9xnl_y(&)&B@)pel_x*b6xMj71fgYE)enSQ4;~~0(p%L>jSzqGLI~*0DE^HK6qTkRwk=e={JUF5*0#6i!>MLXf*gV%YsEFYT6cGpUj8D_0H8BH#RTlWjV*hg?J5FYhT~&bZ*|bwt2ILn0MPz4Y}<SIgM&aAbq{Dw0()A4K`rVA0iV7{UI_T$c#_Qy5AFd$Vi}FpE+B<YIJopw+p)0fv$`Cb|sf7@4CCeSIK6hyS`20BzCUhZ@YE-$4ND9@;+T_m(<K%O3CV~2fLWd%yixTrA6%_p$prt>Ped|GvhvH(yE?Co@o*c71IQL#<oPJT+(0{WsEkV5A(xpM?)M`SwFwzh^sDb8D6qwoM3)!=@laqb%kE!6G^Bk=bG>m!ITC=3jdO5U&BV=Ymxghg(FkK7j7_+IRZu2G&D2C<A*4Yp>Q>1*cvn<+*Wn2Q1sjCY%!<A8mLN9Pfw_n#DhqjVS12O(XZK0z`;UQV@N6w(h^W)%ByUHMba9hKmGd-`gq}dS1+4Gh1Em}!+`xbm+WDmN+Hl*K&S321Ayo6X%IYrZ|Aim^RudZJ+m-0nF;voDt(O>mrU4pV(<!sdmsx=HWE=1Ef(jo-}%DOWWuHbVW90sXlfv|g_M@)RoAk8(ZW|vVX%_;mGxFXuL4*L`gbDO3l+h>lS96fL%sz$B*ND}NXMwdT)dbKaH&Q_znp@A6y1g-hgIK8;SO$XpA3U5PqVTZ&gssJ{nH#!<EGpQb5}3bTl@Nool)EGWSj3~n`e}5UZhmbuqkrQ7M7A?>216<*_y$YwN`-Zi<Ylxy?#X{Y{JUv%rdqwS&pWkJl)O6dqi|$)gmuB@3yahS>m?U1WiqHp?^#QAA^zT8msFG#4QiyuC}Dkpes(HtKr01MLKUsy7W#^w3?tuToS*Uq9Q?2-wBI;(S${-iHW3`M_uJFKxE_>7D<~@V?hjS$}`A_Dr+m6Y_TJCzD4oKmgwj$2#wx}hb~4uq)EBq4ud$~9IoJQPParf$Sd}2#GLS^tR$AwPuegqm7wYuDuaT45rF9T2@@1?jD<`0eVCn6h#a+VF8Y#YUeA`ln7!pI*-DqOFZvqx1z*67c0c2EJmkHvp`NuYG-{7^3Q`H*)%rZ9GFCgwS*8#qdi@2HlZcpfUSk-ms5m^9x!ntzU}7B~*F?Mw(ou#QERoq;YsHIl?cKNv&L?{12pA?26Yspi`<A`zVe%1;KEwHrjXE*swIU*gk%#xg;{Z6`o>rPti}tX|UTmTP89zG|n<<I3206ln7RX|`K%3OO25hs?V!(~^2IzSW=mw2n4tTn?nG@=u^OY-F0Vh?0JuDA`U%6rw@_|s1jcEeh3UGqZR9kw-EV?;*_XS=oG{Cn!#KNeu<rNWn@p<Y5Cx2jYo(|(1A~z54PV=K|z#OkQ21?tq-=e%olS*CdsasgN<zsVhpAf(AB%iV2+X`-v0~uF>`R%@k=X0XF*Ih0`omObS<5|I1j+RY4j3>RAPAO`fwJ0#Xw5BVPgS{`tSxbbrN341H;4w#dofP?4#+Sp5SwQbOuvF-Jfu=l#vJ?HzlZRVA%Q0pN*3eG_x$*T~Mb6h}(=1!Z@N)_T!ygQ0X_8LMta{TghQo|FH{~jZ(uZNbt9-uT(P(5mA5Bg%SnzmKy_}-yYdI^cHzYEYZEBZ{1co8oKG?&jWKbNBQC=ZSa}Uactorcy!9)3)92#p}*qT^H(-YKSAS2r+IP&<*p26P@-P}-o1gNl!D}0&|{cf%74yQ?p0EsJX(xNq77EzokOjFSlYFJ1PHM&kw3fISQ-z=xt|N32VxhmHgd)JS%-C>?&<7BseJ%NhJ_01cdt2b_T;_>8rdMHzGN7t^Ov7>9(lj|GTHa0e|Z!FW%=FKyX2r4GmZ(h4`^X9dy%XD<D-8oxFP!T)2xpDK>rrFVDk%zKj^{Bx4W_4K2>b1^|YweoBnmwIqT8DC4H?MDAy>|1OJ=zhl9XV3!Igr0<(2EdhEpOi(uLuoRsqpD~yM1GG^QsL`SzO)N7&T2lmk_{QJ=?5s8#FQ3Zr-?l?W&G%uYEh+klw#qW<zVbp2qzMf0o)#@Mo#raIMlyHKVMrV=u-n>{&+$U#tEl6U|n^kY$oa!6Y!35mxJ<m^<&?OTVBQ7Rnu<TpJrK?`9}b(p@#nF}TPsUpc*Ni(Q8Y+tr616@*071hubomtvCqy|%CF;i+_AQeDVr4BKSo7pRrIRhku6lDX6P!Fq%5D)h_qONbKoC~E^MP!0CNYpCLZ#Z)q)V_tK4pRV!h#;xnkwS3CPMLqd-jcrd)=5j1EB5Tc;9+O#b*-^Q*8pb8olK1%e7PEb=bB4v-+_-v%#k}!_ET(rvODv{$OiL{0jdL%i8R>*WtW-%t!{?V+WNN6v40(mTUusf4LEHNn+&xRC9Cr>Atq8GLrI@Nd>6s5N2YaL07Es1Lom!d(x;g-=5t%(RL^3>0N9B4x+Fw^*8taGIbW9PL0;Mfw32NoYA*iK#5wm<)OBlF9f4g~FuGSmZSiUc~zgD#ubme%hm}8SMoQi3XQ@Tk?0KXhATG^Fdp!W=@HLK2bv7WobqJNmY%3rTP#1<c+DZ|Ek*-wX*sC>^Jm#_sDMsZM2Gm2=ti*~_q$%S1JENgWy6sYzYC#&<8C#!w7$?Cj?$@*kC`H&tq*6*S?3I0|TVF~C)!%4i?e>>Rafo+9(laPG@bJjk~oOQmKIcu*zXKw}1@RI#8oV9n0+2d(3nD=M5)ftR9ce>gfqAVUy@o{{JQP*C)oylAb^a_*ffc%BeGU-;%I?qR6+B`GZXPW7wFKMP<&a>&6G5Y0~Hq#9DnP&Rsmo(GGe0J7(hN@rMOe5H5nr8*HHq*;;#vXu)v8dQ23liXuXUxuGu`}@fwLR8c1FR0uFvd@AKgL;yDMt4U-=Ofu7mJU}kECkX%Z&T_IM2lSyQ`TstG87zOjTcv+@P=1TK4aJoi>fLhOawMhwY_C*}2$JwwD-X=RBiag>L#lP%@GRLV2<9G);&H8kag;HeyCDNi}9Cv-Nv;{|6{6xj_~3Mwc3}0sdLVA=sTqMB(r1Tmp(l0pJSa37Q~5yf>B?PUs`LaM8pHFloo^@*u6u$1Dv{`%IJ6d8?DuKFcI^E_{*}2_u>zJBKJ@b-0$EuQQ7wR)wj(`doFsh`DO7I#+K2uT~U3G(&k2qKESXy!51=U+i!`U_C;IWdT0=lIGVDIIIZp7d*orVZ*Wjzx<MB*by|W2=Es?!;X+)S#ZxIVmLp@XPDo!iWj`VI>H4FiaoS8O)75U3|w?gf7LCbsk>D!R@fO{LW~QO);L{IR)?RO^g>Gw`mQ&_ES*xG@N6{6rYR-KLc?n+yG*y-)v(ylXNg#t<pAE*Yx+QA{j{Y5*P1l+;MMzZfX<x7a1aIK>{vR6sM`sU;-DKq8Mq2fLMYZUOl|-^iLq>V0c~KM6u3e9ULN{hzBFI5Q?I;&s@F#Fd;$voji*_8kWR92tsN~ihG=~uU?Uc-4s?x0O3Siusxt&=?2gQo;Ub%bn^Dk-f{iF>w{*)Hs0l=AP@*3Q%#7MdVnps87&;WWrExaIxXnwsQN$0|Gkb8pevCfeCQ2;R|65#JYYoH(U(|=z25ix7Xatkskj*{&fsqY;2$fpa_D=^G!Qf?bm>iJ_PFT*+Fos>bEgEo{%Hy!}@gP6S2WY6v(Wr0>u;V~rM85#SLfa!@WU?}#R(%mwIh(*rzcuKG`k4@bDn7GPhp9o}YG8iLivnA$uC<V2k>}alo;mYI@OI`-8^}e@o_;g%%^&f#Z+rese*f*vpEi(-oIjoKZ}|zyWuQgqA({@z$$q55AYlI-Uj*R6U<Bj+A?C@LP4koB{osglyUP4r3fS#tVKNXQZM8TVqC>y*7)KLyE$&0>WB7l#?JRg?)1JuPR68)blf3M1w5(ZQ8El@(a{!GEL?)$)?9Jl^m;XmD)f6M*D4XVcr$I5F0<|tFb_6*EvOzXQ0+`Rx{hHhv@DP|xGeDs%0R*ZN+s&s#a`Z)qv#s@Y5pA$v0Dqs((lVRH#dIIHVP~U#V||1F+q`w-`mOlIWM6qYa;IJP&x$cBdMJIvD8LMKL%FE4bVM1g0T<KbmdU&3m2a1TcyCB6kc^?7Y#_qt&@PAq9Ljix9BmgJ-k|kt0>9R^Q#1#zT^V(1lKVrLaXj6)shEnHss7+##g@rdU3LQ~4_&~&hj@N#@5!qE&~g3M@^K$Uy2o9d?)KwP<}=Lx_X+U*7+}!$!z7qu*q3-I@J+)}B!T3S!~a`yBu2V;%euB|T;4cTGe<3JL8v1tTp{2$8SlqMvOfhbq9yHYO=ru>mqzoYfHe~l#Rjlpl0CRwWHRazV{<~col@jHpUlexa@(BmilpaT%F9D@vPD<U^+^g-vpze{(nB@{r*Qout^!gt+$X{xQzie4?dB9kq-ZZHDvNUD^f>)1zmP^R61{DV<K%cf8D4#L4u6*WhtVRrNKj^OoyGO}p14`buoe-4CH$Fm=Pgb*zHTH)jjLdKKJ;6?!wgQH91x+ouj;~H0OzWlYpXL%a?L>0$nO$xx0+b@IDj&~U5C|;#hhlN;)q28e8r}OPl339yY}Vv_UIaYDyMx6IK-);w4#0uqQS|ls*0)Cp?!IH#OHTEE6qr_7|ax72H>#=ge4nr6eO5Jq}elE5jCD=t`g3Y=-$<0gI4h-ys=7JaR$eZ3Yo#jGPade{b^krUGg*s6D{>Ce_(&zbUfRFi8vG~d(g+1Zm)vwRn7{dv^+${r>{9|7f2Y2tc2d)V-M&YV+_rFj$8}erzF}*7r#pff-Bn6nn5YhaXy$Gn6F<O{i%_vsWd&FLyGvQbD`D9ySBe`Snbko^)nJQgf4<aNy;v!@}|dpfNYU$+l8hA#DceLJx3K_UG>XYTgmvq`7|GmPDOxZ4!jKqW?3;dNn5ct=u6)pjAPE%LjLqb$zVw7aBZB;r*JT@V_rj`Ja94&GyZoipA-X^)1j6dc%7u};dWwiNCGSW<f9L(iv*VMoFCHNVLCqafZACAYG(ydX@WULJBIbaepyVJ29uo5R3a<tlXgsT0!iekOjH0Q5>pyUfOj}N2gQc~PLFqG0(%lFmB8aoNUADwkij$NL*Z<FhF6e66@{Wfx}PFbW^CrNIxzLU+eL3$BLg$+0iy6)0NF`?0y{#4P(iSCF`L!ewd<?2h91icTHL(35`5@-yr9Kv?aoRqLV2^rg`Ql5h)nS6-MO$|WD7A;s2M6E#Aq3qIq3N%<{8#!fXi!qb3nz5fVeAvL=<0x;u%35aiyv7S>k~)EOoh~9wjiXivvzgrb5h^QF=L8D|D^})WZV`>8zg*L~=5ni9H%uV9kIlC}$xB1a<pYpQ|1xQ{?%dU-~3VDLkXvXf(kh*w7Y^)jRd^2<lZA7`ru;%sLGTplh0`gdD{z+kzVboZ1*m5>^KJ9SViy1rk^x47^Tf|1qBUeBK|V!0u%y7{^vGExO@&j^+(XpJx%$Q1bYpn#!&{^--V?sOz0AV|-0rUp`;wo*8zQTn2088(cDhn;jwk_SR+;Y;Eos6^h*ocVR1xihNMpK#Qyl53-U2+){20Jr2A91%^?j=C?O?;D1|j`+s`P!7^Duv4O#|U>#r4(E=rai+ZoxVxC6Y3ma)`ja2*2xX^Byk^#*lUkkMBi|Alo+_Mv4%JDK%4uOSNQ^+TyFP820_F5;tb-e>;EBt6*zkz4#_S#0=xzUDofxoxf8#_CZCpK&)Kv))H3h{6O5O2oq_SNeEaXY?s^VU@X@r}(k>aybNn>TM<5X2l5;zH*5=Ekj?JjXY0wFSgiJ68$ftL=@>1wqX5HZBC>8#iy=B8WTJuisP<-?|3xHsXyNo7c_`vM=<<1;E^nuifn2Ao#A`+>o<;t9^|yxxIOP^P)h$K*VTY>ueIp9hh&q`EFgig&^O!dGqSoLH0&CSqWyE)6UhaZ36K6^{W!V8ynX*2*Ax70N`1vfsD^`0rLpUym9?1Ei<5|fO4aAi>C3~=0?plx*QJol@5pfAu7lhom!CvZ4{)y^I*b*tNN@+KILIi9F{?G_%-`9s1tn3=-zfzNQ}M~)>Uf7qr*Wy4YLzqzLMgQ?`;k^aEv`okC4e>8Oy6`*(gT&tGf1In>sEQyp+KA=yCXKfop)H5(})*y{fbl2-?(FQsgaXRN6;`Q?Tv&Ow|FiY)uhE4`ZfsTsPh_NrK_}lBC)m6rm3VCNf%Id|O0OeoSAKAnbDRgk??xR$yeaN)6AA#BeiC{(tt~wabm;Iu!jYeaL$vfW-6+z=y<Hf)a`%rAhd@<k+&;Z~`=d#!QO<8bUW_29lX2JF(@5Y&qAtP8>V29otE6t{*vBPGVcK{a;Qrl=R8>C*1w0s;;W8ZU8ewJsf(;Lv(jNckSA>_pberF_<exY{r$9Y=mY`525RBmarpwagtIn#eyFFV>d?4VzWqp?UTN<YGfS=2Y7396KRQvLd4iQi>JU<^}InuS~_{b;Af+#E<WcXOhNHM&YOeOs%goQ;ZBq{qD(Ek+p|q<O$-!vB3*>evJm@AjLcvYO$Y~cTZHR@(eGP*ya6#wh#|hX8^Y*lQwfC!H>b26M!d-gqj0P6=|_ZThWrRZYosZVBYbTQFJ>Jpj1sHMX#TjUfXNI?0q6hkyFdJE55ry>5EVZ~W(_e)`lm%{IjY%>Ou<zY7W4J{D0)$RDMg=+6_Qo5`4>eWHPo8$UJU1h2|&tPmW<qTA_cr!;7b)>zwzSGmNUGDl;JF&BUS$O2(J-tiP-NoPODN7d(+R&*NBh#EYK1=1n09zTCRm;Q&|(&XRoj>zoZBCO0`|F)*aSmveT)l>k87LE2XzgsRR9p42msI4uD0fB^?{{k$rC5xcvOpn^!kmcQ-HJeD3m{=UO+eUcOl?{CaouIr;Glk3%JLRF$x|4V1X$o7)@L>E)d;!4UeGX-?@>`+%%vGon%nTb}lqw~?>${-ZS@7NXBz5)Djb(*PT563l9uiBHtCO42*Lf<fZ~UfWidHKP!u1~4~?^V<Z<OFA+*&RS@!If~J2W@khQB69E4a5CaHCD;q10p6k4oZ(j&kFQF1^|@9v?TL(tyK&u_`NX<t$)kEf4wNhLh;O13jx~=r%#pcqOR`n;0Lw#>m9Z1YgR<O+$RFlYA!^5lXo-?evJu7;iC5Vb<}`HqRP_{kCHX`tZLHt+y(^s4BZiGQ;Z&cUnVktyjl3<hDz-D^X?q;UTTeqFW{E_JlR5z{-)<LA*SJT+keK=f8azM9!{V+M1(DYv=mQ=yU=V?vGZWKZa2gZdzBrZqjxJlitd#%k#+6UN|ML^b%7`t5HK$349I%{3HoKzf$RRE|C0<ITEhmJ9)Ks<zA`@kXXI!8yN{V>!?V{;q!@~*LxdIkiF|8Ys{**J4&dG4uuCF#OTyWqSjbF59tPp%ybcyt-yB5mo)46Oit>=j>T-I30(xFP8W7eneH<<LGlf8(U&_(Hn>S$2}<)fRrmT5_8{gAcUqUY_hnQyK67f8cEf41{Lpo@uY9p(6}m6UnB0*|`H5#YIt1czR5?km2*lvHRW(*oPc*X$(0CCEYm%0ST0<i(}Hm!!Q%1HHRBOn4-)w0a#CCtO|(mH?0CK}je6qi7hmh2(sJ?LCR@ESl3cGrYX>{If;oayhdQHW1LSiwsMEMYF3aMq%B<IYFyr7ie(?>z>INKrs^K9VhR)gO!Hj`IqvXtvS!S!r_>-S>~aHLDd*Hacp&O)q|s<RzQvm+s?M}x)RB@xC3gN#Wk@jw_Q+BICj(qOiWWFwTtDFyJE6UTfteH_s3xTn*0w?1pYTun6lyRTyD>O95nR@WeCAfb{B3b&q*hn+V9M~0w3l#3c{fy|4qi|H?tjK7t$bQe^P8`Q|$wH7sQ%0H9U8;q_aa!3*Q~BW{g48z;bt<+T6Mnq_Z`_?C4gfhfND@wX{KVWKE?|lD6|qgUcuu%f{;t8Qx9L?5%S$^vtnuZOY|n(V$#>wmO^2lPWi%`A|ufX)=~;g@Bu>9o*C|NCTlr(dye;8?0%+^RI2(xw~m51IM$`-J&+4T;oTi!bEWaHGDSvqrdB}6iw>7Nny9<Kr-<<sG&nfNqB{r)XCOJWE%jOkZcF!Sq-J-B3-cR+V`%~U7O}>lph0|xfpgj7=lTqJYK^#;e7SIjlQE>%m+Yv&QWoDoiN=Gfj3(tX$1{A!&H|g*%zU0Gk41<WK6>DhER!m#_rm!Aeq;Hao2OTAyyqtvw;%wypENY!P)t;nhX*ML@rr8T*bIUVDmkfqQ&=ajRrGE$GZVxf7tJboKoA+R3(H4ihp*m{(_ni>(E_f{Y}M)V*yM>#a#V_<3KN54ohd@RMTYO66%)u@LEO%kz*+4!pxzQ@Lj~}=*j_$b19>9LrqQ4r+Y9SoX~zCLb}mUS9r{U=<#HAlfNpPOHwGbAA4qwKHYOQk9U|bP(7Y*cBq%<OveA=of{=OMUo+!BK*e2Gc^Fw6j%Kf5b$8&>mrvT99vQo(H@O`1kxm!<;Ko6wSoGZS8v|Eb?4gkTQBk)(9RSNuP54C%jy`U0k_k>&kZcl{OwO~-nw(;1=aRAozpg%$SmCa%}?C8{b|)WCI>8Qd?~>E@#RmchF=N}%3Fp4K3!TBcA)WJfHF3-#XTV+m249?tJf|>FQH+|?oK?>KkgOHq`=ZVm7rYv-YKcJYW1T@0%9+ZykKa;We7Q*3dW?AmGOGsxZdG``ExrR1SNf5zif4BbW-#55tMf+w$>*SCg6pDlx)f<1EWnCZRUI7#XtJ8>cas})SKdLYUMLQy(Zr0PQsp}Ta`G}X!<e2=v{&o9~Sm>^VZF)#XS|Zy|i}Kp@W#qtISilI%PJeUI<TD?ZMVLnR3Unnl>dh<SQE18x{6iBt>>hIlR0QYSvb0q>;Ql4JJaGs=Dq;GliAl{l#zRxE#8z0<~5KBYr9YlBETkqX5fp^0GHYV|DAzN>pZI)Y@~=xLsPdBPu>E7WbGMPm{+izQV9oc4F$ro1)1(HGP_9qUkbw5srxY?CscxZuKGy+WR^0=$7B__fBn$s4cn0k;CzBOo3zUgo=%^v^F)*U};dd?QB}-+2ELw+#ymz;)}StpQ~Lp<4qNt@>B}t-DdW)Rw|Y@&Bt1aMA+0HYK3B3Gy7R9l7^b<SFK24(^S7|4#i5-^+8gbru?B7(1Ci|yF3_VwofD$CBmp&rqsrzNqiTsZkdKy72yy?(Z}UE#)Y?YxNGSN-HJNs!Ze#(ste1oOze)WwQY_hr|F%o-q{q(-w!4uG!=|7Mv8mOU}|l&sV6hwSv+MQu*r;+K$$Kg7+_d}nSAa^jV1D4ifJm?I}#S;*>@C8APywhi=&=t$1!C4Tlc230GVS^fxy{UW{gfRhUm%wo02`!g2BKTKiOQMF?A7}IiEXj&M_<0mf3}L;)=uS#$dJtr?HjY+-=w78*z2FU5KxaHRLXNT)f}8xu(VA)g1da4}fOwP1VJZ=QoH|YP$eBt_@K-J)LK;CP`Rm(bW%lM%jJP(sNQ(%~{!`-80Gf2RFZ>36xk7SHv8&vB&{yRU8AJGdyc(B8<(ewv~mDZ;}VOyD-Dtn&Q|r@lQ=?zS!&B)T<Slw#(-a)TCWHU9ioqaNp4_QG}K55ZyY|lQ>}#X@rbZCfnUmSs$xcZd3TnqnU04zd0qMp_#&qa^frpzI2?uB(}_YWZP%ohyca?()?s|BjdF{8YxryD_38P(%tLik8nLsKrp{Dfq$-!(g`d8zqxVuD*mwB@X^>pHq-*5(VY&E<Ld__4Cw=>9Q}-^X_1*fx_ayhE=EG=%4Xx*O=oF^DH!USg6WuM^eq{Z?tHIh2Hh!dPI|Iy-USpn=oY=N*)I@Yw>iL3jkVZUaKQ=RxI}q(#+YwU!cBHSyCK3D%`tD6oQ*z0>C4iiU`D`w_2O=lP7qbgz9Eyw)>hCH;C4?l@$bWwb{WtO%{JayV28pZO<J(;!<R9TSEE&5lSPHieC9wg9>Df(3EGe!D0|PKm6LAn?;F9=1gVf9;Z1ynO4MZZq3|Fz-+YoR@~@alotf=)Pc}2zFIh}gRbZQmkoH#r=c9NQoc5j%N8uy@9!$av-nOpXx_NEm`PQ{tcUsTgx<a8<bZ`(kYh8Ka>XkpaeQV?9rcrPwj&}y(YN3Ok5j+w7utr^RKDR2V5DwLmTJwPVceEH0SEi}=2OdSzV*pB;8~6JW#R41Rw~03e*bFG*)e~ep8F+mRk}XS3!a*1yg<bLPyr^2gL6(ai4!VcIn`Eq`aTo(h^^>mOPo{p@o38%ZOP%TDK0rNP1^SRkSG8o7CKRAEE>FrI_xhTR4!&%4d;Q9RA)w>IG=xnD15$TaT;J<Xhe!aUJ&X~DOVpXtg;pQ>(PI2*r)Z__uX$(bl<~s}bmgt`L-O(@t-zU8tya;(Paa+}{y6DZo(4#*cn$9ig1?HwIlq6#>p+|Gy*_q%evznTi%9oQNv+K=rpOrZ(7+~m^|o+%w<&lv6Mo-72tTh&xOWQAYy%Q;UkXFRV1|l{Xw;1-=<(>AzK9}xUibbW+zSWIhAj{)JoqFxhjQEw?GJSS>&(0<8D<2YT4X@82zReJAXvaG_t-87XIzhkJP>4)(7W>7HKm9b3#~WdtNPD_I5!=S0i#jA8^rtY3-HGb>4L^J3bQJ>fCzP5WLzv6!pkVs$rn<6eu5`8n4xvUKyY7dg(3jWiwC^Mbi2k=BtnJ<b#_O3+S_UjFz9e&x>DcvRd_CJ%@=@cRS=~HFc>TcKd1Q998Hpe4QH-MrgoWY!XIQ^3^%D(Ni}8dEptS$RiK+z#pqn+s$@H$7#BiOC$m0m(^dsB`;5KY%yM{W(w;Ipg42FD#!jHnRK)6gLRo!_o+*=?*;*ItL3`1t343W695g#-SaoAVrF~`1%=OnjQ!U39aniJI&0nL5W^d~FCdi(V1%@r0y<n<B)?IqbW>nN`fL4iiQ?!}Ku-+utA2FT<xs`P77~<vkvW?4E`9L-SjkU%yTYDGgtmDe9x9^Gn;S+G6uiku6!AFdx^D08<LCsUo0G<bb+Niem=t^Q3zyc;wFO;n^v1<dF=JD_m1p}JVfU&0X07BSw;_0ZD@Wj}r9!(f}g!Y0gNwNZn(Syl}e)r?aeMuTR7@~6@cc>Ue;K|ty!k0aBTE?eNTQjD+d2913;L0vX6{4$u=xq`OBUBBeS|{-+Tm@Yg@p=ME>;rZA<IA5Shi)7N2S<`aqxs1*06~Be`OpuC#;Z)M3TPyQngnsi(q^4-j~LTYkKsa5r^$L_+DR~~;3x%r5DGZ*^x2L2g1VDqf93(yV?XL*_eQycpvGuL3E418(>y^B=s>$m^&{PA@^dqy!}r!@taIC_WuXQF(IXtM!|4zsZUYy@sG^nXh*H5pQ7PkgZ(b0w0>%#0d=Ldet}X{kMMA4NA}NR&qa1MhLWh~P%xWfT9NFmRy;CeO(uJM|7z}*J#6OF{%_VD|kSl?TmCBu>Pc2Ue_Q)|aWT%E3Y&e3w*%aOAkqymyDAE>IWO=usX0W&8u36zefpZgW5aXAjw1g*WJ4s2s%2f@<(Aad;r_JP=ou9_bV#HS7cxew17E#saY%?wjNJeF)0SthF6ZXNSmKwHtcEZXj;LO(ir=GlYv2o$z#roR0^Nn+to_xwkDwCgyYy9(YTSHih-vvXO)4AY*%-)GA(jA~|?i_Fws>!2`Q*m1>)O?M5sV*85vyUNk`C`zR#jE<05ljX*>mck-M-~atayS+3<aygSk9=mg^7<ac<O5EdL;gD_H-LP%f21quNXdw&Sb>0_0M8bUGhv^eu3X=^&AJ3l1;Z<hpH`k>`P*{q&()}Czul4OQSGF~+Te%+S~z;Cwh}it30Z!m7^6jKezQcx^7xT)xy2*e(s-OwgoBlrouRBUr#8_>lCk+3@{;28B*K9SLA-W_tm#C&X&xsks;j!2y^w(=55XZ@$Z9D{Bhusd7^-dj+saY}HQg?Uo66ippLZuE<r*>#$g<2Nmttogl0gU%LhQu@W7coC5LPk;Gsl5n1_sQ~01C4!Y0c9Uxj0-|MR9D7ORn(xctNTg`swS))6k<H>BHAYPRoay=Spq@Rm&L3T%)#FN~1Ae4qQ2Ohd^!2M3fc*%8M)I*APcVCa(_AR?s}dvUHT@`D(>$cnoH7pQN#{{LWCSd1tMT-A#u<cb98akG8Ouj|=|q1!FmlAXT}w&HDGogDHz)a&hE7(dH(@&*>bcR6QPh&9jykU2wa`5m$SGMWF2HVStfl?e(yAO6F(h3SG`%CNqAp!wP~w%W;sz{S<ic8cTX)cSm|bD#jI2aM=Ulk8j~fG6)C#71c)WRvV^J<QA=yU(Meu2RZ5FCG|}<*^+7mt%_O0*r!@gv|9Q8YM$@=+g7jH(jTFLlRT*L8eo}Y+A6>FA&mn1Vn=jvtiza+0)Dj5VgkkEqxkVr{P-w-d=&p)9mN{)eS8uxdJ->q055v(+6UqwPUfG4_1c^bCae;$G1^rH(QU54AwGfS#G78~SXo=G(5aV1!)OppqI6~)TUJ!)xx;0aPv2dHdD)Aq`{Wfm^Il8|b}@%1yox8932IyCCflS%Z);fH7JkrZxlK3bN#;yKGt3z_%lV3q3Z_lKnKx@|jxvqUNz7WGB2U_YgL~FkX+<WJTzJ;UXV4xcSJl%_wyn0s`nI?IlU4gXQeBN}Dw?v*oD8iBn>>kHm3FapmwVhMyE6xRdJ@=^?$S9+`>aCpw9Fx8(Wtp+l(oC(mN(AMIb}_ecaes5IIox3?U_{_i?mo}%~~jpTg=lTTS0f~e8^rN&9>L`hCf_3#}N(sTlcq-ZN4uPENyYxcw=y!Bh4{tOeIYIOFJR(4C#ahQ}gaueKkDWl$jFPVn<{`%@Nh}wl+>>_EoiYE2EJQu$m6CoT^4tHXa7Er0WiIylQDIuc{rBNhL|`N@G6bGhZYC-N`a0YLY-f6euIXVlJhkDm_*NE-RhAbHv-?kC|Z5MDr<_^<JY_$@xww-_)jP+VGgQ<)q9AJZYK}w~KPF@fQ0ZnbXJ{T}`6i+)`PxJi2N!ZMUipFyKU}IkJGS*pOdcwZU8U&YYp^&wMj+60u0o%(b0^z9?%ifI{6$zS)|EMb&)ZvYiMG7h9cazaL^6I@Jy3t*ptkAY=pN=%MO|2jc*fC1a|TdbL{f0Ob>Gbd+WutlCbBM$q3lrdw`krUi0StcG9H4pG}kh&@);J&GDgZY%V=(CEkM8%$i0Flz}0gLt2OleIC4QyO?fHf8qYR6RM+1c8DEe`r`MBYb4yHX3lkKshK-Jh1H|r2CaV5DeOh3bmjYjK#&YNdG4$_ELm)WkHwh07rP0E%bC5wCoU6qA3(@bbzcV>ry!DFp&Ue(d-A^uoK7kkyQ+!zg1{4UhNFx&T22b6o%(|{fpiHxl2&)(uGFQxp3jB^XH!o`t^QLKX>uMxr>cpRgiE^it@*^)7R@yo?p3MzjVH)nI|qcj=cttB&;`{ggzUc;L`a^7yF%{*MBkyyBEVwaN%P3RPU*$8ohdW?t-kuB3$vfF0EW|)ax!>(X*W!dtHMzafw^?ZM&?!q4T@eXCYp+ux7choCZwT4`zvhlKaBcu`7K0gs~o}s17(wq(DBh!J>n(HyZ`Rs4Gm}sIw&R!mH5BkCK+?p~60j#?+hE?63guNBqW{(HV)R2nrSRJrEm<)H3-b0)chr!*e=HUY>^GD`92LYKo7~+~Td&E!bnacR49JyF4|wDoD3I{?}PtC%P|B(w2=V%1xkRd^vMw&A(7B1&Q?rUEy$QNQ?75x!+l*!u&pfybq@-K)L3ftC9We+?9h<O4^MF17E}JO71K+sydyHE7>yDvK6S>WraCKC+be&|9OjSIO}&X?=)}|lY|edl7oLKurwGlRevuE_xW7R!CCAMs}=L46do4hVAS2&91=vW@L(0EaM>~0s^NdOC#sP*ln^Ti5g|)AoeW~dNQ+69&?H+oy{$7UjL8o0X#7{FrJ5++G`jLd?)gds1OcS!V@y`&z{04Q)5xV=(>v21&I+!&h+)yM2TXTbBQ0DKF4^It)Y8JGtRW@n9P6$zlZh6(YCN6@mH9X=E<nu{j%;q1vZfdF&oU)F<#lJpPci2_pB)#?VG8bi;lyg&U;*@n_7J%TaQ<g}=&(~omJzRCqn&9yC42p0IE*JVI+zttzy{6XL|l8$W&UlSmr_}ve0mGW1#W7q6TY%|!ZIOmP9bzX9_^U=AibVGr6sF!uHPjtON^kiMWi<gUlzAU6kb5)k5(*776bv2As5}n7?iBl$h)H(C{++b!6;+%Tkvjcz2>cF<K1=#YmkE}>cuS7bneV70keG$=P|9AGprO5Orx?np0lnl%rO*`Tb(4T>pllHWJ%-3bdW|Xcsqv|dg=m@NcFvYPpAgGjf6s_d^(<bH$&XUPr#7~yU7+<j!f-USBlQgBikRQmP=jBW~K2}=9x+#?OOO!bYb-=d3M#ViPUE8Gxiw%huz*$A|Uneoyx{D6oHR=5z4NpW9ZDXyJ)Yn0|cP0inrIjSIq%D;Oih&qrL77?*e^_nyTc3YIz(a4s#~Lh8b`<a|Tln=K{`8;xT(Rh;_==NPKb}()Y<EhY!>1(I^a9^{Z(F=P9-Jj@`n`$8G^Nnn!43xkJ#pR5#RRE^7^nACNn_ich?p#f4TEqy1qWLvqqC3ryXeyUXpos%|4wSFCej4)4I*jD{inBLWw?^fdZq9i2}g(4Y{n1Be3jRjdQ$<!LyXN&BUyzWNoTvcACgL%YWPIW<TGvg+&=)F~dE{>ZYWW7_79u9@4^B{Ml^l|7t%M;0*j$CK+!pIB?JCAb!IW;(Z|?UhkW+ULjw+`}LpP6rho{x!X!ta}vwt5tbQ$S8^CfueUIa1u`u%l^xpuK<HKmFf}CnT)NFvrKm5>7XY<?a>v+tILk%LCx!6=+&h7q;bsp!2<EWOxjhH6x><32)6Zwz_}<$b6}-$&Vy;a9E=36{8Sz|!E0ZGS}1Hhp7nCUWDLo3I_{#;XU2`DvDY7;YZUJgQ@1Cq*D87;(tR~a2S%Zkcu$e%_Bqtz#Ky+*sZe#%4h0`!bvSP7{722ByBt)7v9KYYa-9g<1Z&%yP3z>U>fYPJ%6GY~ZBn!&Ztk|$fBUE#+Z@TFTiclQ+?{QLTW`Q3<$k8+s8knfK<dDYPLU$Di`k;V#8tF!Ri`kh6{r~q>y!<zVJIC%F&Y(1Vsu*xT7^7{$D{4kA<3BXVv!y|w%rXUnD3k<!nV~!8-sFXW~I8>FjfW}pGjG0I!fbdcQ@>DT!0vp7fiyJr_lt&O}x3^fmRf@u~DjEV9A_Uc;QE>HwsbACc9P&mI4iRF;pX6yG}cUsOw$6y^&>iw5LhoGs9pY($nzey+`^3(t~vaaU07QUzv}>XeQ=2EObjrTHRop1Ov&|E+bkG2Ej1MWNUuBj(@GyUO>h^!wH>L;<xhja8_ZPq;KAO?$)|@SJ8q=IE?ow&`>w@Xl$N_RPn^S6JHtP8Pu5$$9x&x@aw+OW)@`XY51n1)!@30R|a7}<?9v14F)KlH8ntjXn?nw9K{ncA;fFM359I##AzB2S2#=_SI79eHFh}J4=~P7FO+_^G}3Gs&opt2jQeOUgl@Ccs<fg86zx%F7I@c-P~D04+P-%u6hGHp7t+rK@yg|m>p^Fgy2?zfS7jup)il^y-NVk;Ob|UA?gtYQij+{*gjr2Q=Xn_qf8<Qzn7A*0SHgpCIL>0`6Hcp=&8Anc3cFBr-SGA&F|<T{M@23`)wd;5Ju(<T1tX@7TUeAit0r4%%@2lVMyk>6434=kvVdYAxCmUX_;rZEBDL6Ygx{LmhFyxrhHRW0L7qbW{z}281(nLq0CDo@v3zKY%P7CrRpn_7Fy4W#aX;{sV(&nL2_B5Pn2I3mV;Hgh(0d7Wb4a+b!~0=4X6G3oS`j+W*gZcM;7>w;muDDo9(cCPq7=Ia5CIAOD5Ma3A;tv)Y_N|t17QZi43&~wrT$jhP~|AMeGD<pT#iHmzzmW73mX2DPXeXx?wY7)nu>8S>|w@?=o4BMtm`TQ(O(damdA&qEo|uu2|=rcOn}_kQu!-Y=u+z9d#g1)3?M^lW8~ewdTkR=MmTGLFWje-FO`cHa@aNE@`UurC!}l`M%sx6=)(Z#RsvMAU>|2x!*EeWOXCFOqya~3!mcq~y6VjLUO*J7;kZVjah-Zv4Ri>!rh8xdWeFaa1z#o}j1OeL74E@0V)U&YVYd~?`1fJYWG0xWC(bAt#QVK?f0Rx84>&40Id&#7^4R3~0sIT_Pd&y{Jlhp4uOlpPA}EbcaHb&YTZV)p9y3kALOpuv4A;)3r5r8b=TO-2cVIe#X&U4BTHy${iX)^9*~J5c8)k#y+W1CQ-x*K&bIv&YwD*E!0R=;)0)u5f!ocJJ!FVN?W{4u`>;{Q4L>D(&XuFP)dRw9*M|5SloLXcGT3dcmL-tVe_9R6O^ARo$<Ya3{%Q(>I5SXrglZ4?gk%(QmeQL~nGZvE^G7wRi3N;HPMFM@L^agaivwjL{Y3`{LjX3bn#Mre+v@xJ0;Xmz8Em_p4SgoN$^ugR*1#{_iT+th>=FHj)h;@0jQi`3gTBz<Qitbw}nPIJP3bm3^*7Dj)jWf5uGWTGoC*hN&sh>KzEGJTyXO*9Nw9A1cWC}vf1%#y%L&1~KOD(4JT2w<NEPeArOu5?FbPGDNs_SlV9Au^tODMW>nk^a5Rt9X>wa%Psr56)0&|NZ{XNf{g^mi1sm{TFfdN7Uy>e7Lw!ooj`ReSfj+m}(@btH|HA$nVLkis;Fq+CiYH&*P!6v{s$_tLO`=I$LSp==BR|MrBMNt~#x{R#aY-)nRHjLaBf)WQ&@r03$$ni!uCq=P92dTX<aj@pVS4m+1LnZu=$1~Hc0kn9fcD91)+oqv1Tz1H@6amY7x(to8W&V!E&L>`EIBF7>QC{A75`X+dal6>8J0*9Lb=N$Ewr>$tq0x)p`#C-_mHSL~FZHIUm^BIfoUGPClfki-cmmWKEQAJv9sZ7m6ZOgo1UfZ$0&3mW9U<WO?+H9N&JW5XuSOiyjqQnA6cdQVK;DVDnw-1}fQ_WG_+>g!jli9fso4Gf&a<1vI-Ns|JPqEsEjz<x18LgXFG%XKM4Ulfh4tfegHpv_ugr-S%0gWoqFd&N}*wZK;tr!N=6w8V}bwMo0c;7M+OU)QV`31pvMqR3Zqa>Mz$!fjcxbWnYzIlq6vU9i>bwghY%J)unr@a8A4m__Hw|OW-v;row3}s5Hlr6qh%>dtvlr@1oIFQKWic&N&TSEs@+*P%>?BffRI12dqYb!>mP5LD4mxydM0%<VlA&Z9j9!7AYpw?y7i$pjhOlGq)!m_rmgOE$}wCuzxt}TY#iO_Nf1}gpN*n^JNr$2==WM>kLcSYDFI2<lJdk2YrkAiPlt<a&vy`Tr1jwc$&hX<unpJg9#2yD&tj8|uEn&U3OWzewrN&bPzeG8p4Qpht+I*PT5#e$$y2K?ac5=u$rqc=)0!XVw<c>emv?bhYZ&8s&zH*Vc*UAcbw?%k_*DRG(<MFgj^Zp#)mRl-&pG_umx<z7}N0X{P;W^$cS=)v)Qmu_6YJ=b9g$}Hohbh75|S*Ea7A6b4*6SieqzC)pHyJEjvH|&Y?w(>O<_5q#NhG8#4uQdunkZ-KvYAjc=<}k^xmN_bxNPnfCFlH>lrkZjwd*-ZRKzGTy0F&YxVpkd8ojQ-J=pYzQNX>o}U$`R$5?RiN0{@V93rZIe1Bs9=haj^p&_x9I$c={7uoNj>VywJ#$Qi+hV-+GKtJQ+HxkiVBF%DO4;JJM_j@7qPv~28*@Y<*wk7pl<9;B$;@zox>lUiw*3_B#9GuCG=e3tXZV?*B0XV-Xkl&zY)lh>Uh+kUC0c<}*Prn&jf-Nv|WaX}<a%tA(gR_{f?<8@_zF1*HrnN4=^UT}w#rU|d<e4HE0j-%4Xm9rxG2^%NU7`8c+vU6rH?uK})=}|(el<%WCB$FOeG7;$*NRf%AcA0vGZ%cgdZWxljLR4sb(-9CbXFg#OV^B7U7qFCLtrw<2WW<KeAr&Sg?oElcbr8`~!mQDg1xb3%R%n}auTqBG<*JgU)Fex)NXkwq1-`Ro;41<0f)H*jd1wMzUcAO)laR5do1WQnEq}K$S2ru$Itz^%rrd6(Srm63>{f!)iuGI&-4ht`PJ*#jQW3N#Ge?xzsbW(H%EmRon2ck(vH*q*Bja#bmw109m4=*}kAH)tDFPOWZUv8AaIwILUW>=mH24mgCM!rn5s(eq?neito96tQYHmVyPAClwiteo}c`tj!qYRsM3;5Ma#)F7rzyVKWr;wH!YBQdI8N5T5(YY*l2|KjgQQIL$T~u1BlSD+%djZ~MRV2IKQc%%zv1YI^%b+}zgA=ic#OVnP5q#p%<Ro>sIT_qszq!g$U`i1iQI@1`SmD-c1%dE#CyopYQYlJWF5@R~#HCUu)IA`>vclkn9aTx-*ZVE*$*wN5e##_4!gRhcwvfp4=B;|8F(*gjFdt@|4>xf2kPtbL79|Z51LEEn>80pm)t>P|uonTDSBaTKM~0zDb-B9tk5PB;7<Hu{WKQi@!bzOi6YWGnvRH-t%U0N1wt||6U8Q`jK>%8Il|hz18mK`mo>S4<N!SxkM{JyM<>JB>8zmLzK%|UJFAFPJkxuAiwD2Wil^GsHBU;y%G`}io%fyI$69|%A19PBZa3n?mYsx(q5~We9VfY|I_%mmZ*_^A@d`vAIpVkNs%Ex@Yt%}`tX>Bzln!6R(r*vZ1;)#){z8jQMJc(eA6GRNCt1{k8a_>6e038=P<Tw(7^Z-^$HF-QwT5K>}D><Ty=3i0Dx>^=_wVGdJBLQCFQIaK*#P7HsWj9U5*t#g$(iJheF>7rVVTD#q+ff4d5L<2S_TA0PcQ!FW{N2qv8#kXv;lp5vq+Jl&LKuE>TWyQ6Z*TkF6=ZDLy#RG_%sKbs?liM4knTF<4i$Apbk9V~Jv)dxlVCE-j1pLYMxh*g=fPBxX?GWGkaS)jN+m#WMmseUp<c-{d!u$m4=XjEOzA>I4`mXjU0)M-^qI@hqbjBSmbxoqz2$aA<)i0IPCuUPtisHbRW!IBgsV_=HysX6TNSIA{g&V)r4Y)go#NC2n8nU?5KL6JT!&<u30}EMjgkDG2Q@-%h_rH|*^&xhw1GEGvMBf1D5VRI;)!P7d^?x~utwp;kQs&Yz9K`DGY8t<uC9A*IKu^|6CMd80^K(#H%8TNVcr~8j0vKkjD!N&c_01clr0V7fMJjZJ;vRLX3W_*WfwIzL`!#V4%is8L}t84;~?yRW+A+f`=FJ{l<jR9sum!uh9g3b3ZH(2t0rZNO}KgWQ=6@go3}r)dADRM=JGSDfLBt>mw9L{nheR2T44jVcAOqZghiDJkuv{LoESxbL+qUSfF!K`lZHbPDT7JxCtVSRpEVjOqxo0Vv!NfQj(jk3Fv5~ZWei|bqOJDjX&P_hu<+DC3WQ2JoiwSBK1C!9VUf%jyZAqMg%)NS9<*9eg_H`UyGtB{+<K9+eN7~x*?2Db!|jUiukK(legM#EdcR@?*eX!fF1vubRy4PmH?U>pQUfFz*)URzWJU!e(?OWkwitbxDvP}3a8Q%W1+IR33v)$px7YKaSn=vi;Tcs@379iyaF60V!`_)QwOp;Wf?BdhDbxc)cWuL7eO<w?)q*g0$J#@sOQ7wiES{}QFjcz4^k9ZEDlZ^-wh$t{{7y(q;-qFpp$BoLw%es(Ro&<DS$?undLh+)4nmJuL$CDl&iu=z+Id&{wBt|7bPN%*G3H=&Xiq~Qc#idRPD5ll0z6EQH~IjN!D)(QP)n5o1X4FK^2UQ85*aTF4!lop+{HW8rH?#4R_;={X^gxIRvg;=5qKNBk+)MvJKXR%Gum+<VY!0{4>Xe8Zck#M8aO9)s-dZd`K{!Pdb}k+(p$19Uh>iJI_3P9&eZUcrWB94;>TR^W3KoySNw72iXU?=OLHw+Ue4nm?p-T;=VSKhF?;lJXO9-L{SF4HY5px=TppyjErnGF>qo`A0$XrVv@lM)RGMd_U<ks$u>CNl0V(MFDp580J2BEQBDLu<$gxMRW|_EA5ifd@nrGcIg|lK7ForJk;R3^-M`L_E$KYKAWguf*2Jv-(T4l>&E-h?&i4L^z$TN~n<RDw!&W)we6e;IGePp{{BW1t`=FwbGrzo>_Q8jI&&nZpj!=D&I0gg~svlEG$*+3UEA~v%vEW<J6LNw|A2yXjWIlIBErc)nu#z&NBD}ky_oB}cg-Q5^NO946SMIt;U`R)gjDGPg9Qf0nPadf;Xe72`a*q;vYnibpM4@N1U3UqA8125i#oxdCJb3DuuJf+5{1ZR$9J=Sl1_CmZvPF!JNfTzkXXUKz%&II&E%VMSs;`sh_jJIc#=wLxR>l5XXKMYaj=;F$eMV6BhTaM1mee{jQkHy%1P|TexLQwgXmgS~So}((X!gs}z%)fJ4R_C)1={7F8;`3Pdyd|l#ST+*t{Q`q2vVgA^FuqOW33@gj%R0n3LRw*y6bN=vn|QTwTGX<Z(3^7=4$Sp3>T6&0$|Xkl#v!bgL!_2t6<p~^=inR>+?-5^%Uah4f%W37%hKQKtBk!v6A!~7*fj|DambttIMkE7#TI%>7X|z`fTU#pT0!Z#fj8#I!7ALmwFMFa6orhl!C}m*@*zy7x$yK#UDKN8%yKz)7xf%*ZawP-;H}wNeY@C6rF54ZX@}(FeCh1HJ8%pg5tTjK?R&Vxw=Bmgv*VgIWy(mmr&B|RldL5u*vK@%SdDtGmdBOlj+GlwhB4;Y@{awEM3tH#|3PJ{oFgPJ=euiUuXE<#EO3mKT8pNo_$Vme$B@RY%rDG#rd`sH*2u(Wc%JJG6(-ceFnqZvg^dhgEoV5braMYOHmW}Y@dEK`lD5#K5#%8NU@MJ<>yuOllv%UMq5cubg@d4*@d2v6irD{@{>0iEFGlIEuv4!YZduJ7x}8*p%wjdfk@Xj)X*GY})E?TB&C!6|p-vM^7=6r=fA}o9omJIk^PW$-bL!JF5G@7{;&^jeJNHgiU9pp8(rYcbs|;wCzuDJ|QN$R`o20daDPgM%I80}UM3zgnseaXr4coNyq#B`;0Kg%geLw(7pOgYc61Km}M0twg*v(`g)6pQhADZ#;Q460phCc0auHbxbs)Z7P5${hi2aC^%e85IK{5$HcwsGfgS$VqBowTLG6C{(aeAq5mn>9VOpB@L3uywxIGCx)<{@99l=L**=Ss+Vz5w9y{TGkKGo4tsDG+i%i#VVAh_PZo_Q;w_Dm}o;5D68OPeJZ(<xovVeNv5Q|ymu-|qwf9Ly;I)RTX$JsC>*<M)T2~oqUXgv#*&3%zHyeu7?(>$eaiKXd6<zbtZ$dpQmj_R_;OUCArFC?*<}?36dTy}^=+q*eeroZUr@-8BbUV*MkUA4G{#sf4f<e>-d}9n<m(kjcHsY<EN8{IufveS8kv<5evx|Og(f;~U<wK1q(O>9AHlFIHRfqN_PH6^zuO@Szg8Xc5yv)gMm$gwk28Ki#;PuH-?*qaRUo7wl$cjHhM~)N`I@Ia&0k(Q)!+d%9uH<7?S2F*+H{yUAu`i=%6{5b94w5r&S$K4{9+YiDV40bp5GDHT(slcPK|E%m=yIW5*3;-UAPk&;qkGY^jJ>%@Z_Yc2LX!n_Um@LJ)Qv()Zabx%rkjg5p=V}$exDHXw9Z{X5-jUo}4(H;y@hOiQ}C?xXMvcRxo7B3WYkYUtDQ4&a*tAx17pRVAMZ-R7Y&vYW}pCaWI0l8wU49o*9%?!c^X*HZj8Zfw-o{h_if|Nvu6xMQ5IU_{j(=BRdk|bDP<ODqHtX1>Sf!hR*0ZstZk?!hqez+g`kL@6>j!U@SF>ed68?K95u9Ofxo^hFUpuhKFPW+{NL^b2|6Us%vvpvYx?U;9<fAy^3P@s$NdK1*CPv!+PXmuKRd#iVK#6cpNdIP`Lbf$P*S%#DQ^+sWs+CK4`u10;*#>?9T})kBLVj0qQuaW1~q#bfG(zk&$s%$h1sZMkW!PL6!#s!n?){-Jm`j(vZ=UEeq-D>jh$tXs|gF7ggX&_GtMi15uabq;zBOk39EMW{SH|T+PQC;@@u#@jSJvf$fE;)GYe&LFxDYyQu8U181(n&Sc_rby>b<?p$yh=^S5`Hu)H+(k34%Rhkjfz4Rz9GVWKkv@pS<p1-v1{BlfQj%C7-W6UY3$TT7*7x13F5hfm}JEO7ZSf$K(Q%n7j2T)l6c{H`pxO|bmg|9~X(IiPJjgvd}wu|F)^>h-EzZZ%^Z>)uu3)&2k>+=PhL2l%lWDDd2*A)h7#kwV`mt4xJ$n*;C)RH+yYs5LDrGmJ=bGpn3qD>16YH9I#oxy$GL38cf2|4)1wcX5=f8hFV-myp4c<UQp#@&UwLM1zbJxoz^byndK5tQ7HP<uy+JEl@AFVi|3_=qaX^(pCEY|@XpROOH5u8)b_g*85KY3nF@=Gz<BrFxk>fNcdSf5I$-VUi}xXsVSjaV<}y=IN@nq9>rO9?OvkZFT15d?MQF>106E5yK0vudYT2MocCKxDce!KuUoI0j(Qr^(R-?_<!dvJ$dnx|I&EJSXTx))Q1LXIsAAWjw%xTm?OLb-C<PONf->7cLA<y-!Uwu+g8)B@X(s{&&OLw-3QiXI{Bz+rgeI5i!C~FYZg(ukD#DiM&}BZ0?QZ-xS1TA!8tN(v-Cy&k+~EhIBOwe1^&>5g&7XMz*-=)*4u=mR~YNL65!@rMidSY#_T}IIqWfLC+20eyQHMJQ6j&WWR8cgj!9JKXOS<2g%NMKHjh1A79K|Hb0MO4dnA|`-7NqWeYw<4hxSrWQfkbk>A8BedE9F#&(^Xe9Bi~m_BY7u>Y|B=^a@7<V0K}iVteRC7s@R<M|n^c9Z~WVDn93pMoh@&@~MWHoU(@Ow{wcgv?KF!RM0yFldMt}aZFVSGwVcyD4mtt;Ar+G+0QvLiNVk74-Rogi!xrSE6FQGvcTi>D?VZw=mf24q&P*umdE>JdzVLL?~;pGu&}^kM!V4#YpEy*VwSyV*siX7%1%(b*j4e2Ix}0*iU=zMhif_%cJ&nTwgsaoI7b+Tj7Fl!mJy2N5!&hC;|+^?XtEW-tXLpYwI7@9e28W{dhZ;noVC$0dEHhFrp5Pt$L0{ylb@$?E_uo&%$%mU^Fp1Ng~gDe)-kevJPd)5hXV{rf|oty*%5`ks_tSBsGJZhBu>je;H`&eBT$J^Iw$zn;|tqIEKw46Sof!oZJw6AkhK*GI#9Rj?mI}M&LdK~Dy=c<U>4$YbX`G?_0SWceq8vEBWg}hhGQ||eoU5yRdpW#&!WjmC&IOSAX?v#4&U=w0emb4@S+TlwYPMR$0B!2vpiA*_?YAQedKtAs>I@X9&6K!wdqA{jtR*}%IF;PTA4RBUwk?*l26zPDhrS8)bAXF$$B(OrSkD=!Iu|j>3;t%&~+sG{kk%L%shR#%o8#QoYh;`AT?00l;jjk4OA?_>+Ldy*nY?nSld$hl&PNDUJ_9heV6V_rpAks`TQM@x=BN?c_y!STt~G<dB8Gb0oz^x?=h0rIGnWPswr~;>r);jJBcz01#BC}QLkcGssj80q8QO*BBO}Moq#TXqA<y|3B;(<zjx|YYJF>M8~$7O>;Lls$4KQNt|6BLOz5iwWGL|f9?U&+uI^v@pRPK2Jc6o^Rep~d;KvN`(HY>!%F+{1ma=p%Vwt;z2m{`#WT{paLoQ+N=I9l-!Issl@wjR{Qu6D{xY?+Ay=d6Pm@)UmaE!lg>wU-gRFm4%0xcRd>B8E=u1=z1Gzcb9I>UP_K5%rWp6mo7`kE(Td}0lQbT}PUwDHu;`KFbspT>FtzZdRB-LP7f8M*c3<>A3Nmty*?Z7fhg-5jT|H4Q(t-8?AgOVi;vq>ClFNL816!8nz<Fk~)f=xKM_0U|kEW#(}OL(&HTrPU4ygp7)-ok84LU8{$^&QtZ^{FC*I{Y&S9^G|ic^XEG7Bk0%9hmA|!9y~nR*DEp)HAz-c<;u(@%}KDR{&;r!dgG}l?GUE3P;nRVyJ7=`;-0NlF{=qKO4I%>*2>PKQipvQ?9`#@6_J2?6ecZ<E@t<Do>L82Y3Mwk_#P_g?l2hNm&GhxY{?9%&d%7&X%A~O7-S<#ro#$%AvZ=fHvn3DsO1mCU{qN;5R=kX<QTS^J=B_4xJ7CP?WtKbt6Jc(R>9HN5W3Ps4aQ3|%AMV|aztxJmpUsB@HDBH{K6TNZRTcQT!Gkn8#XRe4B<$rL(O^rA<7v5b+yNeAK^D=0Ng6AvnsN@xg<x9Atilbm<bjPY^bNj2}Xj`CiM|;zg>=+%5FB~SmRX!Z$MF6#f^)2E%KU6t75W&vV-l50dPii5N1W{gsK<%J3cYtBy8RcQ{Nc)@?JdG;t4f%#hAw&tJf2C)DcQIRJ7QUB$sg2L7%%p&_Of9rtXKw@9eYn3JBvO&6D9h<+M;VYNGX8igqTHE#5H3Qzn3mS^zF*pXf9IJsImj0Ek$A%&B4qc3^=b`<!PmvS`GvAr^W_^2!i7%)RxV0TGsHA~e3y!*xh!2EXx<vot)WZra#{;l!z=Ze6)`^V-Jqt!uaLU_Zxjby`AW%nR~yH$Xg=<~L={uXBm!sDK4JFP<=SPV%4Y+#b>wBrDTY-tW6kqt!{cGeLyMOMlF8sPb!y9FLd)Qsn*deZL?nrnW0N52m+a=4+E=x9Qc}Uf@}sBAdfQ#W+tlfPY)L`|=dkWB3&q+Ks-|Z`(x<sA=o4%{PB#0<TuCVuq;ktZG>;Jo-p!W<twa=@DD5WIz55K+X=6rNGfN7_^cw>{ZTHk!%7vg1rXY=j_JwH*ej!+PZV~?kBEqempehGQ#pc>stqLKoJ-NudUtRx5b)aPNQiDj<<<H6}X**tzn3LletmRs68GIW}0_H5A&@`wfT4wBIO}1q*;ffUSL+OL_}9ky`U5C8QFytAX@fyhm{xg`1dI$hvCM@let<^7X;SK7BvGok`o|FDLGYsZjBiVV%31l9)*$@PxoV{Ee;hul@5D+1h=Bzfznp^!DMF`98^GH^`kT#R;o_@{=HMTWh30lI$nY>OC~5?)N2r-+j4YgWNEZp@~&Tg_UiRkeU485v3mUjSFayYum0amm98=$+FrQ17#<*b?S&IOHc)WRNyepZin9+@N7hHyIMvMn$1E~rB2P|pDHk;>Vz;}f@=~xj8k*&Gn8O$oRC7VfbveQ0nnDkVZ^BW~8OU4QaWDx6gK!Wf!(0#}AszD%RQ3uw=GGm`$V~#<@rrT~4oO-m%KD&JvpB=`f<#t@T_C%Wv64>bNSI~5Ghh`-@T-Vv_P8~P1%nt94o^JyS3u$Fkw%I_<sjrSv`jX&VCe3=VskqhVkDyCewJ9nrVU|do-@GQR-ee^jk{N`!&6S{e}qLB`$0A?@i2is6k>Xhnp__Brn)2y-95;5^I6{J0;+U@=Rl665+Rfng0KK4Dv^l=-wZ~gMD+-hDnq6@b9(m5Rg4{SokqFjr1WJtmL6YIt0!BH_3cNhf_IAlOX}i<M!h@&+_I71=0|*cqUJpzv%L#;el=5uwbaGb6YJg+ls|qpK@FPMSZkaouY(t+J3CPFS^y``#;EIiDTro1Wt+cy`?*i8T!-^x1mXd!>JU8Z-sN%7-3?b7{+dv|ORkgvxr#oKnxPl#q*1TIT6a6MJ<D*jG0JyB*u_NXHKq2cv9d-ltx4&=xD0MVuakg;wicy@m5n`%%)~V=<ZOTi8lw(PWMjk#ldqvc*r=u?yBpC#<Q}b_`GhOCQ2bIYuw~U<sFQ{}vl_|;(y_8f1FCrlU-@$~9r3o7HJqIqgJdlghCpye3{-Qdn6pu*r>P3N0!W!orYLp=q6Ka3_je;;<n|}QSo_R6f~*iJbLGd(tCZ7wdrY3Q;+pD5I<%;Pwzom<>!a!hQyL>DncZiSnU?MbW6I)C7003cdgU<M+nf^_8dJUgnbobzb*dW4mFR_d#RT6~isWd7tD};oi^lEj@U|MEQZ)J;bCbCgWwkN84r!922_*&x9Mbg`8SE<#I5>OImaDjJJ7gqArj9yO78|$jow6p$7Llsu%o5Tnr((0*JLQ6jLQ&OHuEJ*4N9nbe)p0FrvOKa&Nj<wiMxZh+Ki-(GON-@<1ULu;lFzKjws?^%xb0M>lQw5vqkt}3bEZ;qG^k`JtzONhU%4RBG!aF#91Y|tnroDq+37mciQ)?cp@-7$5V%L~OHo1sMfT|5{`BUpJ6B#XZVdEuWW6jb^$kopJ`B^{m?Fi9glDw#Gn-t}QB7cxQT81u(Ke|hRiYuZ4)EZ^HiNAtniH2n2{qJ4jA-k=15mGxH<LR_+k?MTjJ-mlrH$@r_o=8<=D4rXG2o-PeaT6+5yEYjN|Bc73n~IGz*w{Cs;i3%a<pmTBpBhc@j?$;)6R`;jNVeIT2qj%8<X(8+k+4f-AUNb{3TIeo=nUuM4?3_4KL{uc?!uqGVUVfau0izHeg@7=Cx^C!tZVMpx$h^ozg0jT!|_k-9)3J1n-+9{)pH_yhc6Wy9%sYR;3$^@N3}lMI|<YrVI)Bhl<NfN4FJJ9!uIG%2nyMDAfj(kg6W2-ChJ;lTDl1ve^z`#VnqB!)ZclbVs-CC}|~QIJsNM3$*rQKpMI=XnPfyCP+aXu_qXIqMa$)(Xj=G11weFWx9gbcegjL`+QqLe{m^BV}H~OCX-->Px`H>S}3blwt`vR?45D1TD-7)TI;JS2FXh*HkMpdaV;nply8wp1%Gooygl=14A|Q3*`{^%lwl)|$oHO&0czddU^D_rX@vqA6D?7%C_x|NIWIDtbjVA>U~G*i(|>>~(sYCX;yMlkHY~BBzI8;jxAkjLw1VpgAiUD7+jMrgA$B($pt6imnHnNm8}tYxnEE5GD%`9D-7X*y-KxeJAtquXN1XuT(#*vRmp)|F3lD%q4`Vq09N$;GHe)DE8EEqpsFey6j7ZX6$J0@EF5YKb6V42L&ud?S+M`&F3$^BGlp?udr)nf?sjoI+4(-JdVK|!ShCa3c&(>%-9eVsR^8t{qBVk6W+q&0DtSG5y!8uA;g>ambT9bH0*^PNRMt$~qP(2CyE$3A*Ek+j!`d-|EI;}Apy*vRVLts{Q9hij-Q6oR}YqXDq4nmR-nr?<aupN653KuXO=gfE_c}kPP2zCV!vT`#3W+03e5QmV$X`-r#rLq;k3^^UyQ4)$%D>s~s`^q-yobg-+5o(XFX|&3yu*N8Nw1^p)2(NJ|0iXtY;C9xYefJfG;Mc?%Y0!;~Ju^v$bI2Q%1+d1qw?fS$+os?>KuXi7I~@cQua6PW;BdL`!%P)+@~sV0yY;;GHQPpMpSKU2AP64v+92MCgM;*Q7_ktzsJMec8j|AuI8Kd*1l6{VpZU#<#zr%I7U(D1{3c!s4vaSb`0}T`_Dcm#%ny>W)z&%Z_V&7WIrlE3E&X@^z(Z7KKO_h6S3GfFjJ^?P&Uh;;B;e3K;Uuy>d1m%QNz~|4Rtyno*)EiGF}S{?Gt+Iu>@kvdH=Oqx2hK_lzP-}yaj2D^h#oo9h+PE-lUt3p*b$Zsi8@?>M4q{K%1RD>>I}{XEq^#pJ(;s*=A6i8qikXE_W8aE6jc{2^eat<#BF>dx649ccQfTKk;>E|6Zb4Gs4^kxxS7gBKs}^b*f_$KEnoyw1Pl<25T-PO4*YyB^K4K>*L?D>RQVIBC}U2J4DHk{8w&F=^S%`S)-<Zz%zoCigQuw<_H!bwW-u+9Q7$qE&C6{TGG{XL!D+4YJScD3=E~d3Vnq(vHwrlK6=Z9ADx$#xYh6}7DJrc~!U`H)HSvN3ZDNdJqYzC%W2@3aIHEd<wT(h~bq4hT@Xs063{*>X7fxjZDpMok{O79_GQ;vrtss@3Il0@?At(b{W_gT}IHm2vX&4zEt#MRrfS9mji_MQ11oc5>a5N@tYBC-v91Rc^L-UW=+&-1jtLXkIl0I9;1qFu@_Pcahm0hmT^Ofrxw=?}t?nq<S0Sm1>!w{``n-}D7ZDDgYIk(2TO95_lU=!~A7>X74u7<24rOk;2K$gFPWPxn??uWCa;vVbSvH?_!ZVZh}sSonba8v6zcTp&~2b~!ZS6Fe9{iaJj@td(edj^ag36&P%T(XOzguDpr!p!>7AavhBZhrdq)z+04u3q^Q^sn(Qk7hORIW(He-A+>Y5Mz7^%p1iaz;*ypfZXA*==u^wggw|KtpwQKmUf-OE%86NoN3&d_7}WT%<dy|q~s`<l9ziu(So`$gu~DPklaN}x2>)z=3iV6VnVx`yk)|My3lwpQNzL_<jEq+Qn)aFS!vNGXte}|`sA%Em!EB2+qi!9ZtLdd8&~f_tyk}z+KanEr$uOSY8|aZXbpLf-aGX`^>p|4#?>oVTc5c3+|@hRH*Q{~Tc!W~&Cg0qwQf}v{40(%v9hn(H>-}zQJ=ECkU!eiU4d-3aiH6kLSNWb11ohCdOEW7i&x->@wB4#*lIgQj1n73Ie@(wq!3P+SV&-`|4|npz({{0!3$X-2U#aS9Zqy@mTi9ffI_!}Kq&4g@`P-;_|`OAM4?E?=?IPrHIEzJ2}gKsGniqbU09zpKJ4%_9^f(%J{WUVgdw)QE*MtioQ6Y0^~1~?g-ELL7o6Xm;vkLCi7n6;T<0a@gb9VZNhs8x-aGa5YVypzQ*GzvZ8+-bN^<T^?`b-^zeG<^y09zwHj4L0q6_b`kkmP_2{Y6}0jdEelfDFz6k_Gb+jx%Uf`kn4S?IKa+NkqfhA#2V3(S)NAvZF`y@<H0L^6B`+5d!vUgDWZ$L!Q7O6{RONNkJ8)FN3G{uN4vig9&Rq6)jLn(#yuQZGS67HSA5mL+514aa;P^ojgOd3P@gUkMCN6e?AB+t_RRRh`WH*cN-*Czx*CCifgN8$ox?emt@IIojr3O($WvIt&0CCadfMRFxMf2#y+S^}U$&0k~RuM`j8??o>*(*;==i;HpF2_PTeK?RS%D2k)+skRf7=!!WO1X^%Q#l9IO=RK6?PA;Vuv$-wk+I*NAS1-g8J3>z`xJ$w7=%5%VO$82?xQ!N;c<4Lzxt3POyCR~mecN2h{j$Vg?-E0Bfw${07m}G7^S_wFxaSzRF=xJ{a8UlQo4w=FOLC`*vHmi?#!kU0wk0@E~&c?WA2^6jjf;4rQJQFobNUg{+*$NPH04cm#fp%A<OYe$;Mljfk;S}5*`p(4;2IJiTS7{i>geuuG>8qrmTchFkc!{*wQ8Oe1!0X`6A3#4~-;6xUd$h#8Y>CzN0&UXLO9qpoT|DokWY)D$1<3JeI_#i#dtW)J^Z86SG~}?O52P0Zr?rn(Hpt#N?iMV<N}9|(3E2&4`_Tm$k#=g7+9Z<#x^EzoD##WkrF9z`Wqd%QM6ev+Cq3VDf+gD~Nky}W(`M;gw+rlsO6+*DLDgJ`dQ%=aHP=DhROf8Xb(}dX7oH<c+i1q3`I^-XN<M5f1A>s#aLC5P=Ed{1BJ3b7w*FOW4-A7FTf>%P`-OVu-YF#>+4|Y+oX`6wWl~K=6$9A<WymoP&>QFvS^S|Un>5KIH5WEWYErX&RaytoWZs@T3A3RVcs6CcSm{($u}3u(t%IVes3HYZ$r#wusi<O)YARX>MN?5l$~>J0ec$Gm;DmiZz7y7ad3(VwXXZyovnEY#n!6|$AtIA{ni`cU8LiYvp_B{)jHbP>9<6cP0MZ7jB61n%ts)tv%MdMZV<L)V+sd|~FC=+|U+gBV#c8^h*{+kLf%Xf#f=LyPC%dRkA}vgmsUXZ{EY@)1(vOF}n>DYTIpdnHjufPICI4;<J&+zxIzWk0@6}%~fGTv~_a~TxzUqYGxYMAE8$<i$E-Sr=v5rrE;kU88e+^i~yQC1T$X;{NswDB45Hc7o-BjQ1!l%ldsy;>(=}m{j89G|`qXTL9>Kd_cSH6M#AC9+TeW@0Mk6XG6EG(7ZjKy55E*Y4FyX$e#BcRASJpz;yI?`l9QyU&xtDjylm4NWPc_F*oVAMfwUqj{WWgdt1O9?r4B4H%uaV2eM9D9J+G>ymFYV<BYJbH}>fb`HJ^)7Ib7+{8JO(m+Xdl7Nd5$Zi)Ogo{zk#G*x2X>?WbleM4Q82)RQnc3+5v&|5C<lI$z%U6XbHX+i%9Y5m#Ee`7n6-5_Q3t_gTYq2II8aqUF`po!AqAPWc7w#~(G2{eGnJ~D(o4KMA#A2_Pcybfn-YSVP|C3t7Ky8<0vyG*f`!({wBE*mCg{Q|>>zSBhB+0D3;~l-?Y6K2#TZ45`AT=Pfc0?(aH&E<NVx4QrGd-Fb2%AaTOgyIxmH<nF#)WYb5q#|LmwP}SVDUoM3Y>%#ml{bv{_<FYKxpotw+N{zZULTSiq*z6w+EQ!-8=!(<lvgXcaY3HbrT;Ljh=5B}(MM!&pA`?RBqCJq2d>HE&J6F4X4cupIc)k}c%6R*)>-0=wLSc1`WKvC*{;^|FmkTz$C>i;NWOAs4mIF|Ey|vMf$CY%a1S67Hs3YugwC6PO-t9#wQdHRIYYR38313)Hm+Pl{@{kUc@3JL<DdEXGAucC4hiaF?i9q*^1?^T_FQF>NPk7x{iK*uh&NfcG@P5PV`w5ta0!)OR7L0hcO8j-w!b^3~P~K(Q%g7$s4_7kmBWJ+O~ID(+;=M5xu~k>f+!{;Jvj1Jt~F{V7`IcyP8Sq_Lx$tWLNa>_w;}3uZ8Swy5q+#{=}e4tvgc0ryT3cmxsYgrs|<v%{GH+JTk|9*fx<sp5!bInbRX)^#Zsqi7nfmI%Ko8j-pM(NJ)a;wR2UEYBN=2$N_ipWM$B(dZqN-R)#IAmVZXX?aGRSMY$dgPx_(9xx0SKq?IRN(|GU>M9@(yDzTJC|U=_uEoGteY`cze0c!;F=-p5)%foxd$8mYdtHZ4>CuQ=vRS@8mOehLa{#s<w2UFRIV8>^rQxsMpVh)i?c2^Y8uVBdp0tqFu?}I~T5e9!kHCI?7KdouT1ZA-_oNF5sf_Dz1LYihHNpE+Kvbmdqk!Ty!trD&-torQU_S8jTeZ3yB(~K%@tD>H<BIfxmlfB&_NP45^tX*t_7vc_ZeONy-m|LsBccLG>xAX|uFY_%AbR3p`?%4IExvVvcDF0Ozq*5AK!VBaG`(IS<|K2D8l!?_LIxbx%9mlO4tBxO6tQGFFL_j2q_r`JYtY+VcwqJF+-^-$c9Ohq3uF@tt<q{4S996xywZj~sQFQdnpVq43qeyAj>WdnDog}hKDC{X{v{<PEPJB!O5F2G%z1&}Yfep{65b)$;!<%)>wE|wAXpf^LEXIBR~V;q8@2FyWX`?Bag@`hY7q6FwwBt`oKHfG#nu{Qw2&#sNf)OT1zo_~Av|#4Uxy-+(^}wy4oEwfPJCqC=uL%Uw|o4T%~O4?m#U$sOycPTY5Nh<S)A)U6<!EB7lYpU{-ulIxkl$)<Eh5Q{-v<qd8*%QJo)6MI(+Hs6+?7^2-4V*&;p)^<FplycA`-j@~CI0uU|T^v+Kbq9>IBtnq;b_fec7D=Rnax+6oT>z(Ixr-90-j=>=IfTJK4Br3$t-l~#ecWWDNgUiWTSVm6j%X5Hx6KB`Km<gg_x@=cU}ti-&JSa_tSq+>oj-w(&B?_JXj$+#AN#-w4GEQ}9U92Kx0GcWFTr;~}YE|aBP9{nA(qQJ^Y$}W-=j1X&JvnXc6ztjag;*Q6d?jS+sBOv);R-LC=Dg~8ug_`Ac{Fma$DZJCxcI;qu@vI87yb0SEQoa?8C{QWws#+p=JI_cSe$hbeL@A>9B=nPMr$WJJ?%hkyGNJJBpMn}Ze!Ts*Go@&x$fRsib!^bHd#5t9Yv_%l;hSnUahI;VIuyH&HizZnF<*EjS&#|x1RUrdnhUT(<Sij|jcN`#qwBK_5)p_$_g%oV>U2-Bl}w~zs%IJ|j8ux+rCg_4iTXC872n4U26Hs1YF@JP`gXyMTk;66+tjtX=@5vxVt*>UdK<`VUv<Xv1r(lr{6RHUt=gfE92mK%Z<Rb8GQ(&Yjl?<yUJRkHl|cBdb4-Y~eMQ?A4)jQbr-=w<qC0L-*jpguLVgJb`@t-s4vF0AVQMTKv4*1&@pa*vYeNomMcS=Acp;qO{)|N`3EUFEiL`k*I}s4<_7;%n7vTRjq}QSR0b_e&yR9c}q=NtZo1cBbDr1dw6Cr^?*LkiyIIoSPQ%m}QsLUz$C1z+4d{u$A@4XOC$apgtWoFPQA+>cgrp#uXLl@cj2Sv^se!*?a5z7XmXwY-RZR`B{cDtPNXmmW!-Y}*vz|}ZunQPIxMLEo-FhVkz^SbX{<wX$G^sFZg48ntT&zSJS-I_a85YhZNvOzS^g_{@zt15el03tn&SOx!wPvX}z63+<f2-%9L-Im#^T>bU!9DQzj?xJK)Ws}b1OI-5uUZ|vFkX3a`=J>4NmpEvdb~p<60}?GzH;iEmv@2h}@Go`9RLjbC58OZZ{3w7B^^^gF3kce088weW<&*(o#AVRfm~K275(a4ILPQc16C@kG&6vc-3Xh@y^~|HmSWhz7NQC`Rwx~(@s%9!h$?DKNkf#(k078)`PAz*j%)bLUnRZ%g6K`#s;pe5i(e$j^KpFO0k-9T=*dWm*weX7Z+-%Bex^@8Oi-NcT?{6D6OhQrxu>Y^bS@Yx^TFJ(B{nQd1lcspErdAXkT7%BnCUbv|l4dCrbA(%Sej%z-W94Wt4|vBt^<wK=7XLb5+eg7{q;328BePNBXu!yCsH~m5h;nQylmsACWv*e`rnSC^y-II;5ysPxC$n{qEs2UB@&V-&RJ&$CsYWRBNYo*h-khu>P%%#Yaggr%Q34%Q%${-rKRciLAq5}-HtLg^dmA?5RbDk4cu>_t#(3(znM5T6yS0#pnXVkDOK>)Ot$7s-IDqW;POZQOf=Sz-=(I)BgIgl!PqAe>(QcPy3ht{R2Au@|&8g|U|Iy5qG3UwQGP5<oce!TPbFP@|`kgmkwHmi0fSBvQP+ys)tl2gSr~ArKHgfLw7>Ozx+RzDa*lepknkaKlp${c<S!dApe&p{@qBN|y8g^?G>_N*A0>T-Cf?4?KL5jfwFt*zOeVD>#v+mc|$k!h)!MbU)+3?Tva}<+94!k+<*X65SpkuRe7IxrJz)$`~zb?L{>A2ZE=hx5q=lD$$?ZBSJE<}syC5mF+3UaSdd(D4Re3(UXvw2<(C7F%7D`_+gVN%WJ1%Hh@Nz)1Tc|o+-iP8jcsxwQuvGY8Z4nXxB^$OK4iWl8@66)`s1=tTFnm)DF1->Ba!;8imwEq-Mp}QMQk}zG#Y<+Ql=lR<p>fUcQpYj{jC$3XGos4JL;yJ&8uV&o=)_@MVjL6fK;6!=eNVO+DEOL>bL0qJX&{G(V;xSF^DXJeHV9Yb1gFBOWI_8eBTt7(CU>s3Tbp}DdKg6M|iPgb%9K<+YS{`~e2$J0ufanMp5oV0<U=nE@4`_rJE^*_N_?2*kN+Im<DSnsWAw{4%?{lDtoha>fuo})5o}Z6Wb`GU3{qq;F%p~aHGG9c1tf?0mJOq%F5wA|K<9LroUnfBB*6Z}ZFxu^}$cENL1$cE3ri^D)0WeXn_nLo6yg9%M4F=^d2gc}QQpnwD2fv)VNOhy$D4;Q~$uRjFw{U3Zo}^{}<7o6!(BMho^Gm~!XdQQjO}`X(22lr}FHnj5(SDS~gFQqA93TRE5WF&5!BlCC9|)XbFuC80_eU#Hx`ImwOH0$e0q>{dyFow1QPf4*bUZ-x?O-`ZrD1P68DuaU4*5!e-oQ4)SL2}=D5DK!p5QHi5gMt>SHdZQb|+NE@s7e2!|okmcZ|y;gahp!+(3BSFq#e-`vE8D88zU+N_W};OoM;w0JS<UG2@#0+`~VB)_Re8<4LQ?Xm65s%Zp5=BcvE*g+?Rs^%CxSn4bQ%-%n-TI?j?-PkqI#a`H1E3(TT^-e{cHIBNm0oKP<-Fh**U4G&SMt`A{2pp}PD7oNP}U#MqaM&0f}RBZSSMgsY1@1+PZVRXN6-lM(r!1*za$Bd?kFnI2D_g2PY(-MX9pQJC7mlCm$>$LXcpnE^S?L(w^jotupqtk$i%Q!+S34#>w^zk4_k&gsEhKS2K!T}62oX#kUBQ?((9QCMOj6suaaOqcqn0ihOBEgU&&E}J0pNwY#aBk3N<DA%t<5>^mGjwU+)M*RIS1qOvFkB|p@SceP%!jLnNc&To(i)BR_H2iaP-^Xxs5ROf4hX7p47kJu7~Ci4=oP|q1-$`yybT}dDkZyJ%RckIgAQ>FFbeVM6>%ChsQ*u2zHuD^utv`l5W^3|cOXsY=@sX1q=|9&lc<Y3{v32lB>>6AfrN8P&M|{8PAOiy1W1h{mkc(1CBwqenjroq-RM5hD$Z)oplf8W2VpQ7A$C0}-!gHiGf_;!AuYs;Fv}#UG7t^<l*e%Cl6XIvtt7qsG(Sd+WH%h3)AnvKWK_mkCK0MJCDJw+8<OaN`e$mH42Qv(84DWTDC+m|M5zk|#+1VYKu*R%F^ByCh&e4$gZqIOgP_C5EWvXoNT2{^8cv4Yco0wcyugmrT{Kt4{At~EfUg<nf%WL)AuSg40v|t0e@b9)Fr7`~A@VH^L5T5Dj7l(G6o-@#^}2ivrgUK9yS1}O3(m_ge8o)dGoaF!dqdo!7x_rgML>Lgp4kGB8&i3>(=#x16YZ4d5KSx52swpR^uTMzpIRye3pHC8-(I1(dKubJN3TR<K3uU`IOE{dJ11s>FZLqX%hdWgezT7pp@0`Qx!>K7dU)>Pvegkl`(Y;mW{5bky5ub}hD9nc1PdJuk^^GFC%l;v<qr;afoaNCZ5E7aJDoo#9w)@?Huw>lZw`pd5xVAt#KN;ndlHer#UmZAD1Knn*ilh7T6&M%2*4=IQ$sgL^a+J2`Mwv1V|-Q-x3WF%Q|DPQr=Zu;qi^8~`S`~SVFfh56%BV9lbwz%DWjM9!8DD9=abQF5ax=$ydRDlve5sD54ng~GysK{W-6df`8wBd2fhxt`PL|a(@)mB8AN;G6+CdtavX6)F)Py%IoU4`7uN-DZk*AmpF!!`6nDrKIEyZ`>=ct@KmCLccQ8)vB!f&Q*Qu~%*jfwW-{KU5Eg*a~hCa-=dw(29AYQ_rY4xIn(mB>B*={Qwj?-C7<Z*XHQ+6U%7SH0#E0pQ1#t$`KLivs!uEmop!88d5*Kf%5%%3D8BunY<ZV3G(O0%E5extSd!p6-%Y2E$w-OZ~vTDR}qx^a6GwX0}4f&XFHSfQ2S==d1%<30x1o~9x;cL3)CMFC+pSGw*DBTQ~I>am@NF#Dh{<OY#h4FcK6uwF2pO?P{A88?U%;e7<{KTV@}#(R2I{73_g#E1dS)wg^@Xe^#p;eP3*nl0-Ap1*qY*kA$vY&0J#984?KQidov<Qq5|7;^2HP~cn&4j`x~ZX0;&0{5{OZj2~m6A1R}#R-E5p=;h=AapBQ<{y5m=#2H)+Cg6~AR~H-QMljA{9tmneTGMH&Tl`+^hWBpT}bx|*Ij;dEvZ3?)>4fwnuYZlU}62av}J^V-8#OrD;Yq63dXZtxK9MHH6g-=I`BLJtQ6cRIByw1R*BMCvwm@{wnQ~_7-=!*R;jO{JKqKP)3|V<x^yeaZZHm;7&7+dX+UVZXcIX*NmQR@kjb9{6*=2RW2xDq+J_vOo97!JJe2TCTebq#@xKoe%4!Imc+ks*yP5Yttb}(d97LkNbf-`ic0!9e)U~5lx1hH~D|%UtND_&Jt=uB@voIK~u6tT(?JLFOH2Z`FxW9*Y{;<J;_uX3CuH{Rhy;Rxvagd~~`{7JQ0EdzpvsTGSs?4rB7z|tDCuV6C$vX$Z3}hDrryk_h=h}QUn=a1j0NV~#j!IKcCRpU*FRSFuYPI&N>vpUoV_pD0t?-s};`-f;0BpWm264lH1EHKREP=mmuBixY#*RWY*;h@qq5)kxxU&gzVv}XOifrBA!|F}hjH?K?+@$7fo*h4iTW)UQptoy9@iLm`(U@z-IB=TzGpSY({V2aq+U!Mix)y1wS^^sg3#@8rGG_y}YVA2|Ls^T7_O|k=sJL2AG*{){MMWLh1(kAdM~*R`%w7@6=CXv1#%>`#OT>%<W`v_dSvv@QHbL%yg6I(jr9SlCwaE4|FH~XhIfIVV0#x|X`gXLviOd=2WT+HaDa_fTi)_;@OIve`{@r%9k_qh1JTK{NbD4eX^15Dr|H7CmCNp017TM0liyWmTiyWoi$*gjeeq>kK6p%g|s~jyxq}{9f-;7Qp8~B)3Kc>~T=HoHFeoU_))9VkHUSoM?rgOm?a*1w8LsBaMjPbtP-)XUSp&JZBH0h~XU!&oU_*nNZ)J$@(Y6G&1);W5y=V)YZ8C2#1;F^^+|9s7>`=)9Li%dea&T0*#5ni1ad?jazd#82=aVHp<F_*QUs58J@xFik7NfUKhy7jvF0Dz48X_K$`2*1}qj-q|>5rnSMqx2C-^8yb+M3)qak#hTZ0=ws4%+u^61W4dS&fGdV@OI(#YC+}VCQ8<>yy7D3*phYUEhrXTU{SHyoQ0(#M_b&S(-aT^#}W}~aITO^G>)=B`4$#ipM1TAmj@^LF<hM~8G401F+_6;@+vTY{y4~B(>}fJDHLqWX3+;%0}iBV7QDBsn}pwZDjX`Gf97M+u(Go1$yrpMcKU;>pmye&k0pia2(+Ga8ddX*GavI|u5-+9@v;&gjpHQZ(m>S<2^nrJ{OzcOn{8evFHghCtVO<L;o`j*wMx6mu!wf#{2r_di_Ae)S$ZD1nnm65u2^tcpzsk^MeAaLMalQE(3)T;3oVJ({(>u_jeo%f$<JZ#df1LOWZxhtE&Fr{3TshgyfM=`dxx7>B&@=6>1GWDJU3^yrE|0|)Vvo}W?uTNecY{i`|$AvZ*^5a9K@N1=IopX2-}!b#Ds%7a`=Zo390K!y+(c398Hn!5}k3&f9>uDqmda`U5$XnuP_=V(_!V_snPEK_~q`(Go$YAc>iT|Ugwh4#YSJ)H%!bsGiTuYb4pmqgzQC6xYdv3{bZ&p@w0SQm1@<el6Gw}8C$G66)gcm8dtbXGi$l#QLjxk(rVFO$hsAtjv3S)RXC?LKx@sv;GM<4Rlhp`Ji|NkdKEU0KMp6uX-aF-z)N!|+_-ST(H5B%Qa0CMG6`lCZX08rR2Vg;a8y49?T@FGs!xWRm8#oC%i581Bzx^j*$~yUBo*rz_yUG_0gW$uF()(&RY-N#%ckLw(p60-@TEV91FV6P^m^fKzO)Uk=$F_ZWhr-QK^N`2rTcirRN2)WR!h~T*<xC}<8j5#tniM!B)sZP^()8KWeCvJe%Eq|*?Vfj4w)N4uE$q(otot=Vb%12)z&7L_}f10)@y6q&g%R_M2f=b7N3cREttt?FsKFXbMq6eyZGn|1kUYQCWW6EMK5gJ$;=5WE*nF6?V;zf#$+L_xH}<ikV&xONwgDTRvX^U-FPx#3;C9^6)6mR^6>xu`Qg{U_|7kW`0%Iye(TDehu?ht;dg%gySLwX_`P2~{OV8N`Rz|1{^SSz^5M6C{_vasdib^P9{$r;_>Egl;)=p%jBwr48~NSaUwrq+KYr&IufYI*_x6`^&br22);)bhqUx{S$W@&m>yzhw`1P;7`?r4q)5=fC1{9o20hwYC@o*F&yeJ@F1=F{`_uFqBzWu$!H-Gr<|N83Tw|{f^=Wiaq{;T(1|Cj%I?aP&|KmD)QzOsF`3V#8zhhKc_-M8L6{N>NzdFu!7{Nh*i^>g&~a}+kkUOw|W!5oG?EBO^u?A$w5GNxC}c6{>xtkkOyoMUvCZeZ!pnW2*%!3(r8Ss*oM29tmI#lL*`Z{L6Toi81}`Iqm0?O)&j&TH@e<7aWj|LT{pOo!irW%?zCrHe*6#Bj$PRehf|QBa2@yR;jh-DB*;2gn&=Tqs<Aj(l_P-e>SVQ2}~-rh@<U&Q@z>Keqn#X;|{Ceu%+?v9=kqPmK!7ger8V`)eznuo>r8!L1`Da3ti#v@^hffB=EsC>3pD7BR^=v>yO-GlTyh0EaV!|D#VEnjdFqk>tG3yD^LmPbFfbMrq}sUPX_9v@(-F4jS@JqgpMZ9W=`=)f!%fVE3iP87MMRhN@d|vBQC2uuLx!dE)4?UgOsd2RvE?kru&(em5vsF4xPKZcyn4l_nUr8nSf5h8w!AP3kSoCso0Y0E@AYs<-MDl_eO@xa_R7Y(ur%;C35jWo5e!Zr52>Mh&QB17FEH19lkrVN86HjWzo`<InV$y_XA?C&8p(w6qmoV01UmEsKcOkd<&nSP<jQMPRWWad+@Aa~qIt22(&bGPi}#Yt~oXbF{d)=d$7WfP=76T%$5~n8Xd&Z7q#St|G0WYgM$OVi9Qxi;8Nq<*kO>tRnlWsivyVWG@*g&f=&Vu)+tqc+hrgpS>MiPz6H0l~oD>rY$tCZ1%{c``ns~;3O>yJK3b@la(Ck^F(Se9dbk-Id5a_v^KY$Ra@wXne4`c9<*W3$wp}h5Qwqsxnl?I?_x+7F5$u$Ro9htPF{X#^V@djN2V5%Fm(nGam5qC1ZGGV2X%x`2x2P9DHYLEwJ1yz=HbK|*yVbOVL1wB(P$dFQd(!2b3l>Fr^AYYxZ4=Nlh%*dZa{<(afyBxw8wx%P)W2YD~-*+L;z?w0k9RFt*>um<Y)B4hKFj|Yym1SEdmt_GDSs7J`vO!3qh@+L9J0OAAQhCDq=xaL?2b}>D=a+H@9pJd~)aQ&Fx#FFJ{H7yi{GN{o){9CBW0Rkz^>c(H8ZqMiwo&ud<D*^_SlROq}qeSSu{HUf7<G=Vk>+Uvm%)JH5c`u6x$n*S8C3V!m!XSoc=uqmPC?)j5X%c0qtdD(fOr3(wbRNy8y1KW7XwzM|fMWr?DMjvOoC6ky){-p?O?^}B~Z`RjLo{EzSd!(Tl7_piMFSKoT?XFo$32Kke`93!J|zw_3Q-ucB>K<FuJV!jxpeNOn_oZvdQC;!YJqG&Rzz>jL>57+5|{+#{H|50^`tTpW7xm%l;uU~g0Va|_fz2JD6FRbL-Qj%;TwY5bU+GY!gF()L>IB?07Y2<iJ9#}Ujf&xx?umlI+`Sn-d{q;{-phhjuKm7f}?|)%wG%RdoJ|^Oz7eqm4FDH$K+IWtRep(~T<K2gVa|&~uQnVpy4MH;87UbCG3ro~BhhO~O```K@YSorPFet6pUks_lFkN`%lJ`pb+am4Ty?e>_86C}<I-1cxTWc$qe)slYuKe!pZ*BkX?Jr5(a~`}FIyyyAfuVCOT=V#APThrbqJ+AD(aS4r)xvcy$n($N6<Ws&Z-&+fyb}xrC;n`nZb;cD%w+-?#{j_q&6t+q7WwR(@BY(UZnQ=(2Ke&scmLmqKluTxq~Cq(8xQ~fJBR=HIcRCIy`aXo#BKmOS~A$r6b-1l^}DzKb{+oDn<Y6jk1{{v-2q^?XYay0RYq(HyN+v99C;HMofU2aB*?fG@ZXvE_dxI@%bpP|E}*(X><n&P>;>*Ydr-TC`>Ixp<?K$Fr0UwkN_b_+R?$c)PVoi;Z-()nEE@IkdTcpq3&D)8@HC|w|3qj2&`lMu55ON5DAV+4v$BCEA9x%(R(61~W)%SE$*9^KEp4$jVAa|@i4=*sZ18CVhU*Ks-CHo1KNqNJtg0ILa*Kr8xU3n3s4<?Wz53c(*(_W{bt`L8vBTEL@*)`sY)+Qgy8g)4JBLPmQ;-v&cmrB@5UQE&x#zUeqieeBg$g^~%1#g!4!{ADE!A4VjY(m|ajTuI(|kJ?ob^2*Hmx@Nq8%Icb$zSR`6A;}V{=PT1=W0Z392aGxLSrY2H&v$QCx0+9FHri#hPblC0Sy{sVR-5*-hy#LYDqUyQZ-kHNcx)$Trn|tP7PWZ-CxK`MqSpt%=yoZoIMeP1(@S(q;P7**))Po_Y14{-E~i4F4Q7@E88U<{vyQuX8{1OwnDp&@=V*>*o3PTamI5X+_dRB<G~1^P*%!idt-$xSit5ZBaCQwm_r5+lI{z1;r8>zzho|pp8h>F#}zTO<J6#TJdNwlUETn%N(ri8wU|{d_+Sf8kMjU8a3feF}$)egQZ+$5lAJoR$BAV)ruX{&Mn}W<}|P0lFno%Fmn!GwKCtZ6Lv$>>}wN@rqsX@!e7JRZ4&kef(`G+(@~lQ9LmzZ72%ELmCaZoZ{4=G3vB_(Kr0sysOnml#SIAMTFtMKA2MAf=F^gLfIixg811kdB&o1Bm0-sp(P*z5jxgY`i1j24P`P4<nBe+F*I7qcU6A&~wc&~wsbmHyG!#K|U>-$54IR~s4gy!R1wjEFwH?dtjBTWMV1_qxolUs~UbJDW<{aG$-D?u<#8|s{@aUA(y;@O%+15w`$f{D#*Fve}2q=C85bL8^pw%T{thej{uoKscA*IokLWn6IoSomaAgEeXMKjY@wCM;ui1zf9>rK9>6f|GnK94Lbe&oetcZ!15;}99BS9MW|5;<N!5sY#AIQgyh^_BYe@zHfjGBXLHB=kOs_R3dfqMT2RlIeI%{2q@I<L8F!-mCbJz8nwosc~|Cipo>c>D%stjZ+y_<lA26rEWv5u%hL;Zf&SqS$9sa!Loi_u7ic;o&JNa(&dhR&ZrNIyGSnWrFaBS#hlq;#?Hvwet=g&DdZr~oV_;*`l$%h0HyQ0;dpci`mg50g&d!j5Z~<ByV7Sw-$6wEjTT%E1D1<`$%WT!M+AevujX>nIhI0>q?&Wp_((FE<4HsT5T1DF7e9IbuYdmVCqMe#+kcCC{j&<j(yibW>(5VBw^r&74+Vv4B3frSt_ysV4b7=SCM(5N)JTOI|A}IjnVKAqTZHJ;o}H}FQIf;?&Mz%z*ARtOWM&7Qf2551qu#)Q3c9NT*BB9teai6y5B-)_)Yt%y=E9yHBfxC1I|-RS458^LiRG!NGgvi!Tyuz?h68eY8l$M502!RuAaU(<Arp*XDz;c_M-Wpo4m)kZxr2tM@Q;swoL4L-LTD+PnR_TF<yqj*_bSL#D`|I^SV3`(6$}Oyb%RwvVol9~JRWk}(|KDaIy{PnL=pF>je_)a=#{m!qE3c<&#j4+ec2o6y`TN(;m>|~`0H;Re(9_4fB)Z=!OFuQy!P<lzrr`9@BQ}M?|uJ`cmM4-@BH?sWK%FY{Lg=T_pNUo{`7C(`R(@~e)eZX_(yf$)9W67`#13Z@H_vEo>AZa%{#yN^5L7`JN)wZ(CdY|d-%`aKm7Zzuvr#YeD527_wf5);4T`nHSFcrUp@T0uRr|W>xci>8=^ckLRMYS#lxSy`OYu?<?yv{z5m*m_|n%i-~~$nD*y`s@K?2U%P<XW+&BF9&aZKx@4WTa;g`S9(_8Tlzwp(EKmPtZZ~fJ~Z++q3sdvAFQ@nTTcW=J|U_!?$nw15D|9b7Si%sJVScW{%4?q9U@BiIDDA2=dz5n`G-~HS#4}bc`!!LfFFS-x^^GEOe`a5C`e)sm5-utJ|9scaQ);OT(;hSI14}~0~zVPtxzy9!Ves=ive|+~l-+lO#A8;}F^xju~^#0%dy$&lX|1Dw2^yM#e`)pLl7AS0cBG&LIqXC2=zb;@a4uA4D@c8g=zV`5Mek1LuzQABU{N;ZLJDtz}hqE2FvVvuK|FzE@{_vj;|Mg8CA9nG}|9J2FKRW#U=Xnv{``M2UfA|*`PVfZI6-@wE@bHWO&%1y1&$q7JdGF7EefZ^XAO7aA55M@YfaPd<@Xgl`zwr6PpZ*eY<1hZ>@atbYeDh~G=g)spbotwF0m{DnrPm++^xxTk?VVqJ?eH7_^8V{@s4DCgD0Zf1yXYL~`6ECY%Y64ohM5H6z~Sp(c=!81ML7NA=Kv9%b`<cHae(b9^fGD-!G8<xVC9%tOs~#1b-Gw4JT9+%T2S$H@=Co0wRs;Iv8CYFoK1w@ee2Ele)~Ej6=K%{evj-Pil?ZYPDvi@Qm(%oH^?*qIK=x!D2DO@p0n@%_{XnCXX_7;%+vPMC5GZ^ugc*+C@Wts4v3S!a0Ewb&>;&gTsIS!RR2znDVYv3?|#dbTU!;A_bP0Le*mvM9t3n3_=NXguYE-rTs^@*7lo~fAJ}ejMO_Z(vQ@)T{Rrej&UY@sz@3ls1}BP{uhv^gwYP|R?^r6nqpSIjrs_M2x^H=v-_g{5M^OD8N&R;O6`)HEIA4>YX$K{0R+pxoz#TpO>;Hh`{Z&m0UCG4L2L<XF$B|dKeUGSV$hLSvgD0+QSa=DRQ8_#kwZl<#jwir`9Z3i2U`ViVCga|F0tMVRKlk2ie-6xW17r$dK0wTT?@RxN8IgcB2KfOc4UmC<^;Z-(<8@$5759$szx|s!)?-%ojb9yp_q)uyzVqvEu#g7me)z+$0yD!u0O^6-0T%k5-~JsJd;bStd-%P#-}%MYL|fSBTi*m3?(j#y<`Pi!;TxZQ=NGTN_xjJV+poU?t-SM#ucB1(>;HB5jj!gb;2>z2z*0Z_-hUpx{U)<h^L24rhAgo^kWATor1VIw8CRFS469{ZCIww|tk!mt-u1C{c1OX|BVy>0v2)J$#IfeLr~$eo-i}a4g1L%2Eg^wB80s<?t4D<G68zKxBG?k*m)_$1`+n^>Vn3n7fBgbS6e8`GBoN>J4T$v)sT;LnNKW)(nJ7<WMW}oXl7R#PMQfrgC{_VwdFzYs{pK5oum8#_BZvdFoTmfF<ywz;2mP2?eax&b&8!yRq4mYV5GctmcO`AaD5HwvqmkYemSNCp&Cdw1u96vV>4t*BX~c|uPr<xY@g$u%i=8=hKbY(!x=A8tO=e@n%cI%$da*ShI-ZERud9}yU=d%uu?+suK8Zp!MhSi9G_@2iieKb>uH<Fu?LinMp~~?aq#nLVJ#2(dgLN;A(%lffPDCH^gqrppLb`KRoTk0@Evh~PG!SwPI;)*YE@*dUHY2CKcq{e+Q@ApM`EZysXN}h6(pMKiSojYSl`>FM(8LP;MQ5yCR@x2G91vK9o$<7J@6=Ok&qvSRJEdO=O401p<|IJN{QyH@zM3^742?wT)86&#H$2V@>2UzaXe2|<XDK6Lu3D~Y3i#Q233)fneO`$ZA6xccildP+;CrXEtdm$C`mW|0_g}u>8gYVMEtG}!J&nTFK6)pG!%o;k3$etC5u+8RjFt$z=^&*0E*kiac=I_5pE$XW-*vU~Y44NKW)TQ5<%<GAJ|FiIkjPrHA)$T2Ae+|Iw6D3mogyW1;pY{vZp`!YRYEOodRb~9{HbOt63|x$F;GmJLbGIQLFgwzKTN|>5>FCsW{Rp{&z!#*TMm1s_<x%7p%<-z8kLqpT_@ptQ8I;50xOYAQ7b-8NBQiv8Ok)Xp9aa!K-<iI)~qPNvVi(ovs%DSiz~yCh;i#CKv<hkns)H<=>MngO}N`ejz!_Wf+Q!GqzzHn7kZd?WJ#XU9a+{V$&=(NIyhJw5@U+wB}gst(SLtc)!jg2p+wo9<a^0jB!OP4yQ`~f>#7+`SMy~A0syA!g@LZ4#UpuiM5}mq)5zM}3U55NnpX`(5qfD_LFo<OTpMLpDYxw;o<fzz5|LvHiCGL7=f~)tO|EZNjhj!?cy_%;rcb8oZl<*FS2I26^3*vN4G3ImLsOer(TTb%KN8KXR;#lY-0m-RARDL2>DLFQSKaH{GyQO+-m0fz*&=)+trP!I5j;%ed&ibEusaK*(ZcC6WARcMtxh+OYI0o(49yJsaj#Rf0Etc3-ia4w67|!@HdNEA)54ZTITeH!S9s*GOd$`0X5~g6cz$0#&a$Xjuylggh<M!LvO*0yV#DB+kSW=lFgCNhQ~9#ExH{sRmadteuBj8%mA$=D+cKPBQ|(5g_(shzz}(+-ZN(Y1k7`(*_yN}LcD{VnfI?_LpMJ(OyMEk1)t5@Rh26#rlGB0lh|py$s-E|oSk|NvyvOm^$*9R%G`*0xXBqmQiCE^`8L2-QJk>4&%_H*t6UT2vA%Zcj%0K={n*klyHXX=U-2@6}=YU6AhathNRO4$*ND<$%a`e<cg<q=ialQIZ9xA&`AYOycBA*R1lHfepvi(|8;(5UfwMzhMSwQ$Ys!ne$N|4xmQ62{!jZNx(iML9iZ)ABlEf-!HH%GO5i_2nFg|e?E7v%d@zhg=%w*q%V92s?=b;Ox3bic7(t>e<r7k&#>cGw;GL%zNIf>M<V?Uolon@v=pi^9!{-!dcr(OD2R<cUSBTYAINFALz*RuM>kc#WMng=Z;3Rp&WgkMEc|yH~rySOPn)zSatKxT@;)`GZO~^ZA#e#<L9?Lv7?=crool$?780TqaCd&HHL=O6G#njaF++u%H=t@wC6;l;u1w$=n57!EV&B^qSP!R9#^K#D|RX%$W-1v}|zbO8Uu;yrYc-m9>+$k}fBlkq}OvwVO(lB3evvkm~Iwp4nEt*i=y(USZU_tV!u0aBUVto*AMonG+Ngz|eAC5LJ|lZ85o-XhPN=Y}m!3n$mJL{sKjHZVRm0J9PerV|r<h;MqlZz8m0YWGAAYOoZ{g!b<4+`;A8VppN%W_W$S8dC4iEjxnB2wU#pmH=FWJdAY4Sy}qqyy1uQqy*zOS-7`_cF7=&)HK>Y_G=@Fkb;N!-^)-nC3f+FmCQIdYN3TPC%UyEHa!<Qo<(ifE5q8C0gr^SK1_bZfE1IZ$RsJ~q=i-_(HpA7!3f@0S$CZ2RM~e$BuN^SM`{iWC`f{NMy{D&t%>Mf6?AODy=={^?v;9Mu#vb>M`&s17$TXPM1SrU@mON^!8-{bp5}=Yp?PQMXx_<)00BDML%?cttvpEoe)%|?=WmtOJ;Re#8C|{!t5JjTAYHfo;;^7>#eLTV+R9EPva@H{=ZpxB4{i2g*X*x-u&^-OZVE5d_1wP;ti^m`Zw}RshDdR66qF+A0k3N0)aC~%f$oZPy#o6A;`G-$u9}myY8?T`J!QRE*`Qb(L&B4)!51-HLHL7h{4B6~%vYgK@WlT7DZE{9Qd09bpaCE-+{`fGuI6OZ5czAL4_v)HPXSIt@zaE|({SWqkx_7pBe0+F(bpElblylupuc9{@n>|KiDf;Y=GD?v`a8&IGdOSwSpU>t{uz=_ag^Q`sW2|~<Ug^y{xq{^VNjpZR4JPJz*vWui!U7zf?(?|FfIDb-olHT<Kycuoe20i~bXws`9VWScEwBi~#nWpHc#**&n9tS4s4hjG0rv^`!>>g`5wUV3*{c?(DJu%Z<C{q`;kPA)0+_=owk#kd#yE@D8Ejg>{=Pc7gKEy>)%<pnl4h<i2zv6*5-@y1>4yn}ho|sKr3UZl8L|<Wm5Gxc{zEV$8j-7L{-q{muWlEjxgiphs95Lu<N*gdgjWdqJ1oODKDW3bWW#KIyMQAo`6u4zDR4{1MLL;?TWt6cBqWe-5ii!OJg8cd<VTW3G3@0sON8RM94tu9QZ21v2sHqKs{n`?{Oxq|z@|TAK(5{QaWP5H&!`oc(g;fr7mr$hZ-FVNcdVf?7*?&q)UPIpYD-2&gG_sH7Sna1k@iuW0Vh`Ma{J$|$X)$TBNYJ*i8LY_EYb=VyF@xpd58fMj|fP5Am7ertyja@Ouo(6vy_`+knnzxDn!ava|WkywvML<Fxl+=6mYjcn+-n_Z(c)MZY&~oS45d<T6d|6HAu?+!LTefc>WrXse7H*!GS-FSF2@TnxRniyMvFnu)F&cKmh=7pEN#w&w1OQ?3jZ$wxw&fMcmg<E)N{<Iq*m~b+dITi%b+#SfMp?W3%Bgi@!XcvE$GEp@mV?2#s2<05}0+$v9{aE$Bs_rpt$`mqzXmMxwMduv9?xuhj`?jej6D5Amum5<$Tb2OG>x)GRjOc2;{cwBA0{5#sFN{SQQecN_xPJ?6Lr*Od3x3-9gD_b)6^Z5ptVe%W35&Fnu6HqFEd$hSXyGH^L~NStwn%C-&&%ki}fU6XnCh^5cj|E`h7fTZ>2lGbWK*cDaSW!ZzP&NirD*-*KIs(w}Q)(dk(sf{hK7uFUH-z>K%1oOYD7qg$~op#N*xsktPA3uy6>jW6`4xT=|t<Ph>TpC27?nd~Qu1#$J12>Q^=BOy^Kj`EN5#=$ex5k!^hZP=gsLEyqG8TjFTep5h>rLW83)1iG@n%Wer!KS0t)m>n<n3DYMOL~Muv_{Z`1fhNP<cL-f(-2^5FYtE?I#V3j@p4F0PZu`3lliBJ1QvJ1Zb6ailszjA~%e2RVqS=a8p(YzKIN-SK7fEZ%qPtM*^%g&1<6g^M|ogqe<PBSB<HX84u+=VUa8X-f>barJiB2WLy}hY15kX#zmR^)6d8<u5Z^<5?k;j%NX{KENlG0I9m(60M-Z2WPw~xq0MI$t4K?i^F=YNWk!3J_jr-PnDD`9f5wv;L+yzb!r$Esda-vf0bUDh?aI=(>tdxP3_ho;$pRJF^Bfvb7WhWL9}Afm8LP`3Yjjl(03MG&0P`yRAaq#R+xP+g(t_R(^xE&6!Jyn9$W8()w^DIwaicFbc2=x!3!%jO7hBE?IOc7gY5?&-AKT_zvwSN|ha29D_7gy&OccQ!u+cp%@ch0{NC8L91o+}q@X0)TDLN$eO}1m#o9NTW<w!@@kE=KQ7x~bPt_J)Iy%Ci_-hC*>V*f50dsmmw@9WRyoUr&d$|iXZT+vm@ChwD@D^n44`Y5p+F;iS!o*-(n$rA8-1?PT=K?wr_&iH76Z<eewo`b9T0;@gN^vd{R%5oCi9$FgbVyWZmj79$(QDAmPIy5$de{X4I*uV(?(9iY>D_SWqDdh@e133A4j9iiGm0oXn*AYH!4Y=Ni#(0M1Ka2^R2oNG7<Y7c*s9B|ysg&wWjJ0{=i8Psa6FNEyzJ1Xp5PfhvT$wO>q(oiu`Qp5_n;7H9d38OYbc+cCWLALq3&>*ZVS(!XxWE@36DunY<Jn_(SYf~en1OXMh6Z+(*6Vx9>`Yrz;k%^#e4YEU(Vb#B7-BY^!KH-cYIw`yS-*V>W45WcW56uFMO*1!fug)PZP);o>}*0A9F|jyh_kq_%q+$a7;GTicxXxVxq&mFdCDlrBNNHts_%;C2Q-e+jXt`+(g~xxUwv(wA*YnmcZ7Lp9*e;n{Cp_T_|g-!Lb!w6l1Xudq`q1r9nRYrj+L=79JH*B1U>2^qjCfzVSDKb7aD2_*Oj4Gb3|tbaU^Jjd}_F^p@NW-3UR{)8i7{@oaGDd(68`7(mO&s__Y7|<Kf9g^xNLq(ca0$dC9qCb2#VG#EMOCYI5@-F7j%Nf@H+Q@J7NkL>fhksZwL2sFyK68G)fJme=ttdT{v2mGmX6v8r)V-Ad)*ML=_CQF~Y!*+}Pp3yX$6RtOvR1_Tme66io-Mq4~R^h&K_po0}7C4Ilh<iN-RhUQ@$k=}gHpAGONTSC3Aw#V2$w{BHw$r~<}jmf-;r>HsC`Eg82y&*oU&4^e{c-J+}oCj=AF=xT(<*@Xwl*_z7zy1yI(c#WZI=6hzuMtO`X8|T*32_QnooiihKtv*3F7b|<6L1kjhK><pDfI^4@uU9p0;U!A3E(q&v#b;|thQXw3VD|Q4BRW6G~5{nZGx!bg&Ch@bQ1AA&&y$Z@@gaO*N5M)X)`N~OX5OU5E0`BdModCXRSJ(4)hJh_&o~6FO!!RNYlBYkQ%LuW!E~-Z`pN>u1s=)Kj8x$Ub7^(!5$ra9Ik;b7W7wE;)?Rs1UCmcCUNJedf=;Z*>r1%iqgHSMy#x%Vqz|a7ihgX<y;gG&O<(!{OuTBwqgFrK@VN`MpgS{pPi@6t{ma;ES@fI;)GQ-Q4zLb_V0kkl8?yMU>E)pe7hG1A9ra|b`#huH@ETf3)!Ur)1MkUeZx%k%fSM-UQ!!Q<2GVQgFwO3M1T>uQ32)17YOx2eR_o6ko*dQ*hSbJJ)k+EGU&%Pv~)Cj%?l;0+ji7sJ)7K2mT2PzB22`!3*BgzB-T`8)SD^IRi_DkrQhjy@aI~etn}Gx+^347!;NI(l{AQyx((duT!i}O)|(FEjV>#nFLCg3_wlAu1dY>x!cWa~!tnyl27T1)>!p`Li>u`N#7{}VmUk>Gec-wsTTrbiG{8c$sAJxmk_JodaOKp{NNZDXN?h-tcDM-qCf47)`EtK86f1WF7+06@>eV&{6vi34#EH0~t<Jsu@hLHAs96zmGv1IS4vm|uNpva8h_DA-o<S4=Zjy$}U+5essKS8j`Q=rQ?xE0>bzYhl?^>(UZyx?l+~@q(81P=Yq>KUirBBNEPOo{Pj3th6uQ$Z9yH)GRX*sSms)_69u*U{^&7-AOZbC-L2{d1Ga~aQ8O<EMq#jx?(ADS8G%ZTGR{cV=MTiq&7lw*m4x40aQmk2@a{Y~C@`}Jw?EZJ%a7@Y1Z7QOYX!A?VEf!yE8Peaoio9y7Hb-X{ectow&Y@uq%PLGad!6V^m)2ut3&4%5IvsbX^@3KjJJ)0M+Ns8p@9h|_+SRqtf^X-4q&SC3nbh&}c{fn5`mtE8gF3>s|qqQ)e9-g25ym!Ku@3ga@-)q9-O<Ub8=j-bm*we#WjQnY996~*-Tdp^F2c0zE3&D+wI4Yu4dcGTy1^Ds|DGT#Ek+op97?Lx7zkl<_(eL4(4XhWr*TJhboXi1q+C=$A{yR|~P11<{4F@`W+0ZRLR>l`Yzv))JV7Uji-2G-1&4Zyo8e^`PeAt0G8VDujBmH<saoonJh`84uNX~!D3&E2c{jqw|71jYK0%&UGfi}by?)Rxvothpfb4L0L+ZyazuG<wEnATqicEm8zj_Aijy6g-S1_$r$dd2#dZs;agFJ4trl<<fY{n6j<2t&0cqnvL22XhC3Hg_J4!RA;xD7^JKm_jYuCqRs=x(==c)YJjg!H|!ma|%ft8QtsR%MC3BX<zPl9{4iQubZ96%@Ic~AKy5m{^MKO$N-S_0UtfcNB8jPYAjB^%MC|d{}G2zO~IF1YyAh9A9{sfYOTJ3)f^0A%K`}<06I|U?Q(~_>}x2Zk?Wk^pk8JxrHC%}e<X+tmVmDvJb$&|6(d197@jDm-#`drD5__YwLr@kR&t(RBVfmEbZ+jM_Wls~zDpC^h4Zxz>NdJqryEIqmAb+DDfI1)QyC_bNxFuAIGrFLUsE!n8BnLTM1Y5-)M}F+tl}bZQ_v$tp;2RaxjI`{&zG8MD%Pb{N|9IuZNE~VRrI_j5h)GJBu&-&5suKDT)xn{e0GK4MC%0)^(N}pCQz@>ZZrDwxL+)hCR7w+13}n&>+rpCSs@zt8HHIzfXB@(B^3FS>)ZJRTD~-NfM%;cx7)AfW>sc>du7C*v`?7dhi_~~*}vlDjx#c~oB49`ulZ~Rd;{LTD&A?6V!nbSWs+)BhT&$D`2&^*JkXXDa7Sek(s*_U918~+*j6&^{)`fs-p6}IrWMfUzDFl5VtH_*2}XeLIeIc&-K=kuSv;Bct9rcEGe%f}mZJAZ`=8-oZSC_M7=SU=+M$*RVFtUqmfnX}_%n_+_z=1ykby4%0lklJb>2pO9q8<7t^@KD2Qa}G=Vu}+VV}1~W8~wmE6~oQBbm^yPYS$l^?<h*Nkbpcw0N0JU}cw&+JjS?YldxLglxr+dk!*QT@l~R@|$t>wwj0P^;>l!I&e-%Xv&dLC(?*fmttwIgkMv69kUbwsS_z9Zax(_RAnu(0J@?z8Fgw>+HKy)%b7TeP)#EAl`^`<1o~PRtcd<~L+%d$x{1=+yHcA8L(L|fgO5-gD}?6%N2p&^ukrR`A_3>-?T)((ItBF*F`p=KQyG?phb9#PRwy8ghyzQ3R2qR=cpRnXq{&QRA|sbf{w034l1?e>fRUBxA+5!RbYz?Jf*E=<8(1SCi8<7pX}$EWs5nYyFWnphB+Xo2dOa@4HmKCf>ZP|bdHtg&QhC)kIo*t?FJsB`ZMrs5=~CK6PTrpAK8zNnf|iCc{`zGk2m=aWUk24}t$y?7gx<U%p*L?r)byzv`;-ShpaYy3BzV7oDx`(YQ2jC{|Agk;m#@hJv@XW#R?g<Ie}Ps;n0IFSh$^v3LN^Gas$yeJ#2#F-ZN@{%L1MbA)12KV+m2)rV{9W<@Vn0uVm8r6N^d==Bt;61emQa|iC?dDkBG{kSlpzEg`5^TB57lH<=9428IGDHetiPnfnZMNYrI^>^+A#_J#`4L&8*014*t$y_Wx%?G<#h^$R>vAJQ)38a8)}de`Rc;@io|`JS}8E!aGCp9q<I07=kBRyoFhW(JTBX_Yx|!0`E=3tLfvgd3Uk8vmtZ9=v_JAlp6`^mST^R6A?MEBqrf)3~?&c_s!i_ArbXfjLJ?`!$iNsx(w(Tt#$(-{qkxscsbwMT^()#HeCsVMxLUtlaXO%PlV}r8<JqE41vJC30*c*&3&qMorD@;1wgc&&2NbYir{c~)V$M5ERL?HkBgfkS|j<K<dfEIW3d-DDLk>0XEmZv+o~n0|3{2gA;-7OioUq^O%?)5;Jln9YdRhIqc=LNGstgv%O3+<i2VqX;Us+|69}25I|fz%UuPp2q=fRi)?Een@T!Q$0hRZo-f5J;b27VOOS)`wK^F{8A6GYXOwWo90u4YBfGA@IgI#sH3!(uAo$u4)P8ZMUPE`4ID3igWO7RwCiOF2moi5_^OMHz78O$gKevl3S#(#E~UB%ZW=Ttlv5&p&tSs2t2zE?ci=Gi8GTHfs`XI=7$-M4t^elX9pE&UjI!|1*x8?Elr0-}toS>oTsJ!ut>Q0rT)#<$r}#ixD?xn#>+@l$E-sU=d~M`DJf2uG@B0EUSYPIpL0#mx8lBan>O2@L8FWBDArma8S-zHs9SiO3$uKn~Dui50h#f({!Vqe+hXN&CvXf70#>?W@}5lQNC>Hw;$*R==j#B^SU61?<u|xWF1@{N54N{_8J?hsR`7f^y=7;TIf9rmZ`Qp+_-19dg%4yxl}TQ1SIhBP6LnC^Z`r4f|N}(j8r?jsR78aY=EdJmQL?f^*b;9FoLt^cdTIl2z;=7p=*R;*G8*cr;Rf+TQ!45ue!_9zE5(w~(xj+cS7}xc}+wpzbn+BjFWB8-UxVYnFm^N{UnimACsMSo3?~z714|0AtJkWIjA!VI-8JPkj!?K%I7U07Qo|s5|(>*d`Q#@5JIjbKA`A$9RfTA6{(#X3zzSW2~%k&8dOE7)JCeiU!E~&!=})0wZzYV5MSwskC!dlR)JL;yO?62G$#?m8Gv@XZQPE5DYf=Kq{+-aJyPQj%IZ3Z=`Rs<`C7H&@K55kkTYt-*Uf?x;&Gz1sd0|4O)<`JJ_oI7pI5!_T1b^eq?-k^$w7Hk}|~!8j+2zTmrVKeT{1IXn+XfRmaOEM^1^FIPEoHXgiYA97ljYsxk)(Ok+Z=MleA{e%S}?0265+?ne0x+PE~Ynq1=ZWCnZZciHb}{Bmtc*30d{L+;h*JDy(8f$zAv1$J>k`P;{JFywXffAkQ^>NkP#mNi?B#bs;KB9`q9nyKXIN-AA#M=3Lr_tCkm*u_dGc+-sT-U1=+PvhGpi$}aTQ5p)N^oA$jPQ=+U;vh--PfgU6j)K`7b7pC{UQRK7NR)UEU2tW6(Lgo<(PYIVchGZKR3s(~ITKAcMA7gc#&RA<t&EGGVg2yByg1<H-F6DZAYEFr=AC-MH<8k@)#CLU7-pzbOh~d?QlOp*D@NJ@OD6G*=;97%)<Duq=}DfmCFkPcL=(PZtvHl|8?CWa<NhY9M&~DSy<>$Ob_TbX<(}V1vh7`YiEl`>ZO|sh&DJ;Z<7(K=?E%}=S_j!t1L<sJNB=OcxLUdGyC-||6@pD9KrFM^?>vT1vAz{6UPuD)PV3I)Lfa0N7Xp8Q?UQ@?Y5Y|_ebP%*uNRO{Uu<7USYJ0!(vuX;ge)Ym@9U<;c<Q(!Aqc-bLcWrLrQ!zgoNNn)<i{vv9sXW&x}*qdctH;I&g4Wrt_c%Dw18Pa%ekRQGTK6|Am=th+Jp5-(C?i64dSROXK!qzT%pa4Akh(kpL5BR#7)6WqUuWWt`j&E9&t|*njr})wK(r21|kQ2y~J`gCBajTpTJ-_@@07^Th4If?!s5TK(e8-o?e*~xL$!T{!^JihZ|()36nniEq5iM`Y)yCjY9Kc8n2M|tW_*TXylfsk9Mh0NfK!#n?^rQkB)gRgQivdg?hN9*RYy}e6V17DKwWpYRdb9p~(kWw+Z9EOjVDSFMH$pVzOW)g_R&GCdxK|^n*u?l#fa$>Jm&5MRba`EMfhj7u?Dtf3eAIz9?bmzRtpWSAU3J*YvHR9JTxiU)IZ9RApr3cA+WWSWbjUE63e!L~2h#K83v(fv`V&ZY?mr$dr_5Wxe@qbuph$%OYfts-}IOLe~*)4%lKsue=pz@h}c|R`<jfUrO_a+E2!k>3X>owz=Q4pW-IY=J&sQN6BOsFDL(cX5oHBPq;Z=oV$)qvZ?RK_;%XZ0`nAiK?#F!XCHEE08O5giLL~S<LfPjpr%9aTUWLmd$V1jtMhbS{P>2EkV31oIV#0}RHN7Nx+s7v{y`1Hngdl`{vPN>ku@J>8=^ag$I7>NcH>r$fh&oNM$6y2hA*{yAEWgrUVec^y`A5Y7500Zf4Za{p$hfL!0|v$bRdtkP5KB`LtFDHZo3qh;Y;ftX>-&}_ATA&aMvu|mAEVYAI4N})E#>2WPN-3s3pru25fsCqnlSt<yS<iK0`2&lG&;B8g=tU!?kxat1URtQtPg&ZB+|Oi&w^}zU-@usC*}5W3{5ncm75!FERv^>2yx9>6#qroBm~grq8%pc2=4JRmMl*d)i0s;A*qeA-}+Ic{QjhStyX#_+@KSTW)y9kW~BC9OFjY(KhQRbZ2#kpgP73f#0DlZaA{bI`e!rXd+Xqw8BzfRY-|DhHjcH$V5dF(PN<sN^vrgPbYXFp-qtp)-c7`U8qFY`K=?tN%uzb`%r%AH*W@)3RJ!$6%&f$moHw5R}^(v#l8`2)Oy9*-D4;nl~(JjHHuw4DXLclTTc)gEGNElgeJ*kPHVS7hr&iL0<i>5YOWa8Z9daC)QVJ*r}3!K>9Yi2)Wp%}@NM)8)NBC3@g-spEM2eU6eE&UPlUfkK6#Mu_%B`FFRQ;)9x2XW+uYx1*pTGodJIFH>&j)O<o)qFTg0VFesFM^Qi_y=jcwpHI0dBuK6yh2sYu@#=xMx&xoFLeD&3{4Qc!2m_Q|23iW!B>V@0#an9E<N+tgR{6|h-%%lLK!TnkJBvRqBzyfg3$HWuf%_6D!MIbD+O=9TG-if~&8ANd8s#-(e8w9DE*PjA(--{ACDNo!?1o$^V9H-uETZ0`KVX9=LmW*G6q(=CjnC2mwyc%*j)maci8)jR5oMA@~E!8>J5ipN=cvz+6tw^H-@Cv6{fFWi3sRmL)?OZup#(>ca)6sl(QFQ5^U1Mz&Nt!GO%rBJRn^I}!$nx&?A*lp_<HwuuRm+Np=J(IzDcyyTY<=674AE`&Z-jH}l;`d5Qz%}`5M3T<r%Vf19AL%9Qj4#ooklyWZ>augg;tWK0*^UsRsQ*Xt<B2Fkowtx`9G*xV(??0Cgqh^9grciv-M+t}8(Vz9wQvlmAs5ewi}|A8Fc(M`N;En`m)yy0&6+XLQ74vVHJ(-4^*VhKngxZq4vL-fr9GLiKA<~0H{Eq~i&KI=wiFAVTbpTt3shHMwyjd^f5!?6^s#D(MjVO_Q3jw~U70XY%SS^IlAMv6mfV)anvu4!3rb<<!*d+xKz!|Eo6RXuAfUV+aHQ6OD4$|rwN*Ts6@AIfz}X@Kit?DNgI8$f50u?FpXG42u7pXtR4HEm<Bul*?LYo_^`w>4{1v`E>Do0w_qD^%qP&?UkgH4?ULQF>dB{Psgd-AEArh!X?qlL!k9gEjxbjvi9V$lft$Z>tb^*nX#rxnz-AqftahKP3m*&+#aaibzBC8xFc-vnb+hst30p|nIdI(Dcr%~TP58p=6ff8=?w&P5+D1@CqD#5~W@6<P=(rYXggZvu&q?q%dZv4D&oFcL{8IM?Ilcli4h>qBzY579~{!f0q;ecG8@!HF<6;n~{RTYs?8fCOoJMfN-n?R7N7{*ApFx$gw*{52AuEZ3wnwr>4rA&Z#qR0lR^{*5J8Sih9W6$+EUS>cl<c6Z)-?3IrLkIGzA(53OBVR=dB~xBu<7>B(v+fBfR6j-$fjOd4)HrJ~t|t_I5WicT+L*I-0&}Wh2I7HW@tRkvo<*S&alU2Da{nQJkKei=f1!6utt67ncHpn{w_eW{kGdMGs6=qs#cWl#ii+vru^*%(wuryNugh=&lh=JFgmq^G>*|Ut!WJ(d$D*BPJnIFyJ-3iclAiHpYXX$$FvC(ev9p76#Ex8Ty>oPp=oa1Pvz?uC^cdBsUyfkF`>8~cHbRxLI5?o&6z!C32F-|VoQu4TXZ)5Z()x`p;%jWru?98v6Yv$oz_9p>VE@V8zi1iZNvZ<cg7ji|I|qUQENj{-M%XsQZ~_ZDE1F?t+T?wgxWMIPu_|_7Ski%x4HE-Ku9kRak)P#8{3;`ub3A*I@%|nv*4)v0wd4hQW@p3^H|K9hOzZ}{qlG~+E=nF(+_hR6LKSKS!>W;!g&>|8&_*;FH;8B_m3blh!gpcXY(X;B{C4<?TB>xm0qx#l7?D~OxJ{J3+UzgTdQ}fV3M1g<mXoyLqI%7<>IY#5kFWN6+$afzi2pH?e#jVMqdR`c*QbGA)f0N9XUALG@&)s9NFPvm^bIpnYQV`U0@)23uT1kXL=8X;8~!2fcU@9kay~`o(#E=9e{Z$Zh7I|=zD+r<D=fpw-%4LUcuk)|Y4>eC3N_wSlOCu(-8dh`3dpaz;QyQ_s$hU+2QpGfjnG5E%Q7Mn=Nb6-S1}7)oH`Ar|D|{_+wFFpj>S&ig*+bl9UgLd6a%H03N%h~eO#s^ew7ry$)uY7&ilw*-*d{AJ4<$rKF&x9K^*%+(Rx>~S=I}ff?eER=oVd?gloa5=Q%EohORtIC4TQ_`Ypf~WcR8?sT=SA+_&Dmv8|nU>Ia$+-hSzBLhKYslYEkafVJ<*kJXSvE!tg$3VNPqS1ZXfo}Y;$*)F#Z<MpfsSv}hl82}W)MsI{5Q!a|>6i1w{d=d1+39uDKf15jASh0T3iyqY=#5-eeiPz--4&GHT@oQ}L0v)}Q(5I5%X48}c8tDpkzqtod?p0f)t+j~1j9=tU8hIn@wij-^GumKq$KLM7<?C;1Tb(?C@zCA^Fpc|KG|#d57pcYkksKX9(&Z~To<<q@cw?&+i?0g2_tNpMe<Ks|AMd_b3$}s2oaIkl>KN%oh3roqdecsJ=37vr#@cL$7B6u?PvQ#XpS1UfKOdcxQP#Ax!}E(zXL}b%pH8&BgT2#>!?W`$m0FCcC*+SRO<L0dlERp(3D{)DHgSH-Q}dLV1X&D4D&hIf4L6SqR$2<q$ry|Z&tef%E==*_UlUpB_;By+<nSOWBgS3qo&S0+oGUl_+w`lK<S9Zo)iq=24xV$I>C0E;WkVk6cvmecZ*9+)Wi;{|mns;;u-1OVeP(0pyj-(2x&rv3)6B+3YsEX;R>NYm7?`~*DueHdW0ObI?23#{)(T+{>0Y8|FKE_0ZrZ^|%e?UfErzGl{(y-%y_ar%mLXcih{|w+O8(eFD%P>V2l%J)3q&&40p;f4H=OHt=r&rBHAUH?-dq*&vX$56_mQ#9CDwC>@jEnFEp7kw^X~rVgFO+F@LhX^+*1`bDZmQK)I`(ADw|~IN>hAczRVi~p|``9Z;3EcI*4y4(?`tzew$#{SWK$kjWj%JX?bf4l|4{)b70|Gm_|%=y3G<*E6M@Gw9_?u(XtCG&UfVv=*fjDZ=iW>+@O_YM&hB1>1LV?rLGXCXfc`LJR*VNaEd|imOh@G`Y@r%s;5X^Pg%QHdEEy4MF?63IQ&g=7~0uD;;r9nuAO<@dm8$H;qpKyJf0Wq8eOZl={O(-P?=F3<7nU&``fYa!7C6qRySj)fu`633*Sx`5x&Nx^^Jh2lyGq@MmKyvnUPJuFF+ZKPhG|#kCa}-1KFE=AbRcSrxmGwo2dBOhOqcrJ1)MiZHQzqjA@PYuP?wFOYmBO?sb8(WBxH{=M@D=emy+fJCEM)?f-gsa!{rIl<rdXT`e3al`Hu%sxwHwlAnDdzpBU8izze@4tIXuJE=^VF>EAO+=z6UT==WO$+`4*@|ut6SzggfZ!lD+>Izv0G)0000%}1v-!k##lHvXU{$@zmzWe=--6b6S`S#k<E}3Dp911pt*E}v~mDd1vNomL$qTNg<Fg}PVvj;36aEmjUXaxnTIEzxEt!2*YEB(@vm@ge5r$k;fu^MIj$^h_+zwmo5GX<sd1<u>o*{BS0&8M4EXTk~5^uat5N(>^ilqv2&{`wd{oX+PM45@l{u$(XERfW%sa}h78Dgx{<N`5nu^yx_j?NX*P7r4z~Q(b?Ms`Z7}Qy=G@XASIA#yw>cbz(=O2?zp9x)Q7~Tstyo(M~ySrDrxYPvS9$N6uSCyh<=TR<xf?uJIyiM3XsoO{?Um#5uH)-Q%(-U2m;91QU)UwJJ@1m`L@GMf||!KIBrh$DehE29Vbe`L9yLeiH+^--cZgXeN3W$Gvjr>o%TEau_=9YJqm|h+pCM8=irl+jP`05~L^+V#M%5nrIHh-PXrG*Sq3A<+2}6pS(+4T6Ai6;K`^X8H>s!2Xukox93wtTeB7?6W0x@$=l6*W@=hR%%KZt3x$m4ZD%3V4F6}c_`vaUL$U&dTg+icWFzqgR|hj1m%!#SP$v=wzJg?>6pDbWZ4*#+FCsN8u8oQ$7h*9xlw>qq2`4E#1(2lTBvrC57NNZUeP3Sx=E|rpa5r877me~e;Ewi&_K5c8jb_0fEiZy^xPi)-b>sc;+Q7u>z8w=eezvU8TJvn`t&$jPL^RgDT(}nnWH;<V-aW4uE_ddJ$N<|cY<VYGxq-ToQxY7Gw2Ia}gI;20DlSIwH4SD)=Y(83V~;smSDmvZSD@ZAe@?c+5b|6g5fyb!q;cFJ;&j~i*pPQ|gO%Xh<RTc^>IT^(Of$BXX!JV@JQRonvoW_~leer9m=H|E|3*9Jm52bVgv6B>V1TUFBbFmKH$z!<tl9F!MwRp{p*ZGeaO=X2fQmvrKi9We9xxbORpa4QNuH7c9i{u~JqSCPlD)q}Wwk$RBmnJ*D)h5VOc`ex%Q)!#T97F97IRke&hR_5DPidvM6I>Uk%+Ueji^3yJv1qj|K|Z|sZR>JA5AA;FxC-PE~DG9cqV3)=JV1RI)RI7D+dPyABK0eVLG|~Obwo0tg=R9mK9<vmgVpdcl*XDx}<VuK4%+xF5tgbn`&H?7i0g|7No0&NJEw4`is?g96dq>LV<_T1OIx}ZQa>o!?k1hLr<+sW->%L`n@ma1qL+N0D5zv?a<2NE`v2`MfKlwJVl)BU`SWo3sR60#9Bn@*EJ;#D#s}izJF7*Wxh<_vdkU>$5!9e=~#xpp|$Za-gtP;)5F=tnYNS)KD4;+28vowZUPx~xkF}b{*N2##$p!Gr3s-$KzR~-T?^<xy9z)KQx01OO8~a$nlmVC**eDvg%osWL4XdqRj*7y%99?5TY|v7;NE&mtNKzy!d6*fg-Ku;gH2YySK}R_9Y?Cc%UD(2q<;>lV1e;UQaeh1Cni;GHn2%fH|~O0<CD}+2z07M<JY56xf%+j-D_K=x$bbW1u8iyNHmopLS>@QP5pL^>$p=ZjFB#~b;{5oceca(Xr}}1N{rD(={Vg_r;~+D1;r^*M>CwTzpc*N@}v?SG!HiAnr+DuDPej>^-KZ0DPg|W{!-QHVngJK%5!a3)!t}bA^lP7gCYIv{8Pm}G?Xu8tLP5F7A#)%`6Hqf@x!DTmk=~JVp*4e>0qiF`C6Ny?0LNb%J`^tMLj0wKju8AuxgBHDQZevp~TJj)f`wtJOc6K=Bj)i5vdxRLwJ78=4gziP^jEw5|t(sS)7y=7!?M?9j`#>CtegIP>Bnl{4$o<RA_36C=r2-eo(I9xcpY~NfGgWv$7Q#RQmnTXAw=w30sxXoMn^G9Ku|kqPBEO7m}js8pGVm@FyaJ31>VDsq+Qwj|?+f%_Bk+8VSfxi|HIS@liG@7Ss4K!Wbl+loVXw#bq2R$Vl~CDW8yg8GTocfOWyBt5U6mTD7_tj*f#uJfUFvm<jzc!g?&4FC$pM2&Ms!LRd~rxQij*)A<x$VQ}h&bR;Nm=g(&%p9rR;iUdLcPD!MQNSuu1<W`aTF)t1G$PvO@mmRugc7Ddxyp3VGm2vE~CS*$YqL_in<<BzAfR8t*1?GqX{t62Xxi;%nhUZ2J)eI~wxd~7!a$J0k#kux=+eWuFJa-h^qKnQzHvRHV86ih4pwt{LzD%2qb<IMpv5skYrm=cRny%K=5-&8j3C)!DI@i9l!LIl+{p!``;UWV7Xn9jM_EKD!zFMZj^m@poM#T*sYzPq3*@GxshPQ#jn5&iSknMIhbl4F%rn56Fr+%;MY-o5x@R-iV%d(uJCez6*+wiVzjhB+0EAASlc1aj+$#t4c2qmxhI)ik{DcsaU;l9$iyq#fLa54O0ZBV#D#LWj$liTYWG1D<<CZ^s6*l#s?ylA{)%`}H<g!x*YC73vIJzFEU|MC67koU|(TM8A!aPe4P`^^}foQqv3sWCveCF<$oDYkcggIB8ohJ3zWYRG9p>A4b&6S~k~e`(x%1G_EnW8*|;H5lW&`6PpL3%abXU{@D3MU&UdcyTj?GhLH;Dz#*uP0+BqWH6&rmy9Mw5g@)YyC@;~64@ZyOeo}<%=*2$fnt2^C>!{&Hsy6~DBZMvUT@0N=ZssZO7<18s@x3AQ)DZF>^L5jJedr9*0fF%p9_$T;U6IXVmw>uyxTGTEv<6M1c<}FIz_7MZ#5d;dQ6^K^@lYsfWM-f=T950&h0^~DEFVVJL7G3oF7H*`YK+PO)H%~q3BH&wlzG@(=(X;gNyUi_V#RBp1;>_OQ=&BZu#OAFPLpNm@nSELHjqi<j$_D>7e014LYbXxs8Zuk{f-gL{JJf+KB28RA-uK?MH3Dk!~-p4P8HD$q@Djq+#4EtHmYPqiX&8J+e4oRJRO}Dz-^jIy^h7lN#$hLt^4(m)VPS>`?DaHq)QX7Ln8dpo7sI#X|Yp<_=GeaD^g^AE2g13IO=ex#BOBOiQASBORkIx%Liz2T(ZA@ru$So{{w{kI~pYwuT;boJ?YoV^Y_S(T(6l&205oK&tnXnB#qhH#bN@5Rp=#<EG4tm5RR~=J`}!%ZQ}Ub0gw8?^k37>i54i!tV(8FR}@jS+J;1J6Vn#!q+dC{^)9?-SHZ_c2`!CcQBqB>9m{vU-b1mytcYLwgeR#KgZzkJvhcc&uj+4)Q{SiFnJu4y3=AULZmEM28^tiEdxi&FQp(Yes8SMh%{No;t@RUfqbo+;PPu&Ed+T;k=6<6Y8x1%V+<cDG|RR}qnI*zC20Sby_1u}<LGZ3aT`znUVD0R^zrcg;_&pm_Vj%Jm&1e4$A|nOUPu-%PR|abPiIG5fkC=xzQIlOc|;}41QT*fT7x*$<$bWh$U$pWYNadLMlp2k@&N5GX{g%{%Q9acfy({I3}YL(O0Q}OAP+~hL{7KBMl@>r9ft&=y%0cg#<nPtsDs9eHv2rea7j+P)=_)AVIif8ZbM~JYSt?7@zVjk7RBW9-%vt|RwExjA799J1mPAXPCkE(&JNGtAAj2a^}Ko042c58s~5(B`SL&Jbc&#G;gW3$ge?6-L1>Z4R!}YEFj@qpuYxajo1{+t`nE&#9z5I6%amM~u#ClL2Z`Q-6-nY^k`_zhWx9(so?%R?7n^t!bd$R+r1(=~NbKzx65uX8`Y-smF(k;0d;kGOsN-=8rRS)sblIRI8I*eZXBnHCF|m_<d-I0RUMWfbKCDj#LN|LsF-QoI@@Q#pm2YD-YWUd{P8KjaX@I4KI~cq!prm)JISh^EE4hF&ySYdotDat}5HXz1hEf}W$;k@sgD8b=E;z5$9XHxu?pPzq0itil;-1`T(3M6)qiZ|&%1_k;29FddUfQS7!p(1Oa>zM`^i>m9J>wepfXAR7la#ZgQW6u~RW(-PN&AZ%?Jr~P2EFhARQP%?&=Bi1%yA!v>f{P|#WadEZNe-kjcpWcQT!OxO6nr{3tqRT#OW{|X^sPdPdpRuD&)3;Z@ETvHIL{JrEP0)qihKerk5_iTUVv+rG_MwR=ygE$SSYlV*rd>dm75PBjqxA{J=o&+&}#cvkon;d7v!iKGm+Y^<m6G4R8nBM70#%KkrOosLHD^;FYkWE7}hZIP76cCWpVYpR}O-{%8KZ->4-9$}0}a5{CBESWW^ogDucY`3PEj(Lq{T<+1XvtSIvlO3W|tR(|lVtHa9SQK*Bo#k}A;5lo<Vy}D_o$D^m%64r0W>LV1c`z45nA`Bb`H@tjPx=V@ncK8gwMGT{FJ0*TJ%2fIFiZz{Fic3+SJJR5~N?$SIfKs>F3nJUDY;IAx>0DNNja5^3({d|&s28|z@$b?lj3cA4**}hMC$m1fOmNJ*8KB<;K6@yi-M7LqHT_4p&&{CnA)x_DUy`;*ICql~u0VAc8Cw|q%gu7tcwk-)q}RXjx7*qU<yi#hZYOt>Iag>4R}1QNmv^v3XLl*ZbVcm~d?_auCsiFH9}9O%sXsTbuF&IzKWuuYb42>~R?Svu-G+mUAVPhKvVVc2sy&BYbN9I<ho$pQ%SA^XIkc5_<#2QkMi)g#7iggue9;H`_fbu50l}-S3$OIbezl%+d7W(O0sR`yWpa&z4Ya-4!$<CDg+p#zZyN^U*5(AR2x)LmzEM0x?y&S!29{4elCI1_D<X&VV1|_-@IJ<0*n2vn`!EKbcVO^7=nEEn|LN@D@C*du)8nK4y^F(EtK0xkxdjTn*sx|~kv_}iEXX@i$|P_`tl3PbS*GCv-QiuXTr}&8(o$tNXgY4G6U1548drB&&Ig~ME0vY5IL{`FN6cX5b3SM=djzYXLMAk+{x+_zTA8!9o%WV34A-_f?=`oJ?s;#R&Zw0wr<;)JnYrHVmC7e!(^Z4YT~|6uU#i~b9h<Dfy`<*V(cWWqZeP^`sKP)gT2Z*FPIOTCxss==@d3@QrPp%Py;eAg&-+`MT)L7JDhuEAQ|UPWB^>lgB+a@S7V~^{8$a~T0S4TpZ|J_p7O09K5x3-KgqtMiQp^qqQgNbC`x{&*H)9Bp+3&%Rt=9L|OIO_^c-2z1S85B(K*gb;5xR{p4Xq@Hc!TS>SP9Y8+p;O48tW@T@a4@D9KJ4U)D+>}i`<FKgV$XL;Y=o%M{2$Y8^qei<O}&q*IW~?4L05i>4pl}Rjf%)@*_d=BXLIrWo**&&{xIy)pCC(MgZ!L;2%t^dRX)Gbwmfbd{c9hZhA7qaWG7$uq1uL=@QB1odD&A!S)`{FS9kjm-{m3*5u(@s#R!p$MkozV=*n0>aYB~e{vnG6#r&Q>B`gk5E7A_Z>gIt{&$U0XUkYIx|Tz;l&gb)k%6Wa*H`Xf8^KPAgpV!NLzh^1a5rK1{CuFW37Opef6@+54qgoBq(pu~W5tp7^XX^xJ<=yNyp{-rHl5r~Rx*BJODag?McayBg;A5V$_##<yl<zRSF|cAYE)qSrMw9bhOp&JldovtB{8Y$yHuRS8ld_Pj+Vv(f+J8_kQbUj;k3f|iMItz@AH!g;J6&^*jEGP-mtAik4LZPT7|}BP$C{aqoQ<ljRCZJcs;1IQJ#B1fb!E=GjK{|Pd`=VUv#n|{RGtEa-f#1GrZT13e3NZO$`RGRlt=UsK_O$7@+`~(4Q0~3C4+3?#d}4@?_Q-lsfx$F)&{;yxfg>Vaj8fhu&ki5$2JhOY79MdKB=58*8X5Dd$F~*0dc7dP7R$@Bw9N88LZlJABa^%Z0M+V{4_X&V`aXG-~cAO}$xecN<PW5@uE@9}p?O5};2bqyiYKJVlzthxXgq-bXFYSD1gIwNj1Kk}uLFHHLR^=y#p8@*<TQKx8gcH+>ZvuYb#3_l5R=OaTHnqS8W8OK~BB0Aqj)CF2RAN*VG}v^k^)ji8rZ!7fS1+KO_(7q-UvZIW-Mal#PO&_K_lRdy#ZpmH5Ko<^HO19Yz^MH$iV=j&;v6_~#UP+a>;jX%+~iFL|^i{gAKjNlr3Kz1Kch+>M9W`}RpuG?2~@QAtsXsAu7OR!@4j7H`dshs9jf8y);qG6wBBq28~ST%s1Vmqu*FJH#L)D-+}z$%b|lF`(*v!HG&8WBn20c@z!0D!b!41RVKV}faA(qu-7GQSNYr&_LSjML@3C`u(dLf)Y{6Zlzek>|2_X&X{Qb2|k<JHe0<8iQ^2s9_3MF=Owv`1*RuuJI*i=1PSZF1Z26%7B0sB|Bro@{ue0*p85-FGpXZKyi}J1Fo&qHB~01+R&F*-%I#KF}m_j(YPoBRxY=hf<(xF4u-}07I;te%c_TE?hdryIxtG~<H>G>IUEX&=`VMOe=fHgZ&4goJYMR{DbIMNTG}&yNVOa2IPb^?_v(ahyly417h+$uxUv$jXp%X{E($AQzw)wy?KWjAi_urw6>yvttqod+`})h+Qa40V=M4FV6{?rc%xtan=iF;q^f+priu@|{cATvmLJwr%+x0ERp|C2z_{?>}#Z_Qd(C}phROWq^=N?X`Z2i?tLpj4WhM}S<HFmFfSBWnaUX6SOkle@{dvID)bR{-bLkNPLvdw^T+696D`tlsT<%S`rte-UB-89B-@^nu3c!**DJe&M-bf(Q1%MfdzCE!}$(G`jkt;Q13C<U#g@5fFPvsxpnC3xLtAeQsZ0}&b~?PBj>a$XGxntEnH!0R15U3=rD6^Un${XWG2st5(;r7^C^!i_VPap!%$x3@Sn#8SNfV0ybBFdN|>UMUd_so2+hw?GNm&V0EuWB0(cb|11tj7865@u@4re}--;_q@2PKE;3*q$HAUMl$VbM5-AAK}JB7t>PIN{dRPI^#1rTIyn68X#en>H_5k8Bu{3Z`>gK3BO?l-|BhG%LOf&<`mczSDj(t9)cPZSK?a0WsZCJrSaF)NDaIox{b^~7(`{mfgV#BLm8%lz74YgY59BoaW&)`=7*NmBg7@alm-~2mUDVKtKav5oD*1fOo39(^W$sJ#0EvjR6>f87@KD6+|L>m7tdH}H10KSYT*}&cSGaokSsABu-ddv}mjyw>2aWN|>3ntyo(F<gxNL?lziM^cw52|?x-D?za`P)(V~%s>#h(k*@JUqTerep;8$~$77+mY?n>XDBuBY=Po)%a}UR`R=C|!J{H{1pKS+s<r^2Za}v#v8tr_kLuyo%8qH;U@{{80X-kt!~Gjf7m9z{E%dnICVj#H#m&deuf)dAe;d;zH-3cHN+=a+50+nimktY>L83K_xii*xH)lh)eM=*v8v5MJ_@lP-(-;R!wlgDjj??ZSOugrc0_czeaVYQ}f8Iz3JEM)f0YgTGoNCZ7A1>qPVeKD@m9rUOIIS2H!(EX|RH{K~*I^!B9zBOx;Bd5?EA$8z{9Z1ykiy^d3EP6+?oMV@XfYMd4cAb&%yv1Xw<Q)~jku)lvt6Yk)>pbQKZXuvZO_%Z4e3n-DS$9k(EqxS7wt6fG}Yp4F*>u+&TPpZ!`$cBS53%+N;H)r4B?pLhqIj;U=*tJp{JVw%skX=H0^B-L@<)CZdCR)lmUHsabrOm4&-tw*&8v^5pNbLmu?es<raYIu#FpwvhK=v`{QEm9hGwkgKFS^q%olo@O~^aEXaZlqG8p3&vGx@8(0sU~P1Fa&04L+UgPVbqmei%2gLMdL=2#1T*N3x#_Bm?@^GX#dl}VRU}<^U2=D=d(k=fSzyXnd4iw8T+y47;ct&S(wK`655UtCz)#nLCWknb~4*?E!zk)&y7vP4qd}YOuzR>{f~6gL{Yw8VP4`$o-y%kCVYnrB?XC&*~WAYldhm;5Th7~;Jk30&Ef40^4ajYPGFOP=zv#;MJQoiTdybCGl6CJ8_dcqC}Rmrfy)i^HrqemJ3l|#-#bPtY)|_2^z<IoxMA>_Owu*{iykcXhXX&vRhA4W{+q+UU7YPjWn+VX%uWtJpNUrSiUcc`=yZn_wMwkTA87es6-)fu78j{NM}Ql8-U>xiJ+0iWs%NeEE~4>z_!i}1qmfoP73FCQidJkBeELj~lKSg*Xps8*dUTKayLM0yI1}0#bzr3qFU~iF73!Yz5v-+nfn7cuOb82f1rkE>wQJ?5ZG7BpSy{hY03M|wWtM8qbRxl~M%vg>YpWZ1Ha6M9DlvEP6f1V28^*>_D>p|~R_;Q}cI667o3%Uew^_Yw*+u;hOm8&MM0`;nAGg7mfqba)bNgdNghagO*d;;e4BXRbuwzHHS`=mQ>lT4}8XYvYWUFy}FF)&nc^VCM?B7;nT^G$do_7S|!7x0tz0Vh)qQk%KAAdeLJWw}&zoNzig*2Rh`h2#37@ZyNe>yu51cElkNW1)s$&PN|TaQncLcDi*AQ#LJ1%-Fd4)+c|9u98_QsM}pmhCzg&Nl@D$5xVzvBZm0$AB++np;+EvLr}t*UwGQ48tr9(!fC-XOOepHtZm9y}-#V502o}=z&DOJxs?(`-dmzhjoZdC#->FhrX4iq3<}k=Y(Ny<Y8iEN$z`I=;c<-{5T6@d}wDG^SsQ9^T12PEKh7N47}HYBvh5CP6Sr^VWq-mnMMt=Ah2UIPEF4;Jl8Qy_?B4#3qvFDEz5Bd*YRS{&kWyWPM!h(lKH9WJ9%#VS!S58gN)5?Vjl5Dg~<#Yw+58q#IQDwYb8O#a?=i&<GLnu9VhdA$8|DhB$<_)zH5aEOUyL&QZvX{5V`@fR<8r9nFK`z%?bTFXaOup;wG-;=9!tesh`BY7r?^#jD@DbTr)EQW|&^$g{c=>wi`R1ofvUuSTJqL>p+u%f$Fnm`!$d(FEi7`c5GM@GfraWS+I<m!2%Ota5F2;lElnh*I^c5o*9~fXQfUYC%Iu}aoT&js>trv8<H3ALVwfn>o{gxX6A%WZrgE^<Sg)#%nSg;oW;2hyV6Zk({X9SV$ZW|KgfJ92T+;8Qkdzj=S;vP$i(4EgHiyK-5?cK?j=qbxM2uS@)US*W>_w;?;dPZ*Z{7PnW1NUDWIdp%-j!*l%-yt86HbbH~2=#nqV1jeZoA85ZU$}#$a-QT3BuZlNK{Ggf$EeJI{Rsmez?4AQ>=AmJ=sw1|#rtplh!MOU9b05elGUHK5!C5GhRTAWNJewG1bB9FSmg0Lu51G&f@33<8*A*MYYILKzzd%M92q9L-(_R0dqqZWSbIvUr>&H2_1;guZGpW+7}nI}dZqiy17opQSz!QlRyY37}emo7sNq#xP?sP%zu^av&;hV8jN@oFBgyMxq(JB78%0pE^K1YS8&e1iU;kz0}QtT*YPx0H?kW!eZ#8nUxrh1tKGWl){w3rcDqleb4ZH!?cpuLMLNR>^1_FG-;fPrU(U2X4uS3nFDK_rLaSo5eBgl7-5{n0YC;D#>(Kd#D_>&Y!CJYP-!5{xs`?Px5K%(jf(m74tN-mYOVpQ7X(fnR@Ve_5eB*M010s%4;C}Fk~|30%nOs$b^tZ~#P_`zxd9-L3^ovuA`3`hX1M7$!&*jvX;Np)ZJ?wHxM*a#Yj}Q=qya3r0gwPP*a?db&-62(E3RpoFsXnbA%im}G;-VXZ05aw(U&PaL!YT8I{J2<$aqHXWI*hI4av;RXK9|X5V!ya`<VGz7z99^4F)q72T2mRh8r4g90#!vyi~}_V}qo*?T-yC-c1Mt#|rA?85pxXwsHRa1Wx-HNFJQ=HsB@z3k+2nhkot`c?_i40w%%mQxACs+fMS#_Fge}1Xb5z7TOJrgG1KL;ZR5o7ib#`4A=n7g_GVkg4hEjvtipBb{1Q3B)b4(p7?-*09Tf|K)_!M)N|aN>A8)A1GtX_*jylT0RYGnSYDV`JV|0$Rp2fyU>9tloWK)1i4((#5yl1pANqEhzZN!ekD$3r9k0HQKG0-fJ%HKGY}3ktMX{l?5GWi1o556uxo4X$o^&w_;Z#Y1rE|j6$0v#NTCf^PBWtII8#boQ4fDWBJ>VaMEcBpTpk+B6EWVleRuTeZ2d7AG!7|!;fE+<)GS3E@<2lR?%yJtnsr_wt0X**JoxHHYp~Nmg<_B=Thb;CJKy#qB%(p#QTtF}2m2o@3q2jxS1KZRIfydA7%(k-JWUrVy1gTrX@*3-Dhrpo2fdD%bSlq;e10Q)N-vj~zv>|pgm_-vfV++Vf3>*VU6@K9Oj^Wr2umrCJ3e}Gog%tla&o0-gQ#CwMP1gWs$Ol>hfP*yX1R!`Z6Q{)Wf$UouvzU>>@sI#R<H8A-1u2L&a1e&y0b10_t%E37=Q({~ctJ97a^@!>(D=v(!vU2TDY7fTJ_R-sJrfGJAaaGNV*$Q_7y$wopkey05E3iZ+^I%`fl86jh;5*1fIg;~+Q<#3fOI)9=xG2z204&hU^;=wW{wMd3;a$Qh_4Creg9j5<BU`k(S@w2xo3@_!RDp02d65mYL+Et;3AI<?57P!YL?hm2xk<W55V7f88B3-1*;3sLS`Fqh?ueSeNd)JeA|E!c%QJr>X~pf#2^u7AYg|qi|s6TJOhrh(9MA0g7lNPt`#_;>t!x5^Qpx^h-1)@>$vInK?sM?BAUiY^z5|OWex^HYU(%^@KPzrW=<IAN$#XU7y`!{TEI)k%z-ll$TFHT(ik8QL!glXis`QVLm`ff#iBXIp4mXv80fZ#n>USpSayKW^;}r#Ft@`P_`L)K$P`#WI8#7?3xQVpMhq-NZm}fDvmXjD-rY;)Gq!22UCV2#5q#iV-NZ2i!voTY3}BE3z)OS)9EiYI1AgS0Z-6LlnJ%nAkON%{3<IR=(Eoja&z7?m*XY>|?V1%DX##R!Y8x>FItIil2cg9RItOAZgVPjHC5CFD54D+%>MhI23=?E*5D|YL$ZzM#WXi~pT%UKxsDtlVX$T0D!Hj#%auN^D(=1JGW&qO!5)Lr7fHgn?fj=_h9K;LPg?$MF0+uQ@e<$#vXd8CVwd<^k>4ur(r73C!6F9|SccwW*Io60%BV{Iv+rHt425{(^4ct8<7_iHp=~{5k{64Thq>Ma2S_hx&)p6bcuFs59V5Sm~w*h3}1C1axftCY*?*rczdT|J12S^|e6pW1!=wS*RP!d7~=ZAtX-4z-q4eZMLKFz`$gm6H1kj8=}%z$9W0nl-f=fW81yPX?m0EY?#))6^sSP&;h@hQ(?8Q2eonJwT~lQ0@~os#E15JNA7`L|#Lrh#FB{Fb^dP?$6T;l)mYV}$P}utce4xPg<zSr)+POyEGWA3M+Jm=Gms(8H`Wo5-W%)TiAxEQUJF+=+of0-yo6L*Q|lf!J)AAhHHl2;!&{qe9BHbKt7tJkFR0OhL+i&molGnh0-&4N@7}z<Ao358|Kgf*5TlPGA|a%RsITKtLo10Qg9A7+{b%EC@Z*0P+cppKm$#?>UNeetX+m1gC*WIQdQzrh%I=(*c&$wG%tDyxc_{ifj4_9KnG9fb@ZZLj{5wrzQ#_F!b#Ague_Q(cBG|S(jB3gXJ#pE3O-OutW|>Cjqc6z|k70;q`o!V$8&ZQ^B|5#DL+z-pNu24ujChtlx7G>lvmjD4K*nup68Xknh|82{2EX>m{g(3w*=%;E9u1DA&ZUZD(#8<0cM)?QtwC^=yavAp@E5dxtP9@MKNXSzD*nq?bN)YI-0+m~r4I0RsUskMVE<X$z>0ZCL?JfYr5PkS}02cmOsq!?7R2{s2z+`#`;(bg%%n8Cv!EbN~(;Kmm3Nghd8;3`8-@^3d}=6Qo(g${GF#Y%R0g)Xjm1F?~RC1K6<K`%aWt1BkP<38Wu1)Dni1d6t`5mJcV31A;X44@5D@4HWQU3j*fG33FiI<*2nY4b$+DSImHgvA%}_qEBP7nn{#}Zx{)+C3rVfPaxaFOu}r2S!{-IJ~H?lg7AgsyA47boKp!50*HfQfzX&4Sq2o-V?p3EkdJ;Cz^g?Op)chmo<@obWdnXVJL>SWF!O=QMnxz?IuQWZhc?F;grd-NOef2;AoD<G2k|HHJSX*n0M2?~(-WYF-nYX4|ISRmc8Yz+sFUI#wtNrdT`NXq7RYCDk^pH2J~cCw1h!uQ{7PVhIFy<JNSVwB@d21Zo1qikxAU(E?)7q#Mc1s!rP^?+)m00g2a*@?*@o#@aB$?Xod6f&+%y~?WFN;*tu&AA)B({K$Quw61`?%Z0*@L3rTA7DyL3k|9Y#YR4tpP@au8y2(?Q&Gn1OZ-+s|C!m;rkY6LndDsBmJ34x9^C5(b&+!9gB>6Lnp{Nz2^^chlv35y3iN3%8agqI!ngP$nh7B3qsjI4-bdrU?=ka{w9}t(A$#fO!j%eU0<fU_j1{)Q$ZZ&gzuKUh>*4r6!cAol>@CG}uckOIeb@e_@cjE*}5DU|K-iO&9i(XE{L#3=JF!D8c$(=H#)T00-*22~6y3=W$amjym^f8Ffvj14pL|R0`JGGE)OcKAfLG{((0J*&+>LjtoDx1Jg8N>x0-D0`m)}1@nvyx&K!J`K*~CXyBpg_(tkvCa_j;n8$b$<seGHQDxXcm^x`}=DEqxCKAJ116V;|7<m$SnPtBkv`)RMV>af?198jqT+4AXKzra=Z5JqqnOh(vFwZ5)6IdnNPJwUoa~~wYICesuS(XEf%N6;Rlhl^F{IbICfpGFVD4o1^Kk+*e@)ds*9jxQ&DMjrDRtcn`GCX(F25*}U?-L6ohAbwuO|r}d;$?ebmV_=#!Z=G}$Hs#SrWtj$zU!nRurQ8qWq{n4vArSX8wSPifkwfz0|J??p$xYZ^mQ;(pcAg?1_|IiNJMDr198GJL*RRoAO<vo!@&f8F-4~WRNv&D<2u$48Bn)ZhV9jfMwqiCuz?M9fJ@8*ANHf=7zXNf6N|I~05ID8(eoilP0Py?;LBk1g2ZRI@ectNs4=CEX5xC-28)f^aHw)T)C1*?!4d}~3G~)*gWOC55EXnFsP8Z@$l>fwY!Joa%z#atG8DLTpoFpeEwBs6!4h3$HiLb1EG%C?hbs-OKSMzp@FGhR4{4wW95Rq+H-@Q)p?RSVRLO@;;0G+UVn8jQL?zQ^AW@m|zhe;Rlk3~qbUX0APC--PCOs!{{m^w?;NfD^vQcg4qVHhr86F5$HXu)Eg}`I@fc=PeAq=0Pvs>_Q8;kYwSll#(L&x^x%nW@vyn)68Lj(jj4nq^B-_L+ihHc@1sOK9j0KUa^QU>Qk26Ca7|J%mm8ZV9|3__qlAV3+8jmBuhbQwr|=r5N5o0WJ#m{R0Mk-fGoI|zUqckLLMM90ic>qm~|=Vko3)jnxhw1FMSaToX?V2AJ{@iSx)0KbAD2By{ocr##k?7&JaBaVIGC16=x7BUA89Q)reh7H@qHX7P`KTLoeg|OjaDNs(xfc?%~bVUKK)^gD;&-4S!i47J4CyW-8z)u*gUJBB^`R^Kv_3}_`mc&5-k3AdMc3?h&6qyg;m4HsgFhn#z*j65cd<mos$brX@J~$w2pqBREHI()yX$>U?xd{Fj0xgbh^yp-H7G@02yd-0==^2nI6OI@L>^-pY=%ARRm||y17J_v8BZhLu;^|hK!>P-6Ktq!pRXnK+2t+=eXlgg}z|RwwSurq<Kmqd5v|<)!X$m7qV&Kga8w7p(yEb^|QSDj{mdH*09Ie^F!&z=<q<QAV01KQJzTr4<p4lE+tC%0e%tGHII|o|o+XjeVhVgyJ+_vn6U1z;Ptn(a}<$3M`FX{r96xg<(q%i=TB_5j592baC?D-%pyFhT@gn$zu0O7_3dH?kwADx_E>>VFB;j)6d86L#Q7_burw;+$b6l4P1&OlBEeDv+ifOQ7BI1PYBfOi&94xk{06(<=?z5^2A>j4u!JuMgC#yJC0Ih^nymB8vdNooKGpSl^G?|zv19!U3b;=o$Mw3_7hn**a00-C~f#TIkJ*8}#EUvcT=<K?60_K{y_#(*Dj44b9E3x~c9fEmC==IE#w0LVGW^>Emu9B<`z;KS)-WiZF+s-A$n<=a1E0NuFhH_TyHmRQ6F0?~DVX*X>@w9JHsb_U#+4Ft@Em3QEHvAqO2m^d{I-*TBB#=v1FKLXON_f^Bs1#rMNJ>Un!+(<GL#M=PGd>BS%1BVhPAnt>V$5LSH9p43!Hj6_CH3J^HM}EU3cY+mqR>MC%HdAI~z#;j#*W=U!!4nqVNPzJJkux!}7~Pyq2X$yJs`p|)F&*YR=zYju57%}%#2SuQ4R2CAVU}Z~?k-?fnpkP*xM&{%)J4zh4E0o5lBTv5z^=?-*QGXdtpq01hw*&_VMQ1ArcxC4LY-T4OwUOjKlDKuGjlJrn1P<(270#Uz%qCy^U<Hrv12CyBmm}(vDC5|2oR11P+dJK7p?w!c=Yox7cX7364Tfo1HX3Cc`Ffb*Dmo+WAA3QS`?$*-E29zW5ad9mYBhW%~r$f`TTl{Ng{4{v-$lDLtXrzNj5hBJ6i2r{OinGJ}g4#{9pO$oUQ+RHaY)rod4ba<>StOu?M_WJeHOIi}E`zZ^!k!OTf<Bfzi-1hGq^#-2`FQ&kUFl!vIna$2SB17N!)wCa{W*A1A&SyLk>X>$+wJ4}h`q4Z2hC1#iEhSXRc*;1}&M=n~yu?D3K9(2#fV>h#WwgQ8o2gcA{uf}G0RkiGq~A8k6BQ3(Dy1v!hSS{6?yp#?nG?!n>C&wD2t-;-Lr(tIm0;rMr47jP9$SioQ-N5^*02XP&Smb$3{I}CfKp%m-eh~LO2d+uaetfGtK^GMEfw4fZ-(PS0@aPf3bH!5*8kytGw)a%D9toyIY;+cCwoAF6_rI(j9+gP-W?}yiu)y+ClF$q!VZ2lNeF#>|TVX>P#mKEEmjQ3n0wr;}GI5cvL1)c>`X6jo&xC6)C<)e@9Sgq&EZ5saX2gn&h&zSNri2$KYqPZ+{nPKB49}mrL2}?ZN3E^}z;S&Ta;C0d5vK`uN3vjP~WW^>({Akb%Y!-s>1N^6xpsC5X7_iok?=i<u@g5b(Ync%nftTamCDVt-lZN-;i;Ghx#QcveP_y@Hkzi~NZT9vefgA|Ob^=?#h+WvvVd4j23?~f6Rd6%MA+gu^Jr^ksec)h$n?NZVI2t>~Yk@iNyDYV1+YVT2I4*<tf#apPGy!_Z+Hi)O8F2B>+@E-E#VQB)=DqmuA<+AayYn!<`Jegp>|a*#-_GeT^JJcX`rB{+{r=-$?+=VE3AFE8rtLQFC*qwJCq}<aV6qJdt=VA!ayyW++|OX1d?eNuuwE9($8OGy6y}j3m+rex7<@B<erZtOPoU@N;q2l}q0olmn7~w+1{~x$;HRCta9}Zz{ZX}&!7jB^ko4p4q0k#I96X0M%hH@e7wy})a<jh1w8eRxB6{pn1kGaif7i(!o9^7?yE2nRv7@$3$pK&!qxBr9q~QU>5BTqTVH^Ml1-w;~XCPmbv#=2-yVU|V;Zg3lTAtyXb-CTlyf}bPTqj4DjUb0+1J8xcYOvhS4I35LVUXD&;89{_VeI=U1A3hXa3tp0c1X1CFrLAja|%6*NP}6-CQFvCrjJ_kSm6&~4aivnX`Zz@ZCucb#oXe)etWGsH76=i-2clqc1yW$SLEF;O7gIKXAX@myQ#!o$1}~)t<eFD>9y4&2oQF#WB!hQM-;v+nqY0uGz=R!=FQ9SNzV4qlhrTlMCKEJr!Ce1a&g0!S~iDOz$nwjda-~tI{#%4AjJe0unBkM_+bsNVRkeOKE^U0-Y#Vmptsu{vei;`2buU0tFy6XWkGx(O;-53E{M-vYW|;Dkd8naKVmhyoBg-$4aee_J^!Csify8${D`&LvY&tNz6tnQ_n%v=ng-$Q5F-YEI;krUdasc^D;e=symiG3_g=$Cm8?)NvzZ;6&}fRrNQ1#nbJN(uXfV)HV4GHu1pt4qapd;AAWJb~jE~AfF9%5v_NZ&5XqiFpVAL(dp`G`hFyfg=a<-@veDcVeG9E3*_XrMu48WOw0bHYBgHcy5PUKZF>fCcFIYnB6DGzH)!91%@2D@!#RBxAeQYHPNGzequQg(Nxz18}cy_)oORnK-+t|Jfmet9)$rXr?U;IX4`<Jlw!0QwxvGLK3oD*Z`2!o=JZOAaSXyNw@Vrq(i1Zb3o4<#Z|W+K1ZSbjnk6pmSFNm@8Yv%om(L*7)v$DaCO6@e)(~OeudAX7iwOvgPG`z8cB^p0jLT&S-6^b~h{dnsY0IKW+$ibjeF!Rk<aWV)J3Wk&;9V!+`SH!4POS{WerEtCyUt%C%6EWh^kxBrb1%FcR_2Kfte(`RW5^Ipaaj7xj7zOb3IR2tn+?3|^Qt8M@%~?+PQZN{F!G2m=d36QY!$h*zVRvaIw@`69{q#&Wf!1f<<gid*!GA8B6~Pdx$RWCrWA!X#H?if%|zk&C`Su-zWtbVGdA!&HhGHkduE`hX>K%9qw#ukrvx+16)%gRWGr)2KE<v<B6r4{4`TTCXxBd^VyC&Xd`i)fZC$+SSA>7ooqTh|O}wc_xT0R%Y<>(wy=0B-5&MtV|5IVO{cSX*gKrp-26<Pm+$QZ9yW^3Q01e$u>2lK$w+>$Ea)|Qq?nWn&SkWT*|r3<HAbBkrWVdhNRd20OO}|Vj#-$A4-9zj%<L+IVVCZm*li}CGs(q*jgjBYS*0&^HaCA#T)aEW)Do!cZ*?jmqoAK_v*X~Eix}t%`Yr1GDGjQ<$6YR9Mky8%++cjT(Qj-k3EHS$niEIxT0vx0?3ZnwZN=0rnf|7!r`wyalF8+mbtxqlJt%w-K?`(n^BHpg%)3+41PQfT8RW@6wC9^r>CFJE)EYch~1kq4ypczhauHv$WbIQaM(eU(AMeMr{4}w_D=RO^y}9*ZzvcBBI+0RSoEt#U_mkX<774|R+F^yndfKqK<Cp^1y3DGZg8GpXj>5~vDv<aY{u405INN2zihGv^{A`3-W51?)VpRgjF!%`Kmv^egkDo9ea;6(X*`?HfL8@^M=as}a<XFM+Ao~(D&+3j=aY-0kB8cCrid-Ox<Sf~alA!j*|TE|^Nsndfw!2<Bji`d^so;z9ua(SFki3ZWj01$T%>zf`|RT(uS*s{$9IwxQX<t_<H|PUVg9vxCkjyq3O=qK(Y1c+M{B(iqS&Xhin9O5<9fl%9u{aFs@Kc;J#>!QgbkfC7U7t7C}4c^_`o9tAF~HQx1vAb?3*zdv_e8RyIx~3-mAB~v%fUoQV{bNWE--nP;LR)03Br-dq^3>oG%FoUcjQotNHCD)o8_fgJx{$e7p(2x>mPdd&9WKd|;i~ccH^p@`=|0faKutbWcQv{;+p+{Q2xKI=|SvIOMWa9%KGgiZ{xMub{0Yj_|pQCcwPh;?<gjn_R23tW|9+IA05KQxFO{x0QsV(r~Pcp`;N}SfAknQ8e=fkX$ydI_b$>a?6SW=1&}ARAB)8*F9VI6)vF4WGI^KUvSU}1@r-h@GrroVC<q@`8?AaZ$_;}DC<AduaZ5A`lQ@Ih1wx<fApZ=gSNDJuJPdEIEj4j3d#>anq7`OBDoYknawqxQVKH>^N`cU7cE~;r<g^&1LpR+JOh1;OYds2fiks&v>SVf4&VY4O92!wpW<;Uq`5X`A>ye2M<gnMJ4`7>Hi=`1OLy^d63>LRtBVpK=8bCko2XCuPUU2jEPUgyv(e|1z29J`>|ur?HL&`}!;4=&0aHG^qNSVR(#htC8^Q-KRDP<_W2NIRHtA%JoY0KqN6B6))(S2@lo@Mp+3IEv(wo+MiyK5~TcxsPN>UJu%Xw-~6NOesDnsl|RYeVk-mDW(rxAnoq?O)O4MhCQSv7wmpgPT-5MjFG_)0ddHA$jC^zqZdVVQck1iwNC+Jh>DgPPt1FF(gq%vC^U#A}H=X?{IcBt80?t#22FEH`*3v2P4zFMINrWGwGQK1z(6jn~6P`ameF2!Y7^Q@RqO2WmT!$YIjElG!2}EI0~(P^S^AepF3Cgxlm6nZ2T4^;4~%;IoM0N#IBF%ccA>q(laNS$J?Y?&8m?WQHIM$f|=K{8ciuL=BN$sFfM~VT{kp-=p?Qf1Z7r&F^Qs*;?GQFTqvBz=~m&_|zH`3<WsY6)-JTVWoj8j=x%)n58!By15gPfufT~Z*-+Jx2Z>jrrA}8yb5hpS>dXxgIWetLNB^FNy(_VIFF0bdNJ5RjrlcZOaL}zy}%3#^Xd8)nC*_<k7k7Hn$m)T;vgRIvyPZBeiHE%^Ux|jgEMfkUe%w~m^FAN&fTa$)<451JT+6}abu6*xS;#1pT@?hL*4Kd-aVD1V<ED4_ev$<uZ%4;uINO+1)QoCCMeRXgH5}?VKbgc78QFF3~7p5Se`gKWz%T^lH(G|IMRYlDEs6z%mJStk$i9&`U?YzcN$T&(Wq9Yr&j~(-GgA<D-}Y$-7oR=^_1;mvPsI1fV#JAvXsnr53}>8D+L*9O-uCZ%W^vJplc`odU$qncpSmuw*QM%EuMZlKHC3#IgPv##4O%QNlh=G8{XBp_PfHK8foONCKX{UvszQJZ|k5;e#%cI1%a^5ac94P2qF=hRI`5t48_CGH7y`6cWgP07rl{f=_17(2(Xwb2IjV4$b&f6)1cnd>!BBmiIq9}5D2<C0CdOsr##d@jjQ((F5MXX)UMUJL#oWDe2YV2k%EJg^~71{1YIiB>#9mZ6od3by<Pp;wh%&kBB8>E9VjOfcw)U>QB+d8GWlqMJhW{o7*0@8a1#<kdslxLn;ngR!i`%M8=IJ}MvKd{I}@BaDOHzoZ4=@2nm=tAzf3#;Ux7zkz_XvmmeE0ldg!Wl0RyCB<t{RZLnz)HP{xWc?4HwY@c=d%OkG#IRCbq_p!}yWuF8_34Dp)HF}Kts@I!OdN4~=ZV8}jZ%I;NhzK4mqCrE`ajjIhss&rX6H-z`Vug)+`LvvJ(qE`w*s_N)HgnT-MY33%H8m(^+|L|{z`=2=juJVUfi#Ug^CV7iEJS3)U8HpsONYeQ!DTgk{T>$eq9UK4uqbJ=9rd`%YaY^uNe<r@HF~F-1yAE;-oLW`oFx7p@MO2SR+H(X2^>0T91&Z9oB%`d<(b2)Vs(RV549)IqFqX65V9Cqm+RfTypbLBv-;T|iAPX(j1ceSzBnrZ_F=ZT^7c%W_7Xx(qcMt_nPWLW;fpz0Qe);tAu>A4){P67k-uWS*HncKa%olyAt5X8PvLl(-l~+0Wgq6;ZE)K^%v!{!8E<ObG^Oq0NFQ4B>pFVszK04tgKv+0A|M2PTBh;$BsHXAY=)(tUwe}v`I@r6|Lj`s7IredIbWWo0#o_TGh`eWiZ!~&(bQ+y}ijGbW4*$kMg$J?{e*X=JfmzIcIo$u1gZk;$!;_=`K|oNGb$onye02VC+{-cHT~7yIWbZtBzqb#i4#qt?HYNIqC4}j!EnzaBuYf;^7d1<!u1S@JVi*6%AMyW3s2zd*Arv0k>A{D|`W9AmH3w?CSWfP0`8(0U4o-`AKyG2n@?!i{AM*A}!dK_qCTNYul*v*_S}AB1$SOT8GI&zs2mC_6yS2)!^DLzlT5wp5vyx1rW|3R3XIZ&L8|^EJNcjX{Zchx4A?G6p$uTMxEUhI%Jv}<sXCOyE>gk;&IqgFnI8N}%`u6m3_Y+7^`@eB$<_n%Abu|ZW<%TU`a_8FK`y&mbuE|9Au)ShZnP3}o#`(E)XART472U<|Q@=ylPDQ`(=#Fj=be>X^zgWY91p|a@xc}++(^+)(^Ly*(v%SA7b}>1oesA%($a262U^_6oJML0to>Q1PI$sN5-Ikb3ux&vfE>Xv?w6ybPcz-kPscIXM4G(LfPtYp9F4Nu>k41#PHx}kOt;?*m%YsE9uE5D7t&UYqY-fx_Ts8_=^G+GV<$_4rf=Fxmm~(Xi(SLO|-(Qe@d2nf8UA0b<s+G|u|A4&rQS3WK;1XMqdID6P&hNFa>a0GA<nF?XD2%X}&~ZAgnGIL*5-@7q+Ge{v!!2NQ?;S5Qy#0dW6t;)j9?WN|(^ie<Uqp7(vehcH;Sd__U&4Hf)S<Ya^xn=bvN98tsCRZIutAXpgC7%4Y3T86zC%hE`6ELHQm<PI;0*_^r=!x3e;?5IYGyBYOgP4M(i{q~;iTGlnyJG8u*q&ys!Wt#2K98u5_@Z;_bUu?3X9Vl!G<43=t_q6(J0b;wAp$izUOLhV6V^8yWR+Qp!!0QeR*HK5rR|pJOrdT`uq`2mk*x?Fujkf8?+hXAVec|Uny=;S65v)N=ZtIXgmI@NHHTJhhU;e!P<``>%Pfi`FS?^<>;*UL}C_iz9u|iCnGUk+;&)bR&}AJa0=M1b}rWg3(Jr2V}T9&8ytv0y|{dSen}oy9ZJMS-Fzr~rAh<hyTy=W{A<v^bpI)jGNc;MIK=!xNVcQJkjz=U6n~~S3)e6VF@VEdb6`-GxkI4t*U$I;c!6n6Np}NqGhh(Di1VXYQckWbBx5a_Yxq;7N*CBLDK+QZYP|pXV6S%4Z+|-Cx3O}%bg?ryV!{h7GUQ;X$2_Ta;1!+PN}2!L+?Ou5jb&-R3T#IO2npgM$s1^7htqadIm+93$*d`v!GK6mVu~VJ1Sw0U<im9I3(WlO`7u8_V&0?PWX^Us;DV%_*&Ur(l_J55yWexq@||;74a8f_nx4t*3`qTW(tr<?ka~Q%x=b;ux0)mn3a$M@)@o`WUWl4_SR3Ku6cIp0z($IaIaV^rgNSkg6yz}K+7H-#EKfiroiw5Y=|mKH&7x8kHxuwc95uPYTO%g(Rgs3m6{wlZDI#?6{uA9O>a>LoCV#MSXSOXS3SEvr8!sC=Pc6*z9K^Mp6WhX0z{Kh>aRW4;P5L<kO$Y;^W)BEDUi>K*Z(hz|f1&-h5p`8fa>(VI0yLh(JxGU0{h-fMNx5mYiwp?9g3Z(F+dm?=h*!Io$X-vEFkP!TbWZVrhH%NNmh`WXVREg_XSh^$i-|A2bE)g6uxb~;={*mMbJ+d|6Cg1|tP_nk*;FBjsq^p>a-qv_c|H|fUekvib8P&h@Mm6Adiz#i!nESJQ-CpmeHYxqqG8_caKCa0B6q;$bvuv+{Xy2hG;G!4#Te(OtG?1Ef;Fce+a_td$AU$;m73Ru)mMjS!7de(JfAn39eIvK+_qHdx!;OH95Ye{RaC$&guOIXumGEpMQ^?9Woj9~@FfA;$ps~a!uWO_0o5*6#CZ+IW64flmOtE;`+EPD`$2~|?BViA3DtZO+{!2X@n?N{6IsvuE~Uik@F<flEcgHm&9BI~@Pw=A-)IaSIU-;jFasexL9DT1qOh?DKt>3p6z3s~YY71n18Ak`l(4@zsvB%0z1`bW&G{9><j+ul&8<sK*RvJB%ub82g}Jwz_@84St>^i5*Lust07RwgVG&a6jaJA67N3V)6%+4Nm6q${TUc|+n>j;RHuSvRW&S4>3?S`4VTqW_sNw8v&P#SZmrA%MoN8HNjl1N()*L<vn42?b&(VxbbtQE77@n=B6bE>0kl=%F!)8@Ym3l3yZ2iO9vG-&c<ldFY4VK{L<j%rM;-j!y>Ms17;wlX@Nlhq-rS4WQLWXidRwfSyOWT+%7<jdmC52(df@v>dl1tU%og-42uG$X`7h&aITK$O|s_PPUD@QIZTim457&1ToVl^6#uchWDf7=h6%gcEPyba~Hhy6Ogp>7Eet80q7`eReL6xGMdmPddA$RD&#!i33?V9dirZC0f}PBfulas(ylZ&_k=H+PJ=n};_nNj03aOFtMadkJ=|xmC}3?{nAr-7c}97FTawA=Wsf4HSTD^-!&bxI{>^;l53-a09`zAWFC^bja$(&+k9}<&~0-Hni<Bzj|@xUQaDS{{zvo6BbGF<^UP=V8w#Dv~Qui@hQ!WdDe@oJF!?ox13g9e|QBi58>iTEsBbd;NG-Le)|0K{g=-%UE6fptF@PgO?rWPeLL<hCb*8}C$85d#6a>=acLotfnFU?X9h41Y-2P*EflnqEpd+Yfa37Q$5+okzd{A_&#ytY{TWS|k!Nve=ca~vw^uB-_S}hV{$8xEa&<4om5bVHN2G}#ft<Wjsb&&>VKOUn#pW3%gK%xaVUXIC?rp>-<wMOzRM`Z+^7hkCSFUaQsEwfSXQ&sswF@Wh$HR5lbuMfN{Ustt!yIy96U*n^yab#95Fu9gCR+G3<8Yi7LWXg0!HQh2>{K*x-8L820#KP}QTv&W0v}<UhCuhC{SaCCsOu>|nyi4-R}pZs4Qn=umhyJq7`V>7{8*wSq``H=n;Ex2D2;?sRbr|Rz;inxN%D+THlOE|yVCC*ZVtClr9zqz8G71jm+F-IEK;Kcn(H|^A31f2AXKfteBX&&uZLlY;B5N5skhf?Cf!Bvu@dIFHh3e;cNNU@g(^-8Vb&rSgx`rp??XP2$iyvYD@?2F=yR|;i}5f|H5rkfFx^L6u1DFY-m3c0(1B7vM;5bGrxMp?E8AM1Pg7lX3de7?B-PM^;3FMIR8yUfr!DSqP~2Q%c!(l6y}{j)BlzLXc)Gf#32Qo7Ph?oppFA%X7?KOMqN;62$?d+xrcb3b^{PsVfzyR6j;HptvF^;(s9E+8F+8N9avsu=Li(~&Ozw2=nk3T~$lWy>%IUX{T3!k}2BNZHsI+^nzaDSL<yAFe?~64;fHcap=DupJ{A?y7z&3YL>m^`3r?$F){0Ca0^2I`rQp+c8iv~gA8MR?mgB7uvWUJCf8+gQAGgL=A;6QKe5w@JoCPhnGbl3{npN&Sab1}a{aG?Fl4;aqivG%a5R*Ky^vAWyqp63%F3tmqee8cbJ$}g3CUGAmom5nZ|{BC4Q@9w&)lp?zSUawVdz|k8KzwVxzCO+$<f|~n=bB0?5gzDv`<8&uW62{VVQ!VV^)g&)h%ul&$QfA|SB9;blkznGGSoFDEga1?s8b5EUWtU#KH$k18uXgipr=z7ee@iSro&zToTNT}bRvYu>t$J(ix;XlcljReto7{RF+5AdmLBAw69R6cjC3dkXf1f?sYfI*vHa)Rj&uRZFVsR335$7DxBlm2YF>XAexK>njhlQeY?(5ZK%Y{6r)^kpOCgZg4Y*H=ju-ey6W^^8hGNmV8<@f8U3rdFepa1w;wMZ{J&GKX`usTVqRM4>OQurdjNlQ+#xLjd^|Ml8hWAg7AU-Lu$9EXNK;OFN1I@)IWv}8-TpER^S+=tGcB%MdSsLTCGMY?H(p9YKZ91b);$JXf1ofR|{aO+OB^O0eP)C3J@(_B@hEb}%kK(<*P=R^Cb;)~mlsj$%v;mc#QcOt9W+qFY@ELSaECPvU%%{%&`f|lQ)u@$js0u7d#akOaoTuX@a8{z>6BY~7x^msdA9k31pGKl^3K!G2i^fJFZC_t(^P!yVVCftGV11>xW2-7QagJSRp=h^fuZ?eU<#ULZnmy2KjczT|rcsXQ}Cz1dK0YM8@NqftcmX$;0$#AqmJD@(sXVcjtKhWz|f`U|~I~yFb$9>v;%CsiQbo=G2pWc0W^TwJ?*U?l2XWHZBPo#dv1HhgdKpb$RCFocYRgKrO<t(|x5%x;KTNowF0-uWC;)r)YVR)h&0tZcUknwPXlG_=r91<&#XT-l3@utd5-@t87J{!iTE^_2N!&&Q2L|x5PFvo0%{P=xY1Yr+aCg_MT3KT>stg-v;!X|WSGa;eK)wP+hKPu_^*$PJjFk0c5mEsPTZf>!7*nvlCfh8OK(#xS@q%r$inj;Xc64Hch%+d*wIAmzP`}jFdpA)oVjFD2@I3m(^?ON%jQsx-6A(Q&4`3vlzhP>~Dcv09K!|F&cx5IJnFl*l#UuQ>oW&T~F3QzgwpgNYZ*5WRB`z7vPbP2_V5LZCH=gYp<)7WO%(xKiaEZ1v2EQJHJEKCqgsj#a~MS2oqc92(}KK|+XJ9v3MSrr`K%8leuq;8NHJ6wyZ3svq_b4)(5I%)ydYC6d!!Y*PQHH-6NcJ|Xa|8{Ka7R&&o(!uQP3JdjdoP)fNsg>ItFtuc8Qq@UlVA}FbCu8&5RhGAGYf<SZRDG-n{s)%^S++%AlZ}AXPm9^2sBhQG16tRHU1W#0n)V=Sn?-j{qPZK|waa{GGqa0)3g>rWK2@8kmT_!zJ5-MHIwe)bCk0%~^GYiPDo8;{Uzkr$E4?y+$Ud7IG_AIp_MA2KXIJ^KFYie|rZaRknlEi(MtM@RuJN}lZMggTl$H4N6wg$vYP}}XYf)=8TxiK1F8l8jg<}013+*lVLs44TW(-%(*RRmiR_M|AkJzh;E=0kdqa<#z2{srricTcL5iYOgP#M`Q^>@;zm~qr%!^RgVIJi}yv<GFKF+9@{xt78*I+gZL8TDA+S|r#}A8Ad(%^PV8QPqIo8j`Ahw~4+$gPSm6=uk^rayD4}PNd&dSW~>?zbT5mZVckxZx6vkZ^JAvJ(x@`UAhupjh7oKO$|K?E41utHpouZ7s~X@xnMohSKeqI^P@(tI1|Q@nHrEo<)9KsCEO=U<C2`eWaGb%r=Moiv**p_as(YVPk!VlrD^#5=Y!A1`OWm;LtEZ3Pde2nk1CH2>4}E&NtC>FLln$X5KJ%IP#4S(C;G(Rraj;a!^eYs@MPG2ig}J6wvQeUPP5VAv;+T+o^;xe4zuB~^Qd!l`tVU_^l<n%KOADF@TZu(J{vvE+u=sXB=P*shS_|Hk!9`Jdpdx*grL=&rp3^gh=OfKQnOWJJx_H+T%bLw5ZZz;kiVuMD)_Evno3k^?J~MFi?OQLwR5sHLz<8fBXO-?+Dqz@2BU>28eTJ=7G4jewpa<jO7KOPyT#~=3CsbkTRJ#p<22kdth!HqE<B~4_(qwp+;WjSmt)Bt;z1x%K6=z)tb*x*EC<Q`P;R>P4HNITK<;tT-q7k=1kSp5JxAKZyKZB5Y(JF^Oxjl7EZl(!vzw}CJHLzD4C~qF--l}stGxfNEPrHaAw;uGZP1!EoblqVv5|j^9ad31d+soz$d%qJ!c>bp*rHc{vjFexUUemn%qG6&*4FR=_`h}NvVz~m1MYsc>>?!YIV0%Rn-{7CSkt-W%ZE>&KfZeY)(WnvJhOVS;vjXfhD5*FC;Hg``zpgAOiX+FSDjE%Xa+C&Y!5E+FTxh|cnd?RgE3=7^OUQ00HmR8AcGL$GDa7V*RWLJRlN@~q(Cn0pG(^-Tk;nnIExdzH}!yHxi6X&BfBJzHeJ>)g(%AnM2!K37B>^pX=3zHAg(HW8SWGIRXLx{ptcNd_PjXP2mtL)t?i<0;&E_6>W;Eays#5&yRgOG!~zl$YpH-swReC68y6n{`|T>DS-I&PUefVf6xP53$;qNyyl3oOxi#@rV#hawihE-r^<w3npv=wO2OYd+aRX~inUpk0H5;nPI2asja+Am}4#Lc}3dkr}RE^ax+-@I;!+JhS_FF8d;!yhOGb!k;X{!gFfNP>9MAtqHfq#~xIROEb3Nsc56#J#*y0xTo8FXD@%d41!cdXrtx`dTtP-=y|Hsi2bGh}fT-eR@u<QD7Q@4yGH`sC~V@$l>Y-da-{JWHwFNr8y+FAeG)bxtY>$3xg3<qm2j6Til!BEKF?Rzo0mI+#X$Lcu-6l@9JQJtY`ONCn*p@@srJ#*9O%4m$bE^Kl_9Al`>MR8_f3+Q$q-6=#bJ7|G@A3WK{L#bEC^)5l6N?9l>f#4z-QudH<Wql%oS`Yg34!C5|C0Vf?SC`-i!ha|w}Pl3fm*fNIILC~D;f@dgI{ekcR_83<41zp1Dv&HxynDlVcmjofm%7@xK+j*ASuMN&Kyy}RBTw&UecJwSgJWjk0lp98_7ex9IaN~jL4(5@{*$EV*6CDS->4IW107YJ6*uh{iTdI^QBlG~@IP46a#9^eD&?$78;h1>PhZ%5W>^!#XSiUINoX=PVkae_<qnNWoT;o5^VY^?!pM$})3u^}i6@Sbo2gv^c!Gp!ZF!R_!;A}EZXfm{Ve>%HP)%g)GEO-?5x^(7JIAzx-hrI**(*{9_z8~@Tjt8BB3yCj2OP?gQV^YzfbZoy4M%(!|9xl&Y=ft()_a~uk0RKc0*m68+*;LDjNifZ`#le(qW@7Wjix~S5WLLBCun5lc><X=oAayJ;e%stNd}pV?XJGl(@j)m0q1k>|-}84fE5sDncpyvw%4iQehu59MCpxf}Aq^NsFleX{U)bs3IO#;8k_W{RYg|w~D~$0;kxFd27LGBnBNQ0~zmen!Tt<;kiA1-siKb2kvNTC+RI3>paaVW)avmG><r-!v_y&{gR0kI^5q6QD#}yB29p_dF^u9<l7--ZViL@)%^X<ATL6Z%D3Qourj?Y_;{i>bQb=cU2N~NHp{}xFYTj2RE;c9(OoY8;F=rzJEf}k@V!9Fq})Ha5#c@U_Rl}%<u-^+B+5WWK<eDr!(?Y?Y)5S{`9%c`A)n@ag9Zw>^euUfHOmts{VUUT=(7bC`-UTnV?O%4)jUX5Eg2RpiAl>p<elypGpt#}V7RT97q4g7-fjNJ=Akd74Q;tZsZU47|NvP)bN$^cB}a~-#J-Q7G()jQWBz19?My>`EQntuCK(UuFyZcaN>PBiZwE#&%UqUyWmGmKZ}rbZvE<C97b1~|1`Kvl}>#+v)u*Sc5dIdT=AZY|>UsFe|o4N0#|j6Ed}(i2xkjCR59SJ4L5x~pSzmB6Kec%|6UW)W659n~(G-tfv*m+uga9BOQ6n;fL077wc>gi)y^Z?elt+0tn20IB63tQb<0f=)MW7qDdLI1F{Wu9~lsA3{N)9M=WYmNJ#zH4#&FM1AoTnBrTA(jf{!9|*pq^T`M_g|L?JtHM*Isnsho<GL;Fq;=p0mp$)|Fv=u_f7>el&rT~jEs#Rc503DU1P&>dZDiY~4%kNPggL6XFI!WtPPMJ`Q@~=Shv89hpM79R+l0y>BAom%DXoT4J5-k3z?))~bkS2ruu^MZANX)t!*Mj5ZdfKd48MPfDLhgbg~K*vYKX)P1T->l&@74oHXuHXQT;N?K89{43p~mCK6;MaCkF0N@jM6r{P%yGo(HG@@vk!#Jb=R^36}Qa{6GG65O8|4B)H|Ss57(f5?T}Z|9}3U|2+tn|NQs=8%+M+|60K=ogp^Tqn6dIH6L{o0mK4WYad+{(c#%{e9VSiCD(&kvXqz%7LmEUDLu*cpfdHsyDNN)6$SMcifcGrm1W*Ba6@DJ{6sZ=_&f(5ZlBC5K;IAR-18|(7!ORdo<xmUqDRXE#_jdGD}*0Z?*f=>m=&Fd=flfCD%ljVm}QKmYO$mU<Waa-VwDA<LH5enWLiK1Y9$D%&7+`Vh+YGmWvWwZFsYVmRh3#!Gap{RQI0aNv7pl&T-Q+FdCuHbgX45NZuW!8c;4p}M{%e5FnX3AEB((QMf5gBF-~y*3^<K2egfr=4mZn7FVMM}vUpwP0gU#~uZ-+tS`8ZQ_?Uy{>iUC|!}#d1XGg%+#!}6e$1*aYpyR;`{@=eo47Dvmk;AQXXeLTmW4ul2EtC&YROULGq!HOWj`<sFM)r=Qm?ppZF)uPyFip>zKfZqP1^y2-(o)WI1{I*E(wan@=G+;0ofNS0KpKwM92=j(NhjeLJ@4m;=BJd`vmA{;OhkdR>#H<5Ko@2g7NpvYP7?7s0heUk46Y72PAU?w>D&grK+h2Mavmm!O|<dC(2yzr21gbHE<Q$Qd(3vHpB#2#9ZqOnQAD+$ag`@fWA9n|P({YZC*{g}e>l9CcI*;vQ|5Q9`RG|%Tao#xRKv6M(a|=1vAd1my?JXkar@ahj)4N#4u=8PT7-^^$V;B~V6wV|T|0%l3cUdmku-rBgd+@94=5&J(E{oZv@#~_Hd1iZE72*Z^JA&VCfRA~2ArKYvm#q8fXAnuSq?7y=J}72^ZH2=Cvm$+wT!Z2`EK$y&!)vomBG7V%8<G#9s%>WY*-6ehSk+BKwLBbRZmPlL^xs|iZev$kTob6&>;*gg7alg)pT;%hhqIIQrJFa+#UCTU1m3|x`1xy*@ZeuU8Ljv@w9(6E>@I25G7`9Th<4yvTuYRx9$l>v;2La!u5zLmhTIFO>8)l8^}GXAu$6{pU6jg=p$=~;)QnX5>$)q#U=Q;W0Le_nc@064yn0k1Vk0m8I5-A1t3Eg8tQghqKYbt>i)R#rU5R8oDNOy2yQeOqGA?N7DaGt^<y1YXus2C)IDocI;y|r3-|269z)CNH0r4zZM+keDo`gvfGJ&Kj89S}b3V2OqB00vG@Im$Y&x*=#JtgDgnTd;V>B9OBUHfy>z1MMh*MRfHfuP}&ZaX=y&1DEDma_m%+C+B0ipm7Jl6_lcrza=3;|j&FdA<NM~qdZcm>L6@w4Z?d;eFJj<%LGNjR+fz}NLuO***YT?&t)kXl9K4BoQ$O+P+`xdXW#E5Q7l#rbS9q+)S9K79C)YX1D^kFP%cx&Pzqch5im&;ILoKkZvj-+%t}rAo9=K%rjGrc>IpFH9%6$>^lrJ5W&+ng5Jw)l#d!84S&OF)9T))(Sn`BrZ7$t;1(>Q?bkG6=K@Z+C@o^jKF9k2Hdk2Chm9N$<|?B?P~NL2jl@b{G~()3=_UNa4q3vWw4?c3B5s2qk~iTJBT814V54bW?uR{dtoB8X#Yer3Q|A*@xI-k%!)!uoKo-^SG3oB8i#Z|`s#Qfre~_A&zY~;VA0P8gViD%+=z-5I*oFTm57pP{tB}&a_&=&a#>wADygq^#B)1WDu-;}3!@UQ-@R6len@IgW^y@y<4%cRA7H@B14gl7Ou>8#9T^~=T~a|JDN+K{?DQA1)uD9P<!G6}on*t|f<-7ee>!cN9YvLTl`^~Rk`)Qkn~rDAA5quw`h6qnazR8rH6-j9rwmTR5AXgI(h<Q1`;+`CpQMjUKBTEc_<<Q9Z8=5-I?qy+`7vq)t6eyYEXJN3j?WNVhW0PE5~ZV%$62&)q3}s^*jv-5b#o4T=KFo<>?qZK&o^kKYD3_~)Tvx`3fv#m4=OwD-Ez^Eb#z<vymm_+Np(*u<nWeNEs3r`>%rhJe8D7138ie5rG^I>N}cGFDq31FF1egBv?p@b&OLDg3+fXiz#~!cHjKNZZ-lZ>t2ZhOz9p(%)p9AGEn<v%+S%QSaf`(GBJ?J)Q+~uP<*bMpW5ltACEye3TWRpzCvJcqX!e<WYEgBd{Av}N1|Ag@C)Z>zb7Sc^Q;u)bJ4s#R^sBA=LN5<hm_lV&ij{)onpIj&HyNroB&6PC!`5Dxq#crEe-cAf8+obNuKKRsEYMWrT}{oN6k*J@+Q1)J6MrrgJJ}D;E)6rb%|)?m=83A9AeFOF$<TW0@S#cVx51_S1<gE$wP(JY(i~y{WvQebeuP#iEw!WP1p84tm7en3=Fqatvwc(S0@0SB=~|(gmZ241yJ0Ce`z>cOPv|T#xlGEuKrCcCVS`I03#F1O9|Vr}0jfDbo##NXC&1cYBG8Wo0J^M!kf36d(JD}g@KPAFidnf4ynYJXLPJtss8rvINOGWIn42?<8)^CIkaicv3W=YN>25Kw0XJ@AV*bYql@Ukh5tlf#Bs09^`grf?Y^>>tVO@K1C4pyZX3ELy*^Bq~*fN#fV76>U6eYnNE>r5&S<l$0*0l=mN)(9TPEmRrbME2(d^jqY=;*WP7&D|1lYSFnl6~&7kZ62Nl~NkNOBmR#ooB6K^|R#`+*cUyI?cPhTBlye>#9f<a+kFfvgYM6#k8qqG0UqP<Dg`r=%WK5Ml_WCNFU)xIs^PjxENY@s~Y{4jo*4jt<SM+M<@I{h<a!{QFjvs&81=TVBi_7kFYdwV1L^ot#D~O@n+`WNsj_MAG_O<9jy!h<HStUiOFegaNKCBjUDW8S_LahotPY<PZi(p?QxtX)pj-^duksCB^^oJfts>3(sCiSsFviU$3;c@i*a!w5?!D8pT3bH$n03+Iyrh5h#Uf_)bpfBp<0^q60CiAa;DIe3=c3=EDk~*D7l`bC~_cib}kEwLg?V$^jn(|ZagN1oip~(-ey}@S<Cd6vWI$-Z0PID*<eII1Dc(pYXhkH=HNVqMAr#?6c|4W^^Se7SPC^g>XW-JcF!T0G{K)l4%IiE8Uk0TBvY!b<<y2|-UW!RuzuWQ#C&CA7>UTZ!ysD<UXyl1Gx#XubM}L3B7^4(=m}gSO^a}z<=%+l%4w+|?w7M22(_M&^ov5x-7+5$^twQtrf3`}QdxS^0|^!nN!qpnoWSdTD^$u*WWM==fTv2fhL83|1vH7n)#8h#d^~kZ1P*xe%|xh0O1pv$Y#F-oXAz4q6xAq)VZvtD+7+inN`jLQ^<XlR&Q~Ebl;1R-5dkvOK$}%X|0NNr1Z?7mQeY=|+w_Y8To%v(;e$nujBHMqPU8C)gvTVs4<3mbrU?wVS4Ft4%!!j3)v`d~v`eC7$~lPJUvmLYK^w4XrE$e_?%2K99bY*_Y*Zqev(kjgqK5$DF3~BKB-%CYEuG2K_zB_uJh7Lvm-0i?8MPDI(SAp3wqa8RWXvp7`uRXWfC%uXTV5S)&3OPF7*o=8%%zM8F@PXpj7!De(!Mp~?BGIfc0PHM7evxZsWWTsY&<crskITA%*Re#?$R+Z$A=DM366!G;g-omj{+|8cVy)D!#;Zl8w#<#hb5?26>8~+XxdO6&B-F`pqaA`2`dD(=Fy#4Isj%+xhU{jQGIpnqIbrz@dAZ?(xBH39r0wR`aKd*`TkL&iT27MHkth8jMUP-?!-5p`1&xuIrO@3Y;I#Py96l|V@%9YZzNMlU(n$cRX*w=ed+CY_%XYVmwt>lH^$r;_f6^ClvLq7MJ7)quHmR$1|Qv}C01M|8U)Ao0n-EvAE^7Q7P2sNyEll_p7@p`4iQA(N5g|DU>@z%hIqb>ho%N{b|V}jqA{TV>{7@-x{r5i!pK-Mb!xY1=eICntRZ3C$@&r|A3nbS>D9aE?_Rv(jEuXq#3pnLP^c~)imOZ}>Nu2y434H&_2|S^C-&Bx8)Z*K#<QiaTg#U7?pZCzy+J=m>ftt1T#OL{2Dh5XtSfc-nzw-JGY{-|WcmJ~&^=nD*Snzzl~c9Z6HW)Ml6UB!e(NqBEONI2kJBn`AD7s;ghZT7tC(;<0RvIRuHR+@xhopP$baJy*r#Dm8#h#fh&y^@zWUi}$xs;(8_pKlkoh%6%PICP+_LFZq>a%mHK;{gJ(>hZ;Zn2tXb7p=uq>hpl!Bda=#gnhoZxBv7{^BZakn2xXW9Jq#VUth6qgIg6vyFP?ft_o?>NG{^b}Y+R37l^#~asr-8BM;;$}z$+1`r5YFIdCPi@c>_->KMDsE-HTXXD0jW*ejuP`e-Ij#Ni`Rg}dKECRI`uzO!t4~Q8aJ$=Bs$Mi+o<s8U89=BJ41^V}7Y@{hjy3>KSwgCn8+LAD%a&4pG;WeHYqR>``BCh^?CW0%qg`${gq^`w8DI)tL|K-?Y=$2{zWnhWRD2X4#gC5TL->0b#Q~$KTp(1UBQP9byv-M{n6cN{`Vc+542t{=C39voIe+XT+@Yr|=9WSE#v|+X%eSr1@c*~7x50k!8H388P%2)`o1pu_kI(<~>P_qIn>TO6b)Z*pr|DDq+{(*P_CnaHGL__^Qk!^>#8wKFC7#!W&4EQj<l~A5m64rkL|ZO_2f>HeFDv*cR0RR;7E6&DQJ3R+bKVfbiW~EsfX$FdkOHc?m9(l_wW?&Rcw-+R>n&UOvNVpj7s(Zy2IDJoA*#Nqq5)%fE`Z9<fjilHgq4`aijKOPX>F7)gd;cbx8{5{Z-k>ouGF){=WS`z6Rk(Vi44bSdC}A}`s|#;TyiCm;VzspQ!V2!goimyY**$N?_a`|@cK{ho_~hR!DN^3D`%HJkgTgikm#*k^V_Z=71al`uo-nf>pIEMD!C8g*VSa1?!-F_|5>bd+<Wz^gKBT77j+m61;IcB#%>jQbdI7{sVJSiyl-vhgZMA`jif-RrYZBv8#po@Lw1Kr8xJljR^{Jwu6ci_bvO-mK6adFx#YeJ6)X_ojd<hEZrw16Tp1ng!hxxidwYjJXk(nfJCq%G$En*0AXW-r+fd>((+vucPnH}wf?zk=Zo-waxoO9VXM_&w@<}3rzgiod8ce&0_ZED;>Rzz}<Lv0h6M>$a_DY^i0#Wz)%WNH#8~SyJ-m&iT4(DrkA6LU+oG35P9uLa?MPMO4HPVf)uC<A^quo0$q3meY#m|+~cUD1YmUe|x55;I0qhvHsdU5Ab#GS45^A5e?N#F^gLF-B!hh4tkodS0ma=q`u&X)lr&PgY3r4|L+OwdeHkyFyIp=|Gzzm-6zcUdL@*2Q)f3Z5=O*Y!Nv6?XZjgdWvlb1V4K9na8QQbYs&rOCKscYISkrwJWjen(Y`-Mti24j-6|*HiB*lIs;g8cP(J&DL!F4WsYt886)l{pqDu47CDK->vpL(me;WJa-_b-Se$*<-51z4%&l*7==b0Dq6SaYYPssV|#xeq+UlSxokWU3htup;cWF4WscIU@&JDe7oeg*lYo~aWZ-lLM*^PEfY?P9WBs!8)6OID^!a=Gi*KeIp}ECLwW&K@1S4uhe6=ixsic0cpgc_se73x;IDq!fb3_gT<%1}@=1zz4GMmpSgHYr8@Zh==wQM)8xDEd{ZVnG_%nwYKVlv$I?j-0q;U&d22T=<G&Q5Ok_ShXmqF66y+8suExsKH>lJMVn_ocFM1Bw0^t4PwG7#DvZgvs9r=&s9GcS2rfjk1(-fU-;cD7s5m2D8~>2t3LX{eEa<eM_T1ouO0E1mxD~RaW#FOQE9s4Ey3n(ziTqtkYkgua?X;u<_VMsjvL$!n<@_h5W=D-E{_?&wvj;g?h8$kRmXlU+Az;sR4$@HFP|sU~~QK43S04zA**cT({T-sNZ_6RMadh_{EO=yw_EOg=-&jgLZ4Q<N0|IFcS?j;R4+fkSSNN;<-kHb#)%Jp%n}bVZ#C$<b-I$R<?x2A2jbQr&Pk2?ZT;%`;??(fSFk^cKtgzDH8;F-ds&FXLaMwynS<VN$u~fr`Qre<nF|1V{8#mB$1s-!++{F?UoHH#;~-v$A#8A?`1@KU{vxfU^`h0hcrCE;DsV{awJF${dm2U&At0>HF7JmASAoGL5sQ@Q91P{G(FPqvzBR<RS8g+*oZ+rzZZ!VzUZ;)nDswuGw8s`(fZe07jv+KwX|xn+djL*=ZIWaab)5+Bo#;W-D;It-fAA)$vTKdlU=zIYFqnlVJGmJhc&Q^om77At?$^N1q$h<;WS?@$Y<Exrd?gs;Eq*79@M>JTwA>Opj3a%AxA{HZlVF}%YFw2iXam;yOVp;T+Pv>(qKcog%uTBw}wjSeC`<rVhDsGy%U4(IbPBWU%o2|Oy;k47kXg9Y~frK>B*F1WqAT~(&4$6diZ%a&_2G!d2o@=<Fe6wqPB1{fhA;ru=25c5PYKK_53U435U}@SS=8IjQkk~=PeLIt)f5{0RpapSdJHjviz1^l`UR8s)ju*dPIY9^pCy$>MYj0CgPnjTBEdq&Zpj3byGMsXY&<fYoavm{1>(KL#Hfgq2&^mxG@FRt~vqrruS0CrJk0K7=lx=EH>816)76jZfv1eSwU!OEctJi0*IlS4r3#5dK*#Jy}Hr~-8F>4R)f;-WoF0DaYKDiwqR-fN%r{F5?mQKW_l+mtq+QewFty^ArM;?z`8ic4D`*xY<`2h;Oz7l1(;WqBJy;}!!GJnxe7HsE1IL?W;!rQbSZ$~OuH6~7dr=@va~|GxUowfUFChS$VYxvkIfva(uKIW^Mu)YpJJNLi}TsictY0LpY`u{W?Gc4C^1Z>mRh%D!T~5<Ya;4<de~!IGFq)|BeIHK9df7uxYe(gmj`rx9@y6%<_IV2YnXIm3tXxC7A6dW)_o;BryagaCJV1CGs(z1Ga%2TE=WogJ+^U)Nd>6rI!R?55|Uu^WcwkfpX78lo0!0Ku@xFHsm`tq3GHJ{hBM4XjF}EivPJ_GwQ}*ep)+g{B6ge2-x~e7$qGK0i6>2%uxA;v4;aYojAehea6yl@L<nEoi+txz?SKflj%Z!)>m?<0#GRiM#)Ex@ka^??l6X6)Kq<t8C|4nINM!nX$r<YP?r&lIX`LsG<K#*Ueaf_iyVG*zlkUhtTwPvb0GnI0bs1<^#Fb5B_9w<9L&mbuTrav;v(F>{bvEA?DNwwo<D<xU|B2sU)q&EMI9&O|Zc==660^XJY{Xq2CU^r|d;WLcR6EC(tV>mIIT2<{jC{7N-&?YNQ6;^R;~F{&Z=Er(l|M`&a;&X?;p|5$Zh6;r)@Ilz<#HW1gHL%LRQnlOS$NVifQZtlmJmZWZCqO~pV$YGP``H~#=cEZL-^cQ9a`-|hDH_&qQ(_y^SX|7?LbJly|4GJ1~D^E{c>~j(dM0FFWB~4WCxW;?VF0OE~{OhDL}VmlC_(Sws^;))8nq2+C(|Rr%uq1IzmU=EEf8%Rz#83^mgs>y07z9`Yzw3t_c<@9h_^!3YRMJri)!Xm24(f;{STs1Rrii4gI9C0hYhti5KE;LWZ?ox{xiB=hKvYpt^h~I5Q{-CW7j;s6J{Irx!H|?SikkOd2XutSQC<Q5OMw?A|BcWetS*5RiYU_Lq?d+9}bE5`uwEv{g?oSQGCA$LBFfeVMAVEggl*%|I&S;!R{r1XyD;8x9dxfsvrFm#JO^3_4<!r711>H}b<x{*5~RNl7VlFSUGi+&SraBL{Ju6{T_sth%CVOEwPjp4TIdL#}+Q_$Hasr60D2OLg4q5GJKV9TJk(b+ywSbu>YxAXsD_tTTMt3bW{r4tQ3qOn2&Qqh?m!jeo!RndLdr&6Fc*ny6X#(uNp186T&tn|qa*nY*o*^1HPxxmRP1Ih^xcnM@$jmOiI;AVLn2WGgmZPchhR_-onMtx|5M@w}I2t-)ByoQ>x&?M;E7)QqNH4|~s8n604Dj)B<@4=wiP4F_7{*o*x%^r%vRA)=dP_H9FWFCSGOIU1kRiBoj;wwfoV))j3X{JO~{W*dP}Lssh}zELK3AyDIcbb%=n?2GgCNp4cZ8yBcXK_uqva=cs`I-@*!3Z!Xr4YRD|TadTvmSFE6mZoG6@fH$B)zpqDeykgy0J2XjR&4eiimd#C+^zcH9{%WYfnp~D5qdDZ6Fj@&%wZErT#ZG;ax_tv3Q^9$`HoODlniQNLcu3C2E%ki?vV;j=TKE-N6WBo7#>qj@p-}vrX*~L_nMIstWdp}E>D$a^sRh3(UJt5%#LqW*;de6O)<wB=|4(Y+@z+^RVZ!dBTe%vl}R~SKuL`Ea>E;DPF_oWv#i2_Eb>&dt#0h5a#_9hmhxagF;pFQp`CRSmV8#(!D~<CYdFp**`d^49W!T!e~0{Xu^OPG&1f~zS`)tWis1A{wt5SB%$Clu7qfVW^^>**L6wD*ZC*6lFxnLieKyQFB%z+}VO|Ut<2iI``=sNzD1cWnS)EbFQ~RpL`#rT9X_`QV;k!S+;8aEQQdD|r0k`_<w6O?Jp8b$~z2Eu@|NQzddtbj^x7!Es-_EE9bh#hKSQ=x;LWdh+&<vxK<k4}jxv-w#`Cvxe<+k!B%<F_=PC`SCe;G%O7w_Nw`r+fdU!Q;Z>#y&=y!rLT`?n{ZgQvY;-+cH?A0lp%rGZ9>?X`PRLVZR}JASA{0W}Xfp4V+-t|fzcCCETzARJ~Tw16k_%h_VKn!DnR0yvRiz<Nz;E=stt9e~9wnf6|SVKfJ@r4h8zh!BLM?Bc43tPz{GJS~rNBJ<LkN1<KrrFpMx)L;N_cyJKzJ6a45m8$v^A@G|5Ey2z-l|?^v1Q`{2jB+RH^<L_qg?(;aNHQ>aV~!tW36pZipqqlQ2LCmJWv#Su&nGQN*UADKKlMv!o}_j>*3{)`-Ryr<La|SuzI^!b{^RFYFB6Y%$(QNH6fLHVB4sKl?<c`6e{ce2;;v6E>A6d?SL&0~x)<ZA^wSE5P>J-32_0a_ASN~_H}iP9j5xE#S^iAt(6CBT^|}~+d)D60+SwyCd1Mbx$#s<uvoJl=X@16tdU2&0x^Pj{0*t46ej7X!z?9%Q!)Oh&$?9?%Mw{ka`auogyuf5*oz7INLl>$R$#@P;@nIT;2mH=0^>DwFw2y{sCsFV0BIP4i)qRv{T+b$wI^49{0WD6!eK4mS+)4W(w94Hphn^-n&hJ~)t6n!U3SU|pk!(m;B=1J-I?RizZtYczcp9^dxz1=-{A5?;WA_zj6e^tnjS|lMw|vpSAPF>HwJW8NLFIYIr?4O&-u($i!*4ia5t61+PvR76g{T$)jzHFR$ZB^GI<`qvrWM5r|Lvvno%5v8XG;&H*jVn4hf{qwbNC;{;0ynQA=<BP@xx{`<^Gep(bF=WWS6JIj5S^=bPtWZ=Vm41Sg0fp$m?Xh5lae!fErS}8l4ygTx+3_S(rusu!%#A9c^`D-};Uk+vOyMhvexnj9H9a!xAybz^S?{<;Q%4nI1S+EnGnu*H)&p8(if1+?eLrmO^t8Am){^fhs&FMJBdcO07)B*;b~xt^BTHA$4~-fS}!fD^|7jc-IJl%K4UJEnDxmLM0i5PC+0++|#nH6!}!UTgqz6Z*x}1{AxT~74>n{t-h4vsc%Y^DVW$w#A(yu?b<m{_WdSBV%)XS29?X$y{xEj>R7?`0ql*ciiJ`>BPK{pwP||mZtY^F6hhbz>}`9SQJsmCk_#4}V@@h>u_irqC@Y?1dJ5$9P;zK33*huB91H$fs?JZ3x843DJwoS{kt|ks#W<VA+G}M`IDw|6w+;={!N`OvyCxzznd=%uzG&RYiqDpPHk48-cd_JO!sZ;^;Kib^P4OG?n4}a<ch;er>A?x026XsdS{lXHdH#Bz`7DZo0(2)(Ndz#-qg%V1t8$4EOqXfxq`Mb{GY)=LN4+ogk>N*ab+G(eRlUkDXzs2w12JuSJ0iApsa~Uvq~aBI&TxTh8{#4j$NCo4P6|92m^AL?I*fK&)w|7NS7z>RQC|JCZbxIUPI6k=NS<WfIjTYMo|38!dJe?;eR(LzIdAOR<>lNyz}1)q_+(Zz2?tl&jj?V%g7f%VV`C)YN;k-^W*FW~Gpi~M&A0O$W6~?tT?uw7w}k|HNJ`MX4V@@;Ts?0JC8+LRKLi*TO24a5h<W6K0v`m=gK7TFMCj*a^uTfuhq9WAnN0W)Itr-wLRLgwNOF5Lt5-%WCrjNN9`|HRdLZC2lUCnm*|O37*=}SHX?A_vrsWQoHxXCj_rAd$=AJ6AvrTDRm{1=jn1hE!@xdH!Lu>5pHe)kKvErd+KG!8x@a^87u!xv*S(w3UH7>U+2|PUPF2%FD%3W6X69=H8eMDOo<snp0f!HAW3ieKdqb7Si_}l9bb{YqfJ$$Oue~jli#NW>QO5Y`)y9@+VgMm!H)G&E8tnhChegyjBO{~N8onK==-t+7VgLmT_RV}}2;BLg(5K;BMOGVYw9m%LmLP?J;Q{DXAc#aoaL$?@**CqoN$_~2Z>koZ|xV?HA6L1aQe00p6Mp!d*Egpc%hI`Hq50P3udJW&*dpRac0Rq_4M(631cHDt~(H7eF`5jWcFI<WRx`k%T<sxm*I_>tO<Kt(~I*+>KJGdxAojn!NRF$ZE8P07xp!09y>>0)s1-}0eccPfCY>1Y8iqJ9A<H7(MEMJS#faID+r+s+5hyRJ<(=goRfXD+?{w`FpW*3+?@iWg9j?~K*_In#6`Jcg1R<2BSoGpZ&b6?)=uJq7OX@DuGYAp^0fvPEnlU=VwR5_1<LryQV>+$94GCe{?!ZI6A5H6B7nw_}Wc^Eg3;eQV`_vsB~!s#$yF0yZTu;FzF=d&W8YH9XC@S`}}PFI(wK#pfBW*o;9C!iT72OdmTLoLX-M&K4l->1!La)dVj+abVeAzwTv#`%W6M$cr444A|Y|4S-jr5mOOql8~1_*Qz-tF)$DvI|zjcrxiIbPXuH=pDjI)hQF6eg@}ICs1!cOU(};d_S|k9rm7??A30;j#Kam3vSM|{y|c6SUuC)_YvZCC|OEip`&p4CqSrsrG#R*Oof>n5BrGm()-_H`g5KhwcGL0!*-YT@kM$$o;HqSoDr-jin<JdNgECQY+uSAbfQ*EK1X|8pQTkk&w=;ka5TU!bL^hDA?5d^7m{{24j<(7PZ)KFa0F9*7@PQ?231zVaE@`PeDkd?DHs8FuS7K``Dht0XY+LYhf>>P-TO-CiOa;+Ny*i~I`*GnE&C6n67q3B3|c|B?--@vrk$kWDG=0)I%v`+z}#IybCO%}97xU-wojwM13c(Fl2tx<=)K5kR)KdaVCp#3g+0=v$~!d?a>Sog1DkZ{Vp+`Q<!S-~9Z;BlWjc>wKEtnHr(qMLt~U79lV6fw_K%*hI_MVyO`$~9Xvera50BftKdEdK#Ya!zKQ&X1GEd5YX?1Octx<^47vVUs`*Zr#flu_|u*asFF23h-m32}+FoG$SCLWSG-G=aoU7le|g$fUxJgC4aT+a2ii#0@Dp;d~A;SEVYxTMw)iHcX~^T`B;JVj^E;v7y#ehsu^!m?+<(2P@)s=;9CM3-g)ffOYXo!hKG&c{GTwvveB!?u0WtH-TxH)BpeKlMQBb|e8cWB~hY(E#XO^);(kyV3%l%Y$q(K10c$Uk*3XguBp%JJJS!1;`tnuzQeoPyN!{Gsx7FU;c0?pbqN<9!I^}8sR+A?ZF>o3swocpv%{yF3?d0Bqe*NzNc^~=`|at3A}`~v)_riOvG&**@XqXsY-WL>7&|wCUfMPRjOm)Pw>5(q8Kz~wxHIIo*cL6P3hjzleT=9`)yNWTu}0}H?XgrnAgUta*taBLRT#y0X9{MO`(4(Dc|3{dXBDDTkyYRO0T^8=J~r9e|!I_W^i(7_;#U=Zn<6b4YTpg!F%zpcP$ZrLz_vyr?28-th$JJ-R>qj?xTgj8&ww>NlJa4-tWctRV?!R$?bg({(V2WKYH|dlpT$Z9<?7GosJ$qJ$?G{asH_Dc=WXM@bux6<6-vj_~_}wqx<Xlz9h|JNfO@8A0Hn-dDMQ`K05839v?nA%CpYVlc!I!!=ui__V6%!at!|t+ec5b(?h6v1pgbgpFSFO9v_XKK7Dk5{r>^+-G1B'}


## 1. Thu thập cấu hình và kiểm tra các API/tensor nhỏ

Không cần sửa ô mã hỗ trợ phía trên. `failed`, `timeout`, `not_installed` được giữ
riêng. Nếu có nhiều Python, kết quả từng interpreter nằm ở `runtimes`; không lấy
package của Python 3.13 làm bằng chứng cho Python 3.10. Wheel tags chỉ kiểm tra
tương thích ABI/platform, chưa xác nhận native CUDA hoặc inference.
DocEnTR/DE-GAN có inventory riêng và đối chiếu kernel metadata với
`learned_documents_runtime.json`. Probe Keras dùng backend Torch trong subprocess
riêng và HDF5 bốn số; không dựng generator hoặc nạp weights. Wheel METADATA và
dependency được đọc mà không cài đặt. Metadata khớp chưa xác minh inference.


In [ ]:
report = collect_environment(
    run_probes=RUN_PROBES, input_root=INPUT_ROOT,
    configured_paths={'models':MODELS_DIR, 'packages':PACKAGES_DIR,
                      'pdfs':DATA_DIR, 'prepared_images':PREPARED_IMAGES},
    runtime_pythons=RUNTIME_PYTHONS, options=CHECK_OPTIONS,
    checkpoint_path=OUTPUT_DIR/'kaggle_environment_report.json')
print(json.dumps(report["hardware"], ensure_ascii=False, indent=2))
print()
print("GPU:", report["tools"].get("gpu", {}).get("stdout", "unavailable"))
print("Torch/CUDA:", report["probes"].get("torch_cuda", {}).get("stdout", "not available"))
print()
print("Các package liên quan:")
for name, version in report["packages"].items():
    print(f"{name:24s} {version or 'not installed'}")
print()
print("Pip dependency check:", report.get("pip_check", {}).get("status"))
print("Runtime VTM từ phiên trước:", report["prior_vtm_runtime_exists"])
print('Trạng thái báo cáo:', report['status'])
print('Input:', json.dumps(report['asset_roots'], ensure_ascii=False, indent=2))
print('Asset:')
for name, details in report['assets'].items():
    print(name, details.get('state', details.get('status')), details.get('required_missing', []))
print('Runtime riêng:', {name:value['status'] for name,value in report['runtimes'].items()})


## 2. Build FlashAttention và kiểm tra UniHIR thật

Phần này mặc định tắt (`RESTORATION_GPU_CHECK='off'`); không cần bật để gửi log
môi trường. Giữ Internet Off và gắn models/packages mới nhất. Phép thử dùng đúng worker,
source upstream, kiểm checkpoint và pipeline Notebook 1 được nhúng có checksum.
Mặc định 4 L4 cùng phiên: kiểm từng GPU và peer access, UniHIR chia model qua
Accelerate với ngân sách weights 18 GiB/card, chừa activation/KV/VAE. VRAM không
tự gộp. Kiểm placement và VRAM trước inference; device map/đỉnh VRAM từng card
được lưu trong log. Một GPU lớn vẫn dùng được khi đổi gpu_ids/placement tương ứng.
Build thành công, CUDA kernel chạy đúng và inference là ba trạng thái riêng.
Kết quả phục hồi chưa chứng minh đúng chữ; original được giữ để đối chiếu.
Phép thử không dùng cache ảnh phục hồi. File log/build receipt được gom vào ZIP,
không chứa weights, venv hoặc source build nặng.


In [ ]:
try:
    report['restoration_gpu_check'] = run_restoration_gpu_check(
        report, mode=RESTORATION_GPU_CHECK, options=RESTORATION_CHECK_OPTIONS,
        payload=GPU_CHECK_PAYLOAD, output_dir=OUTPUT_DIR)
except Exception as error:
    report['restoration_gpu_check'] = {'status':'supervisor_failed', 'error':str(error),
        'traceback':traceback.format_exc(), 'model_inference_verified':False}
report['model_inference_performed'] = bool(report['restoration_gpu_check'].get('model_inference_attempted'))
save_report(report, OUTPUT_DIR)
print(json.dumps(report['restoration_gpu_check'], ensure_ascii=False, indent=2))


## 3. Gửi lại ZIP log

Mặc định gửi **`kaggle_environment_logs.zip`** trong Output; có JSON/TXT, inventory
DocEnTR/DE-GAN, metadata dependency và GPU. Nếu bật GPU check, gửi
**`restoration_check_*_logs.zip`** sau khi chạy xong (kể cả
khi build hoặc inference lỗi). Có JSON/TXT inventory, phase report, build log,
worker log và ảnh original/restored của một trang khi thành công. Nếu chỉ kiểm
inventory, ZIP tên `kaggle_environment_logs.zip`. Nếu bị ngắt, JSON đã lưu một phần vẫn hữu ích; trường
`status`/`complete` cho biết các phép kiểm tra đã kết thúc hay chưa.

Báo cáo giữ toàn bộ distributions/dependencies, ABI, vị trí module thật, log lỗi,
GPU architecture, compiler/header và file asset còn thiếu. Từ dữ liệu này mới
chọn package/runtime cần bổ sung cho UniHIR mà không hạ kernel đang dùng.
Không cần upload lại models/packages chỉ để lấy báo cáo. Nếu Input vừa cập nhật
UniHIR, gắn đúng phiên bản Dataset mới để kiểm tra checkpoint và inventory tương ứng.


In [ ]:
json_path, text_path = save_report(report, OUTPUT_DIR)
print("Báo cáo JSON:", json_path)
print("Báo cáo TXT:", text_path)
log_archive = package_gpu_check_logs(report, OUTPUT_DIR)
print('ZIP log gửi lại:', log_archive)
try:
    from IPython.display import display, FileLink
    display(FileLink(str(json_path)))
    display(FileLink(str(text_path)))
    display(FileLink(str(log_archive)))
except ImportError:
    pass
